# 🔥 Fire Code Regulation Entity Extraction Pipeline

## This pipeline extracts regulatory entities from fire code PDFs using:
- PDF text extraction with OCR fallback
- Semantic chunking
- Entity extraction with context
- Graph Neural Networks (GAT)

ARCHITECTURE OVERVIEW:
Input: PDF Documents → Text Extraction → Chunking → Entity Labeling →
Graph Construction → GNN Training → JSONL Export

---

## 1️⃣ Setup Google Drive

In [ ]:
import threading, time, os, errno, subprocess, logging
from pathlib import Path
from google.colab import drive as _drive

# ── Config ───────────────────────────────────────────────────────────────────
_KEEPALIVE_PATH   = '/content/drive/MyDrive/RegCheckAI/.keepalive'
_PROBE_INTERVAL   = 240
_REMOUNT_RETRIES  = 3
_REMOUNT_WAIT     = 8
_DRIVE_EIO_ERRNOS = {errno.EIO, errno.ENOTCONN}
_ROOT_SENTINEL    = '/content/drive/MyDrive'

# ── Logger reference (set after pipeline init, or use print as fallback) ─────
_keepalive_logger: logging.Logger | None = None

def _klog(msg: str, level: str = 'info'):
    """Write to pipeline logger if available, else print."""
    if _keepalive_logger:
        getattr(_keepalive_logger, level, _keepalive_logger.info)(msg)
    else:
        print(f"[Keepalive] {msg}")

# ── Safe stat probe (never calls os.path.exists — that calls stat internally
#    and can raise OSError on a broken FUSE mount before your except catches it)
def _safe_stat(path: str) -> bool:
    try:
        os.stat(path)
        return True
    except OSError:
        return False

# ── Logger stream heal (mirrors _heal_log_handler in RegulationPipeline) ─────
def _heal_logger_handlers(logger: logging.Logger | None):
    if logger is None:
        return
    for h in logger.handlers:
        # RotatingFileHandler has .baseFilename; plain StreamHandler does not
        if not hasattr(h, 'baseFilename'):
            continue
        stream = getattr(h, 'stream', None)
        if stream is None:
            continue
        try:
            stream.tell()       # raises OSError if FUSE stream is dead
        except OSError:
            try:
                stream.close()
            except Exception:
                pass
            try:
                h.stream = open(h.baseFilename, 'a', encoding='utf-8')
                _klog(f'Healed log handler → {h.baseFilename}')
            except Exception as e:
                print(f'[Keepalive] Could not reopen log file: {e}')

# ── Remount ───────────────────────────────────────────────────────────────────
def remount_drive(
    retries: int = _REMOUNT_RETRIES,
    wait:    int = _REMOUNT_WAIT,
    logger:  logging.Logger | None = None,
) -> bool:
    """
    Lazy-unmount then remount Google Drive.
    Checks if /content/drive is actually mounted before calling fusermount,
    so it never runs blind on an already-clean state.
    Returns True on success.
    """
    _target_logger = logger or _keepalive_logger

    # Only lazy-unmount if the FUSE mount exists
    if _safe_stat(_ROOT_SENTINEL):
        _klog('Running fusermount -uz before remount …')
        subprocess.run(['fusermount', '-uz', '/content/drive'],
                       check=False, capture_output=True)
        time.sleep(2)

    for attempt in range(1, retries + 1):
        _klog(f'Remount attempt {attempt}/{retries} …')
        try:
            _drive.mount('/content/drive', force_remount=True)
        except Exception as exc:
            _klog(f'mount() raised: {exc}', 'warning')
            time.sleep(wait)
            continue

        time.sleep(2)

        if _safe_stat(_ROOT_SENTINEL):
            _heal_logger_handlers(_target_logger)
            _klog('✅ Drive remounted successfully.')
            return True

        _klog(f'Mount appeared to succeed but sentinel missing (attempt {attempt})', 'warning')
        time.sleep(wait)

    _klog('❌ All remount attempts failed — manual runtime restart needed.', 'error')
    return False

# ── Keepalive thread ──────────────────────────────────────────────────────────
def _keep_drive_alive(interval_sec: int = _PROBE_INTERVAL) -> None:
    """
    Every interval_sec:
      1. stat() the root sentinel and the keepalive file (not os.path.exists —
         that can swallow the OSError we need to catch).
      2. Ensure the keepalive directory exists (makedirs, no-op if already there).
      3. Write a fresh timestamp.
      4. On any Drive I/O error, attempt a remount and heal log handlers.
    """
    # Ensure keepalive dir exists once before the loop (Drive must be up at this point)
    try:
        Path(_KEEPALIVE_PATH).parent.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        _klog(f'Could not create keepalive directory: {e}', 'warning')

    consecutive_failures = 0
    MAX_LOG_FAILURES     = 5

    while True:
        time.sleep(interval_sec)

        # ── Probe ─────────────────────────────────────────────────────────
        root_ok = _safe_stat(_ROOT_SENTINEL)

        if not root_ok:
            consecutive_failures += 1
            if consecutive_failures <= MAX_LOG_FAILURES:
                _klog(f'Root sentinel missing (consecutive={consecutive_failures}) — remounting …', 'warning')
            remount_drive()
            continue

        # ── Write timestamp ───────────────────────────────────────────────
        try:
            Path(_KEEPALIVE_PATH).parent.mkdir(parents=True, exist_ok=True)
            with open(_KEEPALIVE_PATH, 'w') as f:
                f.write(str(time.time()))
            consecutive_failures = 0    # reset on clean write

        except OSError as e:
            consecutive_failures += 1
            if consecutive_failures <= MAX_LOG_FAILURES:
                _klog(f'Write failed (errno={e.errno}): {e} — remounting …', 'warning')
            if e.errno in _DRIVE_EIO_ERRNOS:
                remount_drive()
            # Non-EIO OSErrors (permissions etc.) are logged but not remounted

        except Exception as e:
            consecutive_failures += 1
            if consecutive_failures <= MAX_LOG_FAILURES:
                _klog(f'Unexpected error: {e}', 'warning')

# ── Initial mount + start thread ──────────────────────────────────────────────
_drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive mounted!')

_keepalive_thread = threading.Thread(
    target=_keep_drive_alive,
    daemon=True,
    name='DriveKeepalive',
)
_keepalive_thread.start()
print(f'✅ Drive keepalive thread started (probe every {_PROBE_INTERVAL}s)')

Mounted at /content/drive
✅ Google Drive mounted!
✅ Drive keepalive thread started (probe every 240s)


## 2️⃣ Install Dependencies

In [ ]:
def install_package(package_name: str, import_name: str = None):
    """Install package if not available"""
    import_name = import_name or package_name
    try:
        __import__(import_name)
        print(f"✅ {package_name} already installed")
    except ImportError:
        print(f"📦 Installing {package_name}...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"✅ {package_name} installed successfully")
        else:
            print(f"❌ Failed to install {package_name}: {result.stderr}")
# Install system dependencies first (for Poppler and Tesseract)
print("\n🔧 Installing system dependencies...")


🔧 Installing system dependencies...


In [ ]:
import sys
import os
from pathlib import Path
import json
import logging
from logging.handlers import RotatingFileHandler
from typing import Dict, List, Tuple, Optional, Any, Set, Union
from datetime import datetime
import hashlib
import re
import gc
import pickle
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from enum import Enum
import subprocess

In [ ]:
print("\n📦 Installing Python packages...")
packages = [
    ("pdfplumber", "pdfplumber"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("spacy", "spacy"),
    ("torch", "torch"),
    ("torch-geometric", "torch_geometric"),
    ("sentence-transformers", "sentence_transformers"),
    ("scikit-learn", "sklearn"),
    ("tqdm", "tqdm"),
    ("networkx", "networkx"),
    ("matplotlib", "matplotlib"),
    ("pdf2image", "pdf2image"),
    ("pytesseract", "pytesseract"),
    ("imbalanced-learn", "imbalanced-learn"),
    ("difflib", "difflib"),
    ("openpyxl", "openpyxl"),
    ("bisect", "bisect"),
]

for pkg, imp in packages:
    install_package(pkg, imp)


📦 Installing Python packages...
✅ pdfplumber already installed
✅ pandas already installed
✅ numpy already installed
✅ spacy already installed
✅ torch already installed
✅ torch-geometric already installed
✅ sentence-transformers already installed
✅ scikit-learn already installed
✅ tqdm already installed
✅ networkx already installed
✅ matplotlib already installed
✅ pdf2image already installed
✅ pytesseract already installed
📦 Installing imbalanced-learn...
✅ imbalanced-learn installed successfully
✅ difflib already installed
✅ openpyxl already installed
✅ bisect already installed


In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'poppler-utils', 'tesseract-ocr'],
               capture_output=True)
print("✅ System dependencies installed (Poppler & Tesseract)")

✅ System dependencies installed (Poppler & Tesseract)


## 3️⃣ Load Pipeline Code

### IMPORTS

In [ ]:
from __future__ import annotations
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import pandas as pd
import numpy as np
import random
import inspect
import gc
import torch
assert os.path.exists("/content/drive/MyDrive/RegCheckAI/Checkpoints"), \
    "Drive not mounted or path wrong!"
from tqdm import tqdm
import pdfplumber
import spacy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, classification_report
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from pdf2image import convert_from_path
import pytesseract
from urllib.parse import quote
import bisect
import traceback
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Circle
from matplotlib.ticker import MaxNLocator

In [ ]:
# Suppress all pdfminer warnings
logging.getLogger('pdfminer').setLevel(logging.ERROR)
logging.getLogger('pdfminer.pdfinterp').setLevel(logging.ERROR)

###SETUP LOGGER

In [ ]:
def setup_logger(log_path: str) -> logging.Logger:
    logger = logging.getLogger("RegulationExtraction")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in logger.handlers[:]:
        try:
            handler.close()
        except Exception:
            pass
        logger.removeHandler(handler)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
    logger.addHandler(console_handler)

    try:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        file_handler = RotatingFileHandler(
            log_path, maxBytes=10 * 1024 * 1024, backupCount=5
        )
        file_handler.setFormatter(
            logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        )
        logger.addHandler(file_handler)
    except OSError as e:
        logger.warning(
            f"⚠️  Could not attach log file ({e}) — console logging only. "
            f"Remount Drive and call setup_logger() again to restore."
        )

    return logger

### SPACY SETUP

In [ ]:
print("\n📚 Setting up spaCy...")
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("   Downloading spaCy model...")
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

nlp.max_length = 5_000_000
print(f"✅ spaCy configured (max_length: {nlp.max_length:,})")


📚 Setting up spaCy...
✅ spaCy configured (max_length: 5,000,000)


### ENTITY TYPES (27 CLASSES)

In [ ]:
REGULATION_ENTITY_TYPES = {
    # Critical Safety
    "FIRE_RATING":          0,
    "HAZARD_LEVEL":         1,
    "PENETRATION":          2,

    # Safety Equipment & Components
    "EQUIPMENT":            3,
    "COMPONENT":            4,
    "MATERIAL":             5,
    "FACILITY":             6,

    # Regulatory References
    "STANDARD":             7,
    "AUTHORITY":            8,
    "CERTIFICATION":        9,

    # Building Classification
    "BUILDING_TYPE":        10,
    "OCCUPANCY":            11,

    # Measurements - Primary
    "DIMENSION":            12,
    "AREA":                 13,
    "VOLUME":               14,
    "CAPACITY":             15,
    "PERCENTAGE":           16,

    # Measurements - Additional
    "EGRESS_DISTANCE":      17,
    "SEPARATION_DISTANCE":  18,
    "CLEARANCE":            19,
    "PRESSURE":             20,
    "FLOW_RATE":            21,
    "TEMPERATURE":          22,

    # Location & Spatial
    "LOCATION":             23,

    # Operational Requirements
    "INSPECTION_FREQUENCY": 24,
    "TESTING_REQUIREMENT":  25,

    # Special tokens
    "NON_ENTITY":           26,
    "CHUNK_ROOT":           -1,
}


In [ ]:
_SENTENCE_HINTS: Dict[str, List[str]] = {
    'FIRE_RATING':          ['fire-rated', 'fire rated', 'fire resistance', 'hour', 'hr', 'class a', 'class b', 'class c'],
    'EQUIPMENT':            ['sprinkler', 'detector', 'extinguisher', 'alarm', 'panel', 'suppression', 'standpipe', 'hose'],
    'COMPONENT':            ['valve', 'fitting', 'coupling', 'pipe', 'duct', 'damper', 'seal', 'gasket'],
    'MATERIAL':             ['concrete', 'steel', 'gypsum', 'wood', 'masonry', 'glass', 'membrane', 'coating'],
    'TESTING_REQUIREMENT':  ['test', 'testing', 'tested', 'verify', 'verification', 'annual test', 'monthly test', 'functional test'],
    'INSPECTION_FREQUENCY': ['inspect', 'inspection', 'annually', 'monthly', 'weekly', 'quarterly', 'every year', 'semi-annual'],
    'OCCUPANCY':            ['occupancy', 'use group', 'group a', 'group b', 'group e', 'group f', 'group h', 'group i', 'group m', 'group r', 'group s'],
    'BUILDING_TYPE':        ['construction type', 'type i', 'type ii', 'type iii', 'type iv', 'type v', 'noncombustible', 'combustible'],
    'EGRESS_DISTANCE':      ['travel distance', 'exit access', 'common path', 'dead end', 'egress', 'means of egress'],
    'SEPARATION_DISTANCE':  ['separation', 'setback', 'distance between', 'minimum distance', 'feet from', 'metres from'],
    'STANDARD':             ['nfpa', 'astm', 'ansi', 'ul', 'ibc', 'ifc', 'standard', 'listed', 'labeled', 'approved'],
    'HAZARD_LEVEL':         ['hazard', 'high-hazard', 'low-hazard', 'ordinary hazard', 'extra hazard', 'combustible', 'flammable'],
    'AUTHORITY':            ['authority having jurisdiction', 'ahj', 'fire marshal', 'fire department', 'building official', 'commissioner'],
    'PRESSURE':             ['psi', 'kpa', 'bar', 'pressure', 'static pressure', 'residual pressure'],
    'FLOW_RATE':            ['gpm', 'gph', 'lpm', 'flow rate', 'flow demand', 'discharge'],
    'TEMPERATURE':          ['temperature', 'fahrenheit', 'celsius', '°f', '°c', 'degrees', 'thermal', 'heat'],
}

In [ ]:
_PLAUSIBILITY_REQUIRED: Dict[str, List[str]] = {
    'FIRE_RATING':   ['hour', 'hr', 'class', 'rating', 'rated', 'resistance'],
    'PERCENTAGE':    ['%', 'percent'],
    'FLOW_RATE':     ['gpm', 'gph', 'lpm', 'flow', 'l/min', 'litre'],
    'TEMPERATURE':   ['°', 'fahrenheit', 'celsius', 'deg', '°f', '°c'],
    'PRESSURE':      ['psi', 'bar', 'kpa', 'mpa', 'pressure', 'pa'],
    'AREA':          ['sq', 'ft²', 'm²', 'square', 'sf', 'sqm'],
    'VOLUME':        ['gal', 'litre', 'liter', 'cubic', 'ft³', 'm³'],
    'EGRESS_DISTANCE':     ['ft', 'feet', 'm', 'metre', 'meter', 'distance', 'travel'],
    'SEPARATION_DISTANCE': ['ft', 'feet', 'm', 'metre', 'meter', 'distance', 'separation'],
    'CLEARANCE':           ['ft', 'feet', 'm', 'metre', 'meter', 'clearance', 'clear'],
}

In [ ]:
_LEADING_STOP: frozenset = frozenset({
    'the', 'a', 'an', 'of', 'in', 'at', 'to', 'for', 'and', 'or',
    'with', 'by', 'from', 'that', 'this', 'those', 'these', 'any',
    'all', 'each', 'such', 'no', 'not', 'be', 'is', 'are', 'was',
    'were', 'been', 'being',
})
_TRAILING_STOP: frozenset = frozenset({
    'the', 'a', 'an', 'of', 'in', 'at', 'to', 'for', 'and', 'or',
    'with', 'by', 'from', 'that', 'on', 'within', 'more', 'than',
    'requirements', 'requirement', 'limitations', 'limitation',
    'as', 'per', 'where', 'which', 'when', 'if',
})

_MEASUREMENT_TYPES: frozenset = frozenset({
    'DIMENSION', 'AREA', 'VOLUME', 'PRESSURE', 'FLOW_RATE',
    'TEMPERATURE', 'CAPACITY', 'PERCENTAGE', 'FIRE_RATING',
    'SEPARATION_DISTANCE', 'EGRESS_DISTANCE', 'CLEARANCE',
})

In [ ]:
ENTITY_ID_TO_TYPE = {v: k for k, v in REGULATION_ENTITY_TYPES.items()}
NUM_ENTITY_TYPES = len([v for v in REGULATION_ENTITY_TYPES.values() if v >= 0])
_NON_ENTITY_ID = REGULATION_ENTITY_TYPES["NON_ENTITY"]

print(f"✅ Loaded {NUM_ENTITY_TYPES} entity types, Non-entity ID is {_NON_ENTITY_ID}")

✅ Loaded 27 entity types, Non-entity ID is 26


### CONFIGURATION

In [ ]:
def create_config():
    return {
        # Input paths
        "train_pdf_dir": "/content/drive/MyDrive/RegCheckAI/PDF",
        "test_pdf_path": "/content/drive/MyDrive/RegCheckAI/New Regulation/SBC-Code-801-2018-unlocked-2-001-300.pdf",
        "Val_pdf_path": "/content/drive/MyDrive/RegCheckAI/PDF/International-Fire-Code.pdf",

        # Output paths
        "base_path": "/content/drive/MyDrive/RegCheckAI",
        "output_dir": "/content/drive/MyDrive/RegCheckAI/Output",
        "model_dir": "/content/drive/MyDrive/RegCheckAI/Models",
        "visual_dir": "/content/drive/MyDrive/RegCheckAI/Visualizations",
        "log_dir": "/content/drive/MyDrive/RegCheckAI/Logger",

        #Text Chuncking
        "chunk_size": 500,
        "overlap": 50,

        # Training parameters
        "test_size": 0.2,
        "random_state": 42,
        "batch_size": 32,
        "epochs": [10, 15, 20, 20],
        "learning_rate": 0.0005,
        "checkpoint_dir": "/content/drive/MyDrive/RegCheckAI/Checkpoints",
        "use_checkpoints": True,

        # Model parameters
        "embedding_dim": 795,
        "focal_gamma":   2.0,
        "hidden_dim": 384,
        "num_heads": 8,
        "dropout": 0.6,
        "patience": 7,
        "min_delta": 0.002,
        "background_label_id": 26,
        "num_entity_types": NUM_ENTITY_TYPES,
        "warmup_steps": 100,
        "max_grad_norm": 1.0,
        "inverted_dropout": [0.30, 0.25, 0.20, 0.20],
    }

### PDF TEXT PREPROCESSOR

In [ ]:
class TextPreprocessor:
    def __init__(self, logger):
        self.logger = logger

        # A "fill run" is any mix of ASCII periods, Unicode ellipsis (…),
        # em/en dashes used as leaders, and whitespace — at least 2 chars so
        # a lone decimal point inside a sentence is never matched.
        _FILL = r'(?:[.\u2026\u2013\u2014\s]{2,})'

        # A page reference: single number, range (28-29 / 28–29), roman
        # numeral, or appendix-style ref (A-1).
        _PAGE = r'(?:\d+(?:[-\u2013\u2014]\d+)?|[ivxlcdmIVXLCDM]{1,6}|[A-Z]-\d+)'

        self.toc_patterns = [
            # ── Lines WITH a fill run (ASCII dots, Unicode ellipsis, or mixed) ─
            # Generic: any title text <fill> page
            rf'(?i)^.+{_FILL}\s*{_PAGE}\s*$',

            # Fill-only + page  ("……… 5")
            rf'^\s*{_FILL}\s*{_PAGE}\s*$',

            # Structural keyword at start, with fill
            rf'(?i)^(?:Chapter|Section|Part|Appendix|Annex|Schedule|Exhibit|Table|Figure)'
            rf'\s+[\w.\-]+\b.*{_FILL}\s*{_PAGE}\s*$',

            # ── Lines WITHOUT any fill run ─────────────────────────────────────
            # Structural keyword + title + page (no dots at all).
            # Covers: "Appendix D - Fire and Life Safety Checklist  28-29"
            rf'(?i)^(?:Chapter|Section|Part|Appendix|Annex|Schedule|Exhibit|Table|Figure)'
            rf'\s+[\w.\-]+[\w\s\-\(\):,]*\s+{_PAGE}\s*$',

            # Numbered section without fill:  "1.2  Introduction  5"
            rf'(?i)^\d+(?:\.\d+)*\s+[\w\s\-\(\):,]{{3,}}\s+{_PAGE}\s*$',

            # Multi-level numbered:  "1.2.3  Title  45-47"
            rf'(?i)^\d+\.\d+\.\d+\s+[\w\s\-\(\):,]{{2,}}\s+{_PAGE}\s*$',

            # ── Isolated fill-only lines (a pure row of dots / ellipsis) ──────
            rf'^\s*[.\u2026]{{3,}}\s*$',

            # ── Blank-line-surrounded short titles ────────────────────────────
            r'(?:\n\s*){3,}[A-Za-z0-9][A-Za-z0-9\s\-\(\)]{2,50}(?:\n\s*){3,}',
        ]

        self.header_footer_patterns = [
            r'(?i)^\s*Page\s+\d+\s*$',
            r'(?i)^\s*Page\s+\d+\s+of\s+\d+\s*$',
            r'^\s*\d+\s*$',
            r'(?i)^(?:Chapter|Section|Part|Appendix)\s+[\w.\-]+\s*$',
            r'^[A-Z]{2,}-\d+-\d+\s*$',
            r'^[A-Z0-9_\- ]{5,}\s*$',
        ]

        # Copyright / legal notice patterns (full lines)
        self.copyright_line_patterns = [
            # © 2024 Acme Corp  /  Copyright (C) 2024  /  Copyright © …
            r'(?i)^\s*(?:©|copyright\s*(?:\(c\)|©)?)\s*.{0,120}$',
            # All rights reserved (standalone line or trailing)
            r'(?i)^\s*all\s+rights\s+reserved\.?\s*$',
            # Printed in … / Published by …
            r'(?i)^\s*(?:printed|published)\s+(?:in|by)\b.{0,100}$',
            # ISBN / ISSN lines
            r'(?i)^\s*(?:isbn|issn)[\s:\-][\d\-X ]{5,}\s*$',
        ]

        self.compiled_toc = [
            re.compile(p, re.MULTILINE | re.IGNORECASE) for p in self.toc_patterns
        ]
        self.compiled_headers = [
            re.compile(p, re.MULTILINE | re.IGNORECASE) for p in self.header_footer_patterns
        ]
        self.compiled_copyright_lines = [
            re.compile(p, re.MULTILINE | re.IGNORECASE) for p in self.copyright_line_patterns
        ]

        # ------------------------------------------------------------------ #
        # Spaced-letter title:  "I N T R O D U C T I O N"                    #
        # Each visible character is separated by exactly one space, and the  #
        # whole token is at least 5 chars long (i.e. 3 letters minimum).     #
        # ------------------------------------------------------------------ #
        self._spaced_title_re = re.compile(
            r'^\s*[A-Za-z0-9](?:\s[A-Za-z0-9]){4,}\s*$'
        )

        # ALL-CAPS title guard: must contain at least 2 words so that lone
        # acronyms (e.g. "PDF", "USA") are not removed.
        self._allcaps_title_re = re.compile(
            r'^[A-Z][A-Z0-9\s\-\(\):]{4,}$'
        )

    # ---------------------------------------------------------------------- #
    # TOC removal                                                            #
    # ---------------------------------------------------------------------- #
    def remove_toc_entries(self, text: str) -> str:
        lines = text.split('\n')
        cleaned_lines = []
        removed_count = 0
        for line in lines:
            is_toc = any(p.match(line.strip()) for p in self.compiled_toc)
            if is_toc:
                removed_count += 1
            else:
                cleaned_lines.append(line)

        if removed_count > 0:
            self.logger.info(f"   Removed {removed_count} TOC entries")
        return '\n'.join(cleaned_lines)

    # ---------------------------------------------------------------------- #
    # Header / footer removal                                                #
    # ---------------------------------------------------------------------- #
    def remove_headers_footers(self, text: str) -> str:
        lines = text.split('\n')
        cleaned_lines = []
        removed_count = 0
        for line in lines:
            is_header_footer = any(p.match(line.strip()) for p in self.compiled_headers)
            if is_header_footer:
                removed_count += 1
            else:
                cleaned_lines.append(line)

        if removed_count > 0:
            self.logger.info(f"   Removed {removed_count} headers/footers")
        return '\n'.join(cleaned_lines)

    # ---------------------------------------------------------------------- #
    # Copyright line removal                                                 #
    # ---------------------------------------------------------------------- #
    def remove_copyright_lines(self, text: str) -> str:
        lines = text.split('\n')
        cleaned_lines = []
        removed_count = 0
        for line in lines:
            is_copyright = any(p.match(line.strip()) for p in self.compiled_copyright_lines)
            if is_copyright:
                removed_count += 1
            else:
                cleaned_lines.append(line)

        if removed_count > 0:
            self.logger.info(f"   Removed {removed_count} copyright lines")
        return '\n'.join(cleaned_lines)

    # ---------------------------------------------------------------------- #
    # ALL-CAPS and spaced-letter title removal                               #
    # ---------------------------------------------------------------------- #
    def remove_styled_titles(self, text: str) -> str:
        """
        Remove lines that are:
          - Written entirely in CAPITAL LETTERS (>= 2 words, to spare acronyms)
          - Written with a space between every letter  (e.g. "I N T R O")
        """
        lines = text.split('\n')
        cleaned_lines = []
        removed_count = 0

        for line in lines:
            stripped = line.strip()

            # --- spaced-letter check first (more specific) ---
            if self._spaced_title_re.match(stripped):
                removed_count += 1
                continue

            # --- ALL-CAPS check: must have at least two "words" ---
            if self._allcaps_title_re.match(stripped):
                words = stripped.split()
                if len(words) >= 2:          # skip lone acronyms / codes
                    removed_count += 1
                    continue

            cleaned_lines.append(line)

        if removed_count > 0:
            self.logger.info(f"   Removed {removed_count} styled title lines")
        return '\n'.join(cleaned_lines)

    # ---------------------------------------------------------------------- #
    # Non-English character removal                                            #
    # ---------------------------------------------------------------------- #
    def remove_non_english(self, text: str) -> str:
        """
        Strip characters that are not part of the English writing system.

        Kept:
          - ASCII letters and digits          (A-Z, a-z, 0-9)
          - Common punctuation and symbols    (!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~)
          - Whitespace                        (space, tab, newline, carriage return)

        Removed (examples):
          - Arabic / Hebrew / CJK / Cyrillic / Greek scripts
          - Accented / diacritic Latin letters outside ASCII (é, ñ, ü, …)
          - Any other Unicode codepoint above U+007E that isn't whitespace

        Design note: characters are replaced with a space rather than deleted
        outright so that adjacent ASCII words separated only by a non-English
        run are not accidentally joined into a single token.  Normalise_whitespace
        (called later in the pipeline) collapses the extra spaces.
        """
        # Replace every character outside printable ASCII (except whitespace)
        # with a single space.
        cleaned = re.sub(r'[^\x20-\x7E\t\n\r]', ' ', text)

        removed_count = sum(1 for a, b in zip(text, cleaned) if a != b)
        # Count characters that exist in text but were replaced
        # (zip stops at the shorter; lengths are equal so this is correct)
        if removed_count > 0:
            self.logger.info(f"   Removed {removed_count} non-English characters")

        return cleaned

    # ---------------------------------------------------------------------- #
    # General artifact removal                                               #
    # ---------------------------------------------------------------------- #
    def remove_artifacts(self, text: str) -> str:
        # ── Bullet-only lines ─────────────────────────────────────────────────
        text = re.sub(r'^\s*[•▪▫■□●○◆◇★☆‣⁃]+\s*$', '', text, flags=re.MULTILINE)

        # ── StandardsInfo / ANSI / NFPA / ICC license tracking codes ─────────
        # e.g. "--`,`,,,,,,`,,,,`,````,`,,,,,``-`-`,,`,,`,`,,`---"
        text = re.sub(r'^\s*[-`,]{5,}\s*$', '', text, flags=re.MULTILINE)

        # ── Horizontal rules (dashes, underscores, equals, tildes) ───────────
        text = re.sub(r'^\s*[-_=~*#]{3,}\s*$', '', text, flags=re.MULTILINE)

        # ── Repeated punctuation runs (e.g. ".....", "-----", "=====") ────────
        text = re.sub(r'([.=*~])\1{4,}', '', text)

        # ── Form feed → newline ───────────────────────────────────────────────
        text = text.replace('\f', '\n')

        # ── Zero-width / BOM / invisible Unicode characters ───────────────────
        text = re.sub(r'[\u200b-\u200d\u200e\u200f\ufeff\u00ad]', '', text)  # soft-hyphen too

        # ── Inline copyright / trademark symbols ──────────────────────────────
        text = re.sub(r'[®™℠©]', '', text)

        # ── Enclosed alphanumerics U+2460–U+24FF (circled numbers, etc.) ──────
        text = re.sub(r'[\u2460-\u24ff]', '', text)

        # ── Standalone special Unicode punctuation ────────────────────────────
        # Arrows, box-drawing, block elements, geometric shapes
        text = re.sub(r'[\u2190-\u21ff]', ' ', text)   # arrows
        text = re.sub(r'[\u2500-\u257f]', ' ', text)   # box drawing
        text = re.sub(r'[\u2580-\u259f]', ' ', text)   # block elements
        text = re.sub(r'[\u25a0-\u25ff]', ' ', text)   # geometric shapes
        text = re.sub(r'[\u2600-\u26ff]', ' ', text)   # miscellaneous symbols
        text = re.sub(r'[\u2700-\u27bf]', ' ', text)   # dingbats

        # ── PDF ligature / glyph artefacts ────────────────────────────────────
        # Common bad extractions: ﬁ→fi, ﬂ→fl, ﬀ→ff, ﬃ→ffi, ﬄ→ffl
        text = text.replace('\ufb00', 'ff')
        text = text.replace('\ufb01', 'fi')
        text = text.replace('\ufb02', 'fl')
        text = text.replace('\ufb03', 'ffi')
        text = text.replace('\ufb04', 'ffl')
        text = text.replace('\ufb05', 'st')
        text = text.replace('\ufb06', 'st')

        # ── Hyphenation artefacts from PDF line-breaks ────────────────────────
        # "rein- \nforcement" → "reinforcement"
        text = re.sub(r'-\s*\n\s*([a-z])', r'\1', text)

        # ── Spurious lone punctuation lines ───────────────────────────────────
        # Lines containing only punctuation chars (no alphanumeric content)
        text = re.sub(r'^\s*[^a-zA-Z0-9\n]{1,5}\s*$', '', text, flags=re.MULTILINE)

        # ── Page number artifacts (standalone numbers or "- 5 -" style) ───────
        text = re.sub(r'^\s*-\s*\d+\s*-\s*$', '', text, flags=re.MULTILINE)

        # ── Watermark-style repeated word lines ───────────────────────────────
        # e.g. "DRAFT DRAFT DRAFT DRAFT DRAFT"
        text = re.sub(
            r'^\s*(\b\w+\b)(?:\s+\1){3,}\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE
        )

        # ── Scanned-page OCR noise: isolated single characters on a line ──────
        text = re.sub(r'^\s*[^a-zA-Z0-9]\s*$', '', text, flags=re.MULTILINE)

        # ── Column separator artifacts (tab/space runs used as table dividers) ─
        text = re.sub(r'(\t{2,})', ' ', text)           # multiple tabs → single space
        text = re.sub(r'[ \t]+\n', '\n', text)          # trailing whitespace on lines

        return text

    # ---------------------------------------------------------------------- #
    # Whitespace normalisation                                               #
    # ---------------------------------------------------------------------- #
    def normalize_whitespace(self, text: str) -> str:
        text = re.sub(r' {2,}', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        lines = [line.strip() for line in text.split('\n')]
        return '\n'.join(lines)

    # ---------------------------------------------------------------------- #
    # Master pipeline                                                        #
    # ---------------------------------------------------------------------- #
    def clean_text(self, text: str) -> str:
        original_length = len(text)

        text = self.remove_toc_entries(text)
        text = self.remove_headers_footers(text)
        text = self.remove_copyright_lines(text)
        text = self.remove_styled_titles(text)
        text = self.remove_non_english(text)
        text = self.remove_artifacts(text)
        text = self.normalize_whitespace(text)

        cleaned_length = len(text)
        reduction = (original_length - cleaned_length) / original_length * 100
        self.logger.info(
            f"   Cleaned text: {original_length:,} -> {cleaned_length:,} chars "
            f"({reduction:.1f}% reduction)"
        )
        return text

###PDF TEXT EXTRACTOR

In [ ]:
class PDFTextExtractor:

    def __init__(
        self,
        logger:       logging.Logger,
        preprocessor  = None,
        config:       dict = None,
        cache_manager : 'TextCacheManager' = None,
    ):
        self.logger       = logger
        self.preprocessor = preprocessor
        cfg = config or {}
        checkpoint_dir = cfg.get('checkpoint_dir', '')
        cache_dir = os.path.join(checkpoint_dir, 'text_cache')
        os.makedirs(cache_dir, exist_ok=True)

        if cache_manager is not None:
            self.cache_manager = cache_manager
        else:
            cache_dir = os.path.join(cfg.get('checkpoint_dir', ''), 'text_cache')
            os.makedirs(cache_dir, exist_ok=True)
            self.cache_manager = TextCacheManager(
                checkpoint_dir=cache_dir,
                logger=logger,
            )

    # ---------------------------------------------------------------------- #
    # Internal helpers                                                       #
    # ---------------------------------------------------------------------- #

    @staticmethod
    def _safe_int(value) -> int:
        try:
            return int(value)
        except (ValueError, TypeError):
            return -1

    def _log_text_stats(self, label: str, text: str) -> None:
        self.logger.info(f"   {label} text stats:")
        self.logger.info(f"     Length   : {len(text):,} chars")
        self.logger.info(f"     Newlines : {text.count(chr(10))}")
        self.logger.info(f"     Periods  : {text.count('.')}")
        self.logger.info(f"     Sample   : {text[:200]}")

    # ---------------------------------------------------------------------- #
    # Extraction backends                                                      #
    # ---------------------------------------------------------------------- #

    def _extract_pdfplumber(self, pdf_path: str) -> str:
        try:
            pdfminer_logger = logging.getLogger('pdfminer.pdfinterp')
            original_level  = pdfminer_logger.level
            pdfminer_logger.setLevel(logging.ERROR)

            pages_text = []
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    width  = page.width
                    height = page.height

                    # Try two-column extraction first
                    left_bbox  = (0,        0, width / 2, height)
                    right_bbox = (width / 2, 0, width,    height)

                    left_text  = page.crop(left_bbox).extract_text()  or ''
                    right_text = page.crop(right_bbox).extract_text() or ''

                    # Heuristic: if both halves have content, it's two-column
                    if left_text.strip() and right_text.strip():
                        pages_text.append(left_text + '\n' + right_text)
                    else:
                        # Fall back to single-column extraction
                        full_text = page.extract_text() or ''
                        pages_text.append(full_text)

            pdfminer_logger.setLevel(original_level)

            text = '\n'.join(p for p in pages_text if p)
            if text.strip():
                self.logger.info(f"   Extracted {len(text):,} chars via pdfplumber")
                return text
        except Exception as e:
            self.logger.warning(f"pdfplumber failed: {e}")
        return ''

    def _extract_ocr(self, pdf_path: str) -> str:
        self.logger.info("   Trying OCR extraction...")
        try:
            images = convert_from_path(pdf_path, dpi=300)
            pages  = []
            for i, image in enumerate(images):
                page_text = pytesseract.image_to_string(image)
                pages.append(page_text)
                data = pytesseract.image_to_data(
                    image, output_type=pytesseract.Output.DICT
                )
                confidences = [
                    v for v in (self._safe_int(c) for c in data['conf']) if v > 0
                ]
                avg_conf = sum(confidences) / len(confidences) if confidences else 0
                if avg_conf < 60:
                    self.logger.warning(
                        f"   Low OCR confidence ({avg_conf:.0f}%) on page {i}"
                    )
            text = '\n'.join(pages)
            if text.strip():
                self.logger.info(f"   Extracted {len(text):,} chars via OCR")
                return text
        except Exception as e:
            self.logger.error(f"OCR failed: {e}")
        return ''

    # ---------------------------------------------------------------------- #
    # Public: text extraction                                                  #
    # silent=True suppresses per-file logger.info so tqdm bars stay clean     #
    # ---------------------------------------------------------------------- #

    def extract_text_with_cache(self, pdf_path: str, silent: bool = False) -> str:
        def _info(msg: str) -> None:
            if not silent:
                self.logger.info(msg)

        # 1. Cleaned text cache hit — but validate before trusting it
        if self.cache_manager.has_cached(pdf_path, stage='cleaned'):
            cleaned_text = self.cache_manager.load_cached(pdf_path, stage='cleaned')
            if cleaned_text and len(cleaned_text.strip()) >= 100:
                _info(f"   Loaded cleaned text from cache: {len(cleaned_text):,} chars")
                return cleaned_text
            # FIX: stale/poisoned cleaned entry — evict both stages and re-extract
            self.logger.warning(
                f"⚠️  Stale cleaned cache entry for '{Path(pdf_path).name}' "
                f"({len(cleaned_text) if cleaned_text else 0} chars) — evicting."
            )
            try:
                self.cache_manager.delete_cached(pdf_path, stage='cleaned')
                self.cache_manager.delete_cached(pdf_path, stage='raw')
            except Exception:
                pass

        # 2. Obtain raw text — check raw cache first
        raw_text = None
        if self.cache_manager.has_cached(pdf_path, stage='raw'):
            raw_text = self.cache_manager.load_cached(pdf_path, stage='raw')
            # FIX: validate raw cache entry too
            if raw_text and len(raw_text.strip()) < 100:
                self.logger.warning(
                    f"⚠️  Stale raw cache entry for '{Path(pdf_path).name}' — evicting."
                )
                try:
                    self.cache_manager.delete_cached(pdf_path, stage='raw')
                except Exception:
                    pass
                raw_text = None

        if not raw_text:
            _info(f"   Extracting text from: {Path(pdf_path).name}")
            # FIX: log file diagnostics before attempting extraction
            p = Path(pdf_path)
            self.logger.info(
                f"   File: exists={p.exists()}, "
                f"size={p.stat().st_size if p.exists() else 'N/A'} bytes, "
                f"readable={os.access(pdf_path, os.R_OK) if p.exists() else False}"
            )
            raw_text = self._extract_pdfplumber(pdf_path)
            if not raw_text:
                raw_text = self._extract_ocr(pdf_path)
            if not raw_text:
                self.logger.error(f"❌ Failed to extract text from {pdf_path}")
                return ''
            if not silent:
                self._log_text_stats('RAW', raw_text)
            self.cache_manager.save_cached(pdf_path, raw_text, stage='raw')

        # 3. Clean exactly once
        if self.preprocessor:
            cleaned_text = self.preprocessor.clean_text(raw_text)
            reduction = (1 - len(cleaned_text) / max(len(raw_text), 1)) * 100
            if reduction > 50:
                self.logger.warning(
                    f"   ⚠️  Cleaning removed {reduction:.1f}% of text for "
                    f"'{Path(pdf_path).name}'!"
                )
            # FIX: if cleaning wiped the text, fall back to raw
            if not cleaned_text or len(cleaned_text.strip()) < 100:
                self.logger.warning(
                    f"   ⚠️  Cleaned text too short ({len(cleaned_text) if cleaned_text else 0} "
                    f"chars) — falling back to raw text."
                )
                cleaned_text = raw_text
            if cleaned_text.count('.') < len(cleaned_text) / 1000:
                self.logger.warning(
                    "   Very few periods detected — sentence splitting may fail!"
                )
            if not silent:
                self._log_text_stats('CLEANED', cleaned_text)
        else:
            cleaned_text = raw_text

        # 4. Only persist if content is valid — never cache empty strings
        if cleaned_text and len(cleaned_text.strip()) >= 100:
            self.cache_manager.save_cached(pdf_path, cleaned_text, stage='cleaned')
        return cleaned_text

    # ---------------------------------------------------------------------- #
    # Public: chunk caching                                                    #
    # ---------------------------------------------------------------------- #

    def get_chunks_with_cache(
        self,
        pdf_path:     str,
        cleaned_text: str,
        chunker,
        doc_id:       str,
        silent:       bool = False,   # NEW
    ) -> List[Dict]:
        """
        Return chunks for a document, using the chunk cache when available.

        Args:
            silent : suppress logger.info and chunker progress bars so that
                     an outer tqdm loop is not broken.
        """
        if self.cache_manager.has_cached_chunks(pdf_path, cleaned_text):
            chunks = self.cache_manager.load_cached_chunks(pdf_path, cleaned_text)
            if chunks is not None:
                if not silent:
                    self.logger.info(
                        f"   Chunk cache hit: {Path(pdf_path).name} "
                        f"({len(chunks)} chunks)"
                    )
                return chunks

        if not silent:
            self.logger.info(f"   Chunking {Path(pdf_path).name} (no cache)...")

        # Temporarily disable chunker progress bars when silent
        original_show_progress = getattr(chunker, 'show_progress', True)
        if silent:
            chunker.show_progress = False

        try:
            chunks = chunker.chunk_optimal(cleaned_text, doc_id)
        finally:
            # Always restore, even if chunk_optimal raises
            chunker.show_progress = original_show_progress

        if not silent:
            self.logger.info(f"   Created {len(chunks)} chunks")

        self.cache_manager.save_cached_chunks(pdf_path, cleaned_text, chunks)
        return chunks


In [ ]:
def reprocess_cache(
    checkpoint_dir:  str  = None,
    dry_run:    bool = False,
    also_raw:   bool = False,
    force:      bool = False,
    verbose:    bool = False,
) -> dict:
    """
    Re-run TextPreprocessor over every cached text entry and overwrite in place.

    Parameters
    ----------
    checkpoint_dir : str
        Path to the cache directory. If None, reads from the project config.
    dry_run : bool
        If True, log what would change but write nothing.
    also_raw : bool
        If True, re-clean from the 'raw' cache stage rather than 'cleaned'.
        Useful for a fully fresh pass after changing extraction logic.
    force : bool
        If True, write results even when the reduction exceeds 70%.
    verbose : bool
        If True, log files that had no change as well.

    Returns
    -------
    dict with keys: processed, unchanged, skipped, errors
    """

    log = logging.getLogger('reprocess_cache')
    if not log.handlers:
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s  %(levelname)-8s  %(message)s',
            datefmt='%H:%M:%S',
            stream=sys.stdout,
        )

    # ── Resolve cache directory ───────────────────────────────────────────
    if checkpoint_dir is None:
        try:
            checkpoint_dir = config.get('checkpoint_dir', "")+"/text_cache",
        except NameError:
            raise RuntimeError(
                "checkpoint_dir was not provided and 'config' could not be found. "
                "Pass checkpoint_dir='/path/to/cache' explicitly."
            )

    cache_path = Path(checkpoint_dir)
    if not cache_path.exists():
        raise FileNotFoundError(f"Cache directory does not exist: {cache_path}")

    source_stage = 'raw' if also_raw else 'cleaned'

    log.info(f"Cache directory : {cache_path}")
    log.info(f"Source stage    : {source_stage}")
    log.info(f"Dry run         : {dry_run}")
    log.info(f"Force           : {force}")

    # ── Build preprocessor ────────────────────────────────────────────────
    preprocessor = TextPreprocessor(log)

    # ── Discover cache files ──────────────────────────────────────────────
    # TextCacheManager names files: <slug>_<stage>_<hash8>.txt
    # The pattern *_{stage}_*.txt anchors the stage between two underscores
    # so a slug containing the word "raw" or "cleaned" is not falsely matched.
    stage_files = sorted(cache_path.glob(f"*_{source_stage}_*.txt"))

    if not stage_files:
        log.warning(
            f"No '{source_stage}' cache files found in {cache_path}. "
            "Check that the glob pattern matches your TextCacheManager naming."
        )
        return {'processed': 0, 'unchanged': 0, 'skipped': 0, 'errors': 0}

    log.info(f"Found {len(stage_files)} '{source_stage}' cache entries\n")

    SAFETY_THRESHOLD = 0.70
    stats = {'processed': 0, 'unchanged': 0, 'skipped': 0, 'errors': 0}

    for cache_file in stage_files:
        label = cache_file.stem   # used only for log messages

        # ── Load directly from disk ───────────────────────────────────────
        # BUG FIX: we already have the real file path from the glob.
        # Reading it directly avoids the cache manager's key-translation
        # logic, which expects an original PDF path, not a cache stem.
        try:
            source_text = cache_file.read_text(encoding='utf-8')
        except Exception as exc:
            log.error(f"[{label}] Failed to load: {exc}")
            stats['errors'] += 1
            continue

        if not source_text or not source_text.strip():
            log.warning(f"[{label}] Empty — skipping")
            stats['skipped'] += 1
            continue

        original_len = len(source_text)

        # ── Re-clean ──────────────────────────────────────────────────────
        try:
            cleaned_text = preprocessor.clean_text(source_text)
        except Exception as exc:
            log.error(f"[{label}] clean_text() raised: {exc}")
            stats['errors'] += 1
            continue

        new_len   = len(cleaned_text)
        reduction = (original_len - new_len) / max(original_len, 1)

        # ── Safety check ──────────────────────────────────────────────────
        if reduction > SAFETY_THRESHOLD and not force:
            log.warning(
                f"[{label}] SKIPPED — {reduction*100:.1f}% reduction exceeds "
                f"{SAFETY_THRESHOLD*100:.0f}% threshold. Pass force=True to override."
            )
            stats['skipped'] += 1
            continue

        # ── No-op check ───────────────────────────────────────────────────
        if cleaned_text == source_text:
            if verbose:
                log.info(f"[{label}] No change ({original_len:,} chars)")
            stats['unchanged'] += 1
            continue

        # ── Determine output path ─────────────────────────────────────────
        # When re-cleaning from 'raw', write the result to the corresponding
        # 'cleaned' file so the normal pipeline picks it up on the next run.
        # When re-cleaning from 'cleaned', overwrite the same file.
        if also_raw:
            out_file = Path(str(cache_file).replace(
                f'_{source_stage}_', '_cleaned_', 1
            ))
        else:
            out_file = cache_file

        sign   = '-' if new_len <= original_len else '+'
        delta  = abs(original_len - new_len)
        action = 'Would write' if dry_run else 'Saved'

        log.info(
            f"[{label}]  {original_len:>10,} -> {new_len:>10,} chars  "
            f"({sign}{delta:,} / {reduction*100:.1f}% reduction)  "
            f"{action} -> {out_file.name}"
        )

        if not dry_run:
            try:
                out_file.write_text(cleaned_text, encoding='utf-8')
            except Exception as exc:
                log.error(f"[{label}] Failed to write: {exc}")
                stats['errors'] += 1
                continue

        stats['processed'] += 1

    # ── Summary ───────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("REPROCESS CACHE — DONE")
    print("=" * 55)
    print(f"  Source stage : {source_stage}")
    print(f"  Dry run      : {dry_run}")
    print(f"  Files found  : {len(stage_files)}")
    print(f"  Processed    : {stats['processed']}")
    print(f"  Unchanged    : {stats['unchanged']}")
    print(f"  Skipped      : {stats['skipped']}")
    print(f"  Errors       : {stats['errors']}")
    print("=" * 55)
    if dry_run:
        print("Dry run — no files were modified.")

    return stats
#reprocess_cache(
#   checkpoint_dir = config['checkpoint_dir'],
#    dry_run   = False,
#    )

###TEXT CACHE MANAGER

In [ ]:
class TextCacheManager:
"""Cache manager for extracted, preprocessed, and chunked PDF text.

Two independent, file-based caches live side by side inside a single
``checkpoint_dir``:

- **Text cache** (``.txt``): keyed by the source PDF's path and a
  caller-supplied ``stage`` (e.g. ``'raw'`` or ``'cleaned'``).
- **Chunk cache** (``.json``): keyed by the source PDF's path *and*
  a hash of the cleaned text it was built from, so it self-invalidates
  whenever the input text changes.

Cache invalidation combines two strategies: filesystem mtime
comparisons (the PDF is newer than the cache) and content hashing
(the cleaned text, or the cache format version, has changed).

Attributes:
    checkpoint_dir (Path): Directory where all cache files are stored.
    logger (logging.Logger): Logger used for cache hit/miss/save/
        invalidate messages.

Class Attributes:
    CHUNK_CACHE_VERSION (str): Version tag mixed into the chunk
        cache's content hash. Bump this after any change to chunking
        logic to force all existing chunk caches to miss, without
        deleting any files on disk.
"""

    CHUNK_CACHE_VERSION = "v1"

    def __init__(self, checkpoint_dir: str, logger: logging.Logger):
        """Initialize the cache manager and ensure the cache dir exists.

        Args:
            checkpoint_dir: Path to the directory where cache files are
                read from and written to. Created, including any missing
                parent directories, if it doesn't already exist.
            logger: Logger instance used for all cache-related logging.
        """
        self.checkpoint_dir = Path(checkpoint_dir)
        self.logger = logger
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # ---------------------------------------------------------------------- #
    # Internal helpers
    # ---------------------------------------------------------------------- #

    @staticmethod
    def _slugify(text: str) -> str:
        slug = re.sub(r'[^\w]', '_', text)
        slug = re.sub(r'_+', '_', slug).strip('_')
        return slug or 'unnamed'

    def _path_hash(self, pdf_path: str) -> str:
        """8-char MD5 of the resolved absolute path — stable across runs."""
        abs_path = str(Path(pdf_path).resolve())
        return hashlib.md5(abs_path.encode()).hexdigest()[:8]

    def _content_hash(self, cleaned_text: str) -> str:
        key = f"{cleaned_text}|{self.CHUNK_CACHE_VERSION}"
        return hashlib.md5(key.encode()).hexdigest()[:8]

    def _get_cache_path(self, pdf_path: str, stage: str = 'cleaned') -> Path:
        slug = self._slugify(Path(pdf_path).stem)
        return self.checkpoint_dir / f"{slug}_{stage}_{self._path_hash(pdf_path)}.txt"

    def _get_chunk_cache_path(self, pdf_path: str, cleaned_text: str) -> Path:
        slug = self._slugify(Path(pdf_path).stem)
        return self.checkpoint_dir / (
            f"{slug}_chunks"
            f"_{self._path_hash(pdf_path)}"
            f"_{self._content_hash(cleaned_text)}.json"
        )

    def _slug_prefix_glob(self, pdf_path: str) -> str:
        slug = self._slugify(Path(pdf_path).stem)
        path_hash = self._path_hash(pdf_path)
        return f"{slug}_*_{path_hash}*.*"

    def needs_refresh(self, pdf_path: str, stage: str = 'cleaned') -> bool:
        """Return True if cache is missing or older than the PDF file."""
        cache_path = self._get_cache_path(pdf_path, stage)
        if not cache_path.exists():
            return True
        try:
            pdf_mtime = Path(pdf_path).stat().st_mtime
            cache_mtime = cache_path.stat().st_mtime
            return pdf_mtime > cache_mtime
        except OSError:
            return True

    # ---------------------------------------------------------------------- #
    # Text cache (raw / cleaned)
    # ---------------------------------------------------------------------- #

    def has_cached(self, pdf_path: str, stage: str = 'cleaned') -> bool:
        if not self._get_cache_path(pdf_path, stage).exists():
            return False
        # Avoid stale cache if PDF has been modified
        if self.needs_refresh(pdf_path, stage):
            self.logger.info(f"[cache stale] {pdf_path} → re‑extracting")
            return False
        return True

    def load_cached(self, pdf_path: str, stage: str = 'cleaned') -> Optional[str]:
        cache_path = self._get_cache_path(pdf_path, stage)
        try:
            text = cache_path.read_text(encoding='utf-8')
            self.logger.info(f"[cache hit]  {cache_path.name}  ({len(text):,} chars)")
            return text
        except FileNotFoundError:
            return None
        except Exception as e:
            self.logger.warning(f"[cache] Failed to load {cache_path.name}: {e}")
            return None

    def save_cached(self, pdf_path: str, text: str, stage: str = 'cleaned') -> None:
        cache_path = self._get_cache_path(pdf_path, stage)
        try:
            cache_path.write_text(text, encoding='utf-8')
            self.logger.info(f"[cache save] {cache_path.name}  ({len(text):,} chars)")
        except Exception as e:
            self.logger.warning(f"[cache] Failed to save {cache_path.name}: {e}")

    # ---------------------------------------------------------------------- #
    # Chunk cache
    # ---------------------------------------------------------------------- #

    def has_cached_chunks(self, pdf_path: str, cleaned_text: str) -> bool:
        cache_path = self._get_chunk_cache_path(pdf_path, cleaned_text)
        if not cache_path.exists():
            return False
        # Optional: check if PDF is newer than chunk cache
        try:
            pdf_mtime = Path(pdf_path).stat().st_mtime
            cache_mtime = cache_path.stat().st_mtime
            if pdf_mtime > cache_mtime:
                self.logger.info(f"[chunk cache stale] {cache_path.name}")
                return False
        except OSError:
            pass
        return True

    def load_cached_chunks(
        self, pdf_path: str, cleaned_text: str
    ) -> Optional[List[Dict]]:
        cache_path = self._get_chunk_cache_path(pdf_path, cleaned_text)
        try:
            chunks = json.loads(cache_path.read_text(encoding='utf-8'))
            self.logger.info(f"[chunk hit]  {cache_path.name}  ({len(chunks)} chunks)")
            return chunks
        except FileNotFoundError:
            return None
        except json.JSONDecodeError as e:
            self.logger.warning(
                f"[chunk cache] Corrupt JSON in {cache_path.name}, will re-chunk: {e}"
            )
            try:
                cache_path.unlink()
            except OSError:
                pass
            return None
        except Exception as e:
            self.logger.warning(f"[chunk cache] Failed to load {cache_path.name}: {e}")
            return None

    def save_cached_chunks(
        self, pdf_path: str, cleaned_text: str, chunks: List[Dict]
    ) -> None:
        cache_path = self._get_chunk_cache_path(pdf_path, cleaned_text)
        try:
            cache_path.write_text(
                json.dumps(chunks, ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            self.logger.info(f"[chunk save] {cache_path.name}  ({len(chunks)} chunks)")
        except TypeError as e:
            self.logger.warning(
                f"[chunk cache] Chunks are not JSON‑serialisable, skipping: {e}"
            )
        except Exception as e:
            self.logger.warning(f"[chunk cache] Failed to save {cache_path.name}: {e}")

    # ---------------------------------------------------------------------- #
    # Cache management
    # ---------------------------------------------------------------------- #

    def clear_cache(self, pdf_path: str = None) -> int:
        removed = 0
        if pdf_path:
            slug = self._slugify(Path(pdf_path).stem)
            path_hash = self._path_hash(pdf_path)
            for f in self.checkpoint_dir.glob(f"{slug}_*_{path_hash}*.*"):
                f.unlink()
                self.logger.info(f"[cache invalidate] {f.name}")
                removed += 1
        else:
            for f in list(self.checkpoint_dir.glob('*.txt')) + list(self.checkpoint_dir.glob('*.json')):
                f.unlink()
                removed += 1
            self.logger.info(
                f"[cache clear] Removed {removed} files from {self.checkpoint_dir}"
            )
        return removed

    def get_cache_stats(self) -> dict:
        txt_files = list(self.checkpoint_dir.glob('*.txt'))
        json_files = list(self.checkpoint_dir.glob('*.json'))
        total_size = 0
        for f in txt_files + json_files:
            try:
                total_size += f.stat().st_size
            except OSError:
                pass
        return {
            'num_text_files': len(txt_files),
            'num_chunk_files': len(json_files),
            'total_files': len(txt_files) + len(json_files),
            'total_size_mb': round(total_size / (1024 * 1024), 2),
            'checkpoint_dir': str(self.checkpoint_dir),
        }

### TEXT CHUNKING

In [ ]:
class ChunkingStrategy(Enum):
    STRUCTURAL = "structural"  # Use document structure (paragraphs, sections)
    SEMANTIC_LITE = "semantic_lite"  # Semantic with sampling
    SEMANTIC_FULL = "semantic_full"  # Full semantic analysis
    SLIDING_WINDOW = "sliding_window"  # Simple sliding window
    HYBRID = "hybrid"  # Combination approach

In [ ]:
@dataclass
class DocumentProfile:
    """Profile of document characteristics"""
    total_chars: int
    estimated_tokens: int
    paragraph_count: int
    has_structure: bool
    avg_paragraph_length: float
    recommended_strategy: ChunkingStrategy

In [ ]:
class SemanticChunker:
    def __init__(
        self,
        nlp,
        chunk_size:    int   = 500,
        overlap:       int   = 50,
        max_memory_mb: int   = 1024,
        prefer_quality: bool = True,
        show_progress: bool  = True,
    ):
        self.nlp           = nlp
        self.chunk_size    = chunk_size
        self.overlap       = overlap
        self.max_memory_mb = max_memory_mb
        self.prefer_quality = prefer_quality
        self.show_progress = show_progress

        if 'sentencizer' not in nlp.pipe_names:
            nlp.add_pipe('sentencizer')

    # ---------------------------------------------------------------------- #
    # Document profiling                                                       #
    # ---------------------------------------------------------------------- #

    def profile_document(self, text: str) -> 'DocumentProfile':
        total_chars           = len(text)
        estimated_tokens      = total_chars // 5
        paragraphs            = text.split('\n')
        paragraph_count       = len([p for p in paragraphs if p.strip()])
        has_headers           = bool(re.search(r'^#{1,6}\s+.+$', text, re.MULTILINE))
        has_numbered_sections = bool(re.search(r'^\d+\.\s+.+$', text, re.MULTILINE))
        has_structure         = has_headers or has_numbered_sections
        avg_paragraph_length  = total_chars / max(1, paragraph_count)

        if total_chars < 100_000:
            strategy = (
                ChunkingStrategy.SEMANTIC_FULL
                if self.prefer_quality
                else ChunkingStrategy.SLIDING_WINDOW
            )
        elif total_chars < 1_000_000:
            strategy = ChunkingStrategy.HYBRID if has_structure else ChunkingStrategy.SEMANTIC_LITE
        elif total_chars < 10_000_000:
            strategy = ChunkingStrategy.STRUCTURAL if has_structure else ChunkingStrategy.SEMANTIC_LITE
        else:
            strategy = ChunkingStrategy.STRUCTURAL if has_structure else ChunkingStrategy.SLIDING_WINDOW

        return DocumentProfile(
            total_chars=total_chars,
            estimated_tokens=estimated_tokens,
            paragraph_count=paragraph_count,
            has_structure=has_structure,
            avg_paragraph_length=avg_paragraph_length,
            recommended_strategy=strategy,
        )

    # ---------------------------------------------------------------------- #
    # STRUCTURAL chunking                                                      #
    # ---------------------------------------------------------------------- #

    def chunk_structural(self, text: str, doc_id: str) -> List[Dict]:
        logger.info("Using STRUCTURAL chunking strategy")
        chunks   = []
        chunk_id = 0

        # BUG-8 FIX: removed \n\n from the lookahead so we only split on
        # genuine structural markers, not every blank line.
        sections = re.split(r'\n(?=#{1,6}\s+|\d+\.\s+)', text)

        # Track our absolute position inside the original `text` string.
        # We advance search_from after each located piece so we always find
        # the *next* occurrence, not an earlier duplicate.
        search_from = 0

        pbar = tqdm(sections, desc="Processing sections", disable=not self.show_progress)

        for section in pbar:
            if not section.strip():
                # Still advance the position pointer past this empty section
                idx = text.find(section, search_from)
                if idx != -1:
                    search_from = idx + len(section)
                continue

            # Locate this section in the source string
            section_start_in_text = text.find(section, search_from)
            if section_start_in_text == -1:
                # Fallback: keep going; positions for this section will be
                # approximate, but the text content is still correct.
                section_start_in_text = search_from
            search_from = section_start_in_text + len(section)

            paragraphs    = section.split('\n\n')
            current_chunk: List[str] = []
            current_tokens = 0

            # BUG-1 / BUG-2 FIX: track chunk_start relative to the real text
            # rather than accumulating joined-text lengths with a fixed +2.
            para_search_from = section_start_in_text

            def _emit_chunk(para_list: List[str], start: int) -> int:
                """Append a chunk; return the start position for the next chunk."""
                nonlocal chunk_id
                chunk_text = '\n\n'.join(para_list)
                end        = start + len(chunk_text)
                chunks.append({
                    'doc_id':      doc_id,
                    'chunk_id':    f"{doc_id}_chunk_{chunk_id}",
                    'text':        chunk_text,
                    'token_count': sum(len(p.split()) for p in para_list),
                    'strategy':    'structural',
                    'start_char':  start,
                    'end_char':    end,
                })
                chunk_id += 1
                return end

            for para in paragraphs:
                para = para.strip()
                if not para:
                    continue

                para_tokens = len(para.split())

                # Locate this paragraph in the source to get its true offset
                para_pos = text.find(para, para_search_from)
                if para_pos == -1:
                    para_pos = para_search_from
                para_search_from = para_pos + len(para)

                if current_tokens + para_tokens > self.chunk_size and current_chunk:
                    # Emit accumulated chunk; start_char is the position of
                    # the first word in current_chunk inside the source text.
                    chunk_start_pos = text.find(current_chunk[0], section_start_in_text)
                    if chunk_start_pos == -1:
                        chunk_start_pos = section_start_in_text
                    _emit_chunk(current_chunk, chunk_start_pos)

                    if self.overlap > 0:
                        # Re-include the last paragraph for overlap.
                        # The next chunk's start_char is the position of that
                        # last paragraph — NOT chunk_start + len(chunk_text).
                        overlap_para     = current_chunk[-1]
                        overlap_para_pos = text.find(overlap_para, section_start_in_text)
                        if overlap_para_pos == -1:
                            overlap_para_pos = para_pos
                        current_chunk  = [overlap_para, para]
                        current_tokens = len(overlap_para.split()) + para_tokens
                        para_search_from = max(para_search_from, overlap_para_pos)
                    else:
                        current_chunk  = [para]
                        current_tokens = para_tokens
                else:
                    current_chunk.append(para)
                    current_tokens += para_tokens

            pbar.set_postfix({'chunks': chunk_id})

            if current_chunk:
                chunk_start_pos = text.find(current_chunk[0], section_start_in_text)
                if chunk_start_pos == -1:
                    chunk_start_pos = section_start_in_text
                _emit_chunk(current_chunk, chunk_start_pos)

        pbar.close()
        logger.info(f"Created {len(chunks)} chunks using structural strategy")
        return chunks

    # ---------------------------------------------------------------------- #
    # SLIDING WINDOW chunking                                                  #
    # ---------------------------------------------------------------------- #

    def chunk_sliding_window(self, text: str, doc_id: str) -> List[Dict]:
        logger.info("Using SLIDING WINDOW chunking")
        chunks     = []
        chunk_id   = 0
        words      = text.split()
        total_words = len(words)

        # BUG-3 FIX: build a word-start-offset array so we can recover
        # accurate start_char / end_char for every window.
        word_offsets = []
        pos = 0
        for w in words:
            idx = text.find(w, pos)
            word_offsets.append(idx if idx != -1 else pos)
            pos = (idx if idx != -1 else pos) + len(w)

        stride           = max(1, self.chunk_size - self.overlap)
        window_positions = list(range(0, total_words, stride))
        pbar             = tqdm(
            window_positions, desc="Creating chunks", disable=not self.show_progress
        )

        for i in pbar:
            chunk_words = words[i: i + self.chunk_size]

            if len(chunk_words) > self.chunk_size // 4:
                chunk_text  = ' '.join(chunk_words)
                start_char  = word_offsets[i]
                end_word_idx = min(i + len(chunk_words) - 1, total_words - 1)
                end_char    = word_offsets[end_word_idx] + len(words[end_word_idx])

                chunks.append({
                    'doc_id':      doc_id,
                    'chunk_id':    f"{doc_id}_chunk_{chunk_id}",
                    'text':        chunk_text,
                    'token_count': len(chunk_words),
                    'strategy':    'sliding_window',
                    'start_char':  start_char,
                    'end_char':    end_char,
                })
                chunk_id += 1
                pbar.set_postfix({'chunks': chunk_id, 'words': f"{i}/{total_words}"})

        pbar.close()
        logger.info(f"Created {len(chunks)} chunks using sliding_window strategy")
        return chunks

    # ---------------------------------------------------------------------- #
    # SEMANTIC LITE chunking                                                   #
    # ---------------------------------------------------------------------- #

    def chunk_semantic_lite(
        self,
        text:        str,
        doc_id:      str,
        sample_rate: float = 0.2,
        silent:      bool  = False,
        text_offset: int   = 0,
    ) -> List[Dict]:
        """
        Semantic chunking with boundary detection.

        PERF: the original called nlp(sentence) one sentence at a time inside
        get_embedding() — a sequential loop of N full pipeline passes.  This
        version replaces that with a single nlp.pipe() call over all sampled
        sentences before the boundary-detection loop, which is 3–5× faster
        on large documents because spaCy can batch the work efficiently.
        """
        if not silent:
            logger.info(f"Using SEMANTIC LITE chunking (sample rate: {sample_rate})")

        sentences:       List[str] = []
        sentence_starts: List[int] = []
        chunk_size_nlp = min(1_000_000, len(text))

        text_ranges = list(range(0, len(text), chunk_size_nlp))
        pbar = tqdm(
            text_ranges, desc="Extracting sentences",
            disable=(not self.show_progress or silent),
        )

        for i in pbar:
            text_chunk = text[i: i + chunk_size_nlp]

            if i + chunk_size_nlp < len(text) and text_chunk:
                last_period = text_chunk.rfind('. ')
                if last_period > chunk_size_nlp * 0.8:
                    text_chunk = text_chunk[: last_period + 1]

            with self.nlp.select_pipes(enable=['sentencizer']):
                doc = self.nlp(text_chunk[: min(len(text_chunk), self.nlp.max_length)])
                for sent in doc.sents:
                    s = sent.text.strip()
                    if s:
                        sentences.append(s)
                        sentence_starts.append(i + sent.start_char)
                pbar.set_postfix({'sentences': len(sentences)})

        pbar.close()

        total_sentences = len(sentences)
        sample_indices  = set(range(0, total_sentences, max(1, int(1 / sample_rate))))

        # ── PERF: batch-encode all sampled sentences in one nlp.pipe() call ──
        # Previously: get_embedding(idx) called self.nlp(sentences[idx]) inside
        # the boundary-detection loop — one blocking pipeline pass per sentence.
        # Now: we process all sampled sentences upfront in a single pipe() call.
        embeddings_cache: Dict[int, np.ndarray] = {}

        sampled_idx_list  = sorted(sample_indices)
        sampled_sentences = [sentences[i][:1000] for i in sampled_idx_list]

        if sampled_sentences:
            # nlp.pipe() batches work internally and is substantially faster
            # than N sequential nlp() calls.  batch_size=64 is a safe default
            # for the en_core_web_sm model; raise if you have more RAM.
            for idx, doc in zip(
                sampled_idx_list,
                self.nlp.pipe(sampled_sentences, batch_size=64),
            ):
                embeddings_cache[idx] = doc.vector

        def get_embedding(idx: int) -> Optional[np.ndarray]:
            """Cache-only lookup — all sampled embeddings are pre-populated."""
            return embeddings_cache.get(idx)

        # ── Boundary detection (unchanged logic, faster due to pre-populated cache) ──
        boundaries = []
        pbar = tqdm(
            sample_indices, desc="Computing boundaries",
            disable=(not self.show_progress or silent),
        )

        for i in pbar:
            if 0 < i < total_sentences - 1:
                emb1 = get_embedding(i)
                emb2 = get_embedding(i + 1)

                if emb1 is not None and emb2 is not None:
                    if not (np.all(emb1 == 0) or np.all(emb2 == 0)):
                        similarity = cosine_similarity(
                            emb1.reshape(1, -1), emb2.reshape(1, -1)
                        )[0][0]
                        if similarity < 0.7:
                            boundaries.append(i + 1)
                            pbar.set_postfix({'boundaries': len(boundaries)})

        pbar.close()

        boundaries = [0] + sorted(boundaries) + [total_sentences]
        chunks     = []
        chunk_id   = 0

        pbar = tqdm(
            range(len(boundaries) - 1), desc="Creating chunks",
            disable=(not self.show_progress or silent),
        )

        for i in pbar:
            start_idx = boundaries[i]
            end_idx   = boundaries[i + 1]

            chunk_sentences = sentences[start_idx:end_idx]
            if chunk_sentences:
                chunk_text = ' '.join(chunk_sentences)
                abs_start  = text_offset + sentence_starts[start_idx]
                last_sent_idx = min(end_idx, len(sentence_starts)) - 1
                abs_end = (
                    text_offset
                    + sentence_starts[last_sent_idx]
                    + len(sentences[last_sent_idx])
                )
                chunks.append({
                    'doc_id':      doc_id,
                    'chunk_id':    f"{doc_id}_chunk_{chunk_id}",
                    'text':        chunk_text,
                    'token_count': len(chunk_text.split()),
                    'strategy':    'semantic_lite',
                    'start_char':  abs_start,
                    'end_char':    abs_end,
                })
                chunk_id += 1

        pbar.close()
        embeddings_cache.clear()
        gc.collect()

        return chunks

    # ---------------------------------------------------------------------- #
    # HYBRID chunking                                                          #
    # ---------------------------------------------------------------------- #

    def chunk_hybrid(self, text: str, doc_id: str) -> List[Dict]:
        logger.info("Using HYBRID chunking strategy")
        structural_chunks = self.chunk_structural(text, f"{doc_id}_temp")
        refined_chunks    = []
        chunk_id          = 0

        pbar = tqdm(
            structural_chunks, desc="Processing chunks", disable=not self.show_progress
        )

        for struct_chunk in pbar:
            chunk_text = struct_chunk['text']

            if 100 < len(chunk_text.split()) < 1000:
                pbar.set_postfix({'type': 'semantic_lite', 'chunks': chunk_id})

                # BUG-6 FIX: pass the structural chunk's start_char as
                # text_offset so sub-chunks get correct absolute positions.
                sub_chunks = self.chunk_semantic_lite(
                    chunk_text,
                    f"{doc_id}_sub",
                    sample_rate=0.5,
                    silent=True,
                    text_offset=struct_chunk.get('start_char', 0),
                )

                for sub_chunk in sub_chunks:
                    sub_chunk['chunk_id'] = f"{doc_id}_chunk_{chunk_id}"
                    sub_chunk['doc_id']   = doc_id
                    sub_chunk['strategy'] = 'hybrid'
                    refined_chunks.append(sub_chunk)
                    chunk_id += 1
            else:
                pbar.set_postfix({'type': 'structural', 'chunks': chunk_id})
                struct_chunk['chunk_id'] = f"{doc_id}_chunk_{chunk_id}"
                struct_chunk['doc_id']   = doc_id
                struct_chunk['strategy'] = 'hybrid'
                refined_chunks.append(struct_chunk)
                chunk_id += 1

        pbar.close()
        logger.info(f"Created {len(refined_chunks)} chunks using hybrid strategy")
        return refined_chunks

    # ---------------------------------------------------------------------- #
    # Entry point                                                              #
    # ---------------------------------------------------------------------- #

    def chunk_optimal(
        self,
        text:              str,
        doc_id:            str,
        override_strategy: Optional['ChunkingStrategy'] = None,
    ) -> List[Dict]:
        profile = self.profile_document(text)

        logger.info(
            f"Document profile: {profile.total_chars:,} chars, "
            f"{profile.paragraph_count} paragraphs, "
            f"structured: {profile.has_structure}"
        )

        strategy = override_strategy or profile.recommended_strategy
        logger.info(f"Selected strategy: {strategy.value}")

        if strategy == ChunkingStrategy.STRUCTURAL:
            chunks = self.chunk_structural(text, doc_id)
        elif strategy == ChunkingStrategy.SEMANTIC_LITE:
            chunks = self.chunk_semantic_lite(text, doc_id)
        elif strategy == ChunkingStrategy.SEMANTIC_FULL:
            sample_rate = 1.0 if profile.total_chars < 100_000 else 0.3
            chunks = self.chunk_semantic_lite(text, doc_id, sample_rate=sample_rate)
        elif strategy == ChunkingStrategy.SLIDING_WINDOW:
            chunks = self.chunk_sliding_window(text, doc_id)
        else:  # HYBRID
            chunks = self.chunk_hybrid(text, doc_id)

        for chunk in chunks:
            chunk['profile'] = {
                'total_chars':   profile.total_chars,
                'strategy_used': strategy.value,
                'has_structure': profile.has_structure,
            }

        logger.info(f"Created {len(chunks)} chunks using {strategy.value} strategy")
        return chunks

### ENTITY LABELING

In [ ]:
@dataclass
class EntityInfo:
    entity_value: str
    entity_type: str
    type_id: int
    full_sentence: str
    semantic_context: Dict[str, Any]
    start: int
    end: int
    confidence: float
    validation_score: float = 1.0
    is_valid: bool = True
    validation_issues: List[str] = field(default_factory=list)

    def __post_init__(self):
        """Ensure types are Python native (not numpy)"""
        # Convert numpy types to Python native types
        self.is_valid = bool(self.is_valid)
        self.validation_score = float(self.validation_score)
        self.type_id = int(self.type_id)
        self.confidence = float(self.confidence)
        self.start = int(self.start)
        self.end = int(self.end)

    def to_dict(self) -> Dict:
        return {
            'entity_value': self.entity_value,
            'text': self.entity_value,
            'type': self.entity_type,
            'type_id': self.type_id,
            'full_sentence': self.full_sentence,
            'semantic_context': self.semantic_context,
            'start': self.start,
            'end': self.end,
            'confidence': self.confidence,
            'validation_score': self.validation_score,
            'is_valid': self.is_valid,
            'validation_issues': self.validation_issues
        }

In [ ]:
@dataclass
class EntityRelationship:
    relationship_type: str
    source_entity: EntityInfo
    target_entity: EntityInfo
    confidence: float

    def to_dict(self) -> Dict:
        return {
            'type': self.relationship_type,
            'source': self.source_entity.to_dict(),
            'target': self.target_entity.to_dict(),
            'confidence': self.confidence
        }

In [ ]:
@dataclass
class SentenceLabel:
    sentence: str
    sentence_idx: int
    start_char: int
    end_char: int
    labels: List[str]
    confidence_scores: Dict[str, float]
    entities: List[EntityInfo]
    semantic_features: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict:
        return {
            'sentence': self.sentence,
            'sentence_idx': self.sentence_idx,
            'start_char': self.start_char,
            'end_char': self.end_char,
            'labels': self.labels,
            'confidence_scores': self.confidence_scores,
            'entities': [e.to_dict() for e in self.entities],
            'semantic_features': self.semantic_features
        }

In [ ]:
class EntityDeduplicator:
    """
    Remove duplicate and over-segmented entities with intelligent merging.

    FIXES APPLIED:
    1. Raised cross-type IoU threshold to 0.98 — only merge virtually identical spans
    2. Confidence-based merging keeps both entities if both are high confidence
    3. Minimum length check prevents suppressing short but valid entities
    4. Semantic similarity check for related types
    5. Symmetric IoU tolerance — fixes asymmetric expansion bug
    6. O(k) exact deduplication — replaces O(n²) inner loop
    7. merge_entities preserves original type — never creates invalid MERGED_* strings
    """

    # Known type pairs that commonly fire on the same span
    _RELATED_TYPE_GROUPS: List[frozenset] = [
        frozenset({'EQUIPMENT', 'COMPONENT'}),
        frozenset({'MATERIAL', 'COMPONENT'}),
        frozenset({'MATERIAL', 'FIRE_RATING'}),
        frozenset({'FACILITY', 'COMPONENT'}),
        frozenset({'FACILITY', 'LOCATION'}),
        frozenset({'OCCUPANCY', 'BUILDING_TYPE'}),
        frozenset({'CERTIFICATION', 'STANDARD'}),
        frozenset({'PENETRATION', 'COMPONENT'}),
        frozenset({'STANDARD', 'AUTHORITY'}),
        frozenset({'EQUIPMENT', 'FACILITY'}),
        frozenset({'DIMENSION', 'AREA'}),
        frozenset({'DIMENSION', 'VOLUME'}),
        frozenset({'PRESSURE', 'FLOW_RATE'}),
        frozenset({'TEMPERATURE', 'FIRE_RATING'}),
    ]

    # Only merge virtually identical spans across different types
    _CROSS_TYPE_IOU_THRESHOLD: float = 0.98

    # Minimum confidence to keep both overlapping entities
    _HIGH_CONFIDENCE_THRESHOLD: float = 0.85

    # Minimum entity length to protect from suppression
    _MIN_LENGTH_TO_PROTECT: int = 3

    def __init__(self, logger):
        self.logger = logger
        self.stats = {
            'exact_removed':        0,
            'containment_removed':  0,
            'overlap_removed':      0,
            'cross_type_merged':    0,
            'high_conf_preserved':  0,
        }

    # -------------------------------------------------------------------------
    # Public API
    # -------------------------------------------------------------------------

    def deduplicate(self, entities: List[Dict],
                    method: str = 'containment') -> List[Dict]:
        if not entities:
            return []

        self.logger.info(
            f"🧹 Deduplicating {len(entities)} entities (method: {method})"
        )

        if method == 'exact':
            result = self._deduplicate_exact(entities)
        elif method == 'containment':
            result = self._deduplicate_containment(entities)
        elif method == 'overlap':
            result = self._deduplicate_overlap(entities)
        else:
            raise ValueError(f"Unknown deduplication method: {method}")

        removed = len(entities) - len(result)
        self.logger.info(
            f"   Removed {removed} duplicates "
            f"({removed / len(entities) * 100:.1f}%)"
        )
        return result

    def deduplicate_with_stats(self,
                               entities: List[Dict]) -> Tuple[List[Dict], Dict]:
        original_count = len(entities)
        self.stats = {k: 0 for k in self.stats}  # Reset stats

        if original_count == 0:
            self.logger.warning("⚠️  No entities to deduplicate")
            return [], {
                'original_count':           0,
                'final_count':              0,
                'total_removed':            0,
                'exact_duplicates':         0,
                'containment_duplicates':   0,
                'overlap_duplicates':       0,
                'cross_type_merged':        0,
                'high_conf_preserved':      0,
                'removal_rate':             0.0,
            }

        # Stage 1: Exact duplicates
        entities, exact_removed = self._deduplicate_exact_with_count(entities)
        self.stats['exact_removed'] = exact_removed

        # Stage 2: Containment with intelligent merging
        entities, containment_removed = self._deduplicate_containment_with_count(entities)
        self.stats['containment_removed'] = containment_removed

        # Stage 3: Overlap with intelligent threshold
        entities, overlap_removed = self._deduplicate_overlap_with_count(
            entities, overlap_threshold=0.8
        )
        self.stats['overlap_removed'] = overlap_removed

        total_removed = original_count - len(entities)
        removal_rate  = (total_removed / original_count * 100) if original_count > 0 else 0.0

        self.logger.info(f"📊 Deduplication stats:")
        self.logger.info(f"   Exact duplicates removed:       {exact_removed}")
        self.logger.info(f"   Containment duplicates removed: {containment_removed}")
        self.logger.info(f"   Overlap duplicates removed:     {overlap_removed}")
        self.logger.info(f"   Cross-type merges:              {self.stats['cross_type_merged']}")
        self.logger.info(f"   High-confidence preserved:      {self.stats['high_conf_preserved']}")

        return entities, {
            'original_count':           original_count,
            'final_count':              len(entities),
            'total_removed':            total_removed,
            'exact_duplicates':         exact_removed,
            'containment_duplicates':   containment_removed,
            'overlap_duplicates':       overlap_removed,
            'cross_type_merged':        self.stats['cross_type_merged'],
            'high_conf_preserved':      self.stats['high_conf_preserved'],
            'removal_rate':             removal_rate,
        }

    # -------------------------------------------------------------------------
    # Private helpers with counting
    # -------------------------------------------------------------------------

    def _deduplicate_exact_with_count(
        self, entities: List[Dict]
    ) -> Tuple[List[Dict], int]:
        """
        Remove exact and near-position duplicates.

        Uses a two-level lookup:
          - exact_seen: O(1) hash check for perfectly identical keys
          - near_seen:  O(k) scan per (text, type) pair for position-drift duplicates
                        where k = number of prior instances of the same text+type
        """
        exact_seen: set = set()
        # (text_lower, etype) → list of (start, end) for near-position check
        near_seen: Dict[Tuple[str, str], List[Tuple[int, int]]] = {}
        result: List[Dict] = []
        removed = 0

        for entity in entities:
            start      = entity.get('absolute_start', entity.get('start', 0))
            end        = entity.get('absolute_end',   entity.get('end',   0))
            text_lower = entity['entity_value'].lower().strip()
            etype      = entity['type']

            key = (text_lower, etype, start, end)

            # O(1) exact match
            if key in exact_seen:
                removed += 1
                continue

            # O(k) near-position match for the same text+type
            is_near_duplicate = False
            for s, e in near_seen.get((text_lower, etype), []):
                if abs(s - start) < 5 and abs(e - end) < 5:
                    is_near_duplicate = True
                    break

            if is_near_duplicate:
                removed += 1
                continue

            exact_seen.add(key)
            near_seen.setdefault((text_lower, etype), []).append((start, end))
            result.append(entity)

        return result, removed

    def _deduplicate_containment_with_count(
        self, entities: List[Dict]
    ) -> Tuple[List[Dict], int]:
        """
        Remove shorter entities fully contained within longer ones,
        with intelligent rules to preserve important short entities.
        """
        # Sort by start position, then by descending length (longest first)
        sorted_entities = sorted(
            entities,
            key=lambda e: (
                e.get('absolute_start', e.get('start', 0)),
                -(e.get('absolute_end', e.get('end', 0))
                  - e.get('absolute_start', e.get('start', 0))),
            ),
        )

        result:  List[Dict] = []
        removed: int        = 0

        for entity in sorted_entities:
            e_start = entity.get('absolute_start', entity.get('start', 0))
            e_end   = entity.get('absolute_end',   entity.get('end',   0))
            e_type  = entity['type']
            e_text  = entity['entity_value']
            e_conf  = entity.get('confidence', 0.5)
            e_len   = len(e_text.strip())

            # Protect very short but potentially important entities
            if e_len < self._MIN_LENGTH_TO_PROTECT:
                result.append(entity)
                continue

            is_contained    = False
            suppress_reason = None

            for existing in result:
                ex_start = existing.get('absolute_start', existing.get('start', 0))
                ex_end   = existing.get('absolute_end',   existing.get('end',   0))
                ex_type  = existing['type']
                ex_conf  = existing.get('confidence', 0.5)
                ex_text  = existing['entity_value']
                ex_len   = len(ex_text.strip())

                # Check if entity is fully inside existing
                fully_inside = (e_start >= ex_start and e_end <= ex_end)
                if not fully_inside:
                    continue

                # Both high-confidence and different types — keep both
                if (e_conf  >= self._HIGH_CONFIDENCE_THRESHOLD and
                        ex_conf >= self._HIGH_CONFIDENCE_THRESHOLD and
                        e_type  != ex_type):
                    self.stats['high_conf_preserved'] += 1
                    self.logger.debug(
                        f"Preserving both high-confidence entities: "
                        f"'{e_text}' [{e_type}] inside '{ex_text}' [{ex_type}]"
                    )
                    continue

                # Same type — always suppress the shorter contained one
                if e_type == ex_type:
                    is_contained    = True
                    suppress_reason = (
                        f"same type containment (len {e_len} < {ex_len})"
                    )
                    break

                # Known related type pair — suppress only if no new words added
                if self._are_related_types(e_type, ex_type):
                    e_words  = set(e_text.lower().split())
                    ex_words = set(ex_text.lower().split())

                    if e_words - ex_words:
                        # Shorter entity adds new words — keep it
                        self.logger.debug(
                            f"Keeping related type '{e_text}' [{e_type}] "
                            f"— adds new words"
                        )
                        continue

                    is_contained    = True
                    suppress_reason = "related type containment (no new info)"
                    self.stats['cross_type_merged'] += 1
                    break

            if is_contained:
                removed += 1
                self.logger.debug(
                    f"Containment removed: '{entity['entity_value']}' "
                    f"[{e_type}] — {suppress_reason}"
                )
            else:
                result.append(entity)

        return result, removed

    def _deduplicate_overlap_with_count(
        self,
        entities:          List[Dict],
        overlap_threshold: float = 0.8,
    ) -> Tuple[List[Dict], int]:
        """
        Remove entities with high span overlap, with intelligent rules.
        Sorted by confidence descending so higher-confidence entities
        are accepted first.
        """
        sorted_entities = sorted(
            entities,
            key=lambda e: (
                -e.get('confidence', 0.5),
                -(e.get('absolute_end', e.get('end', 0))
                  - e.get('absolute_start', e.get('start', 0))),
            ),
        )

        result:  List[Dict] = []
        removed: int        = 0

        for entity in sorted_entities:
            e_type = entity['type']
            e_text = entity['entity_value']
            e_conf = entity.get('confidence', 0.5)
            e_len  = len(e_text.strip())

            # Protect short entities
            if e_len < self._MIN_LENGTH_TO_PROTECT:
                result.append(entity)
                continue

            suppress        = False
            suppress_reason = None

            for existing in result:
                ex_type = existing['type']
                ex_text = existing['entity_value']
                ex_conf = existing.get('confidence', 0.5)

                iou = self._calculate_iou(entity, existing)

                # Both high-confidence — be very conservative
                if (e_conf  >= self._HIGH_CONFIDENCE_THRESHOLD and
                        ex_conf >= self._HIGH_CONFIDENCE_THRESHOLD):
                    # Only suppress if same type AND virtually identical span
                    if e_type == ex_type and iou >= self._CROSS_TYPE_IOU_THRESHOLD:
                        suppress        = True
                        suppress_reason = (
                            f"high-conf same type near-identical "
                            f"(IoU={iou:.2f})"
                        )
                        break
                    # Different types or below threshold — keep both
                    continue

                # Same type at normal threshold
                if e_type == ex_type and iou >= overlap_threshold:
                    suppress        = True
                    suppress_reason = (
                        f"same type overlap (IoU={iou:.2f} >= {overlap_threshold})"
                    )
                    break

                # Near-identical span across any types — very strict check
                if iou >= self._CROSS_TYPE_IOU_THRESHOLD:
                    e_words  = set(e_text.lower().split())
                    ex_words = set(ex_text.lower().split())
                    word_overlap = (
                        len(e_words & ex_words) /
                        max(len(e_words), len(ex_words), 1)
                    )
                    if word_overlap > 0.8:
                        suppress        = True
                        suppress_reason = (
                            f"near-identical cross-type (IoU={iou:.2f})"
                        )
                        self.stats['cross_type_merged'] += 1
                        break

            if suppress:
                removed += 1
                self.logger.debug(
                    f"Overlap removed: '{entity['entity_value']}' "
                    f"[{e_type}] — {suppress_reason}"
                )
            else:
                result.append(entity)

        return result, removed

    # -------------------------------------------------------------------------
    # Original methods (backward compatibility wrappers)
    # -------------------------------------------------------------------------

    def _deduplicate_exact(self, entities: List[Dict]) -> List[Dict]:
        result, _ = self._deduplicate_exact_with_count(entities)
        return result

    def _deduplicate_containment(self, entities: List[Dict]) -> List[Dict]:
        result, _ = self._deduplicate_containment_with_count(entities)
        return result

    def _deduplicate_overlap(self, entities:          List[Dict],
                             overlap_threshold: float = 0.8) -> List[Dict]:
        result, _ = self._deduplicate_overlap_with_count(entities, overlap_threshold)
        return result

    # -------------------------------------------------------------------------
    # Helpers
    # -------------------------------------------------------------------------

    def _are_related_types(self, type_a: str, type_b: str) -> bool:
        """Return True if two entity types are semantically related."""
        pair = frozenset({type_a, type_b})

        for group in self._RELATED_TYPE_GROUPS:
            if pair <= group:
                return True

        measurement_types = {
            'DIMENSION', 'AREA', 'VOLUME', 'PRESSURE',
            'FLOW_RATE', 'TEMPERATURE', 'CAPACITY',
        }
        if type_a in measurement_types and type_b in measurement_types:
            return True

        building_types = {'BUILDING_TYPE', 'OCCUPANCY', 'FACILITY', 'LOCATION'}
        if type_a in building_types and type_b in building_types:
            return True

        return False

    def _calculate_iou(self, entity1: Dict, entity2: Dict) -> float:
        """
        Calculate Intersection over Union with symmetric tolerance.

        The tolerance is applied only to the intersection window (numerator),
        while the union denominator uses raw (unadjusted) spans. This avoids
        the asymmetric double-expansion bug where both spans were shifted in
        the same direction, producing inflated IoU values.
        """
        start1 = entity1.get('absolute_start', entity1.get('start', 0))
        end1   = entity1.get('absolute_end',   entity1.get('end',   0))
        start2 = entity2.get('absolute_start', entity2.get('start', 0))
        end2   = entity2.get('absolute_end',   entity2.get('end',   0))

        tolerance = 2

        # Raw intersection — used for the union denominator
        raw_inter_start  = max(start1, start2)
        raw_inter_end    = min(end1,   end2)
        raw_intersection = max(0, raw_inter_end - raw_inter_start)

        # Tolerance-expanded intersection — used as numerator
        tol_inter_start = raw_inter_start - tolerance
        tol_inter_end   = raw_inter_end   + tolerance
        if tol_inter_start >= tol_inter_end:
            return 0.0

        intersection = tol_inter_end - tol_inter_start
        union        = (end1 - start1) + (end2 - start2) - raw_intersection
        return intersection / max(union, 1)

    def _get_span_key(self, entity: Dict) -> tuple:
        """Generate a unique key for an entity span."""
        start = entity.get('absolute_start', entity.get('start', 0))
        end   = entity.get('absolute_end',   entity.get('end',   0))
        return (start, end, entity['type'])

    def merge_entities(self, entities: List[Dict]) -> List[Dict]:
        """
        Merge overlapping entities of related types into single records.

        The highest-confidence entity's original 'type' is always preserved
        as the canonical type — this method NEVER creates 'MERGED_*' type
        strings, which would be unmappable by downstream type-ID lookups.
        Merge metadata is stored in 'merged_types' and 'merged_texts' for
        diagnostics only.
        """
        if len(entities) < 2:
            return entities

        groups: List[List[Dict]] = []
        used:   set              = set()

        for i, e1 in enumerate(entities):
            if i in used:
                continue
            group: List[Dict] = [e1]
            used.add(i)

            for j, e2 in enumerate(entities[i + 1:], i + 1):
                if j in used:
                    continue
                if self._calculate_iou(e1, e2) > 0.3:
                    group.append(e2)
                    used.add(j)

            groups.append(group)

        result: List[Dict] = []

        for group in groups:
            if len(group) == 1:
                result.append(group[0])
                continue

            # Pick the best representative by confidence then text length
            best = max(group, key=lambda e: (
                e.get('confidence', 0.5),
                len(e['entity_value']),
            ))

            # Work on a copy — never mutate the original entity dict
            best = dict(best)

            # Store merge metadata; 'type' is NEVER overwritten
            types = {e['type'] for e in group}
            if len(types) > 1:
                best['merged_types'] = sorted(types)
                # best['type'] intentionally left as the canonical winner's type

            texts = {e['entity_value'] for e in group}
            if len(texts) > 1:
                best['merged_texts'] = list(texts)

            # Average confidence across the group
            best['confidence'] = sum(
                e.get('confidence', 0.5) for e in group
            ) / len(group)

            # Expand span to cover all merged entities
            best['start'] = min(e.get('start', 0) for e in group)
            best['end']   = max(e.get('end',   0) for e in group)
            if 'absolute_start' in best:
                best['absolute_start'] = min(
                    e.get('absolute_start', e.get('start', 0)) for e in group
                )
                best['absolute_end'] = max(
                    e.get('absolute_end', e.get('end', 0)) for e in group
                )

            result.append(best)
            self.logger.debug(
                f"Merged {len(group)} entities → "
                f"'{best['entity_value']}' [{best['type']}] "
                f"(merged_types={best.get('merged_types')})"
            )

        return result

In [ ]:
class EntityLabeler:
    r"""
    Regex + spaCy-based entity labeler for regulatory NER.

    Key fixes vs previous version
    ──────────────────────────────
    FIX-OCC-1  OCCUPANCY patterns defined and processed BEFORE BUILDING_TYPE
               in _define_patterns() so the first-match-wins deduplication
               favours the semantically correct label for Group codes and
               occupancy-suffixed phrases.

    FIX-OCC-2  BUILDING_TYPE patterns no longer match "… occupancy" or
               "Group [A-U]-\d …" — those belong exclusively to OCCUPANCY.
               BUILDING_TYPE is now restricted to construction-type language
               (Type I-A, Type II-B, …) and program-type language where the
               trailing word is "building" or "structure", never "occupancy".

    FIX-OCC-3  _resolve_type_conflicts() post-processing step: if the same
               character span is claimed by both BUILDING_TYPE and OCCUPANCY,
               OCCUPANCY wins unconditionally (Group codes, "… occupancy",
               "… use group" always denote occupancy classification).

    FIX-OCC-4  Dict insertion order: OCCUPANCY key is inserted before
               BUILDING_TYPE so the compiled-pattern loop fires OCCUPANCY
               patterns first.

    All other fixes from the previous version (FIX 1–5) are preserved.
    """

    _FIRE_EQUIPMENT_KEYWORDS: frozenset = frozenset({
        'sprinkler', 'detector', 'alarm', 'suppression', 'extinguisher',
        'hydrant', 'standpipe', 'damper', 'hose', 'nozzle', 'valve',
        'sensor', 'panel', 'notification', 'initiating', 'annunciator',
        'sounder', 'strobe', 'horn', 'beacon', 'pull station',
        'heat detector', 'smoke detector', 'flame detector',
        'fire pump', 'jockey pump', 'deluge', 'pre-action', 'dry pipe',
        'wet pipe', 'clean agent', 'halon', 'co2', 'foam system',
        'emergency lighting', 'exit sign', 'emergency generator',
        'key list', 'control panel', 'control unit', 'detection system',
        'suppression system', 'fire hydrant', 'hose connection',
        'control valve', 'backup power', 'emergency power',
        'lighting system', 'notification appliance', 'initiating device',
        'fire alarm', 'smoke control', 'fire damper', 'smoke damper',
        'tamper switch', 'flow switch', 'pressure switch', 'supervisory switch',
        'bell', 'chime', 'speaker', 'visual alarm', 'audible alarm',
        'manual pull station', 'manual call point', 'break glass',
    })

    # FIX 1: Raised base confidence for all types
    _TYPE_BASE_CONFIDENCE: Dict[str, float] = {
        'DIMENSION':            0.90,
        'AREA':                 0.90,
        'VOLUME':               0.90,
        'PRESSURE':             0.90,
        'FLOW_RATE':            0.90,
        'TEMPERATURE':          0.90,
        'FIRE_RATING':          0.90,
        'EGRESS_DISTANCE':      0.90,
        'SEPARATION_DISTANCE':  0.90,
        'CLEARANCE':            0.90,
        'CAPACITY':             0.90,
        'PERCENTAGE':           0.90,
        'STANDARD':             0.90,
        'AUTHORITY':            0.90,
        'CERTIFICATION':        0.90,
        'BUILDING_TYPE':        0.90,
        'OCCUPANCY':            0.90,
        'HAZARD_LEVEL':         0.90,
        'PENETRATION':          0.90,
        'INSPECTION_FREQUENCY': 0.90,
        'TESTING_REQUIREMENT':  0.90,
        'EQUIPMENT':            0.88,
        'MATERIAL':             0.88,
        'COMPONENT':            0.88,
        'FACILITY':             0.88,
        'LOCATION':             0.88,
    }

    _OCCUPANCY_WINS_MARKERS: frozenset = frozenset({
        'occupancy', 'use group', 'group a', 'group b', 'group e',
        'group f', 'group h', 'group i', 'group m', 'group r',
        'group s', 'group u', '-1', '-2', '-3', '-4',
    })
    _CONDITION_TRIGGERS: frozenset = frozenset({
        'if', 'when', 'where', 'unless', 'except', 'provided',
    })
    def __init__(self, nlp):
        self.nlp = nlp
        self.patterns = self._define_patterns()
        self.compiled_patterns = self._compile_patterns()
        self.logger = logging.getLogger(__name__)

    # =========================================================================
    # ARABIC FILTER
    # =========================================================================

    def _is_arabic_dominant(self, text: str, threshold: float = 0.50) -> bool:
        if not text:
            return False
        arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
        return arabic_chars / len(text) > threshold

    # =========================================================================
    # PATTERN DEFINITIONS
    # IMPORTANT: Dict key order determines match priority via seen_entities.
    # OCCUPANCY must appear BEFORE BUILDING_TYPE (FIX-OCC-4).
    # =========================================================================

    def _define_patterns(self) -> Dict[str, List[str]]:
        return {

            # =================================================================
            # PRIMARY FIRE-SAFETY MEASUREMENTS
            # =================================================================

            'FIRE_RATING': [
                r'\b\d+(?:\.\d+)?[-\s]?(?:hour|hr)s?[-\s]?(?:fire[-\s]?)?(?:rating|rated|resistance|resistant)\b',
                r'\b(?:fire[-\s]?)?(?:rating|resistance)\s+(?:of\s+)?\d+(?:\.\d+)?[-\s]?(?:hour|hr)s?\b',
                r'\bClass\s+[ABC]\s+(?:fire[-\s]?)?(?:rating|rated|door|window|assembly)\b',
                r'\b(?:UL|FM\s+\d+|ASTM)\s+(?:\d+[-\s]?hour\s+)?fire[-\s]?rated\b',
                r'\b(?:flame|smoke)\s+spread\s+(?:rating|index|classification)\s+(?:of\s+)?\d+\b',
                r'\b(?:flame|smoke)[-\s]?developed\s+(?:rating|index)\b',
                r'\b(?:non[-\s]?combustible|noncombustible|limited[-\s]?combustible|combustible)\s+(?:construction|material|assembly)\b',
                r'\bfire[-\s]?resistance[-\s]?(?:level|classification|rating)\s+(?:of\s+)?\d+\b',
                r'\bType\s+[IVX]+[-\s]?[AB]\s+(?:construction|building)\b',
                r'\b(?:protected|unprotected)\s+(?:Type\s+[IVX]+)\s+construction\b',
                r'\bR-?\d+\b',
                r'\b\d+\s*hrs?\b',
            ],

            'EGRESS_DISTANCE': [
                r'\b(?:travel|egress|exit\s+access)\s+distance\s+(?:of\s+|not\s+(?:to\s+)?exceed\s+)?\d+\s*(?:m|ft|meters?|feet)\b',
                r'\b\d+\s*(?:m|ft|meters?|feet)\s+(?:maximum\s+)?(?:travel|egress)\s+distance\b',
                r'\b(?:distance|travel)\s+(?:to|from)\s+(?:nearest\s+)?(?:exit|egress\s+door|stairway)\b',
                r'\b\d+\s*(?:m|ft|meters?|feet)\s+(?:to|from)\s+(?:nearest\s+)?(?:exit|egress)\b',
                r'\bcommon\s+path\s+of\s+(?:egress\s+)?travel\b',
                r'\bdead[-\s]?end\s+(?:corridor|aisle|passageway)\s+(?:length|distance|of\s+\d+)?\b',
                r'\bexit\s+access\s+(?:travel\s+)?distance\b',
                r'\bmeans\s+of\s+egress\s+(?:travel\s+)?distance\b',
            ],

            'SEPARATION_DISTANCE': [
                r'\b(?:fire\s+)?separation\s+distance\s+(?:of\s+)?\d+\s*(?:m|ft|meters?|feet)\b',
                r'\b\d+\s*(?:m|ft|meters?|feet)\s+(?:fire\s+)?separation\b',
                r'\b(?:distance|separation)\s+(?:between|from)\s+(?:buildings?|structures?|property\s+line|lot\s+line)\b',
                r'\bspatial\s+separation\s+(?:of\s+\d+\s*(?:m|ft))?\b',
                r'\b\d+\s*(?:m|ft)\s+(?:from|to)\s+(?:property\s+line|lot\s+line|boundary)\b',
            ],

            'CLEARANCE': [
                r'\b(?:minimum|required|maintain(?:ed)?)\s+clearance\s+(?:of\s+)?\d+\s*(?:m|ft|inch(?:es)?|cm|mm)\b',
                r'\b\d+\s*(?:m|ft|inch(?:es)?|cm|mm)\s+(?:minimum\s+)?clearance\b',
                r'\bclear\s+(?:floor\s+space|working\s+space|access\s+space)\s+(?:of\s+)?\d+\b',
                r'\b(?:working|maintenance|access|electrical|equipment)\s+clearance\b',
                r'\b(?:overhead|side|rear|front|vertical|horizontal)\s+clearance\s+(?:of\s+\d+\s*(?:m|ft|inch(?:es)?))?\b',
                r'\b(?:headroom|head\s+room|ceiling\s+height)\s+(?:of\s+)?\d+\s*(?:m|ft|inch(?:es)?|mm)\b',
            ],

            # =================================================================
            # GENERAL MEASUREMENTS
            # =================================================================

            'DIMENSION': [
                r'(?<![Ss]ection\s)(?<![Aa]rticle\s)(?<!\bNFPA\s)(?<!\bSBC\s)(?<!\bUL\s)(?<!\bFM\s)'
                r'\b\d{1,4}(?:\.\d+)?\s*(?:meters?|metres?)\b',

                r'(?<![Ss]ection\s)(?<![Aa]rticle\s)(?<!\bNFPA\s)(?<!\bSBC\s)(?<!\bUL\s)(?<!\bFM\s)'
                r'\b\d{1,4}(?:\.\d+)?\s*(?:feet|foot)\b',

                r'(?<![Ss]ection\s)(?<![Aa]rticle\s)(?<!\bNFPA\s)(?<!\bSBC\s)(?<!\bUL\s)(?<!\bFM\s)'
                r'\b\d{1,3}(?:\.\d+)?\s*(?:inches?|in\.(?=\s))\b',

                r'(?<![Ss]ection\s)(?<![Aa]rticle\s)\b\d{1,4}(?:\.\d+)?\s*(?:mm|cm)\b',

                r'\b\d{1,2}\s*(?:ft|feet)\s+\d{1,2}\s*(?:in|inches?)\b',

                r'\b\d{1,4}(?:\.\d+)?\s*[xX]\s*\d{1,4}(?:\.\d+)?\s*(?:m|ft|cm|mm)\b',

                r'\b(?:width|height|length|depth|thickness|diameter|radius)\s+(?:of\s+)?\d+(?:\.\d+)?\s*(?:m|ft|cm|mm|inches?)\b',
                r'\b(?:minimum|maximum|nominal|required|net|gross)\s+(?:width|height|length|depth|thickness)\s+(?:of\s+)?\d+\b',
                r'\bclear\s+(?:width|height|opening|headroom|span)\s+(?:of\s+)?\d+\b',
                r'\b(?:door|corridor|stair|aisle|exit|window|opening)\s+(?:width|height|clear\s+width)\s+(?:of\s+)?\d+\b',
            ],

            'AREA': [
                r'\b\d+(?:\.\d+)?\s*(?:m²|ft²|m2|ft2|square\s+meters?|square\s+feet|sq\.?\s*(?:m|ft))\b',
                r'\b(?:area|surface|footprint)\s+(?:of\s+)?\d+(?:\.\d+)?\s*(?:m²|ft²|square\s+(?:meters?|feet))\b',
                r'\b(?:gross|net|total|allowable|maximum|minimum)\s+(?:floor|building|compartment|fire)\s+area\b',
                r'\b(?:floor|compartment|fire|egress|refuge)\s+area\b',
                r'\b(?:area|space)\s+per\s+(?:occupant|person)\b',
                r'\b\d+(?:\.\d+)?\s*(?:to|-)\s*\d+(?:\.\d+)?\s*(?:m²|ft²|square\s+(?:meters?|feet))\b',
            ],

            'VOLUME': [
                r'\b\d+(?:\.\d+)?\s*(?:m³|ft³|m3|ft3|cubic\s+meters?|cubic\s+feet|cu\.?\s*(?:m|ft))\b',
                r'\b\d+(?:\.\d+)?\s*(?:liters?|litres?|gallons?|L|gal)\s+(?:capacity|storage|tank|reservoir)\b',
                r'\b(?:tank|container|reservoir)\s+(?:capacity|volume)\s+(?:of\s+)?\d+\b',
                r'\b(?:water\s+)?(?:supply|storage)\s+(?:capacity|volume)\b',
            ],

            'CAPACITY': [
                r'\boccupant\s+load\s+(?:factor\s+)?(?:of\s+)?\d+\b',
                r'\boccupant\s+load\s+(?:factor|calculation|determination)\b',
                r'\b(?:maximum|minimum|design)\s+(?:of\s+)?\d+\s*(?:persons?|people|occupants?)\b',
                r'\b\d+\s*(?:persons?|people|occupants?)\s+(?:maximum|per\s+(?:floor|story|room))\b',
                r'\b\d+\s*(?:persons?|people|occupants?)\s+per\s+(?:\d+\s*)?(?:m²|ft²|square)\b',
                r'\b(?:egress|exit|stair|corridor)\s+capacity\b',
                r'\b(?:accommodate|serve|evacuate)\s+(?:up\s+to\s+)?\d+\s*(?:persons?|people|occupants?)\b',
            ],

            'PERCENTAGE': [
                r'\b\d+(?:\.\d+)?%\s*(?:of\s+)?(?:floor|wall|roof|opening|glazing|area|capacity|load)\b',
                r'\b(?:floor|wall|roof|opening|glazing)\s+(?:area\s+)?(?:of\s+)?\d+(?:\.\d+)?%\b',
                r'\b(?:maximum|minimum|not\s+(?:more|less)\s+than)\s+\d+(?:\.\d+)?(?:%|\s*percent)\b',
                r'\b(?:slope|grade)\s+(?:of\s+)?\d+(?:\.\d+)?(?:%|\s*percent)\b',
                r'\b\d+(?:\.\d+)?(?:%|\s*percent)\s+(?:sprinklered|protected|covered)\b',
                r'\bno\s+part\s+of\b',
                r'\b(?:any|some|all|no)\s+part\b',
            ],

            'PRESSURE': [
                r'\b\d+(?:\.\d+)?\s*(?:psi|psig|bar|kPa|Pa)\b',
                r'\b(?:water|system|operating|working|residual|static|dynamic|flow)\s+pressure\s+(?:of\s+)?\d+\b',
                r'\b(?:minimum|maximum|required|design)\s+pressure\s+(?:of\s+)?\d+\s*(?:psi|bar|kPa)\b',
                r'\bpressure\s+(?:of|at|not\s+less\s+than)\s+\d+\s*(?:psi|bar|kPa)\b',
            ],

            'FLOW_RATE': [
                r'\b\d+(?:\.\d+)?\s*(?:gpm|lpm|L/min|gal/min|gallons?\s+per\s+minute|liters?\s+per\s+minute)\b',
                r'\b(?:flow|flow\s+rate)\s+(?:of\s+)?\d+\s*(?:gpm|lpm)\b',
                r'\b(?:minimum|maximum|required|design)\s+flow\s+(?:rate\s+)?(?:of\s+)?\d+\b',
                r'\b(?:sprinkler|hose|standpipe|hydrant)\s+flow\s+(?:rate\s+)?(?:of\s+\d+)?\b',
                r'\b\d+(?:\.\d+)?\s*(?:gpm|lpm)\s+(?:per|/)\s*(?:ft²|m²)\b',
            ],

            'TEMPERATURE': [
                r'\b\d+(?:\.\d+)?\s*(?:°C|°F|degrees?\s+(?:Celsius|Fahrenheit|[CF]))\b',
                r'\b(?:temperature|thermal)\s+(?:rating|limit)\s+(?:of\s+)?\d+\b',
                r'\b(?:operating|ambient|maximum|minimum|activation)\s+temperature\s+(?:of\s+)?\d+\b',
                r'\b(?:sprinkler|detector)\s+(?:temperature\s+)?(?:rating|activation)\s+(?:of\s+)?\d+\b',
            ],

            # =================================================================
            # BUILDING ELEMENTS & MATERIALS
            # =================================================================

            'MATERIAL': [
                r'\b(?:fire[-\s]?resist(?:ant|ive)|flame[-\s]?retardant|noncombustible|non[-\s]?combustible|combustible|limited[-\s]?combustible)\s+(?:material|construction|assembly|coating|barrier)\b',
                r'\b(?:UL|FM|ASTM)[-\s]?(?:listed|rated|approved)\s+(?:material|product|assembly)\b',
                r'\bgypsum\s+(?:board|wallboard|sheathing|plasterboard|panel)\b',
                r'\bfire[-\s]?rated\s+(?:drywall|sheathing|glass|glazing|panel|board)\b',
                r'\bintumescent\s+(?:material|sealant|coating|strip|seal|paint)\b',
                r'\bmineral\s+(?:fiber|wool|fiber\s+insulation|board)\b',
                r'\b(?:concrete|steel|wood|timber|masonry)\s+(?:assembly|framing\s+member)\b',
                r'\b(?:reinforced|prestressed|precast|post[-\s]?tensioned)\s+concrete\b',
                r'\b(?:structural|cold[-\s]?formed|hot[-\s]?rolled|galvanized)\s+steel\b',
                r'\b(?:heavy|light|mass|cross[-\s]?laminated)\s+timber\b',
                r'\b(?:fire[-\s]?retardant[-\s]?treated|pressure[-\s]?treated|preservative[-\s]?treated)\s+(?:wood|timber|lumber|plywood)\b',
                r'\bmasonry\s+(?:unit|block|wall|construction|veneer|cavity\s+wall)\b',
                r'\b(?:concrete\s+masonry\s+unit|CMU)\b',
                r'\b(?:autoclaved\s+aerated\s+concrete|AAC)\b',
                r'\b(?:calcium\s+silicate|ceramic\s+fiber|rockwool|rock\s+wool|slag\s+wool)\b',
                r'\b(?:spray[-\s]?applied|spray)\s+(?:fire[-\s]?resistant|fireproof(?:ing)?|fire[-\s]?resistive)\s+(?:material|coating|protection|insulation)\b',
                r'\b(?:tempered|wired|fire[-\s]?rated|ceramic|borosilicate)\s+glass\b',
                r'\bfire[-\s]?resistant\s+(?:sealant|caulk|mortar|grout|adhesive)\b',
                r'\b(?:Type\s+[IVX]+[-\s]?[AB]?)\s+(?:concrete|steel|masonry|timber|wood)\b',
                r'\b(?:combustibility|flammability)\s+(?:class|classification|rating)\b',
                r'\b(?:cellulosic|non[-\s]?cellulosic)\s+(?:material|insulation|cladding)\b',
                r'\b(?:Construction|Foundations|Concrete|Masonry)\s+SBC\b',
                r'\bSBC\s+(?:304|305)[-\s](?:CR|CC|AR)\b',
                r'\bMasonry\s+Structures?\b',
                r'\bStructural\s+(?:SBC|Code)\b',
            ],

            'COMPONENT': [
                r'\b(?:fire[-\s]?rated|fire[-\s]?resist(?:ant|ive)|smoke[-\s]?rated)\s+(?:wall|door|window|floor|ceiling|roof|assembly|barrier|partition)\b',
                r'\bfire\s+(?:door|window|shutter|damper|wall|barrier|separation)\b',
                r'\b(?:exit|egress|escape)\s+(?:door|stair(?:way)?|ramp|passage|route)\b',
                r'\b(?:structural|load[-\s]?bearing|non[-\s]?load[-\s]?bearing)\s+(?:wall|column|beam|member)\b',
                r'\b(?:smoke|fire)\s+(?:barrier|partition|damper|wall|compartment)\b',
                r'\b(?:wall|floor|roof|ceiling|door)\s+assembly\b',
                r'\b(?:horizontal|vertical)\s+(?:fire[-\s]?resistant\s+)?(?:assembly|separation|barrier)\b',
                r'\bopening\s+protectives?\b',
                r'\b(?:through|membrane)[-\s]?penetration\s+(?:firestop|protection)\b',
                r'\b(?:stair|elevator|shaft|duct|exit)\s+enclosure\b',
            ],

            'EQUIPMENT': [
                r'\b(?:automatic\s+)?(?:fire\s+)?sprinkler\s+(?:system|head|protection)\b',
                r'\b(?:fire\s+)?suppression\s+system\b',
                r'\bstandpipe\s+(?:system|and\s+hose\s+system)?\b',
                r'\b(?:smoke|heat|flame|fire|carbon\s+monoxide)\s+(?:detector|sensor|alarm)\b',
                r'\b(?:fire\s+)?(?:detection|alarm)\s+(?:system|panel|control\s+panel)\b',
                r'\binitiating\s+device\b',
                r'\bnotification\s+appliance\b',
                r'\b(?:fire\s+)?extinguisher\b',
                r'\bfire\s+hydrant\b',
                r'\bhose\s+(?:cabinet|connection|station|valve)\b',
                r'\bfire\s+department\s+connection\b',
                r'\bemergency\s+(?:lighting|illumination)\s+(?:system|unit)?\b',
                r'\b(?:exit|emergency)\s+(?:light|sign|luminaire)\b',
                r'\bemergency\s+(?:power\s+system|generator|backup\s+power)\b',
                r'\b(?:fire\s+)?alarm\s+control\s+(?:panel|unit)\b',
                r'\b(?:smoke|fire)\s+control\s+system\b',
                r'\b(?:fire|smoke|combination\s+fire[/-]smoke)\s+damper\b',
                r'\b(?:control|supervisory|tamper)\s+(?:valve|switch)\b',
                r'\b(?:UL|FM)[-\s]listed\s+(?:device|equipment|system)\b',
                r'\bKey\s+List\b',
                r'\bControl\s+(?:Panel|Unit|System)\b',
                r'\bDetection\s+System\b',
                r'\bSuppression\s+System\b',
                r'\bFire\s+(?:Pump|Hydrant|Alarm)\b',
                r'\bHose\s+(?:Connection|Valve|Station)\b',
            ],

            # =================================================================
            # FIRE SAFETY SPECIFIC
            # =================================================================

            'PENETRATION': [
                r'\bfirestop(?:ping)?\s+(?:system|material|sealant|device|assembly)\b',
                r'\b(?:fire|smoke)\s+(?:stop|seal|caulk|blocking)\b',
                r'\bpenetration\s+(?:firestop|protection|seal)\b',
                r'\b(?:through|membrane)[-\s]?penetration\s+(?:firestop|protection|seal)?\b',
                r'\b(?:pipe|duct|cable|conduit|wire)\s+penetration\s+(?:firestop|protection|seal)?\b',
                r'\bpenetration\s+(?:through|of|in)\s+(?:fire[-\s]?rated\s+)?(?:wall|floor|barrier|assembly)\b',
                r'\bannular\s+space\s+(?:protection|firestop|seal)\b',
            ],

            'HAZARD_LEVEL': [
                r'\b(?:light|ordinary|extra)\s+hazard\s+(?:(?:occupancy\s+)?Group\s+(?:\d|[IVX]+))?\b',
                r'\b(?:high|moderate|low)[-\s]?hazard\s+(?:occupancy|area|contents|use)\b',
                r'\bGroup\s+[HFS][-\s]?\d\s+(?:occupancy|hazard)\b',
                r'\b(?:high|moderate|low|severe|minimal)[-\s]?risk\s+(?:area|zone|occupancy|classification)\b',
                r'\bhazard\s+(?:category|classification|level|group)\s+(?:[IVX]+|\d+|[A-F])\b',
                r'\b(?:combustible|flammable|explosive|toxic)\s+(?:material|liquid|gas|solid)\s+(?:storage|use|handling)\b',
                r'\b(?:storage|warehouse)\s+hazard\s+(?:classification|category)\b',
            ],

            # =================================================================
            # SPACES & FACILITIES
            # =================================================================

            'FACILITY': [
                r'\bmeans\s+of\s+egress\s+(?:system|route|path|component|width|capacity)?\b',
                r'\bexit\s+(?:access|discharge|passageway|enclosure|stair(?:way)?|ramp|door)\b',
                r'\b(?:exit\s+access|fire[-\s]?rated|smoke[-\s]?proof|pressurized)\s+corridor\b',
                r'\bcorridor\s+(?:width|fire\s+rating|serving|rated)\b',
                r'\b(?:exit|enclosed|interior|exterior|smokeproof|pressurized)\s+stair(?:way|well|enclosure)?\b',
                r'\barea\s+of\s+refuge\b',
                r'\bhorizontal\s+exit\b',
                r'\bsafe\s+(?:area|zone|haven)\s+(?:of\s+refuge)?\b',
                r'\b(?:fire\s+)?(?:command|control)\s+(?:center|room|station)\b',
                r'\bfire\s+pump\s+room\b',
                r'\bsprinkler\s+(?:control|valve)\s+room\b',
            ],

            'LOCATION': [
                r'\b(?:within|not\s+more\s+than)\s+\d+\s*(?:m|ft|meters?|feet)\s+(?:of|from)\s+(?:an?\s+)?(?:exit|stairway|stair|door|opening|property\s+line)\b',
                r'\b(?:on|at|above|below)\s+(?:the\s+)?(?:ground|first|second|third|top|lowest|uppermost|basement)\s+(?:floor|story|level)\b',
                r'\bthroughout\s+(?:the\s+)?(?:building|structure|facility|floor|story)\b',
                r'\b(?:at|near)\s+(?:the\s+)?point\s+of\s+(?:discharge|egress)\b',
                r'\bbuilding\s+perimeter\b',
                r'\bexterior\s+wall\s+(?:of\s+the\s+building)?\b',
                r'\bremotely\s+located\s+from\s+(?:the\s+)?\b',
            ],

            # =================================================================
            # OCCUPANCY — FIX-OCC-1 & FIX-OCC-4:
            # Placed BEFORE BUILDING_TYPE so the seen_entities deduplication
            # correctly assigns Group codes and "… occupancy" phrases to
            # OCCUPANCY first.
            #
            # Rules:
            #   • Group [A-U]-\d codes            → always OCCUPANCY
            #   • "… occupancy" / "… use group"   → always OCCUPANCY
            #   • "… building" / "… structure"    → may be BUILDING_TYPE
            # =================================================================

            'OCCUPANCY': [
                # ── Group codes (most specific — must fire before BUILDING_TYPE) ──
                r'\bGroup\s+[A-U]-\d\b',
                r'\bGroup\s+[A-U]\s+(?:occupancy|use|hazard|classification)\b',
                r'\boccupancy\s+(?:classification|type|group|category)\s+[A-U](?:-\d)?\b',

                # ── Use/occupancy with explicit "occupancy" or "use group" suffix ──
                r'\b(?:assembly|business|educational|factory|hazardous|institutional|mercantile|residential|storage|utility)\s+(?:occupancy|use\s+group|use|group)\b',

                # ── Building-program terms ONLY when followed by "occupancy" ──
                # (these are ambiguous; without "occupancy" they are BUILDING_TYPE)
                r'\b(?:hospital|school|office|warehouse|factory|hotel|dormitory|apartment|clinic|laboratory|library|museum|church|theatre|theater)\s+occupancy\b',

                # ── Occupancy-load and related calculations ──
                r'\boccupant\s+load\s+(?:factor|calculation|determination|of\s+\d+)\b',

                # ── Mixed / separated occupancy ──
                r'\b(?:primary|accessory|incidental|mixed|separated|non[-\s]?separated)\s+(?:occupancy|use)\b',
                r'\bchange\s+of\s+occupancy\b',

                # ── Special care / day-care occupancies ──
                r'\b(?:day\s+care|ambulatory\s+care|residential\s+board|assisted\s+living|memory\s+care)\s+(?:occupancy|facility)\b',

                # ── Generic "… occupancy" catch-all (after specifics above) ──
                r'\b\w+\s+occupancy\b',
            ],

            # =================================================================
            # BUILDING_TYPE — FIX-OCC-2:
            # Patterns restricted to construction-type language and program-type
            # language where the trailing word is "building" or "structure"
            # NEVER "occupancy" (that is OCCUPANCY's domain).
            # Group [A-U]-\d codes are removed from here entirely.
            # =================================================================

            'BUILDING_TYPE': [
                # ── IBC Construction Types (Roman numeral) — canonical BUILDING_TYPE ──
                r'\bType\s+[IVX]+[-\s]?[AB]\s+(?:construction|building|structure)\b',
                r'\bType\s+[IVX]+\s+construction\b',
                r'\bType\s+[IVX]+\s+(?:protected|unprotected)\s+(?:construction|building)\b',

                # ── Building-program terms ONLY when followed by "building" or "structure" ──
                # (explicitly excludes "occupancy" to avoid stomping OCCUPANCY patterns)
                r'\b(?:residential|commercial|industrial|institutional|assembly|educational|healthcare|mercantile|storage)\s+(?:building|structure|facility(?!\s+occupancy))\b',
                r'\b(?:hospital|school|office|warehouse|factory|hotel|dormitory|apartment|clinic|laboratory|library|museum|church|theatre|theater)\s+(?:building|structure)\b',

                # ── Mixed-use and height descriptors ──
                r'\b(?:mixed|multi)[-\s]?use\s+(?:building|structure)\b',
                r'\b(?:high[-\s]?rise|low[-\s]?rise|mid[-\s]?rise)\s+(?:building|structure)\b',

                # ── Special building forms ──
                r'\b(?:covered\s+mall|open\s+mall|atrium|underground|windowless)\s+(?:building|structure)\b',
            ],

            # =================================================================
            # REFERENCES & STANDARDS
            # =================================================================

            'STANDARD': [
                r'\bNFPA\s+\d+[A-Z]?(?:[-:]\d+)?(?:\s*\(\d{4}\))?\b',
                r'\bISO\s+\d+(?:[-:]\d+)*(?:\s*\(\d{4}\))?\b',
                r'\bASTM\s+[A-Z]\d+(?:[-/]?\d+)*(?:\s*\(\d{4}\))?\b',
                r'\bUL\s+\d+[A-Z]?(?:\s*\(\d{4}\))?\b',
                r'\bFM\s+\d+(?:[-.:/]\d+)*\b',
                r'\b(?:ANSI|IEEE|EN|BS|IBC|IFC|IMC|IPC)\s+[A-Z]?\d+(?:[-.:/]\d+)*\b',
                r'\bInternational\s+(?:Building|Fire|Mechanical|Plumbing|Residential|Energy)\s+Code\b',
                r'\b(?:latest|current|applicable)\s+edition\s+of\s+(?:NFPA|ASTM|UL|ISO)\b',
                r'\bSBC\s+(?:Code\s+)?\d{3,4}(?:[-\s][A-Z]{2,4})?\b',
                r'\bSaudi\s+(?:Building|Fire)\s+Code\b',
                r'\bSBC[-\s](?:CR|AR|EN)\b',
                r'\b(?:IBC|IFC|IMC|IPC)\s+(?:Section\s+)?\d+',
                r'\b(?:the\s+)?Saudi\s+(?:Codes?|Standards?)\b',
                r'\b(?:the\s+)?Saudi\s+Code\b',
                r'\b(?:this|the)\s+(?:Saudi\s+)?Code\b',
                r'\bSBC\s+(?:Code\s+)?(?:304|305)[-\s](?:CR|CC|AR)\b',
                r'\bSBC\s+Code\s+\d{3,4}\b',
                r'\b(?:Saudi\s+)?Building\s+Code\s+SBC(?:[-\s]\d{3,4})?\b',
                r'\bSaudi\s+(?:laws|regulations|requirements)\b',
                r'\b(?:the\s+)?International\s+Fire\s+Code\b',
                r'\b(?:the\s+)?(?:NFPA|ASTM|UL|ISO)\s+(?:Standard|Code|Specification)?\s*\d+',
            ],

            'AUTHORITY': [
                r'\bAHJ\b',
                r'\bauthority\s+having\s+jurisdiction\b',
                r'\bfire\s+(?:marshal|chief|inspector|official)\b',
                r'\bbuilding\s+(?:official|inspector|code\s+official)\b',
                r'\bcode\s+(?:official|enforcement\s+officer)\b',
                r'\b(?:approving|approved)\s+(?:authority|agency)\b',
                r'\b(?:local|state|federal)\s+(?:authority|agency)\s+having\s+jurisdiction\b',
                r'\bSaudi\s+(?:laws|authorities|government)\b',
            ],

            'CERTIFICATION': [
                r'\b(?:UL|FM|ASTM|ISO)[-\s]?(?:listed|certified|approved|labeled)\b',
                r'\b(?:listed|certified|approved)\s+(?:by|in\s+accordance\s+with)\s+(?:UL|FM|NFPA|ASTM)\b',
                r'\b(?:third[-\s]?party|independently)\s+(?:certified|tested|verified|listed)\b',
                r'\bcertificate\s+of\s+(?:compliance|conformance|approval|listing)\b',
                r'\b(?:listed|labeled|certified)\s+(?:product|equipment|material|assembly)\b',
            ],

            # =================================================================
            # OPERATIONAL REQUIREMENTS
            # =================================================================

            'INSPECTION_FREQUENCY': [
                r'\b(?:inspected?)\s+(?:at\s+least\s+)?(?:annually|quarterly|monthly|semi[-\s]?annually|weekly)\b',
                r'\b(?:inspected?)\s+(?:at\s+least\s+)?(?:every|each)\s+\d+\s*(?:days?|weeks?|months?|years?)\b',
                r'\b(?:annual|quarterly|monthly|semi[-\s]?annual|weekly)\s+inspection\b',
                r'\binspection\s+(?:and\s+(?:testing|maintenance)\s+)?(?:frequency|schedule|interval)\b',
                r'\b(?:periodic|routine|scheduled)\s+inspection\s+(?:and\s+(?:testing|maintenance))?\b',
                r'\bshall\s+be\s+inspected\s+(?:at\s+least\s+)?(?:annually|quarterly|monthly|every)\b',
            ],

            'TESTING_REQUIREMENT': [
                r'\b(?:tested?|testing)\s+(?:at\s+least\s+)?(?:annually|quarterly|monthly|semi[-\s]?annually)\b',
                r'\b(?:tested?)\s+(?:at\s+least\s+)?(?:every|each)\s+\d+\s*(?:days?|weeks?|months?|years?)\b',
                r'\b(?:annual|quarterly|monthly|semi[-\s]?annual)\s+(?:acceptance\s+)?test(?:ing)?\b',
                r'\bshall\s+be\s+tested\s+(?:at\s+least\s+)?(?:annually|quarterly|monthly|every)\b',
                r'\b(?:acceptance|functional|operational|performance|witness)\s+test(?:ing)?\b',
                r'\btest\s+(?:frequency|schedule|procedure|report|record)\b',
                r'\b(?:flow|pressure|alarm|trip|drain|hydrostatic)\s+test\b',
            ],
        }

    # =========================================================================
    # SENTENCE-LEVEL SEMANTIC CLASSIFICATION
    # =========================================================================

    _OBLIGATION_MODALS = frozenset({'shall', 'must'})
    _PERMISSION_MODALS = frozenset({'may', 'is permitted', 'are permitted'})
    _CONDITION_MARKERS = frozenset({'if', 'when', 'where', 'unless', 'provided'})

    _EXCEPTION_ANCHORS = re.compile(
        r'(?:^|\.\s+|\n)\s*[Ee]xception\s*(?::|No\.|\d+\.?)'
        r'|\b(?:not\s+required|need\s+not\s+comply)\s+(?:where|when|if)\b'
        r'|\bexempt(?:ed)?\s+from\s+(?:the\s+)?(?:requirements?|provisions?)\s+of\b'
        r'|\balternative\s+(?:means|method)\s+of\s+(?:compliance|protection|egress)\b'
        r'|\b(?:except|unless)\s+(?:otherwise\s+)?(?:approved|permitted|required)\s+(?:by|in)\b',
        re.IGNORECASE | re.MULTILINE,
    )
    _PROHIBITION_RE = re.compile(
        r'\bshall\s+not\b'
        r'|\bmust\s+not\b'
        r'|\b(?:is|are)\s+(?:not\s+)?(?:prohibited|forbidden)\b'
        r'|\bno\s+\w+\s+shall\s+be\b'
        r'|\bnot\s+(?:be\s+)?permitted\b',
        re.IGNORECASE,
    )
    _REQUIREMENT_RE = re.compile(
        r'\bshall\s+(?:be\s+)?(?!not\b)'
        r'|\bmust\s+(?:be\s+)?(?!not\b)'
        r'|\b(?:is|are)\s+required\s+(?:to|for|by|in)\b'
        r'|\bin\s+(?:accordance|conformance)\s+with\b'
        r'|\bshall\s+comply\b',
        re.IGNORECASE,
    )
    _CONDITION_RE = re.compile(
        r'\b(?:where|when|if)\s+(?:the\s+)?(?:building\s+height|floor\s+area|occupant\s+load|story)\s+(?:exceeds?|is\s+(?:more|greater|less)\s+than)\s+\d+\b'
        r'|\b(?:in\s+)?(?:sprinklered|non[-\s]?sprinklered|unsprinklered)\s+buildings?\b'
        r'|\bwhere\s+(?:required|permitted)\s+by\s+(?:the\s+)?(?:authority|AHJ|Section|Table)\b'
        r'|\b(?:buildings?|occupancies)\s+(?:of\s+)?(?:Type\s+[IVX]+|Group\s+[A-U]-\d)\b'
        r'|\bwhere\s+(?:the\s+)?(?:building|occupancy|floor)\s+(?:is|contains|exceeds)\b',
        re.IGNORECASE,
    )

    def classify_sentence_semantics(self, sentence: str) -> Dict[str, bool]:
        return {
            'is_requirement': bool(self._REQUIREMENT_RE.search(sentence))
                              and not bool(self._PROHIBITION_RE.search(sentence)),
            'is_prohibition': bool(self._PROHIBITION_RE.search(sentence)),
            'is_exception':   bool(self._EXCEPTION_ANCHORS.search(sentence)),
            'is_condition':   bool(self._CONDITION_RE.search(sentence)),
            'is_permission':  any(
                re.search(r'\b' + re.escape(m) + r'\b', sentence, re.IGNORECASE)
                for m in self._PERMISSION_MODALS
            ),
        }

    # =========================================================================
    # COMPILATION
    # =========================================================================

    def _compile_patterns(self) -> Dict[str, List[re.Pattern]]:
        compiled = {}
        for entity_type, patterns in self.patterns.items():
            compiled[entity_type] = [
                re.compile(p, re.IGNORECASE) for p in patterns
            ]
        return compiled

    def _map_to_regulation_entity_type(self, extracted_type: str) -> int:
        try:
            if extracted_type in REGULATION_ENTITY_TYPES:
                return REGULATION_ENTITY_TYPES[extracted_type]
            return REGULATION_ENTITY_TYPES.get("NON_ENTITY", 26)
        except (KeyError, AttributeError) as e:
            print(f"Warning: Could not map {extracted_type}, using 26. Error: {e}")
            return 26

    # =========================================================================
    # FIX-OCC-3: CONFLICT RESOLUTION
    # When the same character span is claimed by both OCCUPANCY and
    # BUILDING_TYPE, OCCUPANCY wins unconditionally.
    # =========================================================================

    def _resolve_type_conflicts(
        self, entities: List['EntityInfo']
    ) -> List['EntityInfo']:
        """
        Post-processing pass that resolves OCCUPANCY ↔ BUILDING_TYPE conflicts.

        Resolution rules (in priority order):
        1. If two entities share the same (start, end) span:
           - OCCUPANCY beats BUILDING_TYPE unconditionally.
           - All other same-span ties: keep the first entity seen
             (which is the one whose pattern-type was processed first in
             the dict, i.e. already the higher-priority label).
        2. If a BUILDING_TYPE entity's normalised text contains an
           occupancy marker (e.g. "assembly occupancy", "Group A-1"),
           convert it to OCCUPANCY on the spot.
        """
        # ── Pass 1: retype any BUILDING_TYPE that looks like OCCUPANCY ──────
        for e in entities:
            if e.entity_type != 'BUILDING_TYPE':
                continue
            lower = e.entity_value.lower()
            if any(marker in lower for marker in self._OCCUPANCY_WINS_MARKERS):
                e.entity_type     = 'OCCUPANCY'
                e.type_id         = self._map_to_regulation_entity_type('OCCUPANCY')
                e.confidence      = min(1.0, e.confidence)   # no penalty

        # ── Pass 2: deduplicate same-span conflicts ───────────────────────────
        span_map: Dict[Tuple[int, int], 'EntityInfo'] = {}
        for e in entities:
            key = (e.start, e.end)
            if key not in span_map:
                span_map[key] = e
            else:
                existing = span_map[key]
                # OCCUPANCY always beats BUILDING_TYPE for the same span
                if (e.entity_type == 'OCCUPANCY'
                        and existing.entity_type == 'BUILDING_TYPE'):
                    span_map[key] = e
                # BUILDING_TYPE never displaces OCCUPANCY
                # All other conflicts: keep first (higher dict-priority type)

        resolved = list(span_map.values())
        resolved.sort(key=lambda x: x.start)
        return resolved

    # =========================================================================
    # ENTITY EXTRACTION
    # =========================================================================

    def extract_entity_value(self, match_text: str, entity_type: str,
                             match: re.Match) -> Optional['EntityInfo']:
        entity_value = match.group().strip()
        if not entity_value or len(entity_value) < 2:
            return None
        return EntityInfo(
            entity_value=entity_value,
            entity_type=entity_type,
            type_id=self._map_to_regulation_entity_type(entity_type),
            full_sentence="",
            semantic_context={},
            start=match.start(),
            end=match.end(),
            confidence=0.5,
            is_valid=True,
            validation_score=0.5,
            validation_issues=[],
        )

    def extract_entities_from_chunk(
        self,
        chunk_text: str,
        doc=None,
    ) -> List['EntityInfo']:
        """
        Extract all named entities from a text chunk.

        PERF-FIX:
          • sentence_boundaries + starts list built once, reused for all entities
          • ancestor_index built once, passed to _extract_semantic_context
          • doc reused throughout (no redundant nlp() call)
        """
        if self._is_arabic_dominant(chunk_text):
            self.logger.debug(
                f"Skipping Arabic-dominant chunk ({len(chunk_text)} chars)"
            )
            return []

        if doc is None:
            doc = self.nlp(chunk_text)

        # ── Pre-build shared data structures ONCE ────────────────────────
        sentence_boundaries = [(s.start_char, s.end_char) for s in doc.sents]
        sentence_starts     = [s[0] for s in sentence_boundaries]   # PERF-FIX
        ancestor_index      = self._build_ancestor_index(doc)             # PERF-FIX

        entities:   List['EntityInfo'] = []
        seen_spans: set = set()
        seen_keys:  set = set()

        # ── Regex-based extraction ───────────────────────────────────────
        for entity_type, patterns in self.compiled_patterns.items():
            base_conf = self._TYPE_BASE_CONFIDENCE.get(entity_type, 0.85)

            for pattern in patterns:
                for match in pattern.finditer(chunk_text):
                    entity_value = match.group()
                    start, end   = match.start(), match.end()

                    exact_key = (entity_value, start, entity_type)
                    if exact_key in seen_keys:
                        continue
                    seen_keys.add(exact_key)

                    span_key = (start, end)
                    if span_key in seen_spans:
                        continue
                    seen_spans.add(span_key)

                    full_sentence = self._find_containing_sentence(
                        start, end, sentence_boundaries, chunk_text,
                        sentence_starts=sentence_starts,       # PERF-FIX
                    )
                    semantic_context = self._extract_semantic_context(
                        entity_value, full_sentence, entity_type, doc,
                        ancestor_index=ancestor_index,          # PERF-FIX
                    )

                    entities.append(EntityInfo(
                        entity_value=entity_value,
                        entity_type=entity_type,
                        type_id=self._map_to_regulation_entity_type(entity_type),
                        full_sentence=full_sentence,
                        semantic_context=semantic_context,
                        start=start,
                        end=end,
                        confidence=base_conf,
                    ))

        # ── Keyword-gated spaCy NER supplement for EQUIPMENT ─────────────
        for ent in doc.ents:
            if ent.label_ not in ('ORG', 'PRODUCT', 'FAC'):
                continue
            ent_text_lower = ent.text.lower()
            if not any(kw in ent_text_lower for kw in self._FIRE_EQUIPMENT_KEYWORDS):
                self.logger.debug(
                    f"spaCy NER suppressed (no fire keyword): "
                    f"'{ent.text}' [{ent.label_}]"
                )
                continue

            exact_key = (ent.text, ent.start_char, 'EQUIPMENT')
            if exact_key in seen_keys:
                continue
            seen_keys.add(exact_key)

            span_key = (ent.start_char, ent.end_char)
            if span_key in seen_spans:
                continue
            seen_spans.add(span_key)

            full_sentence = self._find_containing_sentence(
                ent.start_char, ent.end_char, sentence_boundaries, chunk_text,
                sentence_starts=sentence_starts,
            )
            semantic_context = self._extract_semantic_context(
                ent.text, full_sentence, 'EQUIPMENT', doc,
                ancestor_index=ancestor_index,
            )
            entities.append(EntityInfo(
                entity_value=ent.text,
                entity_type='EQUIPMENT',
                type_id=self._map_to_regulation_entity_type('EQUIPMENT'),
                full_sentence=full_sentence,
                semantic_context=semantic_context,
                start=ent.start_char,
                end=ent.end_char,
                confidence=0.88,
            ))

        entities = self._resolve_type_conflicts(entities)
        entities = self.validate_entities(entities)
        entities.sort(key=lambda x: x.start)
        return entities

    def extract_raw_spans(
        self,
        text: str,
        doc=None,
    ) -> List['EntityInfo']:
        """
        Return ALL pattern hits with NO validation filtering.
        Used by _build_graphs_with_regex_guarantee.

        PERF-FIX: pre-builds sentence_starts + ancestor_index once.
        """
        if self._is_arabic_dominant(text):
            return []

        if doc is None:
            doc = self.nlp(text)

        sentence_boundaries = [(s.start_char, s.end_char) for s in doc.sents]
        sentence_starts     = [s[0] for s in sentence_boundaries]
        ancestor_index      = self._build_ancestor_index(doc)

        entities:   List['EntityInfo'] = []
        seen_spans: set = set()
        seen_keys:  set = set()

        # ── Regex pass ───────────────────────────────────────────────────
        for entity_type, patterns in self.compiled_patterns.items():
            base_conf = self._TYPE_BASE_CONFIDENCE.get(entity_type, 0.85)

            for pattern in patterns:
                for match in pattern.finditer(text):
                    entity_value = match.group().strip()
                    if not entity_value or len(entity_value) < 2:
                        continue

                    start, end = match.start(), match.end()
                    exact_key  = (entity_value, start, entity_type)
                    if exact_key in seen_keys:
                        continue
                    seen_keys.add(exact_key)

                    span_key = (start, end)
                    if span_key in seen_spans:
                        continue
                    seen_spans.add(span_key)

                    full_sentence = self._find_containing_sentence(
                        start, end, sentence_boundaries, text,
                        sentence_starts=sentence_starts,
                    )
                    semantic_context = self._extract_semantic_context(
                        entity_value, full_sentence, entity_type, doc,
                        ancestor_index=ancestor_index,
                    )

                    entities.append(EntityInfo(
                        entity_value=entity_value,
                        entity_type=entity_type,
                        type_id=self._map_to_regulation_entity_type(entity_type),
                        full_sentence=full_sentence,
                        semantic_context=semantic_context,
                        start=start,
                        end=end,
                        confidence=base_conf,
                        is_valid=False,
                        validation_score=0.0,
                        validation_issues=[],
                    ))

        # ── spaCy NER EQUIPMENT supplement ───────────────────────────────
        for ent in doc.ents:
            if ent.label_ not in ('ORG', 'PRODUCT', 'FAC'):
                continue
            if not any(kw in ent.text.lower() for kw in self._FIRE_EQUIPMENT_KEYWORDS):
                continue

            exact_key = (ent.text, ent.start_char, 'EQUIPMENT')
            if exact_key in seen_keys:
                continue
            seen_keys.add(exact_key)

            span_key = (ent.start_char, ent.end_char)
            if span_key in seen_spans:
                continue
            seen_spans.add(span_key)

            full_sentence = self._find_containing_sentence(
                ent.start_char, ent.end_char, sentence_boundaries, text,
                sentence_starts=sentence_starts,
            )
            semantic_context = self._extract_semantic_context(
                ent.text, full_sentence, 'EQUIPMENT', doc,
                ancestor_index=ancestor_index,
            )
            entities.append(EntityInfo(
                entity_value=ent.text,
                entity_type='EQUIPMENT',
                type_id=self._map_to_regulation_entity_type('EQUIPMENT'),
                full_sentence=full_sentence,
                semantic_context=semantic_context,
                start=ent.start_char,
                end=ent.end_char,
                confidence=0.88,
                is_valid=False,
                validation_score=0.0,
                validation_issues=[],
            ))

        entities = self._resolve_type_conflicts(entities)
        entities.sort(key=lambda x: x.start)
        return entities


    def validate_spans(
        self,
        spans: List['EntityInfo'],
        training_mode: bool = True,
    ) -> List['EntityInfo']:
        """
        Validate raw spans from extract_raw_spans().

        training_mode=True  (default): lenient 0.10 threshold so borderline
                                      regex hits survive as force_regex_entities.
        training_mode=False:           standard 0.25 threshold.
        """
        return self.validate_entities(spans, training_mode=training_mode)

    # =========================================================================
    # TOKEN-LEVEL LABELING FOR GAT
    # =========================================================================

    def _create_token_labels_from(
        self,
        doc,
        entities: List['EntityInfo'],
    ) -> List[Tuple[str, int, int, int]]:
        """
        Map validated entities onto spaCy tokens to produce
        (token_text, type_id, start_char, end_char) tuples.

        PERF-FIX: no nlp() call, no entity re-extraction.
        """
        # Build char → entity lookup from valid entities only
        char_to_entity: Dict[int, 'EntityInfo'] = {}
        for entity in entities:
            if not entity.is_valid:
                continue
            for pos in range(entity.start, entity.end):
                char_to_entity[pos] = entity

        token_labels = []
        for token in doc:
            if token.is_space:
                continue
            t_start = token.idx
            t_end   = t_start + len(token.text)

            # Find entity covering any char in this token's span
            entity_info = next(
                (char_to_entity[pos] for pos in range(t_start, t_end)
                if pos in char_to_entity),
                None,
            )
            type_id = (
                entity_info.type_id if entity_info
                else REGULATION_ENTITY_TYPES["NON_ENTITY"]
            )
            token_labels.append((token.text, type_id, t_start, t_end))

        return token_labels
    def create_token_labels(self, chunk_text: str) -> List[Tuple[str, int, int, int]]:
        """Backward-compatible wrapper — prefer _create_token_labels_from()."""
        doc = self.nlp(chunk_text)
        entities = self.extract_entities_from_chunk(chunk_text, doc=doc)
        return self._create_token_labels_from(doc, entities)

    def create_sentence_labels(
        self,
        chunk_text: str,
        doc=None,
        entities: List['EntityInfo'] = None,
    ) -> List['SentenceLabel']:
        """
        PERF-FIX: accepts optional pre-built doc and entities so callers
        that already have them don't trigger redundant work.
        """
        if doc is None:
            doc = self.nlp(chunk_text)
        if entities is None:
            entities = self.extract_entities_from_chunk(chunk_text, doc=doc)

        sentence_labels = []
        for idx, sent in enumerate(doc.sents):
            s_start, s_end = sent.start_char, sent.end_char
            sent_entities = [
                e for e in entities if s_start <= e.start < s_end and e.is_valid
            ]
            labels = list({e.entity_type for e in sent_entities})
            confidence_scores = {
                et: (
                    sum(e.confidence for e in sent_entities if e.entity_type == et)
                    / sum(1 for e in sent_entities if e.entity_type == et)
                )
                for et in labels
            }
            semantic_features = self._extract_sentence_features(sent.text, sent)
            semantic_features.update(self.classify_sentence_semantics(sent.text))

            sentence_labels.append(SentenceLabel(
                sentence=sent.text.strip(),
                sentence_idx=idx,
                start_char=s_start,
                end_char=s_end,
                labels=labels,
                confidence_scores=confidence_scores,
                entities=sent_entities,
                semantic_features=semantic_features,
            ))

        return sentence_labels

    def label_chunk_for_gat(self, chunk_text: str, doc=None) -> Dict[str, Any]:
        """
        Label a single chunk for GAT training.

        PERF-FIX: accepts optional pre-built spaCy `doc` so the pipeline can
        batch via nlp.pipe() and pass docs in.  Internally reuses the same
        doc and entity list for token labels and sentence semantics — no
        duplicate extraction passes.
        """
        # ── Single nlp() call ────────────────────────────────────────────
        if doc is None:
            doc = self.nlp(chunk_text)

        # ── Single entity-extraction pass ────────────────────────────────
        entities = self.extract_entities_from_chunk(chunk_text, doc=doc)

        # ── Token labels from the SAME doc + entities (no re-extraction) ─
        token_labels = self._create_token_labels_from(doc, entities)

        # ── Sentence semantics from the SAME doc ─────────────────────────
        sentence_semantics = [
            {
                'sentence':   sent.text.strip(),
                'start_char': sent.start_char,
                'end_char':   sent.end_char,
                **self.classify_sentence_semantics(sent.text.strip()),
            }
            for sent in doc.sents
        ]

        return {
            'text':               chunk_text,
            'tokens':             [t[0] for t in token_labels],
            'labels':             [t[1] for t in token_labels],
            'token_positions':    [(t[2], t[3]) for t in token_labels],
            'entities':           entities,
            'doc':                doc,
            'sentence_semantics': sentence_semantics,
        }

    def validate_entities(
        self,
        entities: List['EntityInfo'],
        training_mode: bool = False,
    ) -> List['EntityInfo']:
        """
        Validate entities and set is_valid / validation_score / validation_issues.

        training_mode=True  → is_valid threshold = 0.10  (lenient)
        training_mode=False → is_valid threshold = 0.25  (standard)
        """
        threshold = 0.10 if training_mode else 0.25

        for entity in entities:
            issues = []
            score  = 1.0

            if len(entity.full_sentence) < 10:
                issues.append('sentence_too_short')
                score -= 0.15

            if entity.entity_type == 'DIMENSION' and not re.search(
                    r'\d+', entity.entity_value):
                issues.append('missing_numeric_value')
                score -= 0.20

            elif entity.entity_type == 'STANDARD' and not re.search(
                    r'\d+', entity.entity_value):
                issues.append('missing_section_number')
                score -= 0.15

            if entity.entity_type not in _MEASUREMENT_TYPES:
                if isinstance(entity.semantic_context, dict):
                    if (not entity.semantic_context.get('subject') and
                            not entity.semantic_context.get('action')):
                        issues.append('poor_semantic_context')
                        score -= 0.10

            entity.validation_score  = float(max(0.1, score))
            entity.is_valid          = entity.validation_score >= threshold
            entity.validation_issues = issues

            if not entity.is_valid:
                self.logger.debug(
                    f"Validation failed: '{entity.entity_value}' "
                    f"({entity.entity_type}) "
                    f"score={entity.validation_score:.2f} "
                    f"threshold={threshold:.2f} issues={issues}"
                )

        return entities

    # =========================================================================
    # HELPERS
    # =========================================================================
    @staticmethod
    def _build_ancestor_index(doc, max_dist: int = 3) -> List[Set]:
        ancestor_sets: List[Set] = [set() for _ in doc]
        for tok in doc:
            cur = tok
            for _ in range(max_dist):
                if cur.dep_ == 'ROOT':
                    break
                cur = cur.head
                ancestor_sets[tok.i].add(cur)
        return ancestor_sets

    def _find_containing_sentence(
        self,
        entity_start:     int,
        entity_end:       int,
        sentences:        List[Tuple[int, int]],
        chunk_text:       str,
        sentence_starts:  List[int] = None,      # PERF-FIX: pre-built
    ) -> str:
        if not sentences:
            ctx_start = max(0, entity_start - 150)
            ctx_end   = min(len(chunk_text), entity_end + 150)
            return chunk_text[ctx_start:ctx_end].strip()

        # PERF-FIX: use pre-built list if available
        if sentence_starts is None:
            sentence_starts = [s[0] for s in sentences]

        idx = bisect.bisect_right(sentence_starts, entity_start) - 1

        if idx >= 0:
            s_start, s_end = sentences[idx]
            if s_start <= entity_start < s_end:
                return chunk_text[s_start:s_end].strip()

        # Fallback: context window
        ctx_start = max(0, entity_start - 150)
        ctx_end   = min(len(chunk_text), entity_end + 150)
        return chunk_text[ctx_start:ctx_end].strip()

    def _extract_semantic_context(
        self,
        entity_text:    str,
        full_sentence:  str,
        entity_type:    str,
        doc=None,
        ancestor_index: List[Set] = None,        # PERF-FIX: pre-built
    ) -> Dict[str, Any]:
        if doc is None:
            doc = self.nlp(full_sentence)

        semantic_info: Dict[str, Any] = {
            'subject':         '',
            'action':          '',
            'condition':       '',
            'reference':       '',
            'modifiers':       [],
            'related_values':  [],
            'raw_context':     '',
            'dependency_path': [],
        }

        idx = full_sentence.lower().find(entity_text.lower())
        if idx == -1:
            semantic_info['raw_context'] = "entity not found in sentence"
            return semantic_info

        e_end = idx + len(entity_text)
        entity_tokens = [
            t for t in doc if t.idx < e_end and t.idx + len(t.text) > idx
        ]
        if not entity_tokens:
            semantic_info['raw_context'] = "no matching tokens"
            return semantic_info

        primary = entity_tokens[0]

        # PERF-FIX: reuse pre-built ancestor index if available
        if ancestor_index is None:
            ancestor_index = self._build_ancestor_index(doc)

        def _connected(t1, t2) -> bool:
            """O(1) connectivity check using pre-built ancestor sets."""
            if t1.head == t2 or t2.head == t1:
                return True
            if t2 in ancestor_index[t1.i]:
                return True
            if t1 in ancestor_index[t2.i]:
                return True
            return bool(ancestor_index[t1.i] & ancestor_index[t2.i])

        # ── Single pass over doc ─────────────────────────────────────────
        subjects:       List = []
        actions:        List = []
        condition_toks: List = []
        modifiers:      List[str] = []
        related_values: List[str] = []

        _SUBJ_DEPS = frozenset({'nsubj', 'nsubjpass', 'dobj', 'pobj'})

        for tok in doc:
            connected = _connected(tok, primary)

            if tok.dep_ in _SUBJ_DEPS and connected:
                subjects.append(tok)

            if tok.pos_ == 'VERB' and connected:
                actions.append(tok)

            if tok.text.lower() in self._CONDITION_TRIGGERS:
                condition_toks.extend(list(tok.subtree))

            if tok.pos_ in ('ADJ', 'ADV') and connected:
                modifiers.append(tok.text)

            if (tok.like_num or (len(tok.text) > 0 and tok.text[0].isdigit())) \
                    and tok not in entity_tokens:
                related_values.append(tok.text)

        def _sorted_text(tokens) -> str:
            return ' '.join(
                t.text for t in sorted(set(tokens), key=lambda x: x.i)
            )

        semantic_info['subject']         = _sorted_text(subjects)
        semantic_info['action']          = _sorted_text(actions)
        semantic_info['condition']       = _sorted_text(condition_toks)
        semantic_info['modifiers']       = modifiers
        semantic_info['related_values']  = related_values
        semantic_info['dependency_path'] = self._get_dependency_path(primary)

        parts = [p for p in (
            semantic_info['subject'],
            semantic_info['action'],
            ' '.join(modifiers[:3]),
        ) if p]
        semantic_info['raw_context'] = (
            ' '.join(parts) if parts else "general requirement"
        )
        return semantic_info

    def _extract_sentence_features(self, sentence: str, sent_doc) -> Dict[str, Any]:
        return {
            'has_modal':       any(t.tag_ == 'MD' for t in sent_doc),
            'has_negation':    any(t.dep_ == 'neg' for t in sent_doc),
            'has_condition':   any(t.text.lower() in ('if', 'when', 'unless', 'except')
                                   for t in sent_doc),
            'has_requirement': any(t.text.lower() in ('shall', 'must', 'required')
                                   for t in sent_doc),
            'num_entities':    len([t for t in sent_doc if t.ent_type_]),
            'num_numbers':     len([t for t in sent_doc if t.like_num]),
        }

    def _tokens_connected(self, token1, token2, max_distance: int = 3) -> bool:
        # Cheapest check first
        if token1.head == token2 or token2.head == token1:
            return True

        # Walk token1's ancestor chain up to max_distance steps, collect into set
        ancestors1: set = set()
        cur = token1
        for _ in range(max_distance):
            if cur.dep_ == 'ROOT':
                break
            cur = cur.head
            if cur == token2:          # token2 IS an ancestor of token1
                return True
            ancestors1.add(cur)

        # Walk token2's ancestor chain; stop as soon as we hit a shared ancestor
        cur = token2
        for _ in range(max_distance):
            if cur.dep_ == 'ROOT':
                break
            cur = cur.head
            if cur == token1:          # token1 IS an ancestor of token2
                return True
            if cur in ancestors1:      # shared ancestor found
                return True

        return False

    def _get_dependency_path(self, token) -> List[str]:
        path, current, visited = [], token, set()
        while current and current not in visited:
            path.append(f"{current.text}({current.dep_})")
            visited.add(current)
            if current.dep_ == 'ROOT':
                break
            current = current.head
        return path

###LABELING CHECKPOINT MANAGER

In [ ]:
class LabelingCheckpointManager:
    """
    Manage checkpoints for entity labeling to avoid re-processing.
    Saves:
    - Labeled chunks
    - Entities per chunk
    - Graphs
    - Processing metadata
    - Extracted PDF text (one .txt file per document, in pdf_texts/ subdir)

    Supports incremental saving:
    - entity_{i}.pkl  — one file per labeled chunk
    - graph_{i}.pkl   — one file per built graph
    Both are cleaned up ONLY after the full checkpoint is atomically written.

    Text cache index
    ----------------
    Because doc_ids may contain characters that get sanitised into the filename,
    a companion index file  pdf_texts/doc_id_index.json  maps
    sanitised_stem → original_doc_id so that load_pdf_texts() always returns
    the original keys that the rest of the pipeline expects.
    """

    def __init__(self, checkpoint_dir: str, logger):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.logger = logger

        self.metadata_file = self.checkpoint_dir / "checkpoint_metadata.json"
        self.chunks_file   = self.checkpoint_dir / "labeled_chunks.pkl"
        self.entities_file = self.checkpoint_dir / "chunk_entities.pkl"
        self.graphs_file   = self.checkpoint_dir / "graphs.pkl"
        self.config_file   = self.checkpoint_dir / "config_hash.txt"

        # Per-file text cache — one .txt per document
        self.texts_dir  = self.checkpoint_dir / "text_cache"
        self.texts_dir.mkdir(parents=True, exist_ok=True)

        # Index that maps sanitised stem → original doc_id
        self.text_index_file = self.texts_dir / "doc_id_index.json"

    # -------------------------------------------------------------------------
    # Hashing  (mtime removed — size-only, never invalidated by timestamp)
    # -------------------------------------------------------------------------

    def _compute_config_hash(self, config: Dict, pdf_dir: str) -> str:
        pdf_files = sorted(Path(pdf_dir).glob("*.pdf"))
        pdf_info  = [
            {'name': p.name, 'size': p.stat().st_size}
            for p in pdf_files
        ]
        hash_data = {
            'pdf_files':        pdf_info,
            'chunk_size':       config.get('chunk_size', 500),
            'overlap':          config.get('overlap', 50),
            'entity_types':     list(REGULATION_ENTITY_TYPES.keys()),
            'num_entity_types': NUM_ENTITY_TYPES,
        }
        return hashlib.md5(
            json.dumps(hash_data, sort_keys=True).encode()
        ).hexdigest()

    # -------------------------------------------------------------------------
    # Existence / validity checks
    # -------------------------------------------------------------------------

    def checkpoint_exists(self) -> bool:
        return all(f.exists() for f in [
            self.metadata_file,
            self.chunks_file,
            self.entities_file,
            self.graphs_file,
        ])

    def is_checkpoint_valid(self, config: Dict, pdf_dir: str) -> bool:
        if not self.checkpoint_exists():
            return False
        if not self.config_file.exists():
            return False
        with open(self.config_file, 'r') as f:
            saved_hash = f.read().strip()
        current_hash = self._compute_config_hash(config, pdf_dir)
        is_valid     = saved_hash == current_hash
        if not is_valid:
            self.logger.warning("⚠️  Checkpoint invalid: PDFs or config changed")
        return is_valid

    # -------------------------------------------------------------------------
    # Per-file PDF text cache
    # -------------------------------------------------------------------------

    def _text_path(self, doc_id: str) -> Path:
        """Stable filename for a doc's cached text — sanitised for the FS."""
        safe_id = re.sub(r'[^\w\-]', '_', doc_id)
        return self.texts_dir / f"{safe_id}.txt"

    def _load_text_index(self) -> Dict[str, str]:
        """Load the stem → original_doc_id index, or return empty dict."""
        if not self.text_index_file.exists():
            return {}
        try:
            with open(self.text_index_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            self.logger.warning(f"[text cache] Failed to load index: {e}")
            return {}

    def _save_text_index_entry(self, doc_id: str) -> None:
        """Add a stem → doc_id entry to the index file."""
        safe_id = re.sub(r'[^\w\-]', '_', doc_id)
        index   = self._load_text_index()
        if index.get(safe_id) == doc_id:
            return   # already recorded — nothing to do
        index[safe_id] = doc_id
        try:
            with open(self.text_index_file, 'w', encoding='utf-8') as f:
                json.dump(index, f, indent=2)
        except Exception as e:
            self.logger.warning(f"[text cache] Failed to update index: {e}")

    def has_text_cache(self) -> bool:
        """True if at least one cached text file exists."""
        return any(
            p for p in self.texts_dir.glob("*.txt")
            if p.name != self.text_index_file.name
        )

    def has_cached_text(self, doc_id: str) -> bool:
        """True if a cached text file exists for this specific doc."""
        return self._text_path(doc_id).exists()

    def save_pdf_text(self, doc_id: str, text: str) -> None:
        """Persist extracted text for a single document."""
        path = self._text_path(doc_id)
        try:
            path.write_text(text, encoding='utf-8')
            self._save_text_index_entry(doc_id)
            self.logger.info(
                f"   [text cache] Saved {path.name}  ({len(text):,} chars)"
            )
        except Exception as e:
            self.logger.error(f"❌ Failed to save text for '{doc_id}': {e}")
            raise

    def load_pdf_text(self, doc_id: str) -> Optional[str]:
        """Load cached text for a single document, or None if not cached."""
        path = self._text_path(doc_id)
        try:
            text = path.read_text(encoding='utf-8')
            self.logger.info(
                f"   [text cache] Loaded {path.name}  ({len(text):,} chars)"
            )
            return text
        except FileNotFoundError:
            return None
        except Exception as e:
            self.logger.warning(f"[text cache] Failed to load '{doc_id}': {e}")
            return None

    def save_pdf_texts(self, pdf_texts: Dict[str, str]) -> None:
        """Persist a {doc_id: text} mapping — one file per document."""
        saved, failed = 0, 0
        for doc_id, text in pdf_texts.items():
            try:
                self.save_pdf_text(doc_id, text)
                saved += 1
            except Exception:
                failed += 1
        self.logger.info(
            f"   ✅ Text cache: {saved} saved"
            + (f", {failed} failed" if failed else "")
        )

    def load_pdf_texts(self) -> Optional[Dict[str, str]]:
        """
        Load ALL cached text files and return as {original_doc_id: text}.

        Key restoration priority:
          1. doc_id_index.json  — exact original doc_id recorded at save time
          2. path.stem          — fallback for files saved before the index
                                  existed (preserves backward compatibility)
        """
        txt_files = sorted(
            p for p in self.texts_dir.glob("*.txt")
            if p.name != self.text_index_file.name
        )
        if not txt_files:
            self.logger.info("📭 No PDF text cache found.")
            return None

        index = self._load_text_index()

        pdf_texts, failed = {}, 0
        for path in txt_files:
            try:
                original_doc_id = index.get(path.stem, path.stem)
                pdf_texts[original_doc_id] = path.read_text(encoding='utf-8')
            except Exception as e:
                self.logger.warning(f"[text cache] Failed to load {path.name}: {e}")
                failed += 1

        self.logger.info(
            f"   ✅ Loaded cached text for {len(pdf_texts)} document(s)"
            + (f" ({failed} failed)" if failed else "")
        )
        return pdf_texts if pdf_texts else None

    def get_cached_doc_ids(self) -> List[str]:
        """
        Return the list of original doc_ids that have a cached text file.
        Uses the index for accurate restoration; falls back to stem.
        """
        index = self._load_text_index()
        return [
            index.get(p.stem, p.stem)
            for p in self.texts_dir.glob("*.txt")
            if p.name != self.text_index_file.name
        ]

    def delete_pdf_text(self, doc_id: str) -> bool:
        """Delete the cached text for a single document."""
        path = self._text_path(doc_id)
        if path.exists():
            path.unlink()
            self.logger.info(f"   [text cache] Deleted {path.name}")
            safe_id = re.sub(r'[^\w\-]', '_', doc_id)
            index   = self._load_text_index()
            if safe_id in index:
                del index[safe_id]
                try:
                    with open(self.text_index_file, 'w', encoding='utf-8') as f:
                        json.dump(index, f, indent=2)
                except Exception:
                    pass
            return True
        return False

    # -------------------------------------------------------------------------
    # Incremental entity cache  (one file per labeled chunk)
    # -------------------------------------------------------------------------

    def save_single_entity(self, chunk_index: int, chunk_entity: Dict) -> None:
        """Persist a single labeled chunk as soon as it finishes."""
        path = self.checkpoint_dir / f"entity_{chunk_index}.pkl"
        try:
            with open(path, 'wb') as f:
                pickle.dump(chunk_entity, f)
        except Exception as e:
            self.logger.error(
                f"❌ Failed to save entity for chunk {chunk_index}: {e}"
            )
            raise

    def get_cached_entity_indices(self) -> set:
        """
        Return the set of chunk indices that already have a saved entity file.

        Called ONCE before the labeling loop and cached by the caller —
        do NOT call inside a per-chunk loop.
        """
        return {
            int(p.stem.split("_")[1])
            for p in self.checkpoint_dir.glob("entity_*.pkl")
        }

    def delete_incremental_entities(self) -> None:
        """Remove all per-chunk entity files (call after full checkpoint save)."""
        removed = 0
        for path in self.checkpoint_dir.glob("entity_*.pkl"):
            path.unlink()
            removed += 1
        if removed:
            self.logger.info(
                f"   [entity cache] Deleted {removed:,} incremental entity file(s)"
            )

    # -------------------------------------------------------------------------
    # Incremental graph cache  (one file per built graph)
    # -------------------------------------------------------------------------

    def save_single_graph(self, graph_index: int, graph: Data) -> None:
        """Persist a single graph as soon as it finishes building."""
        path = self.checkpoint_dir / f"graph_{graph_index}.pkl"
        try:
            with open(path, 'wb') as f:
                pickle.dump(graph, f)
        except Exception as e:
            self.logger.error(
                f"❌ Failed to save graph {graph_index}: {e}"
            )
            raise

    def load_incremental_graphs(self, indices: set = None) -> List[Data]:
        if indices is not None:
            graph_files = sorted(
                (
                    p for idx in indices
                    if (p := self.checkpoint_dir / f"graph_{idx}.pkl").exists()
                ),
                key=lambda p: int(p.stem.split("_")[1]),
            )
        else:
            graph_files = sorted(
                self.checkpoint_dir.glob("graph_*.pkl"),
                key=lambda p: int(p.stem.split("_")[1]),
            )

        if not graph_files:
            return []

        print(f"\n💾 Loading {len(graph_files):,} graph file(s) from disk...")

        def _load_one(path):
            with open(path, 'rb') as f:
                return pickle.load(f)

        results, failed = [None] * len(graph_files), 0
        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = {executor.submit(_load_one, p): i for i, p in enumerate(graph_files)}
            for future in tqdm(as_completed(futures), total=len(graph_files),
                              desc="Loading graphs", unit="file"):
                i = futures[future]
                try:
                    results[i] = future.result()
                except Exception as e:
                    self.logger.warning(f"Failed to load {graph_files[i].name}: {e}")
                    failed += 1

        return [r for r in results if r is not None]

    def get_cached_graph_indices(self) -> set:
        """
        Return the set of graph indices that are already saved.

        Called ONCE before the graph-build loop and cached by the caller —
        do NOT call inside a per-chunk loop.
        """
        return {
            int(p.stem.split("_")[1])
            for p in self.checkpoint_dir.glob("graph_*.pkl")
        }

    def delete_incremental_graphs(self) -> None:
        """Remove all per-graph files (call after full checkpoint save)."""
        removed = 0
        for path in self.checkpoint_dir.glob("graph_*.pkl"):
            path.unlink()
            removed += 1
        if removed:
            self.logger.info(
                f"   [graph cache] Deleted {removed:,} incremental graph file(s)"
            )

    # -------------------------------------------------------------------------
    # Save / load full checkpoint  (atomic write — cleanup only on success)
    # -------------------------------------------------------------------------

    def save_checkpoint(
        self,
        chunks:              List[Dict],
        chunk_entities:      List[Dict],
        graphs:              List[Data],
        config:              Dict,
        pdf_dir:             str,
        metadata:            Optional[Dict] = None,
        pdf_texts:           Optional[Dict[str, str]] = None,
        cleanup_incremental: bool = True,
    ) -> None:
        """
        Write the consolidated checkpoint atomically via temp files, then
        delete incremental .pkl files only after every write succeeds.

        Using temp files prevents a partial write from corrupting the last
        good checkpoint if the process is killed mid-save.
        """
        self.logger.info(f"\n💾 Saving checkpoint to: {self.checkpoint_dir}")
        try:
            # ── Write all data files via temp → rename (atomic on POSIX) ──
            def _atomic_pickle(obj, dest: Path) -> None:
                tmp = dest.with_suffix('.tmp')
                with open(tmp, 'wb') as f:
                    pickle.dump(obj, f)
                tmp.replace(dest)   # atomic on Linux/macOS; best-effort on Windows

            _atomic_pickle(chunks,         self.chunks_file)
            self.logger.info(f"   ✅ Saved {len(chunks):,} chunks")

            _atomic_pickle(chunk_entities, self.entities_file)
            self.logger.info(f"   ✅ Saved {len(chunk_entities):,} labeled chunks")

            _atomic_pickle(graphs,         self.graphs_file)
            self.logger.info(f"   ✅ Saved {len(graphs):,} graphs")

            # Optional per-file text cache
            if pdf_texts is not None:
                self.save_pdf_texts(pdf_texts)

            num_cached = (
                len(pdf_texts) if pdf_texts is not None
                else len(self.get_cached_doc_ids())
            )
            meta = {
                'timestamp':          datetime.now().isoformat(),
                'num_chunks':         len(chunks),
                'num_labeled_chunks': len(chunk_entities),
                'num_graphs':         len(graphs),
                'total_entities':     sum(len(ce['entities']) for ce in chunk_entities),
                'pdf_dir':            str(pdf_dir),
                'num_cached_texts':   num_cached,
                **(metadata or {}),
            }
            tmp_meta = self.metadata_file.with_suffix('.tmp')
            with open(tmp_meta, 'w') as f:
                json.dump(meta, f, indent=2)
            tmp_meta.replace(self.metadata_file)
            self.logger.info("   ✅ Saved metadata")

            config_hash = self._compute_config_hash(config, pdf_dir)
            tmp_cfg = self.config_file.with_suffix('.tmp')
            with open(tmp_cfg, 'w') as f:
                f.write(config_hash)
            tmp_cfg.replace(self.config_file)
            self.logger.info("   ✅ Saved config hash")

            # ── All writes succeeded — NOW safe to delete incremental files ──
            if cleanup_incremental:
                self.delete_incremental_entities()
                self.delete_incremental_graphs()

            self.logger.info("\n✅ Checkpoint saved successfully!")

        except Exception as e:
            self.logger.error(f"❌ Failed to save checkpoint: {e}")
            # Clean up any .tmp files left behind
            for tmp in self.checkpoint_dir.glob("*.tmp"):
                try:
                    tmp.unlink()
                except Exception:
                    pass
            raise

    def load_checkpoint(self) -> Optional[Dict]:
        if not self.checkpoint_exists():
            self.logger.warning("❌ No checkpoint found")
            return None

        self.logger.info(f"\n📂 Loading checkpoint from: {self.checkpoint_dir}")
        try:
            with open(self.chunks_file, 'rb') as f:
                chunks = pickle.load(f)
            self.logger.info(f"   ✅ Loaded {len(chunks):,} chunks")

            with open(self.entities_file, 'rb') as f:
                chunk_entities = pickle.load(f)
            self.logger.info(f"   ✅ Loaded {len(chunk_entities):,} labeled chunks")

            with open(self.graphs_file, 'rb') as f:
                graphs = pickle.load(f)
            self.logger.info(f"   ✅ Loaded {len(graphs):,} graphs")

            with open(self.metadata_file, 'r') as f:
                metadata = json.load(f)
            self.logger.info("   ✅ Loaded metadata")

            pdf_texts = self.load_pdf_texts() if self.has_text_cache() else None

            self.logger.info("\n✅ Checkpoint loaded successfully!")
            self.logger.info(f"   Created  : {metadata.get('timestamp', 'unknown')}")
            self.logger.info(f"   Entities : {metadata.get('total_entities', 0):,}")
            if pdf_texts is not None:
                self.logger.info(f"   PDF texts: {len(pdf_texts)} document(s) cached")

            return {
                'chunks':         chunks,
                'chunk_entities': chunk_entities,
                'graphs':         graphs,
                'metadata':       metadata,
                'pdf_texts':      pdf_texts,
            }

        except Exception as e:
            self.logger.error(f"❌ Failed to load checkpoint: {e}")
            return None

    # -------------------------------------------------------------------------
    # Helpers
    # -------------------------------------------------------------------------

    def get_checkpoint_info(self) -> Optional[Dict]:
        if not self.metadata_file.exists():
            return None
        try:
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        except Exception:
            return None

    def delete_checkpoint(self) -> None:
        self.logger.info(f"\n🗑️  Deleting checkpoint from: {self.checkpoint_dir}")
        for file in [
            self.metadata_file,
            self.chunks_file,
            self.entities_file,
            self.graphs_file,
            self.config_file,
        ]:
            if file.exists():
                file.unlink()
                self.logger.info(f"   ✅ Deleted {file.name}")

        self.delete_incremental_entities()
        self.delete_incremental_graphs()

        self.logger.info("✅ Checkpoint deleted")

In [ ]:
def prompt_checkpoint_decision(checkpoint_manager: LabelingCheckpointManager,
                               config: Dict, pdf_dir: str) -> str:
    """
    Prompt user to decide on checkpoint usage.

    FIX: Colab's input() returns a dict (the raw kernel message) instead of a
    str when called programmatically or after a runtime reset.  Calling
    .strip() on that dict raises AttributeError.

    Resolution: every input() call is wrapped in _safe_input(), which checks
    the return type and raises RuntimeError on non-str / EOFError.  If that
    happens inside any loop, the function falls back to
    auto_checkpoint_decision() — no silent prompt, no extra keypress needed.

    Returns:
        'load'  – use existing checkpoint
        'fresh' – start from scratch
    """

    def _safe_input(prompt: str) -> str:
        try:
            result = input(prompt)
        except EOFError:
            raise RuntimeError("no stdin")
        if not isinstance(result, str):
            raise RuntimeError(
                f"input() returned {type(result).__name__} — "
                "non-interactive Colab environment"
            )
        return result.strip()

    def _fallback(prefer_checkpoint: bool = True) -> str:
        print(
            "   ⚠️  Non-interactive environment detected — "
            "delegating to auto_checkpoint_decision()."
        )
        return auto_checkpoint_decision(
            checkpoint_manager, config, pdf_dir,
            prefer_checkpoint=prefer_checkpoint,
        )

    print("\n" + "=" * 80)
    print("💾 CHECKPOINT MANAGEMENT")
    print("=" * 80)

    if not checkpoint_manager.checkpoint_exists():
        print("\n📭 No checkpoint found. Starting from scratch...")
        return 'fresh'

    is_valid = checkpoint_manager.is_checkpoint_valid(config, pdf_dir)
    info     = checkpoint_manager.get_checkpoint_info()

    if info:
        print(f"\n📦 Found checkpoint:")
        print(f"   Created        : {info.get('timestamp', 'unknown')}")
        print(f"   Chunks         : {info.get('num_chunks', 0):,}")
        print(f"   Labeled chunks : {info.get('num_labeled_chunks', 0):,}")
        print(f"   Graphs         : {info.get('num_graphs', 0):,}")
        print(f"   Total entities : {info.get('total_entities', 0):,}")
        print(f"   Status         : {'✅ Valid' if is_valid else '⚠️  Invalid (PDFs or config changed)'}")

    if not is_valid:
        print(f"\n⚠️  Checkpoint is outdated (PDFs or config changed).")
        print(f"   💡 Recommendation: Start from scratch")

        while True:
            try:
                choice = _safe_input(
                    "\nOptions:\n"
                    "  [1] Start from scratch (delete old checkpoint)\n"
                    "  [2] Use invalid checkpoint anyway (risky)\n"
                    "  [3] Exit\n"
                    "Your choice (1-3): "
                )
            except RuntimeError:
                return _fallback(prefer_checkpoint=False)

            if choice == '1':
                checkpoint_manager.delete_checkpoint()
                return 'fresh'
            elif choice == '2':
                print("⚠️  Using invalid checkpoint — results may be incorrect!")
                return 'load'
            elif choice == '3':
                print("👋 Exiting...")
                raise SystemExit(0)
            else:
                print("❌ Invalid choice. Please enter 1, 2, or 3.")

    else:
        while True:
            try:
                choice = _safe_input(
                    "\nOptions:\n"
                    "  [1] Load checkpoint (skip labeling, save time ⚡)\n"
                    "  [2] Start from scratch (fresh labeling)\n"
                    "  [3] Exit\n"
                    "Your choice (1-3): "
                )
            except RuntimeError:
                return _fallback(prefer_checkpoint=True)

            if choice == '1':
                print("\n✅ Loading checkpoint...")
                return 'load'
            elif choice == '2':
                try:
                    confirm = _safe_input(
                        "⚠️  This will delete existing checkpoint. Confirm? (yes/no): "
                    ).lower()
                except RuntimeError:
                    return _fallback(prefer_checkpoint=True)
                if confirm in ('yes', 'y'):
                    checkpoint_manager.delete_checkpoint()
                    return 'fresh'
                else:
                    print("Cancelled. Please choose again.")
            elif choice == '3':
                print("👋 Exiting...")
                raise SystemExit(0)
            else:
                print("❌ Invalid choice. Please enter 1, 2, or 3.")

In [ ]:
def auto_checkpoint_decision(checkpoint_manager: LabelingCheckpointManager,
                            config: Dict, pdf_dir: str,
                            prefer_checkpoint: bool = True) -> str:
    """
    Non-interactive checkpoint decision for automated runs

    Args:
        checkpoint_manager: Checkpoint manager
        config: Configuration
        pdf_dir: PDF directory
        prefer_checkpoint: If True, use checkpoint when valid. If False, always fresh.

    Returns:
        'load' or 'fresh'
    """

    if not checkpoint_manager.checkpoint_exists():
        print("\n📭 No checkpoint found. Starting from scratch...")
        return 'fresh'

    is_valid = checkpoint_manager.is_checkpoint_valid(config, pdf_dir)
    info = checkpoint_manager.get_checkpoint_info()

    if info:
        print(f"\n📦 Found checkpoint:")
        print(f"   Created: {info.get('timestamp', 'unknown')}")
        print(f"   Status: {'✅ Valid' if is_valid else '⚠️  Invalid'}")

    if is_valid and prefer_checkpoint:
        print(f"✅ Using valid checkpoint (set prefer_checkpoint=False to disable)")
        return 'load'
    elif is_valid and not prefer_checkpoint:
        print(f"🔄 Ignoring checkpoint (prefer_checkpoint=False)")
        return 'fresh'
    else:
        print(f"⚠️  Checkpoint invalid. Starting from scratch...")
        checkpoint_manager.delete_checkpoint()
        return 'fresh'

###CANDIDATE PHRASE EXTRACTOR

In [ ]:
class CandidatePhraseExtractor:
    """Extract candidate phrases from regulatory text for GAT inference."""

    # ------------------------------------------------------------------
    # Fallback measurement regex (used when EntityLabeler patterns are
    # unavailable)
    # ------------------------------------------------------------------
    _FALLBACK_MEASUREMENT_RE: re.Pattern = re.compile(
        r'\b\d+(?:\.\d+)?\s*'
        r'(?:meters?|metres?|feet|foot|inches?|(?<!\w)in(?!\w)|cm|mm'
        r'|(?<!\w)ft(?!\w)|(?<!\w)m(?!\w))\b'
        r'|\b\d+\s*(?:ft|feet)\s+\d+\s*(?:in|inches?)\b'
        r'|\b\d+(?:\.\d+)?\s*[xX]\s*\d+(?:\.\d+)?\s*(?:m|ft|cm|mm)\b',
        re.IGNORECASE,
    )

    # ------------------------------------------------------------------
    # Measurement indicator trigger words (FIX-D — optimized)
    # ------------------------------------------------------------------
    _MEASUREMENT_INDICATORS: frozenset = frozenset({
        # dimensional / spatial (keep as is)
        'clearance', 'distance', 'width', 'height', 'depth', 'length',
        'separation', 'spacing', 'opening', 'diameter', 'radius',
        # egress-specific
        'egress', 'travel', 'exit', 'path', 'corridor', 'aisle',
        # performance / capacity
        'capacity', 'rating', 'minimum', 'maximum', 'limit', 'load',
        # physical quantities
        'pressure', 'flow', 'temperature', 'area', 'volume', 'rate',
    })

    # ------------------------------------------------------------------
    # Entity types that need special handling
    # ------------------------------------------------------------------
    _SHORT_ENTITY_TYPES: frozenset = frozenset({
        'FIRE_RATING', 'PERCENTAGE', 'CERTIFICATION',
    })

    _NUMERIC_ENTITY_TYPES: frozenset = frozenset({
        'DIMENSION', 'AREA', 'VOLUME', 'CAPACITY', 'PRESSURE',
        'FLOW_RATE', 'TEMPERATURE',
    })

    # ------------------------------------------------------------------
    def __init__(self, nlp, entity_labeler) -> None:
        self.nlp = nlp
        self.entity_labeler = entity_labeler

        self.has_parser: bool = 'parser' in nlp.pipe_names
        if not self.has_parser:
            print("⚠️  Warning: Dependency parser not available. "
                  "Using POS-based fallback.")

        self._dimension_patterns: list = self._resolve_dimension_patterns()

    # ------------------------------------------------------------------
    # Setup helpers
    # ------------------------------------------------------------------
    def _resolve_dimension_patterns(self) -> list:
        """Return DIMENSION patterns from EntityLabeler, or the fallback."""
        try:
            patterns = self.entity_labeler.compiled_patterns.get('DIMENSION', [])
            if patterns:
                return list(patterns)
        except AttributeError:
            pass
        return [self._FALLBACK_MEASUREMENT_RE]

    # ------------------------------------------------------------------
    # Protected-interval helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _build_protected_intervals(
        candidates: list,
    ) -> Tuple[list, list]:
        """Return sorted start/end lists for training_aligned + measurement_regex spans."""
        intervals = sorted(
            (c['start'], c['end'])
            for c in candidates
            if c.get('source') in ('training_aligned', 'measurement_regex')
        )
        if not intervals:
            return [], []
        starts = [iv[0] for iv in intervals]
        ends   = [iv[1] for iv in intervals]
        return starts, ends

    @staticmethod
    def _in_protected_span(char_pos: int, starts: list, ends: list) -> bool:
        """True if char_pos falls inside any protected interval."""
        if not starts:
            return False
        idx = bisect.bisect_right(starts, char_pos) - 1
        if idx < 0:
            return False
        return char_pos < ends[idx]

    # ------------------------------------------------------------------
    # FIX 1: Span boundary stripping - DISABLED FOR INFERENCE
    # ------------------------------------------------------------------
    @staticmethod
    def _strip_boundary_noise(text: str) -> str:
        """
        Remove leading articles/prepositions and trailing stop-words from
        a candidate phrase.

        Examples
        --------
        "the exit access travel distance"  → "exit access travel distance"
        "access travel distance within the" → "access travel distance"
        "of egress travel distance"         → "egress travel distance"
        """
        tokens = text.split()
        if not tokens:
            return text

        # Strip leading stop-words
        while tokens and tokens[0].lower().rstrip('.,;:') in _LEADING_STOP:
            tokens = tokens[1:]

        # Strip trailing stop-words
        while tokens and tokens[-1].lower().lstrip('.,;:') in _TRAILING_STOP:
            tokens = tokens[:-1]

        return ' '.join(tokens) if tokens else text

    @staticmethod
    def _normalize_candidate_span(
        original_text: str,
        stripped_text: str,
        original_start: int,
        original_end: int,
    ) -> Tuple[str, int, int]:
        """
        Adjust start/end character offsets to match stripped_text within
        original_text.  Falls back to original offsets if the stripped text
        cannot be located (e.g. multi-space normalisation artefacts).
        """
        if stripped_text == original_text:
            return stripped_text, original_start, original_end

        offset = original_text.find(stripped_text)
        if offset == -1:
            # Fallback: stripped text not locatable — return as-is
            return stripped_text, original_start, original_end

        new_start = original_start + offset
        new_end   = new_start + len(stripped_text)
        return stripped_text, new_start, new_end

    def _apply_boundary_strip(self, candidate: Dict) -> Dict:
        """
        Return a (possibly modified) copy of candidate with boundary noise
        removed from text and start/end adjusted accordingly.

        FIX 1: Boundary stripping is ONLY applied during training,
        NEVER during inference. For inference, return candidate unchanged
        to preserve full context for the GAT model.
        """
        # For inference mode (detected by missing 'training_mode' flag),
        # NEVER strip boundaries
        if not candidate.get('training_mode', False):
            return candidate

        # Priority sources are always kept intact
        if candidate.get('source') in ('training_aligned', 'measurement_regex'):
            return candidate

        original_text  = candidate['text']
        stripped_text  = self._strip_boundary_noise(original_text)

        if stripped_text == original_text or not stripped_text:
            return candidate

        stripped_text, new_start, new_end = self._normalize_candidate_span(
            original_text, stripped_text,
            candidate['start'], candidate['end'],
        )

        updated = dict(candidate)
        updated['text']  = stripped_text
        updated['start'] = new_start
        updated['end']   = new_end
        return updated

    # ------------------------------------------------------------------
    # FIX 2: Position-aware substring suppression
    # ------------------------------------------------------------------
    @staticmethod
    def _spans_overlap(
        start_a: int, end_a: int,
        start_b: int, end_b: int,
    ) -> bool:
        """True if two character spans overlap."""
        return start_a < end_b and start_b < end_a

    def _is_substring_of_seen(
        self,
        text:            str,
        char_start:      int,
        char_end:        int,
        seen_texts:      Set[str],
        protected_texts: Set[str],
        seen_positions:  Dict[str, Tuple[int, int]],
    ) -> bool:
        text_lower = text.lower().strip()
        text_len   = len(text_lower)

        # Rule 1: protected spans are never suppressed
        if text_lower in protected_texts:
            return False

        # Rule 2: short numeric texts are always kept
        if any(c.isdigit() for c in text) and text_len <= 8:
            return False

        has_digit = any(c.isdigit() for c in text)

        for seen_lower, (seen_start, seen_end) in seen_positions.items():
            seen_len = len(seen_lower)

            if seen_len <= 3:
                continue

            # ------------------------------------------------------------------
            # PERF: length pre-filter — if neither string can contain the other
            # given the threshold, skip the O(n) string search entirely.
            # threshold is 0.85 for numeric text, 0.70 for alpha text.
            # Containment requires shorter/longer >= threshold.
            # ------------------------------------------------------------------
            shorter = min(text_len, seen_len)
            longer  = max(text_len, seen_len)
            threshold = 0.85 if has_digit else 0.70

            if shorter / longer <= threshold:
                continue    # too different in length — cannot be near-duplicates

            # Exact duplicate at a (possibly different) position
            if seen_lower == text_lower:
                if self._spans_overlap(char_start, char_end, seen_start, seen_end):
                    return True
                continue

            # String containment check — only reached when lengths are similar
            if text_lower not in seen_lower and seen_lower not in text_lower:
                continue

            # Near-duplicate check (shorter/longer already confirmed >= threshold)
            if shorter / longer >= 0.95:
                if self._spans_overlap(char_start, char_end, seen_start, seen_end):
                    return True

        return False

    # ------------------------------------------------------------------
    # FIX 3: Intelligent candidate filtering
    # ------------------------------------------------------------------
    def _should_keep_candidate(self, candidate: Dict) -> bool:
        """
        Determine if a candidate should be kept based on content and length.
        """
        text = candidate['text'].strip()

        # Empty or whitespace only
        if not text:
            return False

        # Priority sources always kept
        if candidate.get('source') in ('training_aligned', 'measurement_regex'):
            return True

        # Check for minimum length
        has_digits = any(c.isdigit() for c in text)

        # Numeric/short entities need special handling
        if has_digits:
            # Keep very short numeric entities (e.g., "2 hr", "R13", "Class A")
            if len(text) <= 8:
                return True
        else:
            # Non-numeric entities need at least 3 chars
            if len(text) < 3:
                return False

        # Check for meaningless single characters or punctuation
        if len(text) == 1 and not text[0].isalnum():
            return False

        return True

    # ------------------------------------------------------------------
    # FIX 4: Optimized measurement indicator windows
    # ------------------------------------------------------------------
    def _get_measurement_window(self, token, indicator: str) -> Tuple[int, int]:
        """
        Get appropriate context window based on the measurement indicator.

        Returns (start_idx, end_idx) tuple.
        """
        # For very specific terms, use very narrow window
        if indicator in {'clearance', 'egress', 'exit'}:
            return (max(0, token.i - 1), min(len(token.doc), token.i + 2))

        # For general measurement terms, use moderate window
        elif indicator in {'distance', 'height', 'width', 'depth'}:
            return (max(0, token.i - 2), min(len(token.doc), token.i + 3))

        # For broader terms, use default window
        else:
            return (max(0, token.i - 2), min(len(token.doc), token.i + 3))

    # ------------------------------------------------------------------
    # Main extraction entry point
    # ------------------------------------------------------------------
    def extract_candidates(self, text: str, training_mode: bool = False) -> List[Dict]:
        """
        Extract candidate phrases from *text* for GAT classification.

        FIX-THRESH: Align inference validation threshold with GraphBuilder's
        training threshold (0.25).  Previously 0.10 was used at inference,
        meaning the GAT received span quality levels it was never trained on,
        causing it to predict NON_ENTITY for those spans and triggering
        regex_fallback.  Now:
          - training_mode=True  → 0.10  (permissive, graph builder filters at 0.25)
          - training_mode=False → 0.25  (matches exactly what the GAT trained on)
        """
        # FIX-THRESH: threshold aligned per mode
        VALIDATION_THRESHOLD = 0.10 if training_mode else 0.25

        try:
            text = text[:self.nlp.max_length]
            doc  = self.nlp(text)

            candidates:      List[Dict]                  = []
            seen_positions:  Set[Tuple[str, int]]        = set()
            seen_pos_map:    Dict[str, Tuple[int, int]]  = {}
            seen_texts:      Set[str]                    = set()
            protected_texts: Set[str]                    = set()

            # ── Strategy 1: training-aligned (regex-labelled) entities ──────
            training_aligned_count = 0
            try:
                raw_spans         = self.entity_labeler.extract_raw_spans(text)
                training_entities = self.entity_labeler.validate_spans(
                    raw_spans, training_mode=False
                )

                for entity in training_entities:
                    # FIX-THRESH: use mode-appropriate threshold instead of
                    # hardcoded 0.10.  At inference this is now 0.25, matching
                    # exactly what GraphBuilder used to build training graphs.
                    if entity.validation_score < VALIDATION_THRESHOLD:
                        continue

                    key = (entity.entity_value.lower().strip(), entity.start)
                    if key in seen_positions:
                        continue

                    candidate = {
                        'text':             entity.entity_value,
                        'start':            entity.start,
                        'end':              entity.end,
                        'type':             f'regex_{entity.entity_type}',
                        'source':           'training_aligned',
                        'expected_type_id': entity.type_id,
                        'expected_type':    entity.entity_type,
                        'validation_score': entity.validation_score,
                        'confidence':       entity.confidence,
                        'training_mode':    training_mode,
                    }

                    candidates.append(candidate)
                    seen_positions.add(key)
                    text_lower = entity.entity_value.lower().strip()
                    seen_texts.add(text_lower)
                    protected_texts.add(text_lower)
                    seen_pos_map[text_lower] = (entity.start, entity.end)
                    training_aligned_count += 1

            except Exception as exc:
                import traceback
                print(f"   ⚠️ Training-aligned extraction failed: {exc}")
                traceback.print_exc()

            # ── Strategy 2: bare measurement patterns ───────────────────────
            dimension_type_id: Optional[int] = REGULATION_ENTITY_TYPES.get('DIMENSION')

            for pattern in self._dimension_patterns:
                for match in pattern.finditer(text):
                    candidate_text = match.group().strip()
                    if len(candidate_text) < 2:
                        continue
                    key = (candidate_text.lower(), match.start())
                    if key in seen_positions:
                        continue

                    candidates.append({
                        'text':             candidate_text,
                        'start':            match.start(),
                        'end':              match.end(),
                        'type':             'bare_measurement',
                        'source':           'measurement_regex',
                        'expected_type_id': dimension_type_id,
                        'expected_type':    'DIMENSION',
                        'validation_score': 0.9,
                        'confidence':       0.85,
                        'training_mode':    training_mode,
                    })
                    seen_positions.add(key)
                    protected_texts.add(candidate_text.lower())
                    seen_pos_map[candidate_text.lower()] = (match.start(), match.end())

            # Build protected intervals AFTER strategies 1 & 2
            p_starts, p_ends = self._build_protected_intervals(candidates)

            # ── Strategy 3: noun chunks / POS sequences ──────────────────────
            if self.has_parser:
                for chunk in doc.noun_chunks:
                    raw_text = chunk.text.strip()
                    key      = (raw_text.lower(), chunk.start_char)

                    candidate = {
                        'text':             raw_text,
                        'start':            chunk.start_char,
                        'end':              chunk.end_char,
                        'type':             'noun_phrase',
                        'source':           'linguistic',
                        'expected_type_id': None,
                        'expected_type':    None,
                        'training_mode':    training_mode,
                    }

                    if not self._should_keep_candidate(candidate):
                        continue
                    if key in seen_positions:
                        continue
                    if self._is_substring_of_seen(
                        raw_text, chunk.start_char, chunk.end_char,
                        seen_texts, protected_texts, seen_pos_map,
                    ):
                        continue

                    if training_mode:
                        candidate = self._apply_boundary_strip(candidate)

                    candidates.append(candidate)
                    seen_positions.add((candidate['text'].lower(), candidate['start']))
                    seen_texts.add(candidate['text'].lower())
                    seen_pos_map[candidate['text'].lower()] = (
                        candidate['start'], candidate['end']
                    )

            else:
                # POS-based fallback when parser unavailable
                current_phrase: list = []
                phrase_start = 0

                for token in doc:
                    if token.pos_ in ('NOUN', 'PROPN', 'ADJ', 'NUM'):
                        if not current_phrase:
                            phrase_start = token.idx
                        current_phrase.append(token.text)
                    else:
                        if current_phrase:
                            raw_text   = ' '.join(current_phrase).strip()
                            phrase_end = phrase_start + len(raw_text)

                            candidate = {
                                'text':             raw_text,
                                'start':            phrase_start,
                                'end':              phrase_end,
                                'type':             'pos_noun_sequence',
                                'source':           'linguistic_fallback',
                                'expected_type_id': None,
                                'expected_type':    None,
                                'training_mode':    training_mode,
                            }

                            if self._should_keep_candidate(candidate):
                                key = (raw_text.lower(), phrase_start)
                                if key not in seen_positions and not self._is_substring_of_seen(
                                    raw_text, phrase_start, phrase_end,
                                    seen_texts, protected_texts, seen_pos_map,
                                ):
                                    if training_mode:
                                        candidate = self._apply_boundary_strip(candidate)
                                    candidates.append(candidate)
                                    seen_positions.add(
                                        (candidate['text'].lower(), candidate['start'])
                                    )
                                    seen_texts.add(candidate['text'].lower())
                                    seen_pos_map[candidate['text'].lower()] = (
                                        candidate['start'], candidate['end']
                                    )
                        current_phrase = []

                # Flush remaining tokens
                if current_phrase:
                    raw_text   = ' '.join(current_phrase).strip()
                    phrase_end = phrase_start + len(raw_text)

                    candidate = {
                        'text':             raw_text,
                        'start':            phrase_start,
                        'end':              phrase_end,
                        'type':             'pos_noun_sequence',
                        'source':           'linguistic_fallback',
                        'expected_type_id': None,
                        'expected_type':    None,
                        'training_mode':    training_mode,
                    }

                    if self._should_keep_candidate(candidate):
                        key = (raw_text.lower(), phrase_start)
                        if key not in seen_positions and not self._is_substring_of_seen(
                            raw_text, phrase_start, phrase_end,
                            seen_texts, protected_texts, seen_pos_map,
                        ):
                            if training_mode:
                                candidate = self._apply_boundary_strip(candidate)
                            candidates.append(candidate)
                            seen_positions.add(
                                (candidate['text'].lower(), candidate['start'])
                            )
                            seen_texts.add(candidate['text'].lower())
                            seen_pos_map[candidate['text'].lower()] = (
                                candidate['start'], candidate['end']
                            )

            # ── Strategy 4: numeric context windows ─────────────────────────
            for token in doc:
                if not (token.like_num or token.is_digit):
                    continue
                if self._in_protected_span(token.idx, p_starts, p_ends):
                    continue

                start_idx = max(0, token.i - 2)
                end_idx   = min(len(doc), token.i + 3)
                phrase    = doc[start_idx:end_idx]
                raw_text  = phrase.text.strip()

                candidate = {
                    'text':             raw_text,
                    'start':            phrase.start_char,
                    'end':              phrase.end_char,
                    'type':             'numeric_context',
                    'source':           'numeric',
                    'expected_type_id': None,
                    'expected_type':    None,
                    'training_mode':    training_mode,
                }

                if not self._should_keep_candidate(candidate):
                    continue

                key = (raw_text.lower(), phrase.start_char)
                if key in seen_positions:
                    continue
                if self._is_substring_of_seen(
                    raw_text, phrase.start_char, phrase.end_char,
                    seen_texts, protected_texts, seen_pos_map,
                ):
                    continue

                if training_mode:
                    candidate = self._apply_boundary_strip(candidate)

                candidates.append(candidate)
                seen_positions.add(
                    (candidate['text'].lower(), candidate['start'])
                )
                seen_texts.add(candidate['text'].lower())
                seen_pos_map[candidate['text'].lower()] = (
                    candidate['start'], candidate['end']
                )

            # ── Strategy 5: spaCy NER spans ─────────────────────────────────
            relevant_ner_labels = {
                'ORG', 'PRODUCT', 'FAC', 'LAW',
                'QUANTITY', 'CARDINAL', 'PERCENT',
            }

            for ent in doc.ents:
                if ent.label_ not in relevant_ner_labels:
                    continue

                candidate = {
                    'text':             ent.text,
                    'start':            ent.start_char,
                    'end':              ent.end_char,
                    'type':             f'spacy_{ent.label_}',
                    'source':           'spacy_ner',
                    'expected_type_id': None,
                    'expected_type':    None,
                    'training_mode':    training_mode,
                }

                if not self._should_keep_candidate(candidate):
                    continue

                key = (ent.text.lower(), ent.start_char)
                if key in seen_positions:
                    continue
                if self._is_substring_of_seen(
                    ent.text, ent.start_char, ent.end_char,
                    seen_texts, protected_texts, seen_pos_map,
                ):
                    continue

                if training_mode:
                    candidate = self._apply_boundary_strip(candidate)

                candidates.append(candidate)
                seen_positions.add(
                    (candidate['text'].lower(), candidate['start'])
                )
                seen_texts.add(candidate['text'].lower())
                seen_pos_map[candidate['text'].lower()] = (
                    candidate['start'], candidate['end']
                )

            # ── Strategy 6: measurement indicator context windows ───────────
            for token in doc:
                indicator = token.text.lower()
                if indicator not in self._MEASUREMENT_INDICATORS:
                    continue

                start_idx, end_idx = self._get_measurement_window(token, indicator)
                phrase   = doc[start_idx:end_idx]
                raw_text = phrase.text.strip()

                candidate = {
                    'text':             raw_text,
                    'start':            phrase.start_char,
                    'end':              phrase.end_char,
                    'type':             'measurement_indicator',
                    'source':           'measurement',
                    'expected_type_id': None,
                    'expected_type':    None,
                    'training_mode':    training_mode,
                }

                if not self._should_keep_candidate(candidate):
                    continue

                key = (raw_text.lower(), phrase.start_char)
                if key in seen_positions:
                    continue
                if self._is_substring_of_seen(
                    raw_text, phrase.start_char, phrase.end_char,
                    seen_texts, protected_texts, seen_pos_map,
                ):
                    continue

                if training_mode:
                    candidate = self._apply_boundary_strip(candidate)

                candidates.append(candidate)
                seen_positions.add(
                    (candidate['text'].lower(), candidate['start'])
                )
                seen_texts.add(candidate['text'].lower())
                seen_pos_map[candidate['text'].lower()] = (
                    candidate['start'], candidate['end']
                )

            return self._sort_candidates(candidates)

        except Exception as exc:
            import traceback
            print(f"⚠️  Error extracting candidates: {exc}")
            traceback.print_exc()
            return []
    # ------------------------------------------------------------------
    # Sorting & statistics
    # ------------------------------------------------------------------
    def _sort_candidates(self, candidates: List[Dict]) -> List[Dict]:
        """Priority sources first, then by character position."""
        priority = []
        others   = []
        for c in candidates:
            if c.get('source') in ('training_aligned', 'measurement_regex'):
                priority.append(c)
            else:
                others.append(c)
        priority.sort(key=lambda x: x['start'])
        others.sort(key=lambda x: x['start'])
        return priority + others

    def get_candidate_stats(self, candidates: List[Dict]) -> Dict[str, int]:
        """Return a summary dict of candidate counts by source."""
        stats: Dict = {
            'total':                    len(candidates),
            'by_source':                {},
            'training_aligned_count':   0,
            'measurement_regex_count':  0,
            'novel_count':              0,
        }
        for c in candidates:
            source = c.get('source', 'unknown')
            stats['by_source'][source] = stats['by_source'].get(source, 0) + 1
            if source == 'training_aligned':
                stats['training_aligned_count'] += 1
            elif source == 'measurement_regex':
                stats['measurement_regex_count'] += 1
            else:
                stats['novel_count'] += 1
        return stats

### GRAPH CONSTRUCTION

In [ ]:
def _sentence_type_hint(sentence: str) -> torch.Tensor:
    """
    Return a one-hot (27-dim) vector whose '1' position corresponds to the
    entity type most strongly suggested by the *containing sentence*.

    Purpose: give the GAT a sentence-level prior so it can distinguish
    'sprinkler head' (EQUIPMENT context) from 'sprinkler head test'
    (TESTING_REQUIREMENT context) without traversing several graph edges.

    Returns a zero vector (all zeros) when no keyword fires, which tells
    the GAT 'no sentence-level hint available' — still valid, just neutral.
    """
    s = sentence.lower()
    best_type: Optional[str] = None
    best_count = 0

    for etype, keywords in _SENTENCE_HINTS.items():
        count = sum(1 for kw in keywords if kw in s)
        if count > best_count:
            best_count = count
            best_type  = etype

    one_hot = torch.zeros(NUM_ENTITY_TYPES, dtype=torch.float)
    if best_type is not None and best_count > 0:
        hint_id = REGULATION_ENTITY_TYPES.get(best_type, -1)
        if 0 <= hint_id < NUM_ENTITY_TYPES:
            one_hot[hint_id] = 1.0

    return one_hot   # shape: (27,)


def _is_plausible(entity_text: str, predicted_type: str) -> bool:
    """
    Return False when the predicted entity type is implausible given the
    *span text alone*.  Used as a post-GAT sanity gate.

    For types not in _PLAUSIBILITY_REQUIRED there is no constraint — the
    function returns True (no evidence of implausibility).

    Why span-only and not sentence?  Because this runs *after* the GAT has
    already seen the sentence-level context; it's a last-resort check for
    obvious contradictions, not a classifier in its own right.
    """
    required = _PLAUSIBILITY_REQUIRED.get(predicted_type)
    if required is None:
        return True   # no rule → assume plausible
    text_lower = entity_text.lower()
    return any(kw in text_lower for kw in required)


def _build_rich_node_embedding(
    text:           str,
    chunk_text:     str,
    start:          int,
    end:            int,
    sentence:       str,
    embedding_model,
    emb_cache:      dict,
    cache_key_fn,
    encode_batch_fn,
) -> np.ndarray:
    """
    Build the rich node feature vector:
        [span_emb (384) | window_emb×0.5 (384) | sentence_hint (27)]
    Total: 795 dimensions.

    Parameters
    ----------
    text            : candidate span text
    chunk_text      : full chunk text the span came from
    start, end      : character offsets of the span within chunk_text
    sentence        : the containing sentence (for hint computation)
    embedding_model : SentenceTransformer instance
    emb_cache       : GraphBuilder._emb_cache dict (shared, mutated in-place)
    cache_key_fn    : GraphBuilder._cache_key method
    encode_batch_fn : GraphBuilder._encode_batch method

    Notes
    -----
    The window is ±50 chars around the span, clamped to chunk boundaries.
    Using encode_batch_fn so both span and window go through the existing
    deduplication + cache logic — no extra model calls for repeated spans.
    """
    # ── Span embedding (384d) ──────────────────────────────────────────────
    span_emb = np.array(encode_batch_fn([text])[0], dtype=np.float32)   # (384,)

    # ── Window embedding (384d) ───────────────────────────────────────────
    w_start = max(0, start - 50)
    w_end   = min(len(chunk_text), end + 50)
    window  = chunk_text[w_start:w_end].strip()

    # If the window is empty or identical to the span, reuse span_emb
    # (avoids an extra model call with no new information)
    if not window or window == text:
        window_emb = span_emb.copy()
    else:
        window_emb = np.array(encode_batch_fn([window])[0], dtype=np.float32)  # (384,)

    # ── Sentence-level type hint (27d) ────────────────────────────────────
    hint = _sentence_type_hint(sentence).numpy()   # (27,)

    # ── Concatenate ───────────────────────────────────────────────────────
    return np.concatenate([span_emb, window_emb * 0.5, hint], axis=0)   # (795,)


In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Parameters
    ----------
    class_weights : pre-computed per-class alpha_t weights (same tensor
                    you previously passed to CrossEntropyLoss).
    gamma         : focusing parameter. gamma=0 → standard CE.
    reduction     : 'none' returns per-node loss (needed for node_weight
                    multiplication in train_epoch).
    """

    def __init__(
        self,
        class_weights: Optional[torch.Tensor] = None,
        gamma:         float                  = 2.0,
        reduction:     str                    = 'none',
    ):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        # Only register a buffer if weights are provided; otherwise, no 'class_weights' attribute exists.
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)

    def forward(
        self,
        logits:  torch.Tensor,   # [N, C]
        targets: torch.Tensor,   # [N]
    ) -> torch.Tensor:
        # Standard CE with class weights (if any)
        weight = getattr(self, 'class_weights', None)
        ce = F.cross_entropy(
            logits,
            targets,
            weight=weight,
            reduction='none',
        )  # [N]

        # p_t = probability assigned to the correct class
        pt = torch.exp(-ce)          # [N], ∈ (0, 1]
        focal_weight = (1.0 - pt) ** self.gamma  # [N]
        loss = focal_weight * ce      # [N]

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss   # 'none'

    def update_weights(self, class_weights: torch.Tensor) -> None:
        """Hot-swap class weights without recreating the loss object."""
        # If the buffer already exists, update its data.
        if hasattr(self, 'class_weights'):
            # Ensure the tensor is on the same device as the existing buffer
            self.class_weights.data = class_weights.to(self.class_weights.device)
        else:
            # Register the buffer for the first time.
            self.register_buffer('class_weights', class_weights)

In [ ]:
class GraphBuilder:
    def __init__(self, embedding_model, entity_labeler=None,
                 max_seq_length: int = 512):
        self.embedding_model   = embedding_model
        self.entity_labeler    = entity_labeler
        self.max_seq_length    = max_seq_length

        # Unified validation threshold — same for training and inference
        self.min_validation_score           = 0.25
        self.min_validation_score_training  = 0.25
        self.min_validation_score_inference = 0.25

        self.num_entity_types    = NUM_ENTITY_TYPES
        self.background_label_id = 26   # NON_ENTITY
        self.chunk_label_id      = -1   # root node, ignored in loss

        self.logger = logger

        # ── Embedding cache ───────────────────────────────────────────────
        # Keyed by the first 200 chars of the text (enough to be unique for
        # short entity strings and negative samples).  Shared across all
        # graph-build calls in a session; caller is responsible for
        # populating / persisting it if cross-session reuse is desired.
        self._emb_cache: Dict[str, np.ndarray] = {}

        # Expanded safe negative words with domain-specific non-entities
        self.safe_negative_words = [
            'the', 'a', 'an', 'and', 'or', 'but', 'nor', 'yet', 'so',
            'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from',
            'into', 'onto', 'as', 'about', 'against', 'between', 'among',
            'through', 'during', 'before', 'after', 'above', 'below',
            'under', 'over',
            'be', 'is', 'are', 'was', 'were', 'been', 'being',
            'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
            'it', 'its', 'they', 'them', 'their', 'this', 'that',
            'these', 'those', 'which', 'who', 'whom', 'whose', 'what',
            'any', 'all', 'each', 'every', 'some', 'no', 'none', 'such',
            'other', 'there', 'here', 'thereof', 'therein', 'hereby',
            'thereby', 'also', 'however', 'therefore', 'thus', 'hence',
            'furthermore', 'moreover',
            # Domain-specific non-entities
            'section', 'figure', 'table', 'appendix', 'annex',
            'paragraph', 'clause', 'subsection', 'chapter', 'part',
        ]

        # Word pattern — accepts plain words and hyphenated terms
        self._word_pattern = re.compile(r'^[a-zA-Z0-9\-]+$')

    # =========================================================================
    # EMBEDDING HELPERS
    # =========================================================================

    def _cache_key(self, text: str) -> str:
        """
        Stable cache key: first 200 chars of the text.
        Entity values and negative samples are almost always shorter than
        200 chars; for long chunk texts the prefix is sufficiently unique.
        """
        return text[:200]

    def _encode_batch(self, texts: List[str]) -> List[np.ndarray]:
        """
        Encode a list of texts, returning one embedding per text.

        * Deduplicates: identical texts are encoded only once.
        * Cache-first: texts already in self._emb_cache skip the model.
        * Remaining texts are encoded in a single model call (batch).

        Returns a list of np.ndarray in the same order as `texts`.
        """
        keys   = [self._cache_key(t) for t in texts]
        needed = []   # (original_index, key, text) for cache-miss items

        for i, (key, text) in enumerate(zip(keys, texts)):
            if key not in self._emb_cache:
                needed.append((i, key, text))

        if needed:
            # Deduplicate by key so each unique string is encoded once
            seen:         Dict[str, int] = {}   # key → index in unique_texts
            unique_texts: List[str]      = []
            unique_keys:  List[str]      = []

            for _, key, text in needed:
                if key not in seen:
                    seen[key] = len(unique_texts)
                    unique_texts.append(text)
                    unique_keys.append(key)

            # Single batched model call
            vectors = self.embedding_model.encode(
                unique_texts,
                batch_size=64,
                show_progress_bar=False,
                convert_to_numpy=True,
            )

            for key, vec in zip(unique_keys, vectors):
                self._emb_cache[key] = vec

        return [self._emb_cache[k] for k in keys]

    # =========================================================================
    # NEGATIVE SAMPLING HELPERS  (unchanged from original)
    # =========================================================================

    def _get_negative_samples(
        self,
        text:        str,
        entities:    List = None,
        num_samples: int  = 3,
    ) -> List[str]:
        entity_texts: set = set()
        entity_spans: set = set()

        if entities:
            for entity in entities:
                if hasattr(entity, 'entity_value'):
                    entity_texts.add(entity.entity_value.lower().strip())
                    for word in entity.entity_value.lower().split():
                        if len(word) > 2:
                            entity_texts.add(word)
                if hasattr(entity, 'start') and hasattr(entity, 'end'):
                    entity_spans.add((entity.start, entity.end))

        negatives:       List[str] = []
        negatives_lower: set       = set()   # PERF: O(1) duplicate check

        words          = text.split()
        word_positions = self._get_word_positions(text)

        if not words:
            return []

        # Strategy 1: safe function words
        words_lower = {w.lower() for w in words}
        for word in self.safe_negative_words:
            if len(negatives) >= num_samples:
                break
            wl = word.lower()
            if wl not in words_lower:
                continue
            if wl in entity_texts:
                continue
            if wl in negatives_lower:          # PERF: was missing; prevents dups
                continue
            if self._word_overlaps_entity(word, word_positions, entity_spans, text):
                continue
            negatives.append(word)
            negatives_lower.add(wl)

        # Strategy 2: random valid single words
        valid_words: List[str] = []
        for w, (start, end) in zip(words, word_positions):
            w_lower = w.lower().strip()
            if (len(w) > 2
                    and w_lower not in entity_texts
                    and w_lower not in self.safe_negative_words
                    and not w.isdigit()
                    and self._word_pattern.match(w)
                    and not self._matches_any_entity_pattern(w)
                    and not self._span_overlaps_entity(start, end, entity_spans)):
                valid_words.append(w)

        needed = num_samples - len(negatives)
        if valid_words and needed > 0:
            import random
            sample_size = min(needed, len(valid_words))
            for w in random.sample(valid_words, sample_size):
                wl = w.lower()
                if wl not in negatives_lower:
                    negatives.append(w)
                    negatives_lower.add(wl)

        # Strategy 3: short 2-3 word phrases from safe zones
        needed = num_samples - len(negatives)
        if needed > 0 and len(words) > 2:
            for i in range(len(words) - 2):
                if len(negatives) >= num_samples:
                    break
                phrase       = ' '.join(words[i:i + 3])
                phrase_start = word_positions[i][0]
                phrase_end   = word_positions[i + 2][1]
                phrase_lower = phrase.lower()

                if phrase_lower in negatives_lower:            # PERF: O(1)
                    continue
                if self._span_overlaps_entity(phrase_start, phrase_end, entity_spans):
                    continue
                if phrase_lower in entity_texts:
                    continue
                if self._matches_any_entity_pattern(phrase):
                    continue
                negatives.append(phrase)
                negatives_lower.add(phrase_lower)

        # Strategy 4: backfill from valid_words
        needed = num_samples - len(negatives)
        if needed > 0 and valid_words:
            for w in valid_words:
                if len(negatives) >= num_samples:
                    break
                if w.lower() not in negatives_lower:           # PERF: O(1) was O(n)
                    negatives.append(w)
                    negatives_lower.add(w.lower())

        return negatives[:num_samples]

    # ------------------------------------------------------------------
    # Span / position helpers  (unchanged)
    # ------------------------------------------------------------------

    def _word_overlaps_entity(self, word: str,
                              word_positions: List[Tuple[int, int]],
                              entity_spans: set,
                              text: str) -> bool:
        if not word or not text:
            return False
        for start, end in word_positions:
            if start < 0 or end > len(text):
                continue
            try:
                if text[start:end] == word:
                    if self._span_overlaps_entity(start, end, entity_spans):
                        return True
            except IndexError:
                continue
        return False

    def _span_overlaps_entity(self, start: int, end: int,
                              entity_spans: set) -> bool:
        for e_start, e_end in entity_spans:
            if max(start, e_start) < min(end, e_end):
                return True
        return False

    def _get_word_positions(self, text: str) -> List[Tuple[int, int]]:
        positions: List[Tuple[int, int]] = []
        pos = 0
        for word in text.split():
            while pos < len(text):
                start = text.find(word, pos)
                if start == -1:
                    pos = len(text)
                    break
                end = start + len(word)
                before_ok = (start == 0 or not text[start - 1].isalnum())
                after_ok  = (end == len(text) or not text[end].isalnum())
                if before_ok and after_ok:
                    positions.append((start, end))
                    pos = end
                    break
                else:
                    pos = start + 1
        return positions

    def _matches_any_entity_pattern(self, text: str) -> bool:
        text_stripped = text.strip()
        if not text_stripped:
            return True
        if not hasattr(self, 'entity_labeler') or self.entity_labeler is None:
            problematic_words = {
                'shall', 'must', 'may', 'might', 'can', 'could', 'should',
                'will', 'would', 'required', 'mandatory', 'necessary',
                'obligatory', 'if', 'when', 'where', 'whenever', 'wherever',
                'unless', 'except', 'provided', 'given', 'subject',
                'prohibited', 'forbidden', 'not', 'banned',
            }
            return text_stripped.lower() in problematic_words
        for entity_type, patterns in self.entity_labeler.compiled_patterns.items():
            for pattern in patterns:
                if pattern.search(text_stripped):
                    return True
        if hasattr(self, '_known_entity_phrases'):
            text_lower = text_stripped.lower()
            for phrase in self._known_entity_phrases:
                if phrase in text_lower or text_lower in phrase:
                    return True
        return False

    def _get_fallback_negative_word(self, chunk_text: str,
                                    entities: List = None) -> str:
        import random
        words          = chunk_text.split()[:50]
        word_positions = self._get_word_positions(chunk_text)[:50]
        entity_texts: set = set()
        entity_spans: set = set()
        if entities:
            for e in entities:
                if hasattr(e, 'entity_value'):
                    entity_texts.add(e.entity_value.lower().strip())
                if hasattr(e, 'start') and hasattr(e, 'end'):
                    entity_spans.add((e.start, e.end))

        def _safe(w: str, start: int, end: int) -> bool:
            return (
                len(w) > 2
                and w.lower() not in entity_texts
                and not self._matches_any_entity_pattern(w)
                and not self._span_overlaps_entity(start, end, entity_spans)
            )

        safe_words = [
            w for w, (s, e) in zip(words, word_positions)
            if _safe(w, s, e) and w.lower() in self.safe_negative_words
        ]
        if safe_words:
            return random.choice(safe_words)

        alpha_words = [
            w for w, (s, e) in zip(words, word_positions)
            if w.isalpha() and _safe(w, s, e)
        ]
        if alpha_words:
            return random.choice(alpha_words)

        any_words = [
            w for w, (s, e) in zip(words, word_positions)
            if _safe(w, s, e)
        ]
        if any_words:
            return random.choice(any_words)

        hard_safe = ['the', 'and', 'for', 'with', 'from', 'that', 'this',
                     'have', 'been', 'section', 'figure', 'table']
        if words:
            candidates = [w for w in words if w.lower() in hard_safe]
            if candidates:
                return random.choice(candidates)
            return random.choice(words)

        return random.choice(hard_safe)

    # =========================================================================
    # TRAINING GRAPH
    # =========================================================================

    def build_graph_for_training(
        self,
        chunk:                Dict,
        entities:             List,
        force_regex_entities: List = None,
        phase:                int  = 0,             # ← NEW: used for phase-scaled ratio
    ) -> Optional['Data']:
        """
        TRAINING MODE: Build a graph from regex-labeled entities + negatives.

        Changes vs. original
        --------------------
        FIX-EMBED : Each entity node now uses a 795-d rich feature vector
                    [span(384) | window×0.5(384) | sentence_hint(27)] instead
                    of a plain 384-d span embedding.  This gives the GAT
                    local syntactic context without changing graph topology.

        FIX-RATIO : Negative sample count is now curriculum-phase-aware.
                    Phase 0: 1:1 entity-to-negative (model learns entity shapes)
                    Phase 1: 1:2 (gradual shift toward real distribution)
                    Phase 2+: 1:3 (approximates inference distribution)
                    This prevents the model from learning to predict NON_ENTITY
                    defensively in early phases when it should be learning what
                    entities look like.

        FIX-HINT  : Sentence-level type hints embedded in each node feature
                    give the GAT a sentence-level prior without extra edges.
        """
        chunk_text = str(chunk.get('text', ''))
        if not chunk_text or len(chunk_text.strip()) < 10:
            return None

        # ── Root node: plain chunk embedding (384d), padded to 795d ──────────
        chunk_emb_raw = np.array(self._encode_batch([chunk_text])[0], dtype=np.float32)
        chunk_emb     = np.concatenate([
            chunk_emb_raw,
            np.zeros(384 + NUM_ENTITY_TYPES, dtype=np.float32),  # window + hint = zeros
        ])

        node_features:  List[np.ndarray]         = [chunk_emb]
        node_labels:    List[int]                = [self.chunk_label_id]
        node_weights:   List[float]              = [0.0]
        node_positions: List[Tuple[int, int]]    = [(0, len(chunk_text))]

        added_spans: set = set()
        added_texts: set = set()
        valid_entities_count = 0

        # ── 1a. Validated entities ────────────────────────────────────────────
        for entity in entities:
            if entity.validation_score < self.min_validation_score:
                continue

            span_key = (entity.start, entity.end)
            text_key = entity.entity_value.lower().strip()

            if span_key in added_spans:
                continue
            if any(self._texts_are_similar(text_key, added) for added in added_texts):
                continue

            # Rich feature: span + window + sentence hint
            feat = _build_rich_node_embedding(
                text            = entity.entity_value,
                chunk_text      = chunk_text,
                start           = entity.start,
                end             = entity.end,
                sentence        = getattr(entity, 'full_sentence', chunk_text),
                embedding_model = self.embedding_model,
                emb_cache       = self._emb_cache,
                cache_key_fn    = self._cache_key,
                encode_batch_fn = self._encode_batch,
            )
            node_features.append(feat)
            node_labels.append(entity.type_id)
            weight = max(0.6, min(1.0, entity.validation_score + 0.2))
            node_weights.append(weight)
            node_positions.append((entity.start, entity.end))

            added_spans.add(span_key)
            added_texts.add(text_key)
            valid_entities_count += 1

        # ── 1b. Force-add regex matches that failed validation ────────────────
        forced_count = 0
        if force_regex_entities:
            for entity in force_regex_entities:
                span_key = (entity.start, entity.end)
                text_key = entity.entity_value.lower().strip()

                if span_key in added_spans:
                    continue
                if any(self._texts_are_similar(text_key, added) for added in added_texts):
                    continue

                feat = _build_rich_node_embedding(
                    text            = entity.entity_value,
                    chunk_text      = chunk_text,
                    start           = entity.start,
                    end             = entity.end,
                    sentence        = getattr(entity, 'full_sentence', chunk_text),
                    embedding_model = self.embedding_model,
                    emb_cache       = self._emb_cache,
                    cache_key_fn    = self._cache_key,
                    encode_batch_fn = self._encode_batch,
                )
                node_features.append(feat)
                node_labels.append(entity.type_id)
                node_weights.append(1.0)
                node_positions.append((entity.start, entity.end))

                added_spans.add(span_key)
                added_texts.add(text_key)
                valid_entities_count += 1
                forced_count += 1

        if forced_count > 0:
            self.logger.debug(
                f"Force-added {forced_count} regex entities that failed validation"
            )

        # ── 2. Negative samples — phase-scaled ratio ──────────────────────────
        # Phase 0 → 1:1  (model learns entity shapes first)
        # Phase 1 → 1:2  (gradual shift)
        # Phase 2+ → 1:3 (approximates inference distribution)
        ratio         = min(3, 1 + phase)
        min_negatives = max(5, valid_entities_count)
        max_negatives = min(60, valid_entities_count * (ratio + 1))
        num_negatives = min(max_negatives, max(min_negatives, valid_entities_count * ratio))

        all_included_entities = list(entities)
        if force_regex_entities:
            all_included_entities.extend(force_regex_entities)

        negatives = self._get_negative_samples(
            chunk_text, all_included_entities, num_samples=num_negatives
        )

        neg_texts_added: set = set()
        for neg_text in negatives:
            neg_text_stripped = neg_text.strip()
            if not neg_text_stripped:
                continue
            text_key = neg_text_stripped.lower()
            if text_key in added_texts or text_key in neg_texts_added:
                continue

            # Negatives: span embedding only; window and hint zeroed out
            neg_emb_raw = np.array(self._encode_batch([neg_text_stripped])[0], dtype=np.float32)
            neg_feat    = np.concatenate([
                neg_emb_raw,
                np.zeros(384 + NUM_ENTITY_TYPES, dtype=np.float32),
            ])
            node_features.append(neg_feat)
            node_labels.append(self.background_label_id)
            node_weights.append(0.3)
            node_positions.append((-1, -1))   # sentinel — no meaningful position
            neg_texts_added.add(text_key)

        # ── 3. Guarantee at least one NON_ENTITY node ─────────────────────────
        has_non_entities = any(lbl == self.background_label_id for lbl in node_labels)
        if not has_non_entities and valid_entities_count > 0:
            fallback = self._get_fallback_negative_word(chunk_text, all_included_entities)
            if fallback and fallback.lower() not in added_texts:
                fb_emb_raw = np.array(self._encode_batch([fallback])[0], dtype=np.float32)
                fb_feat    = np.concatenate([
                    fb_emb_raw,
                    np.zeros(384 + NUM_ENTITY_TYPES, dtype=np.float32),
                ])
                node_features.append(fb_feat)
                node_labels.append(self.background_label_id)
                node_weights.append(0.2)
                node_positions.append((-1, -1))

        if len(node_features) < 2:
            return None

        # ── 4. Edges (unchanged topology) ────────────────────────────────────
        edge_index:    List = []
        edge_weights_: List = []

        for i in range(1, len(node_features)):
            edge_index.append([0, i])
            edge_index.append([i, 0])
            edge_weights_.extend([1.0, 1.0])

        for i in range(len(node_features)):
            edge_index.append([i, i])
            edge_weights_.append(1.0)

        for i in range(1, len(node_features)):
            pos_i = node_positions[i]
            if pos_i[0] == -1:
                continue
            for j in range(i + 1, len(node_features)):
                pos_j = node_positions[j]
                if pos_j[0] == -1:
                    continue
                distance = min(
                    abs(pos_i[1] - pos_j[0]),
                    abs(pos_j[1] - pos_i[0]),
                )
                if distance < 100:
                    weight = 1.0 - (distance / 100.0)
                    edge_index.append([i, j])
                    edge_index.append([j, i])
                    edge_weights_.extend([weight, weight])

        # ── 5. Convert to tensors ─────────────────────────────────────────────
        x = torch.tensor(np.array(node_features, dtype=np.float32), dtype=torch.float)
        x = F.normalize(x, p=2, dim=1)

        edge_index_t = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr    = torch.tensor(edge_weights_, dtype=torch.float).view(-1, 1)

        y           = torch.tensor(node_labels,  dtype=torch.long)
        node_weight = torch.tensor(node_weights, dtype=torch.float)

        from torch_geometric.data import Data
        return Data(
            x=x,
            edge_index=edge_index_t,
            edge_attr=edge_attr,
            y=y,
            node_weight=node_weight,
        )

    def _texts_are_similar(self, text1: str, text2: str,
                           threshold: float = 0.8) -> bool:
        """Return True if two texts are near-duplicates by word-Jaccard."""
        if text1 == text2:
            return True
        words1 = set(text1.split())
        words2 = set(text2.split())
        if not words1 or not words2:
            return False
        return len(words1 & words2) / len(words1 | words2) >= threshold

    # =========================================================================
    # INFERENCE GRAPH
    # =========================================================================

    def build_graph_for_inference(
        self,
        chunk:      Dict,
        candidates: List[Dict],
    ) -> Optional['Data']:
        """
        INFERENCE MODE: Build graph from candidate phrases.

        Changes vs. original
        --------------------
        FIX-EMBED : Each candidate node now carries the 795-d rich feature
                    [span(384) | window×0.5(384) | sentence_hint(27)].
                    Matches the feature space the model was trained on after
                    applying build_graph_for_training with the same fixes.

        FIX-HINT  : Sentence-level type hints baked into node features so
                    GAT doesn't need cross-chunk graph edges for context.

        All filtering, priority ordering, and edge-building logic is
        unchanged from the original.
        """
        chunk_text = str(chunk.get('text', ''))
        if not chunk_text or len(chunk_text.strip()) < 10:
            return None

        if not candidates:
            return None

        PRIORITY_SOURCES = {'training_aligned', 'measurement_regex'}

        # ── Filter empty / punctuation-only candidates ────────────────────────
        valid_candidates = []
        for c in candidates:
            text = c.get('text', '').strip()
            if not text or len(text) < 2:
                continue
            if text.isdigit() or all(not ch.isalnum() for ch in text):
                continue
            valid_candidates.append(c)

        if not valid_candidates:
            return None

        # ── Separate priority vs novel; drop novel that overlap priority ───────
        priority_candidates = [
            c for c in valid_candidates if c.get('source') in PRIORITY_SOURCES
        ]
        novel_candidates_raw = [
            c for c in valid_candidates if c.get('source') not in PRIORITY_SOURCES
        ]

        priority_spans = [
            (c.get('start', 0), c.get('end', 0)) for c in priority_candidates
        ]

        def overlaps_priority(c: Dict) -> bool:
            cs, ce = c.get('start', 0), c.get('end', 0)
            return any(cs < pe and ps < ce for ps, pe in priority_spans)

        novel_candidates = [
            c for c in novel_candidates_raw if not overlaps_priority(c)
        ]

        priority_candidates.sort(key=lambda x: x.get('start', 0))
        novel_candidates.sort(key=lambda x: x.get('start', 0))
        ordered_candidates = priority_candidates + novel_candidates

        # ── Pre-build spaCy doc once for sentence boundary queries ────────────
        # We need sentence boundaries to extract the containing sentence for
        # each candidate (required for _sentence_type_hint).
        try:
            doc = self.entity_labeler.nlp(chunk_text) if self.entity_labeler else None
            sent_boundaries = (
                [(s.start_char, s.end_char) for s in doc.sents] if doc else []
            )
        except Exception:
            doc             = None
            sent_boundaries = []

        def _containing_sentence(start: int, end: int) -> str:
            """Return the sentence that contains (start, end), or empty string."""
            for s_start, s_end in sent_boundaries:
                if s_start <= start and end <= s_end:
                    return chunk_text[s_start:s_end]
            # Fallback: ±150 chars around the span
            return chunk_text[max(0, start - 150): min(len(chunk_text), end + 150)]

        # ── Root node (795-d, window+hint=zeros) ─────────────────────────────
        chunk_emb_raw = np.array(self._encode_batch([chunk_text])[0], dtype=np.float32)
        chunk_emb     = np.concatenate([
            chunk_emb_raw,
            np.zeros(384 + NUM_ENTITY_TYPES, dtype=np.float32),
        ])

        node_features:      List[np.ndarray]    = [chunk_emb]
        node_positions:     List[Tuple[int,int]]= [(0, len(chunk_text))]
        candidate_metadata: List[Dict]          = []

        # ── Candidate nodes (795-d rich feature) ─────────────────────────────
        for candidate in ordered_candidates:
            c_start   = candidate.get('start', 0)
            c_end     = candidate.get('end',   0)
            sentence  = _containing_sentence(c_start, c_end)

            feat = _build_rich_node_embedding(
                text            = candidate['text'],
                chunk_text      = chunk_text,
                start           = c_start,
                end             = c_end,
                sentence        = sentence,
                embedding_model = self.embedding_model,
                emb_cache       = self._emb_cache,
                cache_key_fn    = self._cache_key,
                encode_batch_fn = self._encode_batch,
            )
            node_features.append(feat)
            node_positions.append((c_start, c_end))
            candidate_metadata.append(candidate)

        if len(node_features) < 2:
            return None

        # ── Edges (unchanged topology from original) ──────────────────────────
        edge_index:    List = []
        edge_weights_: List = []

        # Root connections
        for i in range(1, len(node_features)):
            edge_index.append([0, i])
            edge_index.append([i, 0])
            edge_weights_.extend([1.0, 1.0])

        # Self-loops
        for i in range(len(node_features)):
            edge_index.append([i, i])
            edge_weights_.append(1.0)

        # Proximity edges between nearby candidates
        for i in range(1, len(node_features)):
            for j in range(i + 1, len(node_features)):
                pos_i    = node_positions[i]
                pos_j    = node_positions[j]
                distance = min(
                    abs(pos_i[1] - pos_j[0]),
                    abs(pos_j[1] - pos_i[0]),
                )
                if distance < 100:
                    weight = 1.0 - (distance / 100.0)
                    edge_index.append([i, j])
                    edge_index.append([j, i])
                    edge_weights_.extend([weight, weight])

        # ── Convert to tensors ────────────────────────────────────────────────
        x = torch.tensor(np.array(node_features, dtype=np.float32), dtype=torch.float)
        x = F.normalize(x, p=2, dim=1)

        edge_index_t = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr    = torch.tensor(edge_weights_, dtype=torch.float).view(-1, 1)

        from torch_geometric.data import Data
        data                    = Data(x=x, edge_index=edge_index_t, edge_attr=edge_attr)
        data.candidate_metadata = candidate_metadata
        data.node_positions     = node_positions

        data.num_priority         = len(priority_candidates)
        data.num_novel            = len(novel_candidates)
        data.num_novel_dropped    = len(novel_candidates_raw) - len(novel_candidates)
        data.num_training_aligned = sum(
            1 for c in priority_candidates if c.get('source') == 'training_aligned'
        )

        return data

### GNN MODEL

In [ ]:
class RegulationGAT(nn.Module):
    def __init__(
        self,
        input_dim:     int,
        hidden_dim:    int,
        num_classes:   int,
        num_heads:     int   = 4,
        dropout:       float = 0.3,
        num_layers:    int   = 4,
        non_entity_id: int   = 26,
    ):
        super().__init__()

        self.num_layers    = num_layers
        self.hidden_dim    = hidden_dim
        self.input_dim     = input_dim
        self.num_classes   = num_classes
        self.non_entity_id = non_entity_id

        # Input projection
        self.input_proj    = nn.Linear(input_dim, hidden_dim)
        self.input_norm    = nn.LayerNorm(hidden_dim)
        self.input_dropout = nn.Dropout(dropout * 0.5)

        # GAT layers
        self.convs         = nn.ModuleList()
        self.norms         = nn.ModuleList()
        self.residual_proj = nn.ModuleList()

        for i in range(num_layers):
            self.convs.append(
                GATConv(
                    hidden_dim, hidden_dim,
                    heads=num_heads,
                    dropout=dropout,
                    concat=False,
                    negative_slope=0.2,
                )
            )
            self.norms.append(nn.LayerNorm(hidden_dim))
            self.residual_proj.append(nn.Identity())

        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

        self.dropout = nn.Dropout(dropout)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout * 0.3),
            nn.Linear(hidden_dim // 2, num_classes),
        )

        # FIX-A: calibrator now operates on PROBABILITIES (0–1), not raw
        # logits.  Input is the softmax probability vector over the
        # non_entity_id entity classes; Softplus(prob) ∈ (0.69, 1.31) for
        # inputs in [0, 1], giving well-behaved multiplicative scaling.
        # Previously the linear layer took raw logits (−∞ to +∞), making
        # Softplus produce wildly varying factors (≈0 for logit=−5, ≈5 for
        # logit=+5) with no principled relationship to confidence.
        self.confidence_calibrator = nn.Sequential(
            nn.Linear(non_entity_id, non_entity_id),
            nn.Softplus(),
        )

        self._init_weights()

    def _init_weights(self):
        """Xavier init with smaller gain for the final classification layer."""
        final_linear = self.classifier[-1]
        for m in self.modules():
            if isinstance(m, nn.Linear):
                gain = 0.1 if m is final_linear else 1.0
                nn.init.xavier_uniform_(m.weight, gain=gain)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    # ---------------------------------------------------------------------- #
    # Internal helper: shared projection + GAT stack                         #
    # ---------------------------------------------------------------------- #

    def _encode(self, x: torch.Tensor, edge_index: torch.Tensor,
                return_attn: bool = False):
        if x.size(1) != self.input_dim:
            raise ValueError(
                f"Input feature dim mismatch: expected {self.input_dim}, "
                f"got {x.size(1)}"
            )

        x = self.input_proj(x)
        x = self.input_norm(x)
        x = self.input_dropout(x)

        attention_list = []

        for conv, norm, res_proj in zip(self.convs, self.norms, self.residual_proj):
            x_residual = res_proj(x)

            if return_attn:
                x, (edge_idx, attn) = conv(
                    x, edge_index, return_attention_weights=True
                )
                attention_list.append((edge_idx, attn))
            else:
                x = conv(x, edge_index)

            x = norm(x)

            if x.shape == x_residual.shape:
                x = x + x_residual
            else:
                raise RuntimeError(
                    f"Residual shape mismatch: {x.shape} vs {x_residual.shape}."
                )

            x = F.elu(x)
            x = self.dropout(x)

        return x, attention_list

    # ---------------------------------------------------------------------- #
    # Forward                                                                  #
    # ---------------------------------------------------------------------- #

    def forward(
        self,
        data,
        return_attention_weights: bool = False,
        return_embeddings:        bool = False,
    ):
        x          = data.x
        edge_index = data.edge_index
        batch      = getattr(data, 'batch', None)

        x, attention_list = self._encode(
            x, edge_index, return_attn=return_attention_weights
        )

        if batch is not None:
            attention_scores = self.attention_pool(x)
            attn_exp         = torch.exp(
                attention_scores - attention_scores.max()
            )
            denom = torch.zeros_like(attn_exp).scatter_add_(
                0,
                batch.unsqueeze(1).expand_as(attn_exp),
                attn_exp,
            ).clamp(min=1e-10)
            self.last_attention_weights = attn_exp / denom

        logits = self.classifier(x)                            # [N, num_classes]

        # FIX-A: compute softmax probabilities over entity classes FIRST,
        # then feed those probabilities to the calibrator.
        # The old code fed raw logits, making Softplus() output range from
        # ≈0 to ≈5 depending solely on logit magnitude — not meaningful.
        # With probabilities in [0, 1]: Softplus(p) ∈ (0.693, 1.313),
        # so calibration factors are bounded and interpretable.
        entity_logits_raw = logits[:, :self.non_entity_id]     # [N, K]
        entity_probs      = F.softmax(entity_logits_raw, dim=-1)  # [N, K]  ← FIX
        calibration_factors = self.confidence_calibrator(entity_probs)  # [N, K]
        calibrated_entity   = entity_logits_raw * calibration_factors   # [N, K]

        logits = torch.cat(
            [
                calibrated_entity,
                logits[:, self.non_entity_id:],   # NON_ENTITY logit unchanged
            ],
            dim=1,
        )

        if not self.training:
            entity_logits    = logits[:, :self.non_entity_id]                # [N, K]
            max_entity_logit = entity_logits.max(dim=1, keepdim=True)[0]     # [N, 1]
            non_entity_logit = logits[:, self.non_entity_id:self.non_entity_id + 1]  # [N, 1]

            # FIX-B: threshold raised from 0.3 to 1.5.
            # The old threshold (0.3) fired whenever any entity logit exceeded
            # 30% of the NON_ENTITY logit — which is almost always, even for
            # nodes that should be confidently rejected.  This made the
            # NON_ENTITY suppression apply universally, inflating entity
            # probabilities for every node.
            # At 1.5 the suppression only activates when the best entity logit
            # is 1.5× the NON_ENTITY logit, i.e. when the model has already
            # committed to an entity prediction.
            strong_entity_mask = (
                max_entity_logit > 1.5 * non_entity_logit          # ← FIX
            ).squeeze(-1)                                            # [N]

            suppressed = logits[:, self.non_entity_id] * 0.5
            new_non_entity = torch.where(
                strong_entity_mask,
                suppressed,
                logits[:, self.non_entity_id],
            )
            logits = torch.cat(
                [
                    logits[:, :self.non_entity_id],
                    new_non_entity.unsqueeze(1),
                    logits[:, self.non_entity_id + 1:],
                ],
                dim=1,
            )

        outputs = [logits]
        if return_attention_weights:
            outputs.append(attention_list)
        if return_embeddings:
            outputs.append(x)

        return outputs[0] if len(outputs) == 1 else tuple(outputs)

    # ---------------------------------------------------------------------- #
    # Convenience methods                                                      #
    # ---------------------------------------------------------------------- #

    def get_embeddings(self, data) -> torch.Tensor:
        _, embeddings = self.forward(data, return_embeddings=True)
        return embeddings

    def get_predictions_with_confidence(self, data) -> dict:
        # FIX-A (downstream): forward() now produces well-calibrated logits
        # because the calibrator operated on probabilities.  softmax here
        # therefore produces meaningful confidence values.
        logits = self.forward(data)
        probs  = F.softmax(logits, dim=-1)

        preds      = probs.argmax(dim=-1)
        confidence = probs.max(dim=-1)[0]

        temperature           = 1.5
        calibrated_probs      = F.softmax(logits / temperature, dim=-1)
        calibrated_confidence = calibrated_probs.max(dim=-1)[0]

        return {
            'predictions':              preds,
            'confidence':               confidence,
            'calibrated_confidence':    calibrated_confidence,
            'probabilities':            probs,
            'calibrated_probabilities': calibrated_probs,
        }

    def get_attention_patterns(self, data, layer_idx: int = -1):
        _, attention_list = self._encode(
            data.x, data.edge_index, return_attn=True
        )
        if not attention_list:
            return None, None
        edge_idx, attn = attention_list[layer_idx]
        return edge_idx, attn

    def get_gradient_flow(self) -> dict:
        grad_stats = {}
        for name, param in self.named_parameters():
            if param.grad is not None:
                grad_stats[name] = {
                    'mean':    param.grad.abs().mean().item(),
                    'std':     param.grad.std().item(),
                    'max':     param.grad.abs().max().item(),
                    'nonzero': (param.grad != 0).float().mean().item(),
                }
            else:
                grad_stats[name] = None
        return grad_stats

### TRAINING

In [ ]:
class EntityDifficultyScorer:
    """
    Score entity difficulty based on multiple factors (0 = easy, 1 = hard).

    """

    def __init__(self):
        self.type_difficulty: Dict[str, float] = {
            # Very Easy: clear numeric patterns
            'DIMENSION':            0.05,
            'AREA':                 0.05,
            'VOLUME':               0.05,
            'PERCENTAGE':           0.08,

            # Easy: standard measurements
            'CAPACITY':             0.15,
            'PRESSURE':             0.15,
            'FLOW_RATE':            0.15,
            'TEMPERATURE':          0.15,
            'EGRESS_DISTANCE':      0.20,
            'SEPARATION_DISTANCE':  0.20,
            'CLEARANCE':            0.20,

            # Medium: requires context
            'FIRE_RATING':          0.30,
            'MATERIAL':             0.35,
            'COMPONENT':            0.35,
            'EQUIPMENT':            0.40,
            'STANDARD':             0.40,
            'BUILDING_TYPE':        0.45,
            'OCCUPANCY':            0.45,

            # Hard: semantic understanding needed
            'PENETRATION':          0.60,
            'HAZARD_LEVEL':         0.60,
            'FACILITY':             0.65,
            'LOCATION':             0.65,

            # Very Hard: complex context
            'AUTHORITY':            0.70,
            'CERTIFICATION':        0.70,
            'INSPECTION_FREQUENCY': 0.75,
            'TESTING_REQUIREMENT':  0.75,
        }

        self.complexity_patterns = {
            'exception': re.compile(r'\b(?:except|unless|however|but|notwithstanding)\b', re.I),
            'condition': re.compile(r'\b(?:if|when|where|provided|subject to)\b', re.I),
            'reference': re.compile(r'\b(?:per|according to|as per|in accordance with)\b', re.I),
            'multiple':  re.compile(r'\b(?:and/or|and|or|both|either)\b', re.I),
        }

    def score_entity(self, entity) -> float:
        """
        Calculate difficulty score for an entity.

        Factors
        -------
        1. Entity type base difficulty  (30% weight)  ← reduced from 40%
        2. Validation score (inverted)  (25% weight)  ← increased from 20%
        3. Confidence (inverted)        (25% weight)  ← increased from 20%
        4. Semantic-context quality     (10% weight)
        5. Text complexity              (10% weight)

        TUNING CHANGES vs previous version:
        - Clamp range widened [0.2, 0.8] → [0.1, 0.9]: labeler quality is now
          higher after regex hardening, so extreme scores carry real signal and
          should sharpen phase separation rather than being flattened away.
        - Weight redistribution: type_difficulty weight reduced from 0.40 → 0.30,
          and the freed 0.10 split equally between validation_score and confidence
          (each 0.20 → 0.25).  Runtime signals now outweigh the static type table,
          making curriculum ordering adapt to actual model behaviour rather than
          relying on hand-coded priors that may not match your corpus distribution.
        - Fallback defaults are now corpus-aware: validation_score defaults to 0.4
          (slightly pessimistic) instead of 0.5; confidence defaults to 0.4 for
          the same reason.  Entities that have never been validated should lean
          toward harder phases, not sit at the neutral midpoint.
        - Added a rarity_bonus: if the entity type is rare in the training set
          (bottom 20% by frequency), difficulty is nudged up by 0.05 so rare
          classes surface earlier in harder phases and receive more training
          signal.  self._rare_types must be set by CurriculumDataLoader after
          class_distribution is computed (see companion fix).
        """
        scores:  List[float] = []
        weights: List[float] = []

        # 1. Type-based difficulty (default 0.5 for unknown types)
        type_difficulty = self.type_difficulty.get(entity.entity_type, 0.5)
        scores.append(type_difficulty)
        weights.append(0.30)   # ← was 0.40

        # 2. Validation score — low validation = high difficulty
        #    Clamp widened to [0.1, 0.9]: post-hardening, extreme labeler scores
        #    are genuine signals, not artefacts.  Default 0.4 (pessimistic) so
        #    unseen entities lean toward harder phases.
        raw_validation    = float(getattr(entity, 'validation_score', 0.4))  # ← was 0.5
        raw_validation    = max(0.1, min(0.9, raw_validation))               # ← was [0.2, 0.8]
        validation_factor = 1.0 - raw_validation
        scores.append(validation_factor)
        weights.append(0.25)   # ← was 0.20

        # 3. Confidence — low confidence = high difficulty
        #    Same rationale as validation_score above.
        raw_confidence    = float(getattr(entity, 'confidence', 0.4))        # ← was 0.5
        raw_confidence    = max(0.1, min(0.9, raw_confidence))               # ← was [0.2, 0.8]
        confidence_factor = 1.0 - raw_confidence
        scores.append(confidence_factor)
        weights.append(0.25)   # ← was 0.20

        # 4. Context quality
        context = getattr(entity, 'semantic_context', {})
        if isinstance(context, dict):
            context_complexity = self._score_context_complexity(context)
            scores.append(context_complexity)
            weights.append(0.1)

        # 5. Text complexity
        text_complexity = self._score_text_complexity(entity.entity_value)
        scores.append(text_complexity)
        weights.append(0.1)

        # Weighted average
        total_weight = sum(weights)
        final_score  = sum(s * w for s, w in zip(scores, weights)) / total_weight

        # 6. Rarity bonus — nudge rare entity types toward harder phases so they
        #    accumulate more training signal.  _rare_types is populated by
        #    CurriculumDataLoader._compute_class_distribution() after the train
        #    split is known; guard with getattr so the scorer still works standalone.
        rare_types = getattr(self, '_rare_types', set())
        if entity.entity_type in rare_types:
            final_score = min(1.0, final_score + 0.05)

        return float(min(1.0, max(0.0, final_score)))

    def _score_context_complexity(self, context: Dict) -> float:
        """Score how complex the semantic context is."""
        complexity = 0.3  # Base complexity

        full_sentence = context.get('full_sentence', '')
        for pattern_name, pattern in self.complexity_patterns.items():
            if pattern.search(full_sentence):
                complexity += 0.1

        modifiers = context.get('modifiers', [])
        if len(modifiers) > 2:
            complexity += 0.1 * min(1.0, (len(modifiers) - 2) / 3)

        if context.get('condition'):
            complexity += 0.2

        if context.get('negation'):
            complexity += 0.15

        return min(1.0, complexity)

    def _score_text_complexity(self, text: str) -> float:
        """Score complexity based on text characteristics."""
        words = text.split()

        if len(words) == 1:
            return 0.1

        length_factor = min(1.0, len(words) / 10)
        hyphenated    = sum(1 for w in words if '-' in w) / len(words)
        has_numbers   = any(c.isdigit() for c in text)

        complexity  = length_factor * 0.5
        complexity += hyphenated * 0.2

        if has_numbers:
            complexity *= 0.8  # Numbers make identification easier

        return min(1.0, complexity)

    def score_graph(self, graph, entities: list) -> float:
        """
        Score entire graph difficulty (weighted mean of entity difficulties).

        FIX 5: Floor repetition weight at 1 so that short entities such as
        "2 hr" or "R-13" (length < 20) are never silently omitted from the
        difficulty average. Previously int(weight * 10) could be 0 for
        single-character or very short values, dropping them entirely.
        """
        if not entities:
            return 0.5

        entity_scores: List[float] = []
        for e in entities:
            score = self.score_entity(e)
            repetitions = max(1, int(min(2.0, len(e.entity_value) / 20) * 10))
            entity_scores.extend([score] * repetitions)

        return sum(entity_scores) / len(entity_scores) if entity_scores else 0.5

In [ ]:
class CurriculumDataLoader:
    """
    Organise training graphs into difficulty-based phases for curriculum
    learning with improved difficulty scoring and adaptive phase sizing.
    """

    def __init__(self,
                graphs:              list,
                entities_per_graph:  list,
                batch_size:          int  = 8,
                num_phases:          int  = 4,
                adaptive_phases:     bool = True,
                background_label_id: int  = 26,
                train_indices:       list = None):
        """
        FIX (leakage): Accept explicit train_indices so that class_distribution
        and difficulty scoring are computed exclusively on training graphs.
        If train_indices is None we assume ALL graphs are training graphs
        (caller's responsibility to pass only training data).
        """
        self.graphs              = graphs
        self.entities_per_graph  = entities_per_graph
        self.batch_size          = batch_size
        self.num_phases          = num_phases
        self.adaptive_phases     = adaptive_phases
        self.background_label_id = background_label_id

        # FIX: restrict every stat-building operation to training indices only
        if train_indices is not None:
            train_idx_set = set(train_indices)
            self._train_graphs   = [graphs[i]             for i in train_indices]
            self._train_entities = [entities_per_graph[i] for i in train_indices]
        else:
            # Caller passed only training graphs — use everything
            train_idx_set        = set(range(len(graphs)))
            self._train_graphs   = graphs
            self._train_entities = entities_per_graph

        self.difficulty_scorer = EntityDifficultyScorer()

        self.class_distribution = self._compute_class_distribution()
        if self.class_distribution:
            freq_threshold = sorted(self.class_distribution.values())[
                max(0, int(len(self.class_distribution) * 0.20) - 1)
            ]
            rare_type_ids = {
                type_id for type_id, count in self.class_distribution.items()
                if count <= freq_threshold
            }
            self.difficulty_scorer._rare_types = {
                ENTITY_ID_TO_TYPE[tid] for tid in rare_type_ids
                if tid in ENTITY_ID_TO_TYPE
            }
        else:
            self.difficulty_scorer._rare_types = set()

        self.graph_scores       = self._score_all_graphs()
        self.sorted_indices     = self._create_diverse_sorted_indices()
        self._define_phase_boundaries()

    # -------------------------------------------------------------------------
    # Internal helpers
    # -------------------------------------------------------------------------

    def _compute_class_distribution(self) -> Dict[int, int]:
        """
        Compute distribution of entity classes across TRAINING graphs only.

        FIX (leakage #1): previously iterated self.entities_per_graph which
        includes test graphs.  Now uses self._train_entities.
        """
        distribution: Dict[int, int] = defaultdict(int)
        for entities in self._train_entities:          # FIX: train only
            for entity in entities:
                if hasattr(entity, 'type_id'):
                    distribution[entity.type_id] += 1
                elif hasattr(entity, 'type'):
                    try:
                        type_id = REGULATION_ENTITY_TYPES.get(
                            entity.get('type'), self.background_label_id
                        )
                        distribution[type_id] += 1
                    except Exception:
                        pass
        return dict(distribution)

    def _score_all_graphs(self) -> np.ndarray:
        """
        Score graphs with entity-weighted difficulty.

        FIX-DENSITY: Add a NON_ENTITY density component to each graph's
        difficulty score.  Previously scores were based purely on entity
        difficulty, making the loader blind to graphs where negative nodes
        heavily outnumber entity nodes.  Such graphs are harder for the
        model (it must learn to reject most nodes) and should appear in
        later phases.

        The density penalty is:
          non_entity_ratio = non_entity_nodes / total_nodes
          density_penalty  = max(0, non_entity_ratio - 0.5) * 0.3

        This is zero when NON_ENTITY nodes are ≤50% of the graph (easy),
        and rises linearly to 0.15 when NON_ENTITY nodes are 100% (which
        cannot happen in practice but sets the scale).  The 0.3 coefficient
        keeps the penalty modest so it adjusts phase ordering without
        overriding entity-type difficulty entirely.
        """
        scores: List[float] = []

        for graph, entities in zip(self.graphs, self.entities_per_graph):
            base_score = self.difficulty_scorer.score_graph(graph, entities)

            entity_types: List[int] = []
            for entity in entities:
                if hasattr(entity, 'type_id'):
                    entity_types.append(entity.type_id)
                elif hasattr(entity, 'type'):
                    try:
                        type_id = REGULATION_ENTITY_TYPES.get(
                            entity.get('type'), self.background_label_id
                        )
                        entity_types.append(type_id)
                    except Exception:
                        pass

            if entity_types:
                rare_penalty = self._compute_rare_type_penalty(entity_types)
                base_score   = min(1.0, base_score + rare_penalty * 0.2)

            # FIX-DENSITY: compute NON_ENTITY node density from the actual
            # graph tensor labels, not just from the entity list.
            # graph.y contains the ground-truth labels for ALL nodes including
            # negative samples added by GraphBuilder.  The entity list only
            # contains positive entities — it has no visibility into negatives.
            density_penalty = 0.0
            if graph is not None and hasattr(graph, 'y'):
                y            = graph.y
                valid_mask   = y >= 0   # exclude chunk root (-1)
                total_nodes  = valid_mask.sum().item()
                if total_nodes > 0:
                    non_entity_nodes  = (y[valid_mask] == self.background_label_id).sum().item()
                    non_entity_ratio  = non_entity_nodes / total_nodes
                    # Penalty starts accumulating above 50% NON_ENTITY density
                    # and is capped at 0.15 (coefficient 0.3 × max excess 0.5)
                    density_penalty = max(0.0, non_entity_ratio - 0.5) * 0.3

            scores.append(min(1.0, base_score + density_penalty))

        return np.array(scores)



    def _compute_rare_type_penalty(self, entity_types: List[int]) -> float:
        """Compute penalty for graphs containing rare entity types."""
        if not entity_types or not self.class_distribution:
            return 0.0
        try:
            min_count = min(
                self.class_distribution.get(t, 0) for t in entity_types
            )
            max_count = max(self.class_distribution.values())
            return 1.0 - (min_count / max_count) if max_count > 0 else 0.0
        except (ValueError, ZeroDivisionError):
            return 0.0

    def _entity_type_id(self, entity) -> int:
        """Safely resolve an entity to its integer type_id."""
        if hasattr(entity, 'type_id'):
            return entity.type_id
        if hasattr(entity, 'type'):
            try:
                return REGULATION_ENTITY_TYPES.get(
                    entity.get('type'), self.background_label_id
                )
            except Exception:
                pass
        return self.background_label_id

    def _create_diverse_sorted_indices(self) -> np.ndarray:
        """
        Create sorted indices ensuring all entity classes appear in early phases.

        FIX (leakage #3): rare_classes threshold is now derived from
        self.class_distribution (training-only).  Sorting and phase
        construction operate on training graph indices only so test graphs
        are never considered when building the curriculum order.

        FIX 4 (original): Uses NUM_ENTITY_TYPES to define all_entity_classes.
        FIX 5 (original): floor repetition weight at 1 for short entity values.
        """
        # FIX: work on training indices only
        train_indices = [
            i for i in range(len(self.graphs))
            if i in set(range(len(self._train_graphs)))   # only training graphs
        ]
        total = len(train_indices)

        if total == 0:
            return np.array([], dtype=int)

        # Build (index, score, set-of-entity-classes) for every TRAINING graph
        graph_info: List[Tuple[int, float, set]] = []
        for idx in train_indices:
            classes = {
                self._entity_type_id(e)
                for e in self.entities_per_graph[idx]
                if self._entity_type_id(e) != self.background_label_id
            }
            graph_info.append((idx, float(self.graph_scores[idx]), classes))

        # Sort easiest-first globally
        graph_info.sort(key=lambda x: x[1])

        # FIX 4: correct entity class universe — 0 .. NUM_ENTITY_TYPES-1
        all_entity_classes = set(range(NUM_ENTITY_TYPES))

        # FIX (leakage #3): rare threshold uses training distribution only
        rare_classes: set = {
            c for c, count in self.class_distribution.items()   # training-only dist
            if c != self.background_label_id and count < 100
        }

        phase1_set:  set        = set()
        phase1_list: List[int]  = []
        seen_classes: set       = set()

        # --- Priority 1: graphs that contain rare classes --------------------
        rare_graphs = [
            (idx, score, classes)
            for idx, score, classes in graph_info
            if classes & rare_classes
        ]
        rare_graphs.sort(key=lambda x: (-len(x[2] & rare_classes), x[1]))

        for idx, score, classes in rare_graphs:
            if idx not in phase1_set:
                phase1_set.add(idx)
                phase1_list.append(idx)
                seen_classes.update(classes)

        # --- Priority 2: cover any still-missing entity classes --------------
        missing_classes = all_entity_classes - seen_classes
        if missing_classes:
            for idx, score, classes in graph_info:
                if not missing_classes:
                    break
                if classes & missing_classes and idx not in phase1_set:
                    phase1_set.add(idx)
                    phase1_list.append(idx)
                    seen_classes.update(classes)
                    missing_classes = all_entity_classes - seen_classes

        # --- Fill Phase 1 to 25% of total with easiest remaining -------------
        phase1_target = max(1, int(total * 0.25))
        for idx, score, classes in graph_info:
            if len(phase1_list) >= phase1_target:
                break
            if idx not in phase1_set:
                phase1_set.add(idx)
                phase1_list.append(idx)

        # Sort Phase 1 by difficulty (easiest first)
        phase1_list.sort(key=lambda i: self.graph_scores[i])

        # Remaining training graphs sorted by difficulty
        remaining = [
            idx for idx, _, _ in graph_info
            if idx not in phase1_set
        ]
        remaining.sort(key=lambda i: self.graph_scores[i])

        return np.array(phase1_list + remaining, dtype=int)

    def _define_phase_boundaries(self) -> None:
        """Define phase boundaries with adaptive sizing."""
        total = len(self.graphs)

        if total == 0:
            self.phase_boundaries = [0] * self.num_phases
            return

        # 25 / 50 / 75 / 100 %  (adaptive_phases flag kept for API compat)
        fractions = [0.25, 0.50, 0.75, 1.00]
        self.phase_boundaries = [
            max(1, int(total * f)) for f in fractions
        ]
        # Guarantee the last boundary covers every graph
        self.phase_boundaries[-1] = total

        # Ensure monotonically non-decreasing
        for i in range(1, len(self.phase_boundaries)):
            if self.phase_boundaries[i] < self.phase_boundaries[i - 1]:
                self.phase_boundaries[i] = self.phase_boundaries[i - 1]

    # -------------------------------------------------------------------------
    # Public API
    # -------------------------------------------------------------------------
    def get_phase_loader(self, phase: int, shuffle: bool = True):
        """
        Return a DataLoader for the requested curriculum phase.

        FIX-DENSITY: Print NON_ENTITY density stats per phase so training
        logs expose whether early phases are dominated by entity-sparse graphs.
        No functional change — diagnostic addition only.
        """
        if phase < 0 or phase >= len(self.phase_boundaries):
            raise ValueError(
                f"Phase must be 0-{len(self.phase_boundaries) - 1}, got {phase}"
            )

        num_graphs    = self.phase_boundaries[phase]
        phase_indices = self.sorted_indices[:num_graphs]
        phase_graphs  = [self.graphs[i] for i in phase_indices]

        entity_counts:  Dict[int, int] = defaultdict(int)
        unique_classes: set            = set()
        # FIX-DENSITY: track actual node-level NON_ENTITY stats
        total_nodes_all    = 0
        non_entity_nodes_all = 0

        for i in phase_indices:
            for entity in self.entities_per_graph[i]:
                tid = self._entity_type_id(entity)
                entity_counts[tid] += 1
                unique_classes.add(tid)

            # FIX-DENSITY: count from graph tensor labels for accuracy
            g = self.graphs[i]
            if hasattr(g, 'y'):
                valid_mask = g.y >= 0
                total_nodes_all += valid_mask.sum().item()
                non_entity_nodes_all += (
                    g.y[valid_mask] == self.background_label_id
                ).sum().item()

        print(f"\n📖 Phase {phase + 1}: Using {num_graphs} graphs")
        if len(phase_indices) > 0:
            print(
                f"   Difficulty range: "
                f"{self.graph_scores[phase_indices[0]]:.3f} – "
                f"{self.graph_scores[phase_indices[-1]]:.3f}"
            )
            print(f"   Unique classes in phase: {len(unique_classes)}")

            # FIX-DENSITY: surface NON_ENTITY density so training logs are
            # informative about the entity/non-entity mix per phase.
            if total_nodes_all > 0:
                ne_pct = non_entity_nodes_all / total_nodes_all * 100
                entity_pct = 100.0 - ne_pct
                print(
                    f"   Node composition: {entity_pct:.1f}% entity  "
                    f"/ {ne_pct:.1f}% NON_ENTITY  "
                    f"(total nodes: {total_nodes_all:,})"
                )
                if ne_pct > 85:
                    print(
                        f"   ⚠️  High NON_ENTITY density ({ne_pct:.1f}%) — "
                        f"model will see mostly NON_ENTITY nodes in this phase."
                    )

        if entity_counts:
            top_types = sorted(
                entity_counts.items(), key=lambda x: x[1], reverse=True
            )[:5]
            names = []
            for t, c in top_types:
                name = ENTITY_ID_TO_TYPE.get(t, f"Class_{t}")
                names.append(f"{name}({c})")
            print(f"   Top entities: {', '.join(names)}")

        return DataLoader(
            phase_graphs, batch_size=self.batch_size, shuffle=shuffle
        )

    def get_difficulty_stats(self, phase: int) -> Dict:
        """Return detailed statistics for a phase."""
        if phase < 0 or phase >= len(self.phase_boundaries):
            return self._empty_stats(phase)

        num_graphs = self.phase_boundaries[phase]
        if num_graphs == 0:
            return self._empty_stats(phase)

        phase_indices  = self.sorted_indices[:num_graphs]
        type_counts:   Dict[int, int] = defaultdict(int)
        total_entities = 0

        for i in phase_indices:
            for entity in self.entities_per_graph[i]:
                tid = self._entity_type_id(entity)
                type_counts[tid] += 1
                total_entities   += 1

        phase_scores = self.graph_scores[phase_indices]

        return {
            'phase':              phase + 1,
            'num_graphs':         int(num_graphs),
            'total_entities':     total_entities,
            'avg_difficulty':     float(np.mean(phase_scores)),
            'min_difficulty':     float(np.min(phase_scores)),
            'max_difficulty':     float(np.max(phase_scores)),
            'std_difficulty':     float(np.std(phase_scores)),
            'entity_type_counts': dict(type_counts),
            'unique_entity_types': len(type_counts),
        }

    def _empty_stats(self, phase: int) -> Dict:
        return {
            'phase':              phase + 1,
            'num_graphs':         0,
            'total_entities':     0,
            'avg_difficulty':     0.0,
            'min_difficulty':     0.0,
            'max_difficulty':     0.0,
            'std_difficulty':     0.0,
            'entity_type_counts': {},
            'unique_entity_types': 0,
        }

    def get_all_stats(self) -> List[Dict]:
        return [self.get_difficulty_stats(i) for i in range(self.num_phases)]

    def get_phase_recommendations(self) -> Dict:
        """Recommend optimal phase configuration based on data."""
        stats = self.get_all_stats()
        recommendations: Dict = {
            'suggested_epochs':  [],
            'suggested_dropout': [],
            'focus_classes':     [],
        }

        for phase_stat in stats:
            rare_classes = [
                c for c, count in phase_stat['entity_type_counts'].items()
                if count < 10
            ]
            if rare_classes:
                recommendations['suggested_epochs'].append(25)
                recommendations['suggested_dropout'].append(0.2)
                for c in rare_classes[:3]:
                    class_name = ENTITY_ID_TO_TYPE.get(c, f"Class_{c}")
                    if class_name not in recommendations['focus_classes']:
                        recommendations['focus_classes'].append(class_name)
            else:
                recommendations['suggested_epochs'].append(15)
                recommendations['suggested_dropout'].append(0.3)

        return recommendations

In [ ]:
class CurriculumTrainer:
    """
    Trainer with curriculum learning, adaptive class weighting, and
    phase-aware early stopping.

    27-class version: 0–25 entity types, 26 NON_ENTITY, -1 CHUNK_ROOT.
    """

    DROPOUT_SCHEDULE = [0.1, 0.15, 0.2, 0.25]

    def __init__(self, model, config: Dict, logger: logging.Logger):
        """
        CurriculumTrainer.__init__ — identical to original except:
          * self.criterion replaced with FocalLoss(gamma=2.0, reduction='none')
            instead of nn.CrossEntropyLoss(label_smoothing=0.05, reduction='none')
          * self.focal_gamma stored so set_class_weights can hot-swap weights
            into the FocalLoss object without recreating it.

        Usage
        -----
        Monkey-patch before instantiating:
            CurriculumTrainer.__init__ = curriculum_trainer_init
        Or subclass CurriculumTrainer and override __init__.
        """
        self.model  = model
        self.config = config
        self.logger = logger
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

        self.background_label_id = config.get('background_label_id', 26)
        self.num_entity_types    = config.get('num_entity_types', NUM_ENTITY_TYPES)

        self.warmup_steps  = config.get('warmup_steps', 100)
        self.max_grad_norm = config.get('max_grad_norm', 1.0)
        self.min_delta     = config.get('min_delta', 0.001)
        self.patience      = config.get('patience', 7)

        self.optimizer = self._create_optimizer()
        self.scheduler = self._create_scheduler()

        # ── FIX-FOCAL: replace CrossEntropyLoss with FocalLoss ───────────────
        self.focal_gamma = config.get('focal_gamma', 2.0)   # tunable via config
        self.criterion   = FocalLoss(
            class_weights = None,                 # populated by set_class_weights
            gamma         = self.focal_gamma,
            reduction     = 'none',
        )

        self.best_f1                  = 0.0
        self.patience_counter         = 0
        self._global_patience_counter = 0
        self.current_step             = 0


    # -------------------------------------------------------------------------
    # Optimizer / scheduler
    # -------------------------------------------------------------------------

    def _create_optimizer(self):
        gat_params:        List = []
        classifier_params: List = []

        for name, param in self.model.named_parameters():
            if 'classifier' in name:
                classifier_params.append(param)
            else:
                gat_params.append(param)

        return optim.AdamW([
            {'params': gat_params,        'lr': self.config['learning_rate'] * 0.5},
            {'params': classifier_params, 'lr': self.config['learning_rate']},
        ], weight_decay=1e-4)

    def _create_scheduler(self):
        def lr_lambda(current_step: int) -> float:
            if current_step < 20:
                return float(current_step) / 20.0
            return max(0.05, 0.98 ** (current_step - 20))

        return optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)

    # -------------------------------------------------------------------------
    # Class weights
    # -------------------------------------------------------------------------

    def set_class_weights(
        self,
        class_weights:   torch.Tensor,
        from_train_only: bool = False,
    ) -> None:
        """
        Apply pre-computed class weights.

        Identical to original except it calls self.criterion.update_weights()
        so the FocalLoss object picks up the new alpha_t values without the
        caller having to know which loss class is in use.
        """
        if not isinstance(class_weights, torch.Tensor):
            raise TypeError(
                f"class_weights must be a torch.Tensor, got {type(class_weights)}"
            )
        expected = NUM_ENTITY_TYPES
        if class_weights.shape != (expected,):
            raise ValueError(
                f"class_weights must have shape ({expected},), "
                f"got {tuple(class_weights.shape)}"
            )

        if not from_train_only:
            self.logger.warning(
                "⚠️  set_class_weights called without from_train_only=True. "
                "If class_weights were computed on the full dataset this "
                "introduces label-frequency leakage into the loss function."
            )

        smoothed = class_weights.clone()
        smoothed = torch.clamp(smoothed, max=5.0)
        for i in range(expected):
            if class_weights[i] > 0 and smoothed[i] < 0.3:
                smoothed[i] = 0.3

        self.class_weights = smoothed.to(self.device)

        # ── FIX-FOCAL: hot-swap into FocalLoss so it uses updated alpha_t ────
        self.criterion.update_weights(self.class_weights)

        self.logger.info(
            f"✅ set_class_weights: {expected}-class tensor "
            f"(min={smoothed.min():.4f}, max={smoothed.max():.4f}, "
            f"train_only={from_train_only})"
        )

    # -------------------------------------------------------------------------
    # Training — one epoch
    # -------------------------------------------------------------------------

    def train_epoch(self, train_loader, phase: int) -> float:
        """
        Train one epoch using FocalLoss.

        Changes vs. original
        --------------------
        FIX-FOCAL : loss_per_node now comes from self.criterion (FocalLoss)
                    directly, which already handles class weighting internally
                    via the alpha_t buffer.  The old two-branch logic
                    (F.nll_loss with weight= / F.nll_loss without) is replaced
                    by a single self.criterion(out[mask], data.y[mask]) call.

        FIX-FOCAL : The old code applied F.log_softmax then F.nll_loss, which
                    is mathematically identical to F.cross_entropy.  FocalLoss
                    uses F.cross_entropy internally so the log_softmax step is
                    removed — calling it again would double-log the probs.

        Everything else (inverted dropout schedule, gradient clipping,
        NaN guards, node_weight multiplication) is preserved unchanged.
        """
        INVERTED_DROPOUT = self.config.get('inverted_dropout', [0.30, 0.25, 0.20, 0.20])

        self.model.train()
        total_loss    = 0.0
        valid_batches = 0

        if hasattr(self.model, 'dropout'):
            idx = min(phase, len(INVERTED_DROPOUT) - 1)
            self.model.dropout.p = INVERTED_DROPOUT[idx]

        for batch_idx, data in enumerate(train_loader):
            self.current_step += 1
            data = data.to(self.device)

            if torch.isnan(data.x).any() or torch.isinf(data.x).any():
                self.logger.warning(f"Batch {batch_idx}: NaN/Inf in input, skipping")
                continue

            mask = data.y >= 0
            if mask.sum() == 0:
                continue

            self.optimizer.zero_grad()
            out = self.model(data)

            if torch.isnan(out).any() or torch.isinf(out).any():
                self.logger.warning(f"Batch {batch_idx}: NaN/Inf in output, skipping")
                continue

            # ── FIX-FOCAL: single loss call; no log_softmax pre-step ─────────
            # FocalLoss.forward receives raw logits and returns per-node loss.
            loss_per_node = self.criterion(out[mask], data.y[mask])   # [M]

            if hasattr(data, 'node_weight'):
                node_w = data.node_weight[mask].to(self.device)
                loss   = (loss_per_node * node_w).sum() / node_w.sum().clamp(min=1e-8)
            else:
                loss = loss_per_node.mean()

            if torch.isnan(loss) or torch.isinf(loss):
                self.logger.warning(f"Batch {batch_idx}: NaN/Inf loss, skipping")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), self.max_grad_norm
            )
            self.optimizer.step()

            total_loss    += loss.item()
            valid_batches += 1

        if valid_batches == 0:
            self.logger.error("No valid batches in epoch!")
            return float('nan')

        return total_loss / valid_batches

    # -------------------------------------------------------------------------
    # Evaluation
    # -------------------------------------------------------------------------

    def _entity_mask(self, y: torch.Tensor,
                     include_non_entity: bool = False) -> torch.Tensor:
        """
        Build the evaluation mask using self.background_label_id
        """
        if include_non_entity:
            return y >= 0   # All real labels (includes NON_ENTITY = 26)
        return (y >= 0) & (y != self.background_label_id)

    def evaluate(
            self,
            loader,
            include_non_entity:      bool       = False,
            independent_annotations: list       = None,
        ) -> Dict:
            """
            Evaluate on entity classes only (default) or all classes.

            Parameters
            ----------
            loader : DataLoader
                Graphs to evaluate on.  Labels come from regex — see NOTE above.
            include_non_entity : bool
                If True, NON_ENTITY (class 26) is included in accuracy/F1.
            independent_annotations : list[dict] | None
                Optional list of human-annotated examples, each a dict with keys:
                  'chunk_text'   – raw text of the chunk
                  'entity_value' – the span text
                  'entity_type'  – string label, e.g. 'FIRE_RATING'
                When provided, a second 'independent' metrics block is returned
                that is NOT circular with respect to the regex labels.
                Format of independent block mirrors the main block.

            Returns
            -------
            dict with keys:
                accuracy, f1_score, avg_confidence,
                predictions, labels, confidences,
                per_class_counts  ← new: {class_id: pred_count}
                independent       ← new: present only when independent_annotations given
            """
            self.model.eval()
            all_preds:       List[int]   = []
            all_labels:      List[int]   = []
            all_confidences: List[float] = []

            with torch.no_grad():
                for data in loader:
                    data  = data.to(self.device)
                    out   = self.model(data)              # single forward pass
                    probs = F.softmax(out, dim=-1)
                    pred  = out.argmax(dim=1)
                    conf  = probs.max(dim=-1)[0]

                    mask = self._entity_mask(data.y, include_non_entity)
                    all_preds.extend(pred[mask].cpu().numpy())
                    all_labels.extend(data.y[mask].cpu().numpy())
                    all_confidences.extend(conf[mask].cpu().numpy())

            if not all_labels:
                return {
                    'accuracy': 0.0, 'f1_score': 0.0,
                    'avg_confidence': 0.0,
                    'predictions': [], 'labels': [], 'confidences': [],
                    'per_class_counts': {},
                }

            accuracy       = accuracy_score(all_labels, all_preds)
            f1             = f1_score(all_labels, all_preds,
                                      average='weighted', zero_division=0)
            avg_confidence = float(np.mean(all_confidences)) if all_confidences else 0.0
            per_class_counts = dict(Counter(all_preds))

            result = {
                'accuracy':         accuracy,
                'f1_score':         f1,
                'avg_confidence':   avg_confidence,
                'predictions':      all_preds,
                'labels':           all_labels,
                'confidences':      all_confidences,
                'per_class_counts': per_class_counts,
            }

            # ── Optional: evaluate against human annotations ──────────────────
            # This is the ONLY non-circular evaluation path.
            # Each annotation is embedded, a single-node inference graph is built,
            # and the model's prediction is compared to the human label.
            if independent_annotations:
                ind_preds:  List[int]   = []
                ind_labels: List[int]   = []
                ind_confs:  List[float] = []

                for ann in independent_annotations:
                    entity_type = ann.get('entity_type', '')
                    true_id     = REGULATION_ENTITY_TYPES.get(entity_type)
                    if true_id is None or true_id < 0:
                        continue  # skip unknown / CHUNK_ROOT labels

                    chunk_text   = ann.get('chunk_text', '')
                    entity_value = ann.get('entity_value', '')
                    if not chunk_text or not entity_value:
                        continue

                    # Build a minimal single-candidate inference graph
                    candidate = {
                        'text':   entity_value,
                        'start':  chunk_text.find(entity_value),
                        'end':    chunk_text.find(entity_value) + len(entity_value),
                        'source': 'human_annotation',
                    }
                    if candidate['start'] == -1:
                        candidate['start'] = 0
                        candidate['end']   = len(entity_value)

                    chunk_dict = {'text': chunk_text, 'chunk_id': 'annotation'}
                    # graph_builder must be accessible; caller should ensure it is
                    # attached to the trainer (trainer.graph_builder = pipeline.graph_builder)
                    graph_builder = getattr(self, 'graph_builder', None)
                    if graph_builder is None:
                        self.logger.warning(
                            "evaluate(): independent_annotations provided but "
                            "self.graph_builder is not set. "
                            "Set trainer.graph_builder = pipeline.graph_builder "
                            "before calling evaluate() with annotations."
                        )
                        break

                    graph = graph_builder.build_graph_for_inference(
                        chunk_dict, [candidate]
                    )
                    if graph is None:
                        continue

                    graph = graph.to(self.device)
                    with torch.no_grad():
                        out   = self.model(graph)
                        probs = F.softmax(out, dim=-1)

                    # Node 1 is the candidate (node 0 is the chunk root)
                    if out.size(0) < 2:
                        continue
                    pred_id = out[1].argmax().item()
                    conf    = probs[1].max().item()
                    ind_preds.append(pred_id)
                    ind_labels.append(true_id)
                    ind_confs.append(conf)

                if ind_labels:
                    result['independent'] = {
                        'n_samples':     len(ind_labels),
                        'accuracy':      float(accuracy_score(ind_labels, ind_preds)),
                        'f1_score':      float(f1_score(ind_labels, ind_preds,
                                                        average='weighted',
                                                        zero_division=0)),
                        'avg_confidence': float(np.mean(ind_confs)),
                        'predictions':   ind_preds,
                        'labels':        ind_labels,
                        'confidences':   ind_confs,
                    }
                    self.logger.info(
                        f"✅ Independent eval: n={len(ind_labels)} "
                        f"acc={result['independent']['accuracy']:.4f} "
                        f"f1={result['independent']['f1_score']:.4f}"
                    )
                else:
                    self.logger.warning(
                        "evaluate(): independent_annotations were provided but "
                        "no valid samples could be processed (check entity_type "
                        "strings and chunk_text/entity_value fields)."
                    )

            return result

    def train_with_curriculum(
            self,
            curriculum_loader,
            val_loader,
            test_loader              = None,
            epochs_per_phase:  List[int] = None,
            independent_annotations: list       = None,
        ) -> Dict:
            """
            Train with curriculum learning across all phases.

            Parameters
            ----------
            curriculum_loader : CurriculumDataLoader
            val_loader        : DataLoader  — regex-labeled val split
            test_loader       : DataLoader  — regex-labeled test split (evaluated once)
            epochs_per_phase  : list[int]
            independent_annotations : list[dict] | None
                Human-annotated examples in the format expected by evaluate().
                When provided, the checkpoint metric incorporates a non-circular
                independent F1 component (weight 0.4); the regex-based components
                are down-weighted correspondingly.
                Without annotations the checkpoint metric is the original
                combined_f1 = 0.6 * entity_f1 + 0.4 * boundary_f1.
            """
            if epochs_per_phase is None:
                epochs_per_phase = self.config.get('epochs', [15, 20, 25, 30])

            use_independent = bool(independent_annotations)

            history: Dict = {
                'train_loss':             [],
                'val_accuracy':           [],
                'val_f1':                 [],
                'val_f1_with_non_entity': [],
                'val_boundary_f1':        [],
                'val_combined_f1':        [],
                'val_independent_f1':     [],   # new: non-circular metric
                'phase_transitions':      [],
                'learning_rates':         [],
                'confidence_scores':      [],
            }

            total_epochs = 0

            print(f"\n🚀 Starting Curriculum Training on {self.device}")
            print(f"   Total phases:      {curriculum_loader.num_phases}")
            print(f"   Epochs per phase:  {epochs_per_phase}")
            print(f"   Total epochs:      {sum(epochs_per_phase)}")
            print(
                f"   Num classes:       {self.num_entity_types} "
                f"(0–{self.num_entity_types - 1}=entities, "
                f"{self.background_label_id}=NON_ENTITY)"
            )
            print(f"   Warmup steps:      {self.warmup_steps}")

            if use_independent:
                print(
                    f"   Checkpoint metric: 0.4 * entity_f1 + 0.2 * boundary_f1 "
                    f"+ 0.4 * independent_f1  "
                    f"({len(independent_annotations)} human-annotated samples)"
                )
            else:
                print(
                    f"   Checkpoint metric: 0.6 * entity_f1 + 0.4 * boundary_f1  "
                    f"⚠️  CIRCULAR — both components use regex-derived labels.\n"
                    f"   Provide independent_annotations to break circularity."
                )

            for phase in range(curriculum_loader.num_phases):
                print(f"\n{'=' * 80}")
                print(f"📚 PHASE {phase + 1}/{curriculum_loader.num_phases}")
                print(f"{'=' * 80}")

                stats = curriculum_loader.get_difficulty_stats(phase)
                print(f"\n📊 Phase Statistics:")
                print(f"   Graphs:     {stats['num_graphs']}")
                print(f"   Entities:   {stats['total_entities']}")
                print(
                    f"   Difficulty: {stats['avg_difficulty']:.3f} "
                    f"({stats['min_difficulty']:.3f} – "
                    f"{stats['max_difficulty']:.3f})"
                )

                phase_loader           = curriculum_loader.get_phase_loader(phase, shuffle=True)
                phase_epochs           = epochs_per_phase[phase]
                phase_best_combined    = 0.0
                phase_patience_counter = 0

                phase_best_state = {
                    k: v.cpu().clone() for k, v in self.model.state_dict().items()
                }

                for epoch in range(phase_epochs):
                    total_epochs += 1

                    train_loss = self.train_epoch(phase_loader, phase)
                    if np.isnan(train_loss):
                        self.logger.error(f"Training failed at epoch {total_epochs}")
                        break

                    # ── Regex-based metrics (circular, but tracked for visibility) ──
                    val_metrics = self.evaluate(
                        val_loader,
                        include_non_entity=False,
                        independent_annotations=None,
                    )
                    val_metrics_full = self.evaluate(
                        val_loader,
                        include_non_entity=True,
                        independent_annotations=None,
                    )
                    boundary_f1 = self._compute_boundary_f1(
                        val_metrics_full['labels'],
                        val_metrics_full['predictions'],
                    )
                    entity_f1 = val_metrics['f1_score']

                    # ── Independent (non-circular) metric ─────────────────────────
                    independent_f1 = 0.0
                    if use_independent:
                        ind_result = self.evaluate(
                            val_loader,
                            include_non_entity=False,
                            independent_annotations=independent_annotations,
                        )
                        independent_f1 = ind_result.get('independent', {}).get('f1_score', 0.0)

                    # ── Checkpoint metric ──────────────────────────────────────────
                    if use_independent:
                        # Non-circular: independent component carries 40% weight
                        combined_f1 = (
                            0.4 * entity_f1
                            + 0.2 * boundary_f1
                            + 0.4 * independent_f1
                        )
                    else:
                        # Circular fallback (original formula)
                        combined_f1 = 0.6 * entity_f1 + 0.4 * boundary_f1

                    old_lr = self.optimizer.param_groups[0]['lr']
                    self.scheduler.step()
                    new_lr = self.optimizer.param_groups[0]['lr']
                    if old_lr != new_lr:
                        self.logger.info(f"📉 LR: {old_lr:.6f} → {new_lr:.6f}")

                    history['train_loss'].append(train_loss)
                    history['val_accuracy'].append(val_metrics['accuracy'])
                    history['val_f1'].append(entity_f1)
                    history['val_f1_with_non_entity'].append(val_metrics_full['f1_score'])
                    history['val_boundary_f1'].append(boundary_f1)
                    history['val_combined_f1'].append(combined_f1)
                    history['val_independent_f1'].append(independent_f1)
                    history['learning_rates'].append(new_lr)
                    history['confidence_scores'].append(val_metrics.get('avg_confidence', 0.0))

                    if (epoch + 1) % 5 == 0 or epoch == phase_epochs - 1:
                        ind_str = f"IndepF1: {independent_f1:.4f} | " if use_independent else ""
                        print(
                            f"   Epoch {total_epochs:3d} "
                            f"(P{phase + 1}:{epoch + 1:2d}/{phase_epochs}) | "
                            f"Loss: {train_loss:.4f} | "
                            f"Acc: {val_metrics['accuracy']:.4f} | "
                            f"EntityF1: {entity_f1:.4f} | "
                            f"BoundaryF1: {boundary_f1:.4f} | "
                            f"{ind_str}"
                            f"CombinedF1: {combined_f1:.4f} | "
                            f"LR: {new_lr:.6f}"
                        )

                    # ── Phase early stopping ───────────────────────────────────────
                    if combined_f1 > phase_best_combined + self.min_delta:
                        phase_best_combined = combined_f1
                        phase_best_state = {
                            k: v.cpu().clone()
                            for k, v in self.model.state_dict().items()
                        }
                        phase_patience_counter = 0
                    else:
                        phase_patience_counter += 1

                    # ── Global best checkpoint ────────────────────────────────────
                    if combined_f1 > self.best_f1 + self.min_delta:
                        self.best_f1 = combined_f1
                        model_path   = f"{self.config['model_dir']}/best_model.pt"
                        torch.save(
                            {
                                'epoch':                total_epochs,
                                'phase':                phase,
                                'model_state_dict':     self.model.state_dict(),
                                'optimizer_state_dict': self.optimizer.state_dict(),
                                'scheduler_state_dict': self.scheduler.state_dict(),
                                'f1_score':             self.best_f1,
                                'entity_f1':            entity_f1,
                                'boundary_f1':          boundary_f1,
                                'independent_f1':       independent_f1,
                                'used_independent':     use_independent,
                                'num_classes':          self.num_entity_types,
                                'background_label_id':  self.background_label_id,
                            },
                            model_path,
                        )
                        self.logger.info(
                            f"✅ Saved best model  "
                            f"combined={self.best_f1:.4f}  "
                            f"entity={entity_f1:.4f}  "
                            f"boundary={boundary_f1:.4f}"
                            + (f"  independent={independent_f1:.4f}" if use_independent else "")
                        )
                        self._global_patience_counter = 0
                    else:
                        self._global_patience_counter += 1

                    if phase_patience_counter >= self.patience:
                        print(
                            f"\n⚠️  Phase {phase + 1} early stopping "
                            f"(no combined F1 improvement > {self.min_delta} "
                            f"for {self.patience} epochs)"
                        )
                        break

                self.model.load_state_dict(
                    {k: v.to(self.device) for k, v in phase_best_state.items()}
                )
                self.logger.info(
                    f"🔄 Phase {phase + 1}: restored phase-best weights "
                    f"(combined F1={phase_best_combined:.4f})"
                )

                history['phase_transitions'].append(total_epochs)

                print(f"\n✅ Phase {phase + 1} Complete!")
                print(f"   Best combined F1 in phase : {phase_best_combined:.4f}")
                print(f"   Global best combined F1   : {self.best_f1:.4f}")

            if test_loader is not None:
                print(f"\n{'=' * 80}")
                print(f"🧪 FINAL TEST SET EVALUATION (held-out, single pass)")
                print(f"{'=' * 80}")
                test_metrics      = self.evaluate(test_loader, include_non_entity=False)
                test_metrics_full = self.evaluate(test_loader, include_non_entity=True)
                test_boundary_f1  = self._compute_boundary_f1(
                    test_metrics_full['labels'],
                    test_metrics_full['predictions'],
                )
                test_independent_f1 = 0.0
                if use_independent:
                    ind_test = self.evaluate(
                        test_loader,
                        include_non_entity=False,
                        independent_annotations=independent_annotations,
                    )
                    test_independent_f1 = ind_test.get('independent', {}).get('f1_score', 0.0)
                    print(f"   Test Independent F1 : {test_independent_f1:.4f}  ← non-circular")

                print(f"   Test Entity F1   : {test_metrics['f1_score']:.4f}  (vs regex labels)")
                print(f"   Test Boundary F1 : {test_boundary_f1:.4f}  (vs regex labels)")
                print(f"   Test Accuracy    : {test_metrics['accuracy']:.4f}")
                history['final_test_metrics'] = {
                    **test_metrics,
                    'boundary_f1':    test_boundary_f1,
                    'independent_f1': test_independent_f1,
                }

            print(f"\n{'=' * 80}")
            print(f"🎉 Curriculum Training Complete!")
            print(f"{'=' * 80}")
            print(f"   Total epochs     : {total_epochs}")
            print(f"   Best combined F1 : {self.best_f1:.4f}")
            if not use_independent:
                print(
                    f"\n   ⚠️  NOTE: Best combined F1 is measured against regex-derived\n"
                    f"   labels.  It reflects how well the model mimics the regex,\n"
                    f"   not its real NER accuracy.  Add independent_annotations\n"
                    f"   to get a meaningful checkpoint metric."
                )

            return history

    def _compute_boundary_f1(
        self,
        labels:      List[int],
        predictions: List[int],
    ) -> float:
        """
        Compute F1 score for the NON_ENTITY class specifically.

        This is the boundary discrimination metric — how well the model
        distinguishes entity nodes from NON_ENTITY nodes.

        A model that always predicts NON_ENTITY gets recall=1.0 but
        precision=0.0 here, so it cannot game this metric by collapsing.
        A model that never predicts NON_ENTITY gets precision=undefined
        (treated as 0.0) and recall=0.0.

        Returns 0.0 if there are no NON_ENTITY labels in the evaluation set
        (which should not happen but guards against empty-loader edge cases).
        """
        if not labels or not predictions:
            return 0.0

        labels_arr = np.array(labels)
        preds_arr  = np.array(predictions)

        ne_id = self.background_label_id

        tp = int(((preds_arr == ne_id) & (labels_arr == ne_id)).sum())
        fp = int(((preds_arr == ne_id) & (labels_arr != ne_id)).sum())
        fn = int(((preds_arr != ne_id) & (labels_arr == ne_id)).sum())

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        if precision + recall == 0.0:
            return 0.0

        return float(2.0 * precision * recall / (precision + recall))

    def get_detailed_metrics(self, loader) -> Dict:
        """Detailed per-class metrics for entity classes (0 to NUM_ENTITY_TYPES-1)."""
        self.model.eval()
        all_preds:       List[int]   = []
        all_labels:      List[int]   = []
        all_confidences: List[float] = []

        with torch.no_grad():
            for data in loader:
                data  = data.to(self.device)
                out   = self.model(data)
                probs = F.softmax(out, dim=-1)
                pred  = out.argmax(dim=1)
                conf  = probs.max(dim=-1)[0]

                mask = self._entity_mask(data.y, include_non_entity=False)
                all_preds.extend(pred[mask].cpu().numpy())
                all_labels.extend(data.y[mask].cpu().numpy())
                all_confidences.extend(conf[mask].cpu().numpy())

        if not all_labels:
            return {'error': 'No valid labels'}

        accuracy    = accuracy_score(all_labels, all_preds)
        f1_weighted = f1_score(all_labels, all_preds,
                               average='weighted', zero_division=0)
        f1_macro    = f1_score(all_labels, all_preds,
                               average='macro', zero_division=0)

        entity_label_ids = list(range(self.num_entity_types))
        precision, recall, f1, support = precision_recall_fscore_support(
            all_labels, all_preds,
            average=None,
            zero_division=0,
            labels=entity_label_ids,
        )

        per_class: Dict = {}
        for class_id in entity_label_ids:
            if support[class_id] > 0:
                name = ENTITY_ID_TO_TYPE.get(class_id, f"CLASS_{class_id}")
                class_confidences = [
                    all_confidences[i]
                    for i, p in enumerate(all_preds)
                    if p == class_id
                ]
                avg_conf = float(np.mean(class_confidences)) if class_confidences else 0.0
                per_class[name] = {
                    'precision':      float(precision[class_id]),
                    'recall':         float(recall[class_id]),
                    'f1':             float(f1[class_id]),
                    'support':        int(support[class_id]),
                    'avg_confidence': avg_conf,
                }

        cm = confusion_matrix(all_labels, all_preds, labels=entity_label_ids)
        return {
            'accuracy':         accuracy,
            'f1_weighted':      f1_weighted,
            'f1_macro':         f1_macro,
            'per_class':        per_class,
            'confusion_matrix': cm.tolist(),
            'predictions':      all_preds,
            'labels':           all_labels,
            'confidences':      all_confidences,
        }

    # -------------------------------------------------------------------------
    # Prediction analysis
    # -------------------------------------------------------------------------

    def analyze_model_predictions(self, test_loader, label_encoder) -> Dict:
        """
        Analyse model predictions on the test set.
        Prediction distribution table includes NON_ENTITY so
        misclassifications are visible.
        """
        self.model.eval()
        all_predictions:  List = []
        all_labels:       List = []
        all_confidences:  List = []

        print(f"\n🔍 Analysing predictions on "
              f"{len(test_loader.dataset)} test samples...")

        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Analysing"):
                batch       = batch.to(self.device)
                logits      = self.model(batch)
                probs       = F.softmax(logits, dim=-1)
                predictions = torch.argmax(probs, dim=-1)
                confidences = torch.max(probs, dim=-1)[0]

                # Include NON_ENTITY; exclude CHUNK_ROOT (-1)
                valid_mask = (batch.y >= 0) & (batch.y <= self.background_label_id)

                if valid_mask.sum() > 0:
                    all_predictions.extend(predictions[valid_mask].cpu().numpy())
                    all_labels.extend(batch.y[valid_mask].cpu().numpy())
                    all_confidences.extend(confidences[valid_mask].cpu().numpy())

        all_predictions = np.array(all_predictions)
        all_labels      = np.array(all_labels)
        all_confidences = np.array(all_confidences)

        print(f"\n📊 Prediction Distribution:")
        pred_counts  = Counter(all_predictions)
        label_counts = Counter(all_labels)

        print(f"\n{'Class':<30} {'True':<10} {'Predicted':<10} {'Diff':<10} {'Conf':<10}")
        print("-" * 70)
        for class_id in sorted(set(all_labels) | set(all_predictions)):
            class_name = ENTITY_ID_TO_TYPE.get(int(class_id), f"Class_{class_id}")
            true_count = label_counts.get(class_id, 0)
            pred_count = pred_counts.get(class_id, 0)
            diff       = pred_count - true_count
            class_conf = all_confidences[all_predictions == class_id]
            avg_conf   = float(np.mean(class_conf)) if len(class_conf) > 0 else 0.0
            print(
                f"{class_name:<30} {true_count:<10} {pred_count:<10} "
                f"{diff:+d}       {avg_conf:.3f}"
            )

        print(f"\n📈 Confidence Statistics:")
        print(f"   Mean:   {np.mean(all_confidences):.4f}")
        print(f"   Median: {np.median(all_confidences):.4f}")
        print(f"   Min:    {np.min(all_confidences):.4f}")
        print(f"   Max:    {np.max(all_confidences):.4f}")

        print(f"\n📊 Per-Class Confidence:")
        for class_id in sorted(set(all_predictions)):
            name       = ENTITY_ID_TO_TYPE.get(int(class_id), f"Class_{class_id}")
            class_mask = all_predictions == class_id
            if class_mask.sum() > 0:
                conf = all_confidences[class_mask]
                print(f"   {name:<30} Mean: {np.mean(conf):.4f} (n={len(conf)})")

        unique_entity_predictions = len(
            {p for p in all_predictions if p != self.background_label_id}
        )
        non_entity_predicted = pred_counts.get(self.background_label_id, 0)
        non_entity_true      = label_counts.get(self.background_label_id, 0)

        print(f"\n🎯 Model Health Check:")
        print(f"   Total entity classes:     {self.num_entity_types}")
        print(f"   Entity classes predicted: {unique_entity_predictions}")

        if non_entity_true > 0:
            recall_pct = non_entity_predicted / non_entity_true * 100
            print(
                f"   NON_ENTITY: {non_entity_predicted} predicted / "
                f"{non_entity_true} true ({recall_pct:.1f}% recall)"
            )

        low_conf_threshold = 0.3
        low_conf_count = int(np.sum(all_confidences < low_conf_threshold))
        low_conf_pct   = (
            low_conf_count / len(all_confidences) * 100
            if len(all_confidences) > 0 else 0.0
        )
        print(
            f"   Low-confidence (<{low_conf_threshold}): "
            f"{low_conf_count} ({low_conf_pct:.1f}%)"
        )

        if unique_entity_predictions < self.num_entity_types * 0.5:
            print(
                f"   ⚠️  WARNING: Only predicting "
                f"{unique_entity_predictions}/{self.num_entity_types} "
                f"entity classes — possible collapse."
            )
        else:
            print(f"   ✅ Model predicting diverse entity classes")

        print(f"\n🔀 Most Common Prediction Errors:")
        confusion_pairs: List[Tuple] = []
        for true_label in set(all_labels):
            for pred_label in set(all_predictions):
                if true_label != pred_label:
                    mask  = (all_labels == true_label) & (all_predictions == pred_label)
                    count = int(mask.sum())
                    if count > 0:
                        confusion_pairs.append((true_label, pred_label, count))

        confusion_pairs.sort(key=lambda x: x[2], reverse=True)
        for true_label, pred_label, count in confusion_pairs[:10]:
            t_name = ENTITY_ID_TO_TYPE.get(int(true_label), f"Class_{true_label}")
            p_name = ENTITY_ID_TO_TYPE.get(int(pred_label), f"Class_{pred_label}")
            print(f"   {t_name} → {p_name}: {count} errors")

        return {
            'predictions':         all_predictions,
            'labels':              all_labels,
            'confidences':         all_confidences,
            'unique_predictions':  unique_entity_predictions,
            'total_classes':       self.num_entity_types,
            'low_conf_count':      low_conf_count,
            'low_conf_percentage': low_conf_pct,
        }

###FUZZY ENTITY MATCHER

In [ ]:
class FuzzyEntityMatcher:
    """
    Match entities with fuzzy span and type matching.
    Returns both detailed analysis and backward-compatible simple format.

    28-class version — with improved matching logic for better overlap detection.
    """

    def __init__(self, logger):
        self.logger = logger

        # Define related entity type groups (Semantic Equivalents)
        # 27-class version: enhanced groups for better matching
        self.type_groups = [
            # Measurements — Spatial
            {'DIMENSION', 'AREA', 'VOLUME'},
            {'CLEARANCE', 'EGRESS_DISTANCE', 'SEPARATION_DISTANCE'},

            # Physical Components - EXPANDED
            {'COMPONENT', 'MATERIAL', 'EQUIPMENT', 'PENETRATION', 'FACILITY'},

            # References & Standards
            {'STANDARD', 'AUTHORITY', 'CERTIFICATION'},

            # Building Classification
            {'BUILDING_TYPE', 'OCCUPANCY', 'LOCATION'},

            # Quantitative Measures
            {'CAPACITY', 'PERCENTAGE', 'PRESSURE', 'FLOW_RATE', 'TEMPERATURE', 'FIRE_RATING'},

            # Operational Requirements
            {'TESTING_REQUIREMENT', 'INSPECTION_FREQUENCY'},

            # Risk & Safety
            {'HAZARD_LEVEL'},

            # FIX 1: Add explicit EQUIPMENT-FACILITY relationship
            {'EQUIPMENT', 'FACILITY'},

            # FIX 2: Add STANDARD-CODE relationship
            {'STANDARD', 'CODE'},
        ]

        # Build type similarity map with graduated scores
        self.type_similarity = self._build_type_similarity_map()

    def _build_type_similarity_map(self) -> Dict[Tuple[str, str], float]:
        """Build map of type pair similarities with graduated scores."""
        similarity: Dict[Tuple[str, str], float] = {}

        for group in self.type_groups:
            group_list = list(group)
            for i, t1 in enumerate(group_list):
                for t2 in group_list[i + 1:]:

                    if {t1, t2} <= {'DIMENSION', 'AREA', 'VOLUME'}:
                        score = 0.90
                    elif {t1, t2} <= {'CLEARANCE', 'EGRESS_DISTANCE', 'SEPARATION_DISTANCE'}:
                        score = 0.85
                    elif {t1, t2} <= {'COMPONENT', 'MATERIAL', 'EQUIPMENT', 'FACILITY'}:
                        score = 0.80
                    else:
                        score = 0.75

                    # setdefault: first group wins, duplicates from later groups ignored
                    similarity.setdefault((t1, t2), score)
                    similarity.setdefault((t2, t1), score)

        return similarity

    def get_type_similarity(self, type1: str, type2: str) -> float:
        """Get similarity score between two entity types."""
        if type1 == type2:
            return 1.0

        # Handle case where one is CODE and one is STANDARD
        if 'CODE' in [type1, type2] and 'STANDARD' in [type1, type2]:
            return 0.95

        return self.type_similarity.get((type1, type2), 0.0)

    def calculate_span_overlap(self, entity1: Dict, entity2: Dict) -> float:
        """Calculate IoU with symmetric tolerance for offset drift in either direction."""
        start1 = entity1.get('absolute_start', entity1.get('start', 0))
        end1   = entity1.get('absolute_end',   entity1.get('end', 0))
        start2 = entity2.get('absolute_start', entity2.get('start', 0))
        end2   = entity2.get('absolute_end',   entity2.get('end', 0))

        # Symmetric: expand intersection window by tolerance on both sides
        # Union stays on original spans to avoid inflating the denominator
        tolerance = 3
        intersection_start = max(start1, start2) - tolerance
        intersection_end   = min(end1,   end2)   + tolerance

        if intersection_start >= intersection_end:
            return 0.0

        intersection = intersection_end - intersection_start
        union = (end1 - start1) + (end2 - start2) - max(0, min(end1, end2) - max(start1, start2))
        return intersection / max(union, 1)

    def calculate_text_similarity(self, entity1: Dict, entity2: Dict) -> float:
        """
        Calculate text similarity for entities with low span overlap.
        Useful for entities that are near but not exactly overlapping.
        """
        text1 = entity1.get('entity_value', '').lower().strip()
        text2 = entity2.get('entity_value', '').lower().strip()

        if not text1 or not text2:
            return 0.0

        # Simple word overlap
        words1 = set(text1.split())
        words2 = set(text2.split())

        if not words1 or not words2:
            return 0.0

        intersection = words1 & words2
        union = words1 | words2

        return len(intersection) / len(union)

    def entities_match(
        self,
        entity1: Dict,
        entity2: Dict,
        iou_threshold: float = 0.5,
        type_threshold: float = 0.8,
        text_similarity_threshold: float = 0.7,
    ) -> bool:
        """
        Enhanced matching with text similarity fallback.
        """
        # Primary: Span overlap + type similarity
        iou = self.calculate_span_overlap(entity1, entity2)
        type_sim = self.get_type_similarity(entity1['type'], entity2['type'])

        if iou >= iou_threshold and type_sim >= type_threshold:
            return True

        # FIX 5: Fallback for near misses - text similarity
        if iou >= 0.3:  # Some overlap but not enough
            text_sim = self.calculate_text_similarity(entity1, entity2)
            if text_sim >= text_similarity_threshold and type_sim >= 0.6:
                self.logger.debug(
                    f"Text similarity match: '{entity1.get('entity_value')}' [{entity1['type']}] "
                    f"vs '{entity2.get('entity_value')}' [{entity2['type']}] "
                    f"(iou={iou:.2f}, text_sim={text_sim:.2f})"
                )
                return True

        return False

    def find_matches(
        self,
        regex_entities: List[Dict],
        gat_entities: List[Dict],
        iou_threshold: float = 0.5,
        type_threshold: float = 0.8,
        return_detailed: bool = False,
    ) -> Union[Tuple[Set, Set, Set], Dict]:
        """
        Find matching entities between regex and GAT extractions.

        Returns:
            (matched_pairs, regex_only_indices, gat_only_indices)
            or detailed dict if return_detailed=True
        """
        matched_pairs: Set[Tuple[int, int]] = set()
        regex_matched_indices: Set[int] = set()
        gat_matched_indices: Set[int] = set()

        # FIX 6: Improved spatial indexing with overlapping windows
        gat_by_position: Dict[int, List[Tuple[int, Dict]]] = defaultdict(list)
        for gat_idx, gat_entity in enumerate(gat_entities):
            start_pos = gat_entity.get('absolute_start', gat_entity.get('start', 0))
            end_pos = gat_entity.get('absolute_end', gat_entity.get('end', 0))

            # Index by multiple buckets for better coverage
            for bucket in range(start_pos // 100, (end_pos // 100) + 1):
                gat_by_position[bucket].append((gat_idx, gat_entity))

        # Find best matches with multiple candidate consideration
        for regex_idx, regex_entity in enumerate(regex_entities):
            start_pos = regex_entity.get('absolute_start', regex_entity.get('start', 0))
            end_pos   = regex_entity.get('absolute_end',   regex_entity.get('end', 0))

            # Deduplicate candidates across buckets with a seen set
            seen_gat_ids: Set[int] = set()
            candidates = []
            for bucket in range((start_pos // 100) - 1, (end_pos // 100) + 2):
                for gat_idx, gat_entity in gat_by_position.get(bucket, []):
                    if gat_idx in gat_matched_indices:
                        continue
                    if gat_idx in seen_gat_ids:          # ← NEW: skip duplicates
                        continue
                    seen_gat_ids.add(gat_idx)
                    candidates.append((gat_idx, gat_entity))

            if not candidates:
                continue

            # Score all candidates
            scored_candidates = []
            for gat_idx, gat_entity in candidates:
                if self.entities_match(
                    regex_entity, gat_entity, iou_threshold, type_threshold
                ):
                    iou = self.calculate_span_overlap(regex_entity, gat_entity)
                    type_sim = self.get_type_similarity(
                        regex_entity['type'], gat_entity['type']
                    )
                    text_sim = self.calculate_text_similarity(regex_entity, gat_entity)

                    # Weighted score
                    score = (iou * 0.4 + type_sim * 0.3 + text_sim * 0.3)

                    # FIX 7: Bonus for exact type match
                    if regex_entity['type'] == gat_entity['type']:
                        score += 0.1

                    scored_candidates.append((score, gat_idx, gat_entity))

            if scored_candidates:
                # Take best match
                scored_candidates.sort(reverse=True)
                best_score, best_idx, best_entity = scored_candidates[0]

                matched_pairs.add((regex_idx, best_idx))
                regex_matched_indices.add(regex_idx)
                gat_matched_indices.add(best_idx)

                self.logger.debug(
                    f"Match found: '{regex_entity.get('entity_value')}' [{regex_entity['type']}] "
                    f"<-> '{best_entity.get('entity_value')}' [{best_entity['type']}] "
                    f"(score={best_score:.2f})"
                )

        regex_only = set(range(len(regex_entities))) - regex_matched_indices
        gat_only   = set(range(len(gat_entities)))   - gat_matched_indices

        if return_detailed:
            return self._create_detailed_results(
                regex_entities, gat_entities, matched_pairs, regex_only, gat_only
            )

        return matched_pairs, regex_only, gat_only

    def _create_detailed_results(
        self,
        regex_entities: List[Dict],
        gat_entities: List[Dict],
        matched_pairs: Set[Tuple[int, int]],
        regex_only: Set[int],
        gat_only: Set[int],
        high_confidence_threshold: float = 0.85,
    ) -> Dict[str, Any]:
        """Create detailed match results with categorization."""

        # Analyze matches
        match_details = []
        for regex_idx, gat_idx in matched_pairs:
            regex_ent = regex_entities[regex_idx]
            gat_ent = gat_entities[gat_idx]

            match_details.append({
                'regex_idx': regex_idx,
                'gat_idx': gat_idx,
                'regex_text': regex_ent.get('entity_value'),
                'gat_text': gat_ent.get('entity_value'),
                'regex_type': regex_ent['type'],
                'gat_type': gat_ent['type'],
                'iou': self.calculate_span_overlap(regex_ent, gat_ent),
                'type_similarity': self.get_type_similarity(
                    regex_ent['type'], gat_ent['type']
                ),
                'text_similarity': self.calculate_text_similarity(regex_ent, gat_ent),
                'regex_confidence': regex_ent.get('confidence', 1.0),
                'gat_confidence': gat_ent.get('confidence', 0.0),
            })

        # Categorize regex-only
        regex_only_high_conf = set()
        regex_only_low_conf = set()
        for idx in regex_only:
            conf = regex_entities[idx].get('confidence', 1.0)
            if conf >= high_confidence_threshold:
                regex_only_high_conf.add(idx)
            else:
                regex_only_low_conf.add(idx)

        # Categorize gat-only
        gat_only_high_conf = set()
        gat_only_low_conf = set()
        for idx in gat_only:
            conf = gat_entities[idx].get('confidence', 0.0)
            if conf >= high_confidence_threshold:
                gat_only_high_conf.add(idx)
            else:
                gat_only_low_conf.add(idx)

        # FIX 8: Add quality metrics
        total_regex = len(regex_entities)
        total_gat = len(gat_entities)
        matches = len(matched_pairs)

        precision = matches / total_gat if total_gat > 0 else 0
        recall = matches / total_regex if total_regex > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        return {
            'matched_pairs': matched_pairs,
            'match_details': match_details,
            'regex_only_high_conf': regex_only_high_conf,
            'regex_only_low_conf': regex_only_low_conf,
            'gat_only_high_conf': gat_only_high_conf,
            'gat_only_low_conf': gat_only_low_conf,
            'match_quality': {
                'precision': precision,
                'recall': recall,
                'f1_score': f1,
                'matches': matches,
                'total_regex': total_regex,
                'total_gat': total_gat,
            },
            'type_group_performance': self._analyze_type_group_performance(
                regex_entities, gat_entities, matched_pairs
            ),
        }

    def _analyze_type_group_performance(
        self,
        regex_entities: List[Dict],
        gat_entities: List[Dict],
        matched_pairs: Set[Tuple[int, int]],
    ) -> Dict[str, Dict]:
        """Analyze performance by type group."""
        group_stats = {}

        # Initialize stats for each group
        for i, group in enumerate(self.type_groups):
            group_name = f"Group_{i}"
            group_stats[group_name] = {
                'types': list(group),
                'regex_count': 0,
                'gat_count': 0,
                'matches': 0,
                'precision': 0,
                'recall': 0,
            }

        # Count regex entities by group
        for regex_idx, regex_ent in enumerate(regex_entities):
            for group_name, stats in group_stats.items():
                if regex_ent['type'] in stats['types']:
                    stats['regex_count'] += 1
                    break

        # Count GAT entities by group
        for gat_idx, gat_ent in enumerate(gat_entities):
            for group_name, stats in group_stats.items():
                if gat_ent['type'] in stats['types']:
                    stats['gat_count'] += 1
                    break

        # Count matches by group
        for regex_idx, gat_idx in matched_pairs:
            regex_type = regex_entities[regex_idx]['type']
            for group_name, stats in group_stats.items():
                if regex_type in stats['types']:
                    stats['matches'] += 1
                    break

        # Calculate metrics
        for stats in group_stats.values():
            if stats['gat_count'] > 0:
                stats['precision'] = stats['matches'] / stats['gat_count']
            if stats['regex_count'] > 0:
                stats['recall'] = stats['matches'] / stats['regex_count']

        return group_stats

    def find_matches_detailed(
        self,
        regex_entities: List[Dict],
        gat_entities: List[Dict],
        iou_threshold: float = 0.5,
        type_threshold: float = 0.8,
        high_confidence_threshold: float = 0.85,
    ) -> Dict[str, Any]:
        """
        Legacy method for backward compatibility.
        Now just calls the enhanced find_matches with return_detailed=True.
        """
        return self.find_matches(
            regex_entities,
            gat_entities,
            iou_threshold,
            type_threshold,
            return_detailed=True
        )

### VISUALIZATION

In [ ]:
PALETTE = {
    "bg":        "#ffffff",
    "panel":     "#f7f8fa",
    "border":    "#dde1e7",
    "text":      "#1a1d23",
    "muted":     "#6b7280",
    "blue":      "#2563eb",
    "orange":    "#ea580c",
    "green":     "#16a34a",
    "purple":    "#7c3aed",
    "yellow":    "#d97706",
    "teal":      "#0891b2",
    "pink":      "#db2777",
    "grid":      "#e5e7eb",
    # FIX 1: Add entity-type specific colors for consistency
    "standard":  "#7c3aed",  # Purple for STANDARD
    "equipment": "#ea580c",  # Orange for EQUIPMENT
    "dimension": "#0891b2",  # Teal for DIMENSION
    "material":  "#16a34a",  # Green for MATERIAL
}

CMAP_BLUE   = mpl.colors.LinearSegmentedColormap.from_list("blu",  ["#dbeafe", "#2563eb"])
CMAP_DIVERG = mpl.colors.LinearSegmentedColormap.from_list("div",  ["#ea580c", "#f9fafb", "#2563eb"])


def _apply_theme() -> None:
    mpl.rcParams.update({
        "figure.facecolor":  PALETTE["bg"],
        "axes.facecolor":    PALETTE["panel"],
        "axes.edgecolor":    PALETTE["border"],
        "axes.labelcolor":   PALETTE["text"],
        "axes.titlecolor":   PALETTE["text"],
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "axes.grid":         True,
        "grid.color":        PALETTE["grid"],
        "grid.linewidth":    0.7,
        "grid.alpha":        1.0,
        "xtick.color":       PALETTE["muted"],
        "ytick.color":       PALETTE["muted"],
        "xtick.labelsize":   9,
        "ytick.labelsize":   9,
        "text.color":        PALETTE["text"],
        "legend.facecolor":  PALETTE["bg"],
        "legend.edgecolor":  PALETTE["border"],
        "legend.labelcolor": PALETTE["text"],
        "legend.framealpha": 1.0,
        "font.family":       "DejaVu Sans",
        "figure.dpi":        150,
        "savefig.facecolor": PALETTE["bg"],
        "savefig.dpi":       200,
    })


_apply_theme()

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _save(fig: plt.Figure, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, bbox_inches="tight", pad_inches=0.3, facecolor=PALETTE["bg"])
    plt.close(fig)
    print(f"  ✅  Saved → {path}")


def _title_block(ax: plt.Axes, title: str, subtitle: str = "") -> None:
    """Write a styled title + optional subtitle above an axes."""
    ax.set_title(title, fontsize=12, fontweight="bold",
                 color=PALETTE["text"], pad=10 if not subtitle else 4)
    if subtitle:
        ax.text(0.5, 1.015, subtitle, transform=ax.transAxes,
                ha="center", va="bottom", fontsize=8,
                color=PALETTE["muted"], style="italic")


def _add_value_labels(ax: plt.Axes, bars, fmt: str = "{:.0f}",
                      color: str | None = None, fontsize: int = 9,
                      offset_frac: float = 0.012) -> None:
    ymax = ax.get_ylim()[1]
    for bar in bars:
        h = bar.get_height()
        if h == 0:
            continue
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + ymax * offset_frac,
                fmt.format(h),
                ha="center", va="bottom",
                fontsize=fontsize, fontweight="bold",
                color=color or PALETTE["muted"])


def _phase_vline(ax: plt.Axes, transitions: List[int]) -> None:
    for i, pt in enumerate(transitions):
        ax.axvline(x=pt, color=PALETTE["orange"], lw=1.2,
                   linestyle="--", alpha=0.6,
                   label="Phase transition" if i == 0 else "")


def _smooth(values: List[float], window: int = 5) -> np.ndarray:
    """Simple moving-average smoother for training curves."""
    arr = np.array(values, dtype=float)
    if len(arr) < window:
        return arr
    kernel = np.ones(window) / window
    padded = np.pad(arr, (window // 2, window // 2), mode="edge")
    return np.convolve(padded, kernel, mode="valid")[: len(arr)]


# ─────────────────────────────────────────────────────────────────────────────
# MAIN CLASS
# ─────────────────────────────────────────────────────────────────────────────

class Visualizer:
    """
    Enhanced Visualizer for RegCheckAI.
    All plots use a clean white theme suitable for thesis/publication.
    """

    # ------------------------------------------------------------------
    # INIT
    # ------------------------------------------------------------------

    def __init__(self, config: Dict):
        self.config = config
        self.visual_dir    = Path(config["visual_dir"])
        os.makedirs(self.visual_dir, exist_ok=True)

        self.output_dir = Path(config.get("output_dir", "./output"))
        self.comparison_dir = self.output_dir / "comparison" / "visualizations"
        self.comparison_dir.mkdir(parents=True, exist_ok=True)

    # ══════════════════════════════════════════════════════════════════
    # 1. CURRICULUM TRAINING HISTORY
    # ══════════════════════════════════════════════════════════════════

    def plot_curriculum_history(self, history: Dict, filename: Optional[str] = None) -> str:
        """4-panel training dashboard with phase-transition bands."""

        # ── KEY ALIASING: support both 'test_*' and 'val_*' naming conventions ──
        def _h(primary: str, *fallbacks: str) -> list:
            for k in (primary, *fallbacks):
                if k in history and history[k]:
                    return list(history[k])
            raise KeyError(
                f"History dict missing '{primary}' (also tried {fallbacks}). "
                f"Available keys: {list(history.keys())}"
            )

        train_loss     = _h("train_loss")
        test_accuracy  = _h("test_accuracy",  "val_accuracy",  "accuracy")
        test_f1        = _h("test_f1",        "val_f1",        "f1")
        learning_rates = _h("learning_rates", "lr")

        fig = plt.figure(figsize=(18, 10))
        fig.suptitle("RegCheckAI  ·  Curriculum Training History",
                    fontsize=16, fontweight="bold", color=PALETTE["text"], y=0.98)

        gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.32)
        axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

        n      = len(train_loss)
        epochs = range(1, n + 1)
        pts    = history.get("phase_transitions", [])

        # Shade curriculum phases
        phase_colors = ["#dbeafe", "#ede9fe", "#ffedd5", "#dcfce7"]
        phase_edge   = [PALETTE["blue"], PALETTE["purple"],
                        PALETTE["orange"], PALETTE["green"]]
        boundaries   = [0] + pts + [n]
        for ax in axes:
            for i in range(len(boundaries) - 1):
                ax.axvspan(boundaries[i] + 0.5, boundaries[i + 1] + 0.5,
                          alpha=0.45, color=phase_colors[i % len(phase_colors)],
                          zorder=0)

        # ── Loss ──────────────────────────────────────────────
        axes[0].plot(epochs, train_loss,
                    lw=2.2, color=PALETTE["blue"], label="Train Loss", zorder=3)
        _phase_vline(axes[0], pts)
        _title_block(axes[0], "Training Loss", "lower = better")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
        axes[0].legend(fontsize=8)

        # ── Accuracy ──────────────────────────────────────────
        axes[1].plot(epochs, test_accuracy,
                    lw=2.2, color=PALETTE["green"], label="Test Accuracy", zorder=3)
        _phase_vline(axes[1], pts)
        best_acc_ep = int(np.argmax(test_accuracy)) + 1
        best_acc    = max(test_accuracy)
        axes[1].scatter([best_acc_ep], [best_acc], color=PALETTE["yellow"],
                        s=80, zorder=5, label=f"Best {best_acc:.3f}",
                        edgecolors=PALETTE["text"], linewidths=0.8)
        _title_block(axes[1], "Test Accuracy", "higher = better")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
        axes[1].legend(fontsize=8)

        # ── F1 ────────────────────────────────────────────────
        axes[2].plot(epochs, test_f1,
                    lw=2.2, color=PALETTE["orange"], label="Test F1", zorder=3)
        _phase_vline(axes[2], pts)
        best_f1_ep = int(np.argmax(test_f1)) + 1
        best_f1    = max(test_f1)
        axes[2].scatter([best_f1_ep], [best_f1], color=PALETTE["yellow"],
                        s=80, zorder=5, label=f"Best {best_f1:.3f}",
                        edgecolors=PALETTE["text"], linewidths=0.8)
        axes[2].axhline(y=best_f1, color=PALETTE["yellow"],
                        lw=0.9, linestyle=":", alpha=0.7)
        _title_block(axes[2], "Test F1 Score", "higher = better")
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1 Score")
        axes[2].legend(fontsize=8)

        # ── Learning Rate ─────────────────────────────────────
        axes[3].plot(epochs, learning_rates,
                    lw=2.2, color=PALETTE["purple"], label="LR", zorder=3)
        _phase_vline(axes[3], pts)
        axes[3].set_yscale("log")
        _title_block(axes[3], "Learning Rate Schedule", "log scale")
        axes[3].set_xlabel("Epoch"); axes[3].set_ylabel("Learning Rate")
        axes[3].legend(fontsize=8)

        # Phase legend
        handles = [mpatches.Patch(color=phase_colors[i % len(phase_colors)],
                                  edgecolor=phase_edge[i % len(phase_edge)],
                                  linewidth=1.2,
                                  label=f"Phase {i + 1}")
                  for i in range(len(boundaries) - 1)]
        fig.legend(handles=handles, loc="lower center", ncol=4,
                  fontsize=9, framealpha=1.0,
                  bbox_to_anchor=(0.5, -0.01))

        path = filename if filename else f"{self.visual_dir}/curriculum_training_history.png"
        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 2. STANDARD TRAINING HISTORY
    # ══════════════════════════════════════════════════════════════════

    def plot_training_history(self, history: Dict, filename: Optional[str] = None) -> str:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle("Training History", fontsize=14, fontweight="bold",
                     color=PALETTE["text"])

        configs = [
            ("train_loss",    "Training Loss",    PALETTE["blue"],   "Loss"),
            ("test_accuracy", "Test Accuracy",    PALETTE["green"],  "Accuracy"),
            ("test_f1",       "Test F1 Score",    PALETTE["orange"], "F1"),
        ]
        for ax, (key, title, color, ylabel) in zip(axes, configs):
            vals = history[key]
            epochs = range(1, len(vals) + 1)
            ax.fill_between(epochs, vals, alpha=0.10, color=color)
            ax.plot(epochs, vals, lw=2.2, color=color)
            best_i = (np.argmin if "loss" in key else np.argmax)(vals)
            ax.scatter([best_i + 1], [vals[best_i]],
                       color=PALETTE["yellow"], s=70, zorder=5,
                       edgecolors=PALETTE["text"], linewidths=0.8,
                       label=f"Best: {vals[best_i]:.4f}")
            _title_block(ax, title)
            ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
            ax.legend(fontsize=8)

        fig.tight_layout(rect=[0, 0, 1, 0.95])

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = f"{self.visual_dir}/training_history.png"

        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 2b. TRAINING CURVES
    # ══════════════════════════════════════════════════════════════════

    def plot_training_curves(
        self,
        history: Dict,
        smooth_window: int = 5,
        filename: Optional[str] = None,
    ) -> str:
        """
        Publication-quality training-curve dashboard with six panels.
        """
        n      = len(history["train_loss"])
        epochs = np.arange(1, n + 1)
        pts    = history.get("phase_transitions", [])

        phase_colors = ["#dbeafe", "#ede9fe", "#ffedd5", "#dcfce7"]
        phase_edge   = [PALETTE["blue"], PALETTE["purple"],
                        PALETTE["orange"], PALETTE["green"]]
        boundaries   = [0] + pts + [n]

        fig = plt.figure(figsize=(22, 12))
        fig.suptitle(
            "RegCheckAI  ·  Training Curves  (Train vs Validation)",
            fontsize=16, fontweight="bold", color=PALETTE["text"], y=0.99,
        )
        gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.32,
                               top=0.93, bottom=0.08, left=0.06, right=0.97)

        panel_cfg = [
            (0, 0, "train_loss",       "val_loss",       "Loss",               "Loss",       True),
            (0, 1, "train_accuracy",   "val_accuracy",   "Accuracy",           "Accuracy",   False),
            (0, 2, "train_f1",         "val_f1",         "F1 Score",           "F1",         False),
            (1, 0, "train_precision",  "val_precision",  "Precision",          "Precision",  False),
            (1, 1, "train_recall",     "val_recall",     "Recall",             "Recall",     False),
            (1, 2, "learning_rates",   None,             "Learning Rate",      "LR",         True),
        ]

        train_color = PALETTE["blue"]
        val_color   = PALETTE["orange"]
        best_color  = PALETTE["yellow"]

        for row, col, tk, vk, title, ylabel, lower in panel_cfg:
            ax = fig.add_subplot(gs[row, col])

            # Phase shading
            for i in range(len(boundaries) - 1):
                ax.axvspan(boundaries[i] + 0.5, boundaries[i + 1] + 0.5,
                           alpha=0.30, color=phase_colors[i % len(phase_colors)],
                           zorder=0)

            _phase_vline(ax, pts)

            train_vals = history.get(tk)
            if train_vals is None:
                ax.axis("off")
                continue

            train_vals = list(train_vals)
            train_sm   = _smooth(train_vals, smooth_window)

            if vk is None:
                ax.plot(epochs, train_vals, lw=2.0, color=PALETTE["purple"],
                        alpha=0.35, zorder=2)
                ax.plot(epochs, train_sm, lw=2.5, color=PALETTE["purple"],
                        zorder=3, label="LR")
                ax.set_yscale("log")
                _title_block(ax, title, "log scale")
                ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
                ax.legend(fontsize=8)
                continue

            ax.plot(epochs, train_vals, lw=1.2, color=train_color,
                    alpha=0.25, zorder=2)
            ax.plot(epochs, train_sm,   lw=2.4, color=train_color,
                    zorder=4, label="Train (smoothed)")

            val_vals = history.get(vk)
            if val_vals is not None:
                val_vals = list(val_vals)
                val_sm   = _smooth(val_vals, smooth_window)
                ax.plot(epochs, val_vals, lw=1.2, color=val_color,
                        alpha=0.25, zorder=2)
                ax.plot(epochs, val_sm,   lw=2.4, color=val_color,
                        zorder=4, label="Val (smoothed)", linestyle="--")

                best_i = int(np.argmin(val_sm)) if lower else int(np.argmax(val_sm))
                best_v = val_sm[best_i]
                ax.scatter([epochs[best_i]], [best_v],
                           s=90, color=best_color, zorder=6,
                           edgecolors=PALETTE["text"], linewidths=0.8,
                           label=f"Best val: {best_v:.4f}")
                ax.axhline(best_v, color=best_color, lw=0.9,
                           linestyle=":", alpha=0.7)

                final_gap = abs(train_sm[-1] - val_sm[-1])
                ax.text(0.98, 0.03, f"Δfinal={final_gap:.3f}",
                        transform=ax.transAxes, ha="right", va="bottom",
                        fontsize=7.5, color=PALETTE["muted"], style="italic")
            else:
                best_i = int(np.argmin(train_sm)) if lower else int(np.argmax(train_sm))
                best_v = train_sm[best_i]
                ax.scatter([epochs[best_i]], [best_v],
                           s=90, color=best_color, zorder=6,
                           edgecolors=PALETTE["text"], linewidths=0.8,
                           label=f"Best: {best_v:.4f}")

            if val_vals is not None:
                val_sm_arr   = _smooth(val_vals,   smooth_window)
                ax.fill_between(epochs, train_sm, val_sm_arr,
                                alpha=0.07, color=PALETTE["purple"])

            subtitle = "lower = better" if lower else "higher = better"
            _title_block(ax, title, subtitle)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(ylabel)
            ax.legend(fontsize=8, loc="upper right" if not lower else "upper left")

        phase_handles = [
            mpatches.Patch(color=phase_colors[i % len(phase_colors)],
                           edgecolor=phase_edge[i % len(phase_edge)],
                           linewidth=1.2, label=f"Phase {i + 1}")
            for i in range(len(boundaries) - 1)
        ]
        if phase_handles:
            fig.legend(handles=phase_handles, loc="lower center", ncol=len(phase_handles),
                       fontsize=9, framealpha=1.0, bbox_to_anchor=(0.5, 0.0))

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = f"{self.visual_dir}/training_curves.png"

        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 3. CONFUSION MATRIX
    # ══════════════════════════════════════════════════════════════════

    def plot_confusion_matrix_enhanced(
        self,
        y_true,
        y_pred,
        label_map: Optional[Dict[int, str]] = None,
        normalize: str = "true",
        filename: Optional[str] = None,
        top_n: Optional[int] = None,
    ) -> str:
        """
        Side-by-side confusion matrix dashboard.
        """
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        all_labels = sorted(set(y_true) | set(y_pred))
        names      = ([label_map[i] for i in all_labels]
                      if label_map else [str(i) for i in all_labels])

        if top_n is not None and top_n < len(all_labels):
            freq = Counter(y_true.tolist())
            top_ids = [lbl for lbl, _ in freq.most_common(top_n)]
            mask    = np.isin(y_true, top_ids) & np.isin(y_pred, top_ids)
            y_true  = y_true[mask]
            y_pred  = y_pred[mask]
            all_labels = sorted(set(y_true) | set(y_pred))
            names   = ([label_map[i] for i in all_labels]
                       if label_map else [str(i) for i in all_labels])

        cm_raw = confusion_matrix(y_true, y_pred, labels=all_labels)
        n_cls  = len(all_labels)

        if normalize in ("true", "row", True):
            s    = cm_raw.sum(axis=1, keepdims=True); s[s == 0] = 1
            cm_n = cm_raw.astype(float) / s
            norm_label = "Row-normalised  (Recall per class)"
        elif normalize in ("pred", "col"):
            s    = cm_raw.sum(axis=0, keepdims=True); s[s == 0] = 1
            cm_n = cm_raw.astype(float) / s
            norm_label = "Col-normalised  (Precision per class)"
        else:
            cm_n = cm_raw.astype(float)
            norm_label = "Raw counts"

        report   = classification_report(y_true, y_pred, labels=all_labels,
                                         output_dict=True, zero_division=0)
        prec_lst = [report.get(str(l), {}).get("precision", 0.0) for l in all_labels]
        rec_lst  = [report.get(str(l), {}).get("recall",    0.0) for l in all_labels]
        f1_lst   = [report.get(str(l), {}).get("f1-score",  0.0) for l in all_labels]

        cell  = max(0.50, min(0.75, 12.0 / max(n_cls, 1)))
        hm_w  = n_cls * cell
        bar_w = max(8.0, n_cls * 0.38)
        fig_w = hm_w + bar_w + 2.5

        fig   = plt.figure(figsize=(fig_w, max(9, n_cls * cell)))
        fig.suptitle(
            "RegCheckAI  ·  Confusion Matrix  &  Per-Class Metrics",
            fontsize=14, fontweight="bold", color=PALETTE["text"], y=1.00,
        )
        gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.08,
                               width_ratios=[hm_w, bar_w])

        ax_hm = fig.add_subplot(gs[0, 0])
        im    = ax_hm.imshow(cm_n, cmap=CMAP_BLUE, aspect="auto", vmin=0, vmax=1)

        cbar = fig.colorbar(im, ax=ax_hm, fraction=0.03, pad=0.015)
        cbar.ax.tick_params(labelsize=8, color=PALETTE["muted"])
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color=PALETTE["muted"])

        ax_hm.set_xticks(range(n_cls))
        ax_hm.set_yticks(range(n_cls))
        ax_hm.set_xticklabels(names, rotation=45, ha="right",
                               fontsize=max(5.5, min(8, 90 / n_cls)))
        ax_hm.set_yticklabels(names, fontsize=max(5.5, min(8, 90 / n_cls)))
        ax_hm.set_xlabel("Predicted", fontsize=11, fontweight="bold", color=PALETTE["text"])
        ax_hm.set_ylabel("True", fontsize=11, fontweight="bold", color=PALETTE["text"])
        _title_block(ax_hm, "Confusion Matrix", norm_label)

        thresh = 0.5
        skip_zero = n_cls > 20
        for i, j in np.ndindex(cm_n.shape):
            v = cm_n[i, j]
            if skip_zero and v == 0:
                continue
            label_txt = (f"{v:.2f}\n({cm_raw[i,j]})"
                         if normalize else f"{cm_raw[i,j]:d}")
            ax_hm.text(j, i, label_txt, ha="center", va="center",
                       fontsize=max(5, min(7.5, 70 / n_cls)),
                       color="white" if v > thresh else PALETTE["text"],
                       fontweight="bold" if v > thresh else "normal")

        ax_bar = fig.add_subplot(gs[0, 1])
        ax_bar.set_facecolor(PALETTE["panel"])

        sort_order  = np.argsort(f1_lst)
        sorted_names = [names[i]    for i in sort_order]
        sorted_prec  = [prec_lst[i] for i in sort_order]
        sorted_rec   = [rec_lst[i]  for i in sort_order]
        sorted_f1    = [f1_lst[i]   for i in sort_order]

        y_pos = np.arange(n_cls)
        h     = max(0.18, min(0.28, 6.0 / n_cls))

        ax_bar.barh(y_pos + h, sorted_prec, height=h,
                    color=PALETTE["blue"], alpha=0.85,
                    label="Precision", edgecolor="none")
        ax_bar.barh(y_pos, sorted_rec, height=h,
                    color=PALETTE["teal"], alpha=0.85,
                    label="Recall", edgecolor="none")
        ax_bar.barh(y_pos - h, sorted_f1, height=h,
                    color=PALETTE["orange"], alpha=0.85,
                    label="F1", edgecolor="none")

        ax_bar.set_yticks(y_pos)
        ax_bar.set_yticklabels(sorted_names,
                               fontsize=max(5.5, min(8, 90 / n_cls)))
        ax_bar.set_xlim(0, 1.18)
        ax_bar.set_xlabel("Score", fontsize=10)
        ax_bar.axvline(0.5, color=PALETTE["border"], lw=0.9,
                       linestyle="--", alpha=0.8)
        ax_bar.axvline(1.0, color=PALETTE["border"], lw=0.9,
                       linestyle="--", alpha=0.8)

        for i, f1v in enumerate(sorted_f1):
            if f1v > 0.01:
                ax_bar.text(f1v + 0.015, y_pos[i] - h / 2,
                            f"{f1v:.2f}",
                            va="center", fontsize=max(5, min(7, 70 / n_cls)),
                            color=PALETTE["orange"], fontweight="bold")

        _title_block(ax_bar, "Per-Class  P / R / F1",
                     "sorted by F1  ·  worst→best (bottom→top)")
        ax_bar.legend(fontsize=8, loc="lower right",
                      framealpha=1.0, edgecolor=PALETTE["border"])

        macro = report.get("macro avg", {})
        footer = (
            f"Macro  P={macro.get('precision', 0):.3f}  "
            f"R={macro.get('recall', 0):.3f}  "
            f"F1={macro.get('f1-score', 0):.3f}   |   "
            f"Support = {int(macro.get('support', len(y_true)))}"
        )
        fig.text(0.5, -0.01, footer, ha="center", fontsize=9,
                 color=PALETTE["muted"], style="italic")

        fig.tight_layout(rect=[0, 0.02, 1, 0.98])

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = f"{self.visual_dir}/confusion_matrix_enhanced.png"

        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 4. ENTITY DISTRIBUTION
    # ══════════════════════════════════════════════════════════════════

    def plot_entity_distribution(self, entities: List[Dict], filename: Optional[str] = None) -> str:
        """Entity distribution plot with proper path handling."""
        counts = Counter(e["type"] for e in entities)
        types, vals = zip(*sorted(counts.items(), key=lambda x: x[1], reverse=True))

        n = len(types)
        colors = plt.cm.Blues(np.linspace(0.35, 0.85, n))

        fig, ax = plt.subplots(figsize=(max(14, n * 0.55), 7))
        bars = ax.bar(range(n), vals, color=colors, width=0.7,
                      edgecolor=PALETTE["border"], linewidth=0.6)

        ax.set_xticks(range(n))
        ax.set_xticklabels(types, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Count", fontsize=11)
        _title_block(ax, "Extracted Entity Distribution",
                    f"{len(entities):,} total entities · {n} types")
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        _add_value_labels(ax, bars, fontsize=8)

        fig.tight_layout()

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = f"{self.visual_dir}/entity_distribution.png"

        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 5. COMPARISON DASHBOARD
    # ══════════════════════════════════════════════════════════════════

    def plot_comparison_dashboard(self,
                                  regex_entities: List[Dict],
                                  gat_entities:   List[Dict],
                                  comparison_stats: Dict,
                                  filename: Optional[str] = None) -> str:
        """Comparison dashboard with proper path handling."""
        fig = plt.figure(figsize=(22, 14))
        fig.patch.set_facecolor(PALETTE["bg"])
        fig.suptitle("Regex  vs  GAT  ·  Entity Extraction Analysis",
                     fontsize=17, fontweight="bold",
                     color=PALETTE["text"], y=0.99)

        gs = gridspec.GridSpec(3, 3, figure=fig,
                               hspace=0.55, wspace=0.38,
                               top=0.94, bottom=0.07,
                               left=0.06, right=0.97)

        relaxed = comparison_stats.get("relaxed", comparison_stats)

        kpi_ax = fig.add_subplot(gs[0, :])
        kpi_ax.set_axis_off()
        kpi_ax.set_facecolor(PALETTE["bg"])

        kpi_data = [
            ("Regex Entities",   comparison_stats.get("total_regex", 0),  PALETTE["blue"],   "After dedup"),
            ("GAT Entities",     comparison_stats.get("total_gat",   0),  PALETTE["orange"], "After dedup"),
            ("Found by Both",    relaxed.get("found_by_both",        0),  PALETTE["green"],  "Relaxed match"),
            ("Regex Only",       relaxed.get("regex_only",           0),  PALETTE["muted"],  "Relaxed match"),
            ("GAT Novel",        relaxed.get("gat_only",             0),  PALETTE["purple"], "New discoveries"),
            ("GAT Improvement",  f"+{comparison_stats.get('gat_improvement', 0):.1f}%",
                                                                           PALETTE["yellow"], "vs Regex baseline"),
        ]

        for idx, (label, value, color, sub) in enumerate(kpi_data):
            x = (idx + 0.5) / len(kpi_data)
            rect = FancyBboxPatch((x - 0.077, 0.05), 0.148, 0.88,
                                  boxstyle="round,pad=0.02",
                                  linewidth=1.5, edgecolor=PALETTE["border"],
                                  facecolor=PALETTE["panel"],
                                  transform=kpi_ax.transAxes, clip_on=False)
            kpi_ax.add_patch(rect)
            accent = FancyBboxPatch((x - 0.077, 0.82), 0.148, 0.11,
                                    boxstyle="round,pad=0.01",
                                    linewidth=0, edgecolor="none",
                                    facecolor=color, alpha=0.18,
                                    transform=kpi_ax.transAxes, clip_on=False)
            kpi_ax.add_patch(accent)
            kpi_ax.text(x, 0.64, str(value) if isinstance(value, str) else f"{value:,}",
                        transform=kpi_ax.transAxes,
                        ha="center", va="center",
                        fontsize=22, fontweight="bold", color=color)
            kpi_ax.text(x, 0.40, label,
                        transform=kpi_ax.transAxes,
                        ha="center", fontsize=9, fontweight="bold",
                        color=PALETTE["text"])
            kpi_ax.text(x, 0.20, sub,
                        transform=kpi_ax.transAxes,
                        ha="center", fontsize=7.5,
                        color=PALETTE["muted"])

        ax_cnt = fig.add_subplot(gs[1, 0])
        cats   = ["Regex\nTotal", "GAT\nTotal", "Both\n(Relaxed)", "GAT\nNovel"]
        vals   = [comparison_stats.get("total_regex", 0),
                  comparison_stats.get("total_gat",   0),
                  relaxed.get("found_by_both", 0),
                  relaxed.get("gat_only",      0)]
        colors_cnt = [PALETTE["blue"], PALETTE["orange"],
                      PALETTE["green"], PALETTE["purple"]]
        bars = ax_cnt.bar(cats, vals, color=colors_cnt, width=0.55, edgecolor="none")
        _add_value_labels(ax_cnt, bars)
        _title_block(ax_cnt, "Entity Count Overview")
        ax_cnt.set_ylabel("Count")

        ax_conf = fig.add_subplot(gs[1, 1])
        methods = ["Regex", "GAT"]
        confs   = [comparison_stats.get("regex_avg_confidence", 0),
                   comparison_stats.get("gat_avg_confidence",   0)]
        bars = ax_conf.bar(methods, confs,
                           color=[PALETTE["blue"], PALETTE["orange"]],
                           width=0.45, edgecolor="none")
        ax_conf.set_ylim(0, 1.15)
        ax_conf.axhline(1.0, color=PALETTE["border"], lw=1.0, linestyle="--")
        _add_value_labels(ax_conf, bars, fmt="{:.3f}")
        _title_block(ax_conf, "Average Confidence Score")
        ax_conf.set_ylabel("Score")

        ax_mtch = fig.add_subplot(gs[1, 2])
        strict  = comparison_stats.get("strict", comparison_stats)

        methods_m = ["Strict\n(IoU≥0.8)", "Relaxed\n(IoU≥0.5)"]
        both_v  = [strict.get("found_by_both", 0), relaxed.get("found_by_both", 0)]
        ro_v    = [strict.get("regex_only",    0), relaxed.get("regex_only",    0)]
        go_v    = [strict.get("gat_only",      0), relaxed.get("gat_only",      0)]

        x = np.arange(len(methods_m))
        w = 0.45
        b1 = ax_mtch.bar(x, both_v, w, label="Both",      color=PALETTE["green"],  edgecolor="none")
        b2 = ax_mtch.bar(x, ro_v,   w, bottom=both_v,     label="Regex only", color=PALETTE["blue"],   edgecolor="none")
        b3 = ax_mtch.bar(x, go_v,   w,
                         bottom=[a+b for a,b in zip(both_v, ro_v)],
                         label="GAT only", color=PALETTE["purple"], edgecolor="none")
        ax_mtch.set_xticks(x); ax_mtch.set_xticklabels(methods_m, fontsize=9)
        ax_mtch.legend(fontsize=7, loc="upper right",
                       framealpha=1.0, edgecolor=PALETTE["border"])
        _title_block(ax_mtch, "Strict vs Relaxed Matching")
        ax_mtch.set_ylabel("Entity Count")

        ax_types = fig.add_subplot(gs[2, :])
        rt = comparison_stats.get("regex_type_distribution", {})
        gt = comparison_stats.get("gat_type_distribution",   {})

        all_t  = set(rt) | set(gt)
        totals = {t: rt.get(t, 0) + gt.get(t, 0) for t in all_t}
        top10  = [t for t, _ in sorted(totals.items(), key=lambda x: x[1], reverse=True)[:12]]

        x  = np.arange(len(top10))
        w2 = 0.38
        b1 = ax_types.bar(x - w2/2, [rt.get(t,0) for t in top10], w2,
                          label="Regex", color=PALETTE["blue"],
                          edgecolor="none")
        b2 = ax_types.bar(x + w2/2, [gt.get(t,0) for t in top10], w2,
                          label="GAT",   color=PALETTE["orange"],
                          edgecolor="none")

        for bar in list(b1) + list(b2):
            h = bar.get_height()
            if h > 0:
                ax_types.text(bar.get_x() + bar.get_width()/2, h + 1,
                              str(int(h)), ha="center", va="bottom",
                              fontsize=7.5, color=PALETTE["muted"])

        ax_types.set_xticks(x)
        ax_types.set_xticklabels(top10, rotation=35, ha="right", fontsize=9)
        ax_types.legend(fontsize=9, framealpha=1.0, edgecolor=PALETTE["border"])
        ax_types.set_ylabel("Count")
        _title_block(ax_types, "Top 12 Entity Types  ·  Regex vs GAT",
                     "sorted by combined count")

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = str(self.comparison_dir / "comparison_dashboard.png")

        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 6. VENN DIAGRAM
    # ══════════════════════════════════════════════════════════════════

    def plot_comparison_venn(self,
                         regex_entities: List[Dict],
                         gat_entities:   List[Dict],
                         relaxed_stats:  Optional[Dict] = None,
                         filename:       Optional[str] = None) -> Dict:

        """
        Create a Venn diagram showing overlap between regex and GAT entities.

        If relaxed_stats is provided, it must contain the keys:
            'found_by_both', 'regex_only', 'gat_only', 'total_regex', 'total_gat'
        Those numbers (from fuzzy matching) will be displayed.
        Otherwise, overlap is computed by exact text matching (legacy).
        """
        if relaxed_stats:
            both  = relaxed_stats['found_by_both']
            ro    = relaxed_stats['regex_only']
            go    = relaxed_stats['gat_only']
            total_regex = relaxed_stats.get('total_regex', len(regex_entities))
            total_gat   = relaxed_stats.get('total_gat',   len(gat_entities))
            pct   = both / total_regex * 100 if total_regex else 0.0
        else:
            # Fallback: exact text match
            def _texts(ents):
                return {e.get("entity_value", e.get("text", str(e)))
                        if isinstance(e, dict) else str(e) for e in ents}
            rs = _texts(regex_entities)
            gs = _texts(gat_entities)
            both = rs & gs
            ro   = rs - gs
            go   = gs - rs
            total_regex = len(rs)
            total_gat   = len(gs)
            pct   = len(both) / total_regex * 100 if total_regex else 0
        fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                                 gridspec_kw={"width_ratios": [1.4, 1]})
        fig.suptitle("Entity Overlap  ·  Regex vs GAT",
                     fontsize=14, fontweight="bold", color=PALETTE["text"])

        ax = axes[0]
        ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.set_aspect("equal")
        ax.set_facecolor(PALETTE["bg"])
        ax.axis("off")

        c1 = Circle((3.5, 3.5), 2.6, color=PALETTE["blue"],   alpha=0.12)
        c2 = Circle((6.5, 3.5), 2.6, color=PALETTE["orange"], alpha=0.12)
        ax.add_patch(c1); ax.add_patch(c2)
        for cx, col in [(3.5, PALETTE["blue"]), (6.5, PALETTE["orange"])]:
            ax.add_patch(Circle((cx, 3.5), 2.6, fill=False,
                                edgecolor=col, linewidth=2.2))
        _ro   = ro   if isinstance(ro,   int) else len(ro)
        _both = both if isinstance(both, int) else len(both)
        _go   = go   if isinstance(go,   int) else len(go)
        for xp, txt, val, col in [
            (2.0, "Regex Only", f"{_ro:,}",   PALETTE["blue"]),
            (5.0, "Both",       f"{_both:,}", PALETTE["text"]),
            (8.0, "GAT Only",   f"{_go:,}",  PALETTE["orange"]),
        ]:
            ax.text(xp, 4.2, txt, ha="center", fontsize=11,
                    fontweight="bold", color=col)
            ax.text(xp, 3.4, val, ha="center", fontsize=24,
                    fontweight="bold", color=col)

        ax.text(5.0, 2.6, f"{pct:.1f}% overlap",
                ha="center", fontsize=9, color=PALETTE["muted"])

        ax.text(3.5, 0.5, "REGEX", ha="center", fontsize=10,
                color=PALETTE["blue"], fontweight="bold")
        ax.text(6.5, 0.5, "GAT",   ha="center", fontsize=10,
                color=PALETTE["orange"], fontweight="bold")

        ax2 = axes[1]
        ax2.set_facecolor(PALETTE["bg"])
        ax2.axis("off")

        rows = [
            ("Regex total",     f"{total_regex:,}",                          PALETTE["blue"]),
            ("GAT total",       f"{total_gat:,}",                            PALETTE["orange"]),
            ("Found by both",   f"{_both:,}",                                PALETTE["green"]),
            ("Regex only",      f"{_ro:,}",                                  PALETTE["muted"]),
            ("GAT novel",       f"{_go:,}",                                  PALETTE["purple"]),
            ("Overlap %",       f"{pct:.1f}%",                               PALETTE["yellow"]),
            ("GAT improvement", f"+{_go / max(total_regex, 1) * 100:.1f}%",  PALETTE["orange"]),
        ]
        y = 0.92
        for label, val, col in rows:
            rect = FancyBboxPatch((0.03, y - 0.07), 0.94, 0.09,
                                  boxstyle="round,pad=0.01",
                                  linewidth=1, edgecolor=PALETTE["border"],
                                  facecolor=PALETTE["panel"],
                                  transform=ax2.transAxes, clip_on=False)
            ax2.add_patch(rect)
            ax2.text(0.10, y - 0.02, label,
                     transform=ax2.transAxes,
                     fontsize=10, color=PALETTE["text"])
            ax2.text(0.93, y - 0.02, val,
                     transform=ax2.transAxes,
                     fontsize=11, fontweight="bold",
                     color=col, ha="right")
            y -= 0.12

        fig.tight_layout(rect=[0, 0, 1, 0.95])

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = str(self.comparison_dir / "entity_comparison_venn.png")

        _save(fig, path)

        return {
            "regex_only":       _ro,
            "gat_only":         _go,
            "found_by_both":    _both,
            "total_regex":      total_regex,
            "total_gat":        total_gat,
            "overlap_percentage": pct,
        }

    # ══════════════════════════════════════════════════════════════════
    # 7. GAT SOURCE BREAKDOWN
    # ══════════════════════════════════════════════════════════════════

    def plot_gat_source_breakdown(self,
                                  gat_entities: List[Dict],
                                  filename: Optional[str] = None) -> str:
        """Horizontal bar + pie showing entity sources in GAT output."""
        source_counts = Counter(e.get("source", "unknown") for e in gat_entities)
        labels  = list(source_counts.keys())
        counts  = [source_counts[l] for l in labels]
        total   = sum(counts)
        pcts    = [c / total * 100 for c in counts]

        source_colors = {
            "gat_confirmed":        PALETTE["green"],
            "regex_fallback":       PALETTE["blue"],
            "gat_reclassified":     PALETTE["purple"],
            "gat_novel":            PALETTE["orange"],
            "gat_regex_confirmed":  PALETTE["yellow"],
            "gat_training_aligned": PALETTE["teal"],
        }
        bar_colors = [source_colors.get(l, PALETTE["muted"]) for l in labels]

        order      = sorted(range(len(labels)), key=lambda i: counts[i], reverse=True)
        labels     = [labels[i]     for i in order]
        counts     = [counts[i]     for i in order]
        pcts       = [pcts[i]       for i in order]
        bar_colors = [bar_colors[i] for i in order]

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle("GAT Entity Source Breakdown",
                     fontsize=14, fontweight="bold", color=PALETTE["text"])

        y = np.arange(len(labels))
        hbars = axes[0].barh(y, counts, color=bar_colors, height=0.62, edgecolor="none")
        axes[0].set_yticks(y)
        axes[0].set_yticklabels(labels, fontsize=9)
        axes[0].invert_yaxis()
        axes[0].set_xlabel("Entity Count")
        axes[0].spines["left"].set_visible(False)
        axes[0].spines["top"].set_visible(False)
        axes[0].spines["right"].set_visible(False)
        _title_block(axes[0], "Count by Source")

        for bar, pct in zip(hbars, pcts):
            w = bar.get_width()
            axes[0].text(w + max(counts) * 0.01, bar.get_y() + bar.get_height()/2,
                         f"{int(w):,}  ({pct:.1f}%)",
                         va="center", fontsize=8.5,
                         color=PALETTE["text"])

        wedges, texts, autos = axes[1].pie(
            counts, labels=labels, colors=bar_colors,
            autopct="%1.1f%%", startangle=140,
            pctdistance=0.78, labeldistance=1.12,
            wedgeprops=dict(edgecolor=PALETTE["bg"], linewidth=2.0))

        for t in texts:  t.set_fontsize(8);   t.set_color(PALETTE["text"])
        for a in autos:  a.set_fontsize(7.5); a.set_color("white"); a.set_fontweight("bold")

        _title_block(axes[1], "Proportion by Source")
        axes[1].set_aspect("equal")

        fig.tight_layout(rect=[0, 0, 1, 0.95])

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = str(self.comparison_dir / "gat_source_breakdown.png")

        _save(fig, path)
        return path

    # ══════════════════════════════════════════════════════════════════
    # 8. CLASS BALANCE CHART
    # ══════════════════════════════════════════════════════════════════

    def plot_class_balance(
        self,
        entity_count: int,
        non_entity_count: int,
        class_counts: Optional[Dict[int, int]] = None,
        label_map: Optional[Dict[int, str]] = None,
        filename: Optional[str] = None,
    ) -> str:
        """Class balance chart with proper path handling."""
        has_detail = class_counts is not None and len(class_counts) > 0

        fig, axes = plt.subplots(1, 2 if has_detail else 1,
                                 figsize=(18 if has_detail else 7, 6))
        if not has_detail:
            axes = [axes]

        fig.suptitle("Training Class Balance  ·  RegulationGAT",
                     fontsize=14, fontweight="bold", color=PALETTE["text"])

        total    = entity_count + non_entity_count
        sizes    = [entity_count, non_entity_count]
        clrs     = [PALETTE["blue"], PALETTE["orange"]]
        lbls     = [f"Entities\n(cls 0-25)", f"NON_ENTITY\n(cls 26)"]
        wedges, _, autos = axes[0].pie(
            sizes, labels=lbls, colors=clrs,
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(width=0.55, edgecolor=PALETTE["bg"], linewidth=2.5),
            pctdistance=0.75)
        for a in autos:
            a.set_fontsize(11); a.set_fontweight("bold"); a.set_color("white")
        axes[0].text(0, 0, f"{total:,}\ntotal",
                     ha="center", va="center",
                     fontsize=12, color=PALETTE["text"], fontweight="bold")
        ratio = entity_count / max(non_entity_count, 1)
        axes[0].set_title(f"Entity vs Non-Entity Balance\nRatio {ratio:.2f}:1",
                          fontsize=11, color=PALETTE["text"])

        if has_detail:
            top_n  = 20
            sorted_cls = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
            cls_ids = [c for c, _ in sorted_cls]
            cls_cnt = [v for _, v in sorted_cls]
            cls_lbl = [label_map.get(c, f"C{c}") if label_map else f"C{c}"
                       for c in cls_ids]

            colors = []
            for cnt in cls_cnt:
                if cnt < 50:
                    colors.append(PALETTE["orange"])
                else:
                    colors.append(PALETTE["blue"])

            bars  = axes[1].bar(range(len(cls_ids)), cls_cnt, color=colors,
                                edgecolor="none")
            axes[1].set_xticks(range(len(cls_ids)))
            axes[1].set_xticklabels(cls_lbl, rotation=45, ha="right", fontsize=8)
            axes[1].set_ylabel("Count")
            _title_block(axes[1], f"Top {top_n} Entity Classes",
                         "training set distribution (orange = sparse)")
            _add_value_labels(axes[1], bars, fontsize=7)

        fig.tight_layout(rect=[0, 0, 1, 0.95])

        # FIX: Handle filename properly
        if filename:
            path = filename
        else:
            path = f"{self.visual_dir}/class_balance.png"

        _save(fig, path)
        return path

### MAIN PIPELINE

In [ ]:
class RegulationPipeline:

    _STANDARD_NOISE_RE: list = [
        re.compile(r'^\s*(the\s+)?[Ss]audi\s+(Codes?|Standards?)\s*$'),
        re.compile(r'^\s*[Cc]ode\s+[Rr]equirements?\s*$'),
        re.compile(r'^\s*[Aa]rabic\s+[Cc]ode\s+[Pp]rovisions?\s*$'),
        re.compile(r'^\s*[A-Z]\s*\n'),
        re.compile(r'[^\x00-\x7F]{2,}'),
        re.compile(r'^\s*\d+\s*$'),
    ]

    @staticmethod
    def _is_standard_noise(entity_value: str) -> bool:
        v = entity_value.strip()
        return any(p.search(v) for p in RegulationPipeline._STANDARD_NOISE_RE)

    _CLASS_CONFIDENCE_FLOORS: Dict[str, float] = {
        'TESTING_REQUIREMENT':  0.55,
        'CERTIFICATION':        0.55,
        'PENETRATION':          0.50,
        'CAPACITY':             0.50,
        'INSPECTION_FREQUENCY': 0.45,
        'HAZARD_LEVEL':         0.45,
        'BUILDING_TYPE':        0.40,
        'EQUIPMENT':            0.25,
        'SEPARATION_DISTANCE':  0.40,
        'EGRESS_DISTANCE':      0.35,
        'CLEARANCE':            0.25,
        'OCCUPANCY':            0.40,
        'AUTHORITY':            0.35,
        'FLOW_RATE':            0.35,
        'PERCENTAGE':           0.30,
        'TEMPERATURE':          0.25,
        'PRESSURE':             0.25,
        'VOLUME':               0.22,
        'FIRE_RATING':          0.25,
        'AREA':                 0.25,
        'MATERIAL':             0.20,
        'DIMENSION':            0.20,
        'FACILITY':             0.25,
        'LOCATION':             0.30,
        'COMPONENT':            0.22,
        'STANDARD':             0.20,
    }

    _NOVEL_CONFIDENCE_PENALTY: float = 0.0
    _ABSOLUTE_FLOOR:           float = 0.15

    # =========================================================================
    # CONSTRUCTION
    # =========================================================================

    def __init__(self, config: Dict, logger: logging.Logger):
        self.config = config
        self.logger = logger

        for key in ['output_dir', 'model_dir', 'visual_dir']:
            os.makedirs(config[key], exist_ok=True)

        self.output_dir = Path(config['output_dir'])
        self.model_dir  = Path(config['model_dir'])
        self.visual_dir  = Path(config['visual_dir'])

        self.comparison_dir = self.output_dir / 'comparison'
        self.comparison_dir.mkdir(exist_ok=True)

        # FIX (#4): define checkpoint_dir BEFORE any code that uses it
        checkpoint_dir = config['checkpoint_dir']
        Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

        text_cache_dir = Path(checkpoint_dir) / 'text_cache'
        text_cache_dir.mkdir(parents=True, exist_ok=True)
        self.text_cache_manager = TextCacheManager(str(text_cache_dir), logger)

        self.preprocessor  = TextPreprocessor(logger)
        self.pdf_extractor = PDFTextExtractor(
            logger,
            self.preprocessor,
            config=config,
            cache_manager=self.text_cache_manager,
        )

        self.chunker = SemanticChunker(
            nlp=nlp,
            chunk_size=500,
            overlap=50,
            max_memory_mb=50000,
            prefer_quality=True,
        )

        self.labeler = EntityLabeler(nlp)

        print("\n🔤 Loading embedding model...")
        self.embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

        self.graph_builder = GraphBuilder(
            self.embedding_model,
            entity_labeler=self.labeler,
        )
        self.graph_builder_version = self._get_graph_builder_version()

        self.visualizer    = Visualizer(config)

        self.candidate_extractor = CandidatePhraseExtractor(nlp, self.labeler)

        # FIX (#4): checkpoint_dir already defined above — no redefinition needed
        self.checkpoint_manager = LabelingCheckpointManager(checkpoint_dir, logger)

        self.deduplicator  = EntityDeduplicator(logger)
        self.fuzzy_matcher = FuzzyEntityMatcher(logger)

        self.model         = None
        self.trainer       = None
        self.label_encoder = None

        self.logger.info("✅ Pipeline initialized")
        self.logger.info(f"   GraphBuilder version : {self.graph_builder_version}")
        self.logger.info(f"   Checkpoint dir       : {checkpoint_dir}")
        self.logger.info(f"   Text cache dir       : {text_cache_dir}")

    def _ensure_drive_mounted(self, probe_path: str = None, retries: int = 3) -> bool:
        ROOT_SENTINEL = "/content/drive/MyDrive"
        if probe_path is None:
            probe_path = ROOT_SENTINEL

        def _heal_log_handler():
            for h in self.logger.handlers:
                if hasattr(h, "stream") and h.stream is not None:
                    try:
                        h.stream.tell()
                    except OSError:
                        try:
                            h.stream.close()
                        except Exception:
                            pass
                        try:
                            h.stream = open(h.baseFilename, "a", encoding="utf-8")
                        except Exception:
                            pass

        def _probe(path: str) -> bool:
            try:
                os.stat(path)
                return True
            except (FileNotFoundError, OSError):
                return False

        if _probe(ROOT_SENTINEL) and _probe(probe_path):
            return True

        for attempt in range(1, retries + 1):
            print(f"[Drive] remount attempt {attempt}/{retries} …")
            try:
                _drive.mount("/content/drive", force_remount=True)
            except Exception as exc:
                print(f"[Drive] mount() raised: {exc}")

            _heal_log_handler()
            time.sleep(2)

            if _probe(ROOT_SENTINEL) and _probe(probe_path):
                self.logger.info("✅ Drive remounted successfully.")
                return True

            if _probe(ROOT_SENTINEL) and not _probe(probe_path):
                self.logger.error(
                    f"❌ Drive is mounted but file not found: '{probe_path}'\n"
                    f"   Check that the filename/folder path is correct."
                )
                return False

        self.logger.error(f"❌ Drive still unreachable after {retries} attempts.")
        return False

    def _flush_memory(self, label: str = ""):
        """Release Python + CUDA memory between pipeline steps."""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if label:
            print(f"   🧹 Memory flushed after: {label}")
    # =========================================================================
    # VERSION HASH
    # =========================================================================

    def _strip_boundary_noise(self, text: str) -> str:
        tokens = text.split()
        if not tokens:
            return text
        while tokens and tokens[0].lower().rstrip('.,;:') in _LEADING_STOP:
            tokens = tokens[1:]
        while tokens and tokens[-1].lower().lstrip('.,;:') in _TRAILING_STOP:
            tokens = tokens[:-1]
        return ' '.join(tokens) if tokens else text

    def _normalize_span(
        self,
        original_text:  str,
        stripped_text:  str,
        original_start: int,
        original_end:   int,
    ) -> Tuple[str, int, int]:
        if stripped_text == original_text:
            return stripped_text, original_start, original_end
        offset = original_text.find(stripped_text)
        if offset == -1:
            return stripped_text, original_start, original_end
        new_start = original_start + offset
        new_end   = new_start + len(stripped_text)
        return stripped_text, new_start, new_end

    def _get_graph_builder_version(self) -> str:
        try:
            source_parts = [
                inspect.getsource(GraphBuilder.build_graph_for_training),
                inspect.getsource(GraphBuilder._get_negative_samples),
            ]
            if hasattr(GraphBuilder, '_matches_any_entity_pattern'):
                source_parts.append(
                    inspect.getsource(GraphBuilder._matches_any_entity_pattern)
                )
            version_hash = hashlib.md5(''.join(source_parts).encode()).hexdigest()[:8]
            return f"v1.1_{version_hash}"
        except Exception as e:
            self.logger.warning(f"Could not generate GraphBuilder version: {e}")
            return "v1.0_unknown"

    # =========================================================================
    # TEXT CACHE HELPER  (TextCacheManager is the single source of truth)
    # =========================================================================

    def _get_text_for_pdf(self, pdf_path: str, silent: bool = False) -> Optional[str]:
        # Level 1: shared TextCacheManager hit
        # FIX: after Fix 1, self.text_cache_manager IS pdf_extractor.cache_manager
        # so this check and extract_text_with_cache's check are now identical objects.
        if self.text_cache_manager.has_cached(pdf_path, stage='cleaned'):
            text = self.text_cache_manager.load_cached(pdf_path, stage='cleaned')
            if text and len(text.strip()) >= 100:
                return text
            self.logger.warning(
                f"⚠️  Cache entry for '{Path(pdf_path).name}' is "
                f"{'empty' if not text else f'too short ({len(text)} chars)'} — evicting."
            )
            try:
                self.text_cache_manager.delete_cached(pdf_path, stage='cleaned')
                self.text_cache_manager.delete_cached(pdf_path, stage='raw')
            except Exception:
                pass

        # Level 2: full extraction via shared cache manager
        p = Path(pdf_path)
        if not p.exists():
            self.logger.error(f"❌ PDF not found: {pdf_path}")
            return None

        try:
            # FIX: always silent=False so extraction failures are visible;
            # tqdm bars are not the concern here — correct failure diagnosis is.
            text = self.pdf_extractor.extract_text_with_cache(pdf_path, silent=False)
        except Exception as e:
            self.logger.error(f"❌ Extraction raised for '{p.name}': {e}")
            return None

        if not text or len(text.strip()) < 100:
            self.logger.error(
                f"❌ '{p.name}' yielded {len(text) if text else 0} chars.\n"
                f"   File exists : {p.exists()}\n"
                f"   File size   : {p.stat().st_size if p.exists() else 'N/A'} bytes\n"
                f"   Readable    : {os.access(pdf_path, os.R_OK)}\n"
                f"   Likely cause: scanned/image PDF (needs OCR), FUSE mount issue, "
                f"or preprocessor wiped content."
            )
            return None

        # FIX: extract_text_with_cache already saves to cache — no double-save needed.
        # The original saved here unconditionally which was redundant after Fix 2
        # added save_cached inside extract_text_with_cache on success.
        return text

    # =========================================================================
    # DATA PREPARATION
    # =========================================================================

    def prepare_training_data(
        self,
        interactive:          bool = True,
        prefer_checkpoint:    bool = True,
        force_rebuild_graphs: bool = False,
    ) -> Tuple[object, DataLoader, DataLoader, LabelEncoder]:

        print("\n" + "=" * 80)
        print("STEP 1: PREPARING CURRICULUM TRAINING DATA")
        print("=" * 80)

        pdf_dir = self.config['train_pdf_dir']

        if interactive:
            decision = prompt_checkpoint_decision(
                self.checkpoint_manager, self.config, pdf_dir
            )
        else:
            decision = auto_checkpoint_decision(
                self.checkpoint_manager, self.config, pdf_dir, prefer_checkpoint
            )

        all_chunks:         list = []
        chunk_entities:     list = []
        graphs:             list = []
        entities_per_graph: list = []

        # ── Alignment helper ─────────────────────────────────────────────
        def _align(g_list: list, e_list: list, label: str):
            if len(g_list) != len(e_list):
                shorter = min(len(g_list), len(e_list))
                self.logger.warning(
                    f"⚠️  {label}: graphs ({len(g_list)}) and entities_per_graph "
                    f"({len(e_list)}) differ — truncating both to {shorter}. "
                    f"This is a checkpoint integrity issue; consider a fresh build."
                )
                return g_list[:shorter], e_list[:shorter]
            return g_list, e_list

        # ── Load from checkpoint ──────────────────────────────────────────
        if decision == 'load':
            checkpoint_data = self.checkpoint_manager.load_checkpoint()

            if checkpoint_data is None:
                self.logger.error("❌ Failed to load checkpoint. Starting fresh...")
                decision = 'fresh'
            else:
                all_chunks     = checkpoint_data['chunks']
                chunk_entities = checkpoint_data['chunk_entities']

                stats = self.text_cache_manager.get_cache_stats()
                if stats['num_text_files'] > 0:
                    print(
                        f"   📄 Text cache: {stats['num_text_files']} file(s) "
                        f"({stats['total_size_mb']} MB) in {stats['checkpoint_dir']}"
                    )

                metadata           = checkpoint_data.get('metadata', {})
                saved_gb_version   = metadata.get('graph_builder_version')
                current_gb_version = getattr(self, 'graph_builder_version', None)

                version_changed = (
                    saved_gb_version is not None
                    and current_gb_version is not None
                    and saved_gb_version != current_gb_version
                )
                if version_changed:
                    print(f"\n⚠️  GraphBuilder version changed!")
                    print(f"   Checkpoint: {saved_gb_version}  →  Current: {current_gb_version}")
                    force_rebuild_graphs = True

                saved_graphs = checkpoint_data.get('graphs', [])

                if not saved_graphs:
                    print(
                        f"\n⚠️  Checkpoint contains no graphs "
                        f"({len(chunk_entities)} label set(s) available)."
                    )
                    print("🔄 Building graphs from saved labels...")
                    graphs, entities_per_graph = self._rebuild_graphs_from_checkpoint(
                        chunk_entities
                    )
                    graphs, entities_per_graph = _align(
                        graphs, entities_per_graph, "rebuild (no saved graphs)"
                    )
                    print("\n💾 Saving updated checkpoint with graphs...")
                    self._save_updated_checkpoint(
                        all_chunks, chunk_entities, graphs, entities_per_graph,
                        pdf_dir, metadata.get('pdf_count', 0),
                    )

                elif force_rebuild_graphs:
                    print("\n🔄 Rebuilding graphs with updated GraphBuilder...")
                    graphs, entities_per_graph = self._rebuild_graphs_from_checkpoint(
                        chunk_entities
                    )
                    graphs, entities_per_graph = _align(
                        graphs, entities_per_graph, "rebuild (version change)"
                    )
                    print("\n💾 Saving updated checkpoint with rebuilt graphs...")
                    self._save_updated_checkpoint(
                        all_chunks, chunk_entities, graphs, entities_per_graph,
                        pdf_dir, metadata.get('pdf_count', 0),
                    )

                else:
                    raw_entities = [ce['entities'] for ce in chunk_entities]
                    graphs, entities_per_graph = _align(
                        saved_graphs, raw_entities, "load (graphs vs chunk_entities)"
                    )

                self.logger.info(
                    f"\n✅ Loaded from checkpoint: "
                    f"{len(all_chunks)} chunks, {len(graphs)} graphs, "
                    f"{len(entities_per_graph)} entity lists"
                )

        # ── Start from scratch ────────────────────────────────────────────
        if decision == 'fresh':
            print(f"\n📂 Training folder: {pdf_dir}")

            pdf_files = sorted(Path(pdf_dir).glob("*.pdf"))
            if not pdf_files:
                raise ValueError("No PDFs found in training directory!")

            print(f"   Found {len(pdf_files)} PDF(s)")
            print("\n✂️  Extracting and chunking documents...")

            all_chunks   = []
            total_docs   = 0
            skipped_docs = 0

            pbar = tqdm(pdf_files, desc="Extracting & chunking")
            for pdf_path in pbar:
                doc_id = pdf_path.stem
                pbar.set_postfix({'doc': doc_id[:35]})

                text = self._get_text_for_pdf(str(pdf_path))
                if text is None:
                    tqdm.write(f"⚠️  Skipping '{doc_id}' — insufficient text")
                    skipped_docs += 1
                    continue

                chunks = None
                if self.text_cache_manager.has_cached_chunks(str(pdf_path), text):
                    chunks = self.text_cache_manager.load_cached_chunks(str(pdf_path), text)
                    if chunks:
                        tqdm.write(f"   ✅ Chunks from cache ({len(chunks)})")

                if not chunks:
                    chunks = self.pdf_extractor.get_chunks_with_cache(
                        str(pdf_path), text, self.chunker, doc_id, silent=True
                    )
                    if chunks:
                        self.text_cache_manager.save_cached_chunks(str(pdf_path), text, chunks)
                all_chunks.extend(chunks)
                total_docs += 1

            pbar.close()

            if not all_chunks:
                raise ValueError("No chunks produced — check PDF extraction!")

            print(
                f"   Created {len(all_chunks)} chunks from {total_docs} document(s)"
                + (f"  ({skipped_docs} skipped)" if skipped_docs else "")
            )

            print("\n🏷️ Labeling entities (this may take a while)...")

            cached_entity_indices = self.checkpoint_manager.get_cached_entity_indices()
            cached_graph_indices  = self.checkpoint_manager.get_cached_graph_indices()

            # FIX-ENTITY-MAP: split cached indices into two groups:
            #   needs_load  — have entity PKL but NO graph PKL yet (orphans)
            #   already_complete — have BOTH entity and graph PKLs
            # Previously only `needs_load` was loaded into chunk_entities_map.
            # `already_complete` indices were silently skipped by the labeling
            # loop and never entered chunk_entities_map, so _build_graphs_with_
            # regex_guarantee received an entities_list with gaps at those
            # positions — causing empty entity lists on every previously-cached
            # graph when it was reloaded.
            needs_load       = cached_entity_indices - cached_graph_indices
            already_complete = cached_entity_indices & cached_graph_indices

            print(
                f"   📊 Entity/graph cache status:\n"
                f"        Cached entity files : {len(cached_entity_indices):,}\n"
                f"        Cached graph files  : {len(cached_graph_indices):,}\n"
                f"        Orphaned (need load): {len(needs_load):,}\n"
                f"        Complete (both PKLs): {len(already_complete):,}\n"
                f"        Truly new (no PKL)  : "
                f"{len(set(range(len(all_chunks))) - cached_entity_indices):,}"
            )

            chunk_entities_map: Dict[int, Dict] = {}

            # FIX-ENTITY-MAP: load ALL cached entity PKLs — both orphans and
            # already-complete indices — so chunk_entities_map is fully
            # populated before the labeling loop runs.
            # Orphans need their entity data so graphs can be built for them.
            # Already-complete indices need their entity data so the downstream
            # entities_list passed to _build_graphs_with_regex_guarantee has a
            # valid, non-empty entry for every graph index that will be loaded
            # from disk.
            all_indices_to_load = cached_entity_indices  # was: `needs_load` only

            if all_indices_to_load:
                entity_pkls = sorted(
                    (
                        p for idx in all_indices_to_load
                        if (p := self.checkpoint_manager.checkpoint_dir / f"entity_{idx}.pkl").exists()
                    ),
                    key=lambda p: int(p.stem.split("_")[1]),
                )

                load_label = (
                    f"Loading {len(entity_pkls):,} cached entity file(s) "
                    f"({len(needs_load):,} orphaned + {len(already_complete):,} complete)..."
                )
                print(f"\n💾 {load_label}")

                def _load_one(path: Path):
                    idx = int(path.stem.split("_")[1])
                    with open(path, 'rb') as f:
                        return idx, pickle.load(f)

                failed = 0
                with ThreadPoolExecutor(max_workers=8) as executor:
                    futures = {executor.submit(_load_one, p): p for p in entity_pkls}
                    for future in tqdm(as_completed(futures), total=len(entity_pkls),
                                      desc="Loading cached entities", unit="file"):
                        try:
                            idx, data = future.result()
                            chunk_entities_map[idx] = data
                        except Exception as e:
                            self.logger.warning(f"Failed to load {futures[future].name}: {e}")
                            failed += 1

                if failed:
                    self.logger.warning(f"⚠️  {failed} entity file(s) failed to load")

                # FIX-ENTITY-MAP: warn if any already_complete index failed to
                # load — its graph will be reloaded from disk but will have an
                # empty entity list, which is the exact bug we are fixing.
                missing_complete = already_complete - set(chunk_entities_map.keys())
                if missing_complete:
                    self.logger.warning(
                        f"⚠️  FIX-ENTITY-MAP: {len(missing_complete)} 'complete' index(es) "
                        f"could not be loaded from entity PKLs "
                        f"(indices: {sorted(missing_complete)[:10]}"
                        f"{'…' if len(missing_complete) > 10 else ''}).\n"
                        f"   Their graphs will train with empty entity lists.\n"
                        f"   Resolution: delete affected graph_*.pkl files and re-run "
                        f"so those indices are re-labeled and re-built from scratch."
                    )
            else:
                print("   ℹ️  No cached entities found — will label all chunks from scratch")

            # ── Labeler probe ─────────────────────────────────────────────
            _probe_chunk = next(
                (c for c in all_chunks if c.get('text', '').strip()),
                None,
            )
            if _probe_chunk is not None:
                try:
                    _probe_ents = self.labeler.extract_entities_from_chunk(
                        _probe_chunk['text']
                    )
                except Exception as _probe_exc:
                    raise RuntimeError(
                        f"❌ Labeler probe raised an exception on the first chunk:\n"
                        f"   {_probe_exc}\n"
                        f"   Fix the labeler before re-running."
                    ) from _probe_exc

                if not _probe_ents:
                    self.logger.warning(
                        f"⚠️  Labeler probe: 0 entities found in first chunk "
                        f"(len={len(_probe_chunk['text'])} chars).\n"
                        f"   First 300 chars: {_probe_chunk['text'][:300]!r}\n"
                        f"   compiled_patterns keys: "
                        f"{list(self.labeler.compiled_patterns.keys()) if hasattr(self.labeler, 'compiled_patterns') else 'N/A'}\n"
                        f"   This may be fine if the first chunk is a TOC / header page, "
                        f"but if ALL chunks produce 0 entities the build will abort."
                    )
                else:
                    print(
                        f"   ✅ Labeler probe: {len(_probe_ents)} entities in first chunk "
                        f"— labeler is working"
                    )
            else:
                raise ValueError(
                    "❌ All chunks have empty 'text' fields — "
                    "check PDF extraction and chunking."
                )

            new_labeled = 0
            for i, chunk in enumerate(tqdm(all_chunks, desc="Labeling")):
                if i in cached_entity_indices:
                    # FIX-ENTITY-MAP: skip re-labeling (entity PKL already exists)
                    # but do NOT skip silently — if the load above failed for this
                    # index, warn explicitly rather than leaving a silent gap.
                    if i not in chunk_entities_map:
                        self.logger.warning(
                            f"⚠️  FIX-ENTITY-MAP: chunk {i} is in cached_entity_indices "
                            f"but was NOT loaded into chunk_entities_map (PKL missing or "
                            f"corrupt).  This chunk will be re-labeled now to fill the gap."
                        )
                        # Fall through to re-label rather than leaving a gap
                    else:
                        continue
                entities = self.labeler.extract_entities_from_chunk(chunk['text'])
                if entities:
                    ce = {'chunk': chunk, 'entities': entities}
                    self.checkpoint_manager.save_single_entity(i, ce)
                    chunk_entities_map[i] = ce
                    new_labeled += 1

            # ── Post-labeling guard ───────────────────────────────────────
            if not chunk_entities_map:
                _sample_texts = [
                    c.get('text', '')[:200]
                    for c in all_chunks[:5]
                    if c.get('text', '').strip()
                ]
                _pat_keys = (
                    list(self.labeler.compiled_patterns.keys())
                    if hasattr(self.labeler, 'compiled_patterns')
                    else 'attribute missing'
                )
                raise ValueError(
                    f"❌ Labeling produced 0 chunks with entities out of "
                    f"{len(all_chunks)} total chunks.\n"
                    f"   Every call to extract_entities_from_chunk() returned empty.\n\n"
                    f"   Diagnostic info:\n"
                    f"     total chunks          : {len(all_chunks)}\n"
                    f"     cached_entity_indices : {len(cached_entity_indices)}\n"
                    f"     compiled_patterns keys: {_pat_keys}\n"
                    f"     sample chunk texts    :\n"
                    + "\n".join(f"       [{i}] {t!r}" for i, t in enumerate(_sample_texts))
                    + f"\n\n   Likely causes:\n"
                    f"     1. self.labeler.compiled_patterns is empty or mis-keyed.\n"
                    f"     2. All chunk texts are empty / whitespace after preprocessing.\n"
                    f"     3. The NLP model loaded in self.labeler.nlp is wrong.\n"
                    f"   Fix: run `self.labeler.extract_entities_from_chunk("
                    f"all_chunks[0]['text'])` manually and inspect the output."
                )

            # FIX-ENTITY-MAP: report coverage so the user can confirm the fix
            # is working — all_chunks coverage vs cached graph coverage.
            loaded_count   = sum(1 for i in range(len(all_chunks)) if i in chunk_entities_map)
            coverage_pct   = loaded_count / max(len(all_chunks), 1) * 100
            print(
                f"   Found {len(chunk_entities_map):,} chunks with entities "
                f"({loaded_count:,}/{len(all_chunks):,} chunks covered, {coverage_pct:.1f}%)"
                + (f"  ({new_labeled:,} newly labeled)" if new_labeled else "")
            )

            # FIX-ENTITY-MAP: warn if there is a significant gap between
            # cached_graph_indices and what chunk_entities_map now covers.
            # A large gap means graphs will be loaded from disk with empty
            # entity lists even after this fix — the only resolution is a
            # fresh entity re-label for the missing indices.
            graphs_without_entities = cached_graph_indices - set(chunk_entities_map.keys())
            if graphs_without_entities:
                self.logger.warning(
                    f"⚠️  FIX-ENTITY-MAP: {len(graphs_without_entities)} cached graph(s) "
                    f"have no matching entry in chunk_entities_map even after loading all "
                    f"entity PKLs.  These graphs will train with empty entity lists.\n"
                    f"   Affected indices (first 10): "
                    f"{sorted(graphs_without_entities)[:10]}"
                    f"{'…' if len(graphs_without_entities) > 10 else ''}\n"
                    f"   Resolution: delete those graph_*.pkl files and re-run so they "
                    f"are rebuilt with correct entity lists."
                )

            chunk_entities = [
                chunk_entities_map[i]
                for i in sorted(chunk_entities_map)
            ]

            print("\n🕸️ Building graphs...")
            graphs, entities_per_graph, skipped = self._build_graphs_with_regex_guarantee(
                [ce['chunk']    for ce in chunk_entities],
                [ce['entities'] for ce in chunk_entities],
            )
            graphs, entities_per_graph = _align(
                graphs, entities_per_graph, "fresh build"
            )
            print(f"   Built {len(graphs)} valid graphs (skipped {skipped} invalid)")

            # ── Post-build guard ──────────────────────────────────────────
            if not graphs:
                raise ValueError(
                    f"❌ Graph building produced 0 valid graphs from "
                    f"{len(chunk_entities)} labeled chunks "
                    f"({skipped} skipped).\n"
                    f"   Likely causes:\n"
                    f"     1. All chunks have fewer than 2 nodes after graph construction\n"
                    f"        (single-node graphs are discarded).\n"
                    f"     2. graph_builder.build_graph_for_training() is returning None\n"
                    f"        for every chunk — add a print inside it to confirm.\n"
                    f"     3. The embedding model produced NaN/zero vectors for all nodes.\n"
                    f"   Fix: run `self.graph_builder.build_graph_for_training("
                    f"chunk_entities[0]['chunk'], chunk_entities[0]['entities'])` "
                    f"manually and inspect the result."
                )

            checkpoint_chunk_entities = [
                {'chunk': chunk_entities[i]['chunk'], 'entities': ents}
                for i, ents in enumerate(entities_per_graph)
                if i < len(chunk_entities)
            ]

            print("\n💾 Saving checkpoint for future use...")
            self.checkpoint_manager.save_checkpoint(
                chunks=all_chunks,
                chunk_entities=checkpoint_chunk_entities,
                graphs=graphs,
                config=self.config,
                pdf_dir=pdf_dir,
                metadata={
                    'pdf_count':             total_docs,
                    'skipped_docs':          skipped_docs,
                    'skipped_graphs':        skipped,
                    'graph_builder_version': getattr(self, 'graph_builder_version', 'v1.0'),
                },
                cleanup_incremental=True,
            )

        # ── Defensive guards ──────────────────────────────────────────────
        if decision not in ('load', 'fresh'):
            raise ValueError(
                f"prepare_training_data: unrecognised decision value "
                f"'{decision}'. Expected 'load' or 'fresh'."
            )

        if not graphs:
            raise ValueError("No valid graphs were created!")
        if not entities_per_graph:
            raise ValueError("No entities found!")

        graphs, entities_per_graph = _align(
            graphs, entities_per_graph, "pre-split final check"
        )

        # ── Three-way split ───────────────────────────────────────────────
        print("\n📊 Splitting data (train / val / test)...")

        test_size             = self.config['test_size']
        val_size_of_remaining = test_size / (1.0 - test_size)

        all_indices = list(range(len(graphs)))

        train_val_idx, test_idx = train_test_split(
            all_indices,
            test_size=test_size,
            random_state=self.config['random_state'],
        )
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=val_size_of_remaining,
            random_state=self.config['random_state'],
        )

        train_graphs   = [graphs[i]             for i in train_idx]
        train_entities = [entities_per_graph[i] for i in train_idx]
        val_graphs     = [graphs[i]             for i in val_idx]
        test_graphs    = [graphs[i]             for i in test_idx]

        all_labels: list = []
        for g in train_graphs:
            all_labels.extend(g.y[g.y >= 0].cpu().numpy())

        class_counts = Counter(all_labels)
        print("\n📊 Training class distribution:")
        for class_id, count in sorted(
            class_counts.items(), key=lambda x: x[1], reverse=True
        ):
            class_name = ENTITY_ID_TO_TYPE.get(
                int(class_id), f"UNKNOWN_{int(class_id)}"
            )
            print(f"  {class_name:20} : {count:5}")

        entity_count     = sum(1 for l in all_labels if 0 <= l <= 25)
        non_entity_count = sum(1 for l in all_labels if l == _NON_ENTITY_ID)
        print(f"\n📊 Entity vs Non-Entity Balance (train split only):")
        print(f"   Entity labels (0-25)    : {entity_count:,}")
        print(f"   NON_ENTITY labels ({_NON_ENTITY_ID})  : {non_entity_count:,}")
        print(f"   Ratio: {entity_count / max(1, non_entity_count):.2f}:1")

        curriculum_loader = CurriculumDataLoader(
            graphs=train_graphs,
            entities_per_graph=train_entities,
            batch_size=self.config['batch_size'],
            num_phases=4,
            adaptive_phases=True,
        )

        val_loader  = DataLoader(
            val_graphs,  batch_size=self.config['batch_size'], shuffle=False
        )
        test_loader = DataLoader(
            test_graphs, batch_size=self.config['batch_size'], shuffle=False
        )

        print(f"\n   Train : {len(train_graphs)} graphs  ({len(train_idx)/len(graphs)*100:.1f}%)")
        print(f"   Val   : {len(val_graphs)} graphs  ({len(val_idx)/len(graphs)*100:.1f}%)")
        print(f"   Test  : {len(test_graphs)} graphs  ({len(test_idx)/len(graphs)*100:.1f}%)")

        label_encoder = LabelEncoder()
        valid_classes = [
            ENTITY_ID_TO_TYPE[i]
            for i in range(NUM_ENTITY_TYPES)
            if i in ENTITY_ID_TO_TYPE
        ]
        label_encoder.fit(valid_classes)

        assert len(label_encoder.classes_) == NUM_ENTITY_TYPES, (
            f"❌ FATAL: encoder has {len(label_encoder.classes_)} classes "
            f"but model has {NUM_ENTITY_TYPES}.  "
            f"Check ENTITY_ID_TO_TYPE for IDs outside 0…{NUM_ENTITY_TYPES - 1}."
        )

        print(f"\n✅ Label encoder created:")
        print(f"   Total classes : {len(label_encoder.classes_)}  "
              f"(must equal NUM_ENTITY_TYPES={NUM_ENTITY_TYPES})")
        print(f"   Classes       : {label_encoder.classes_[:10]}...")

        self.label_encoder = label_encoder
        return curriculum_loader, val_loader, test_loader, label_encoder


    def _build_graphs_with_regex_guarantee(
        self,
        chunks:        List[Dict],
        entities_list: List[List],
    ) -> Tuple[List['Data'], List[List], int]:
        """
        Build training graphs ensuring ALL regex-matched spans are represented.

        PERF: ONE nlp() call per chunk.

        FIX-B: warn about entity-map gaps after loading incremental graphs.
        FIX-SKIP: guard against the case where cached_graph_indices equals
        ALL indices in this call's range — which previously caused the loop
        to skip every chunk and return 0 new graphs without any warning.
        """
        skipped = 0
        stats   = {'total_validated': 0, 'total_forced': 0}

        _has_raw_spans = hasattr(self.labeler, 'extract_raw_spans')

        cached_graph_indices = self.checkpoint_manager.get_cached_graph_indices()

        indices_in_call = set(range(len(chunks)))
        already_cached  = indices_in_call & cached_graph_indices
        truly_new       = indices_in_call - cached_graph_indices

        if cached_graph_indices:
            print(
                f"   ⏭️  {len(already_cached):,} of {len(chunks):,} chunk(s) already have "
                f"cached graphs — skipping those, building {len(truly_new):,} new one(s)"
            )
        else:
            print(f"   🔨 No cached graphs found — building all {len(chunks):,} chunk(s) from scratch")

        if not truly_new and cached_graph_indices:
            print(
                f"   ✅ All {len(chunks):,} graphs already cached — "
                f"loading from disk without rebuilding"
            )
            # PERF: pass cached_graph_indices directly — no glob needed
            graphs = self.checkpoint_manager.load_incremental_graphs(
                indices=cached_graph_indices
            )
            sorted_graph_indices = sorted(cached_graph_indices)
            entities_map = {
                i: (entities_list[i] if i < len(entities_list) else [])
                for i in sorted_graph_indices
            }
            entities_per_graph = [entities_map.get(idx, []) for idx in sorted_graph_indices]
            return graphs, entities_per_graph, skipped

        if not truly_new and not cached_graph_indices:
            self.logger.warning(
                "⚠️  _build_graphs_with_regex_guarantee called with 0 chunks "
                "and 0 cached graphs.  Returning empty lists.  "
                "Check that chunk_entities is non-empty before calling this method."
            )
            return [], [], 0

        entities_map: Dict[int, list] = {}

        for i, (chunk, preextracted_entities) in enumerate(tqdm(
            zip(chunks, entities_list),
            total=len(chunks),
            desc="Building",
        )):
            if i in cached_graph_indices:
                entities_map[i] = preextracted_entities or []
                continue

            chunk_text = chunk.get('text', '')
            if not chunk_text or len(chunk_text.strip()) < 10:
                skipped += 1
                continue

            try:
                if _has_raw_spans:
                    doc       = self.labeler.nlp(chunk_text)
                    raw_spans = self.labeler.extract_raw_spans(chunk_text, doc=doc)
                    if not raw_spans:
                        skipped += 1
                        continue
                    validated = self.labeler.validate_spans(
                        raw_spans, training_mode=True
                    )
                else:
                    raw_entities = self.labeler.extract_entities_from_chunk(chunk_text)
                    if not raw_entities:
                        skipped += 1
                        continue
                    validated = raw_entities

            except Exception as exc:
                self.logger.warning(
                    f"Entity extraction failed: {exc} -- using preextracted fallback"
                )
                validated = preextracted_entities or []
                if not validated:
                    skipped += 1
                    continue

            passing = [e for e in validated if     getattr(e, 'is_valid', True)]
            failing = [e for e in validated if not getattr(e, 'is_valid', True)]

            def _keep(entity) -> bool:
                if getattr(entity, 'entity_type', None) == 'STANDARD':
                    return not self._is_standard_noise(
                        getattr(entity, 'entity_value', '')
                        or getattr(entity, 'text', '')
                    )
                return True

            passing = [e for e in passing if _keep(e)]
            failing = [e for e in failing if _keep(e)]

            graph = self.graph_builder.build_graph_for_training(
                chunk,
                entities=passing,
                force_regex_entities=failing,
            )

            if graph is not None and isinstance(graph, Data):
                if graph.x.size(0) > 1 and graph.y.size(0) > 1:
                    combined_entities = passing + failing
                    self.checkpoint_manager.save_single_graph(i, graph)
                    entities_map[i] = combined_entities
                    stats['total_validated'] += len(passing)
                    stats['total_forced']    += len(failing)
                else:
                    skipped += 1
            else:
                skipped += 1

        newly_built = {k for k in entities_map if k not in cached_graph_indices}
        if truly_new and not newly_built:
            _sample = [
                (i, chunks[i].get('text', '')[:200])
                for i in sorted(truly_new)[:5]
                if i < len(chunks)
            ]
            _sample_str = "\n".join(f"       [{i}] {t!r}" for i, t in _sample)

            _short_chunks = [
                i for i in truly_new
                if i < len(chunks)
                and len(chunks[i].get('text', '').strip()) < 10
            ]
            _all_short = len(_short_chunks) == len(truly_new)

            if cached_graph_indices and _all_short:
                self.logger.warning(
                    f"⚠️  _build_graphs_with_regex_guarantee: {len(truly_new)} "
                    f"chunk(s) flagged as new but all are < 10 chars — likely "
                    f"PDF extraction fragments.  Skipping them and continuing "
                    f"with {len(cached_graph_indices):,} cached graphs.\n"
                    f"   Fragment samples:\n{_sample_str}"
                )
            else:
                raise ValueError(
                    f"❌ _build_graphs_with_regex_guarantee: attempted to build "
                    f"{len(truly_new)} new graph(s) but produced 0.\n"
                    f"   All {len(truly_new)} chunks were skipped.\n\n"
                    f"   Diagnostic info:\n"
                    f"     skipped counter        : {skipped}\n"
                    f"     has extract_raw_spans  : {_has_raw_spans}\n"
                    f"     sample chunk texts     :\n"
                    + _sample_str
                    + f"\n\n   Likely causes:\n"
                    f"     1. extract_raw_spans() / extract_entities_from_chunk() "
                    f"returns empty for every chunk.\n"
                    f"     2. build_graph_for_training() returns None or a single-node "
                    f"graph for every chunk.\n"
                    f"     3. All chunk texts are shorter than 10 chars after stripping.\n"
                    f"   Fix: manually run:\n"
                    f"     self.labeler.extract_raw_spans(chunks[0]['text'])\n"
                    f"     self.graph_builder.build_graph_for_training(\n"
                    f"         chunks[0], entities=<result above>)"
                )

        all_graph_indices = cached_graph_indices | set(entities_map.keys())
        graphs = self.checkpoint_manager.load_incremental_graphs(indices=all_graph_indices)

        sorted_graph_indices = sorted(all_graph_indices)

        missing_indices = [idx for idx in sorted_graph_indices if idx not in entities_map]
        if missing_indices:
            self.logger.warning(
                f"⚠️  FIX-B: {len(missing_indices)} graph(s) have no entry in "
                f"entities_map (indices: {missing_indices[:10]}"
                f"{'…' if len(missing_indices) > 10 else ''}).\n"
                f"   These graphs will train with empty entity lists.\n"
                f"   Cause: graph_*.pkl files from a previous run whose "
                f"entity_*.pkl was deleted or never produced.\n"
                f"   Resolution: run with force_rebuild_graphs=True to rebuild."
            )

        entities_per_graph = [entities_map.get(idx, []) for idx in sorted_graph_indices]

        new_count  = len(newly_built)
        total_ents = stats['total_validated'] + stats['total_forced']
        force_pct  = stats['total_forced'] / max(1, total_ents) * 100

        print(f"\n   Graph Build Summary:")
        print(f"      Graphs built this run      : {new_count:,}")
        print(f"      Total graphs (incl. cached): {len(graphs):,}")
        print(f"      Total entities in graphs   : {total_ents:,}")
        print(f"        Validated (passing)  : {stats['total_validated']:,}")
        print(f"        Force-added (failing): {stats['total_forced']:,}  "
              f"({force_pct:.1f}%)")
        if missing_indices:
            print(f"      ⚠️  Entity-map gaps        : {len(missing_indices):,}  "
                  f"(see logger warning above)")
        if stats['total_forced'] == 0:
            print("      force_added=0 -- EntityLabeler may lack extract_raw_spans().")

        return graphs, entities_per_graph, skipped

    # =========================================================================
    # GRAPH REBUILD HELPERS
    # =========================================================================

    def _rebuild_graphs_from_checkpoint(
        self, chunk_entities: List[Dict]
    ) -> Tuple[List[Data], List[List]]:
        chunks   = [ce.get('chunk')        for ce in chunk_entities if ce.get('chunk')]
        entities = [ce.get('entities', []) for ce in chunk_entities if ce.get('chunk')]
        graphs, entities_per_graph, skipped = self._build_graphs_with_regex_guarantee(
            chunks, entities
        )
        print(f"   ✅ Rebuilt {len(graphs)} valid graphs (skipped {skipped})")
        if graphs:
            sg = graphs[0]
            ul = sg.y.unique().tolist()
            print(f"\n   📊 Sample rebuilt graph:")
            print(f"      Nodes: {sg.x.size(0)}")
            print(f"      Unique labels: {ul}")
            print(f"      Has entities (0-25): {any(0 <= l <= 25 for l in ul)}")
            print(f"      Has NON_ENTITY ({_NON_ENTITY_ID}): {_NON_ENTITY_ID in ul}")
        return graphs, entities_per_graph

    def _save_updated_checkpoint(
        self,
        all_chunks:         List[Dict],
        chunk_entities:     List[Dict],
        graphs:             List[Data],
        entities_per_graph: List[List],
        pdf_dir:            str,
        pdf_count:          int,
    ):
        # FIX-C: zip(chunk_entities, entities_per_graph) silently truncated to
        # the shorter list in the original.  Align with a warning first so the
        # caller is informed of any mismatch rather than silently losing data.
        if len(chunk_entities) != len(entities_per_graph):
            shorter = min(len(chunk_entities), len(entities_per_graph))
            self.logger.warning(
                f"⚠️  FIX-C _save_updated_checkpoint: chunk_entities "
                f"({len(chunk_entities)}) and entities_per_graph "
                f"({len(entities_per_graph)}) differ — truncating both to "
                f"{shorter}. This is a checkpoint integrity issue; consider "
                f"a fresh build."
            )
            chunk_entities     = chunk_entities[:shorter]
            entities_per_graph = entities_per_graph[:shorter]

        checkpoint_chunk_entities = []
        for ce, entity_list in zip(chunk_entities, entities_per_graph):
            chunk = ce.get('chunk')
            if chunk is None or not ce.get('entities'):
                continue
            checkpoint_chunk_entities.append(
                {'chunk': chunk, 'entities': entity_list}
            )

        self.checkpoint_manager.save_checkpoint(
            chunks=all_chunks,
            chunk_entities=checkpoint_chunk_entities,
            graphs=graphs,
            config=self.config,
            pdf_dir=pdf_dir,
            metadata={
                'pdf_count':             pdf_count,
                'skipped_graphs':        len(chunk_entities) - len(graphs),
                'graph_builder_version': getattr(self, 'graph_builder_version', 'v1.0'),
                'rebuilt':               True,
            },
            cleanup_incremental=True,
        )
        self.logger.info(
            f"   ✅ Saved updated checkpoint with {len(graphs)} graphs"
        )


    # =========================================================================
    # MODEL CREATION & TRAINING
    # =========================================================================

    def _create_model(self) -> object:
        model = RegulationGAT(
            input_dim=self.config['embedding_dim'],
            hidden_dim=self.config['hidden_dim'],
            num_classes=NUM_ENTITY_TYPES,
            num_heads=self.config['num_heads'],
            dropout=self.config['dropout'],
            non_entity_id=_NON_ENTITY_ID,
        )
        print(f"\n✅ Model initialized:")
        print(f"   Input dim  : {self.config['embedding_dim']}")
        print(f"   Hidden dim : {self.config['hidden_dim']}")
        print(f"   Num classes: {NUM_ENTITY_TYPES} (0-25=entities, {_NON_ENTITY_ID}=NON_ENTITY)")
        print(f"   Num heads  : {self.config['num_heads']}")
        return model

    def _load_best_model(self, model) -> object:
        best_model_path = self.model_dir / 'best_model.pt'

        # FIX-E-1: map_location ensures a GPU checkpoint loads cleanly on a
        # CPU-only machine (and vice-versa) instead of raising RuntimeError.
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # FIX-E-2: weights_only=True suppresses the FutureWarning in PyTorch
        # 2.x and is the hard requirement in 2.6+.  We try the safe form first
        # and fall back for older checkpoints that embedded non-tensor objects
        # (e.g. the optimizer state dict with Python objects) — those only arise
        # if the checkpoint was saved with a pre-2.0 idiom.
        try:
            checkpoint = torch.load(
                best_model_path,
                map_location=device,
                weights_only=True,
            )
        except Exception as e:
            self.logger.warning(
                f"⚠️  FIX-E: torch.load with weights_only=True failed "
                f"({e}); retrying with weights_only=False. "
                f"Re-save the checkpoint to suppress this warning."
            )
            checkpoint = torch.load(
                best_model_path,
                map_location=device,
                weights_only=False,
            )

        model.load_state_dict(checkpoint['model_state_dict'])
        self.logger.info(f"✅ Loaded best model from {best_model_path} (device={device})")
        return model

    def _inspect_training_sample(self, phase_0_loader):
        print("\n🔍 INSPECTING TRAINING SAMPLE...")
        sample_data = next(iter(phase_0_loader))
        print(f"Sample graph shape: {sample_data.x.shape}")
        print(f"Sample labels shape: {sample_data.y.shape}")
        print(f"Sample labels (first 20): {sample_data.y[:20]}")
        valid_labels = sample_data.y[sample_data.y >= 0]
        print(f"Unique labels in sample: {valid_labels.unique()}")
        if _NON_ENTITY_ID in valid_labels:
            n = (valid_labels == _NON_ENTITY_ID).sum().item()
            print(f"✅ Found {n} NON_ENTITY labels (class {_NON_ENTITY_ID})")
        else:
            print(f"⚠️  WARNING: No NON_ENTITY labels (class {_NON_ENTITY_ID}) found!")
        n_ents = (valid_labels < _NON_ENTITY_ID).sum().item()
        print(f"✅ Found {n_ents} entity labels (classes 0-{_NON_ENTITY_ID - 1})")
        print("\nLabel distribution in sample:")
        for label in valid_labels.unique().sort()[0]:
            count = (valid_labels == label).sum().item()
            name  = ENTITY_ID_TO_TYPE.get(int(label), f"CLASS_{int(label)}")
            print(f"   {name:<30} (class {int(label):2d}): {count:4d}")

    def _run_diagnostic_analysis(self, test_loader):
        print("\n" + "=" * 80)
        print("🔍 RUNNING POST-TRAINING DIAGNOSTIC ANALYSIS")
        print("=" * 80)
        if hasattr(self.trainer, 'analyze_model_predictions'):
            try:
                device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                self.model = self.model.to(device)
                self.model.eval()
                self.trainer.analyze_model_predictions(
                    test_loader=test_loader,
                    label_encoder=self.label_encoder,
                )
                print("✅ Diagnostic analysis complete")
            except Exception as e:
                self.logger.warning(f"⚠️  Diagnostic analysis failed: {e}")
        else:
            self.logger.warning(
                "⚠️  analyze_model_predictions not available in trainer"
            )

    def train_model(
            self,
            train_loader,
            val_loader,
            test_loader,
        ) -> Tuple[object, Dict]:
            """
            FIX (#2): set_class_weights now called with from_train_only=True
            because _calculate_class_weights uses phase_0_loader which is
            derived exclusively from training graphs.

            FIX (#9): val_loader passed to train_with_curriculum for per-epoch
            early stopping; test_loader passed for the single post-training
            evaluation only.

            FIX (#10): plot_confusion_matrix_enhanced moved out of the
            plot_training_curves except block so it always runs after training,
            guarded by test_loader and ENTITY_ID_TO_TYPE instead of label_encoder.
            """
            print("\n" + "=" * 80)
            print("STEP 2: TRAINING MODEL WITH CURRICULUM LEARNING")
            print("=" * 80)

            model        = self._create_model()
            self.trainer = CurriculumTrainer(model, self.config, self.logger)

            phase_0_loader = train_loader.get_phase_loader(0, shuffle=False)

            class_weights = self._calculate_class_weights(phase_0_loader)
            self.trainer.set_class_weights(class_weights, from_train_only=True)

            self._inspect_training_sample(phase_0_loader)

            raw_epochs = self.config.get('epochs', [15, 20, 25, 30])
            if isinstance(raw_epochs, int):
                per_phase        = max(1, raw_epochs // 4)
                epochs_per_phase = [per_phase] * 4
                self.logger.warning(
                    f"config['epochs'] is int ({raw_epochs}); "
                    f"expanding to {epochs_per_phase} for 4-phase curriculum."
                )
            elif isinstance(raw_epochs, (list, tuple)):
                epochs_per_phase = list(raw_epochs)
            else:
                self.logger.error(
                    f"config['epochs'] has unexpected type {type(raw_epochs)}; "
                    f"falling back to [15, 20, 25, 30]."
                )
                epochs_per_phase = [15, 20, 25, 30]

            history = self.trainer.train_with_curriculum(
                curriculum_loader=train_loader,
                val_loader=val_loader,
                test_loader=test_loader,
                epochs_per_phase=epochs_per_phase,
            )

            model      = self._load_best_model(model)
            self.model = model

            self._run_diagnostic_analysis(test_loader)

            try:
                self.visualizer.plot_curriculum_history(history)
            except KeyError as e:
                self.logger.warning(
                    f"⚠️  plot_curriculum_history skipped — missing key {e}. "
                    f"Available history keys: {list(history.keys())}"
                )
            except Exception as e:
                self.logger.warning(f"⚠️  plot_curriculum_history failed: {e}")

            try:
                self.visualizer.plot_training_curves(history)
            except Exception as e:
                self.logger.warning(f"⚠️  plot_training_curves failed: {e}")

            if test_loader is not None and ENTITY_ID_TO_TYPE:
                try:
                    y_true, y_pred = self._get_test_predictions(test_loader)
                    cm_path = self.visual_dir / "confusion_matrix.png"
                    cm_path.parent.mkdir(parents=True, exist_ok=True)
                    self.visualizer.plot_confusion_matrix_enhanced(
                        y_true, y_pred,
                        label_map=ENTITY_ID_TO_TYPE,
                        filename=str(cm_path),
                    )
                    self.logger.info(f"✅ Confusion matrix saved to {cm_path}")
                except Exception as e:
                    self.logger.warning(
                        f"⚠️  plot_confusion_matrix_enhanced failed: {e}"
                    )
            else:
                self.logger.warning(
                    "⚠️  Confusion matrix skipped — "
                    f"test_loader={'None' if test_loader is None else 'OK'}, "
                    f"ENTITY_ID_TO_TYPE={'empty' if not ENTITY_ID_TO_TYPE else 'OK'}."
                )

            print(f"\n✅ Training complete:")
            print(f"   Best val F1 Score : {self.trainer.best_f1:.4f}")
            print(f"   Total epochs      : {len(history['train_loss'])}")
            return model, history

    def _get_test_predictions(self, test_loader):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.eval()
        all_preds:  list = []
        all_labels: list = []
        with torch.no_grad():
            for batch in test_loader:
                batch  = batch.to(device)
                logits = self.model(batch)
                preds  = logits.argmax(dim=1)
                mask   = batch.y >= 0
                all_preds.extend(preds[mask].cpu().numpy())
                all_labels.extend(batch.y[mask].cpu().numpy())
        return np.array(all_labels), np.array(all_preds)

    # =========================================================================
    # CLASS WEIGHTS
    # =========================================================================

    def _calculate_class_weights(self, phase_loader) -> torch.Tensor:
        NON_ENTITY_WEIGHT_FLOOR = 0.30
        WEIGHT_MAX              = 5.0
        WEIGHT_MIN              = 0.15
        SPARSE_THRESHOLD        = 30
        SPARSE_CLASS_WEIGHT_CAP = 3.0

        all_labels: list = []
        for batch in phase_loader:
            labels = batch.y
            all_labels.extend(labels[labels >= 0].cpu().numpy().tolist())

        total        = max(len(all_labels), 1)
        class_counts: Counter = Counter(all_labels)

        nonzero_counts = [
            class_counts[c]
            for c in range(NUM_ENTITY_TYPES)
            if class_counts.get(c, 0) > 0
        ]
        if not nonzero_counts:
            self.logger.error("❌ All class counts are zero!")
            return torch.ones(NUM_ENTITY_TYPES)
        median_count = float(np.median(nonzero_counts))

        weights = torch.zeros(NUM_ENTITY_TYPES)

        print("\n⚖️  Calculating class weights (SQRT-DAMPENED + sparse cap)...")
        print(f"   Method : weight = sqrt(median_count / class_count)")
        print(f"   Median count       : {median_count:.0f}")
        print(f"   Max cap            : {WEIGHT_MAX}")
        print(f"   Sparse cap (n<{SPARSE_THRESHOLD:2d}) : {SPARSE_CLASS_WEIGHT_CAP}")
        print(f"   NON_ENTITY floor   : {NON_ENTITY_WEIGHT_FLOOR}")
        print(
            f"\n📊 NON_ENTITY class ({_NON_ENTITY_ID}): "
            f"{class_counts.get(_NON_ENTITY_ID, 0):,} samples "
            f"({class_counts.get(_NON_ENTITY_ID, 0) / total * 100:.1f}%)"
        )
        print(f"\n📊 Class Distribution (total samples: {total:,}):")
        print(
            f"{'Class ID':<10} {'Class Name':<30} {'Count':<8} {'Pct':<8} "
            f"{'Raw w':<8} {'Final w':<8} {'Note'}"
        )
        print("-" * 90)

        for class_id in range(NUM_ENTITY_TYPES):
            count      = class_counts.get(class_id, 0)
            freq       = count / total if count > 0 else 0
            class_name = ENTITY_ID_TO_TYPE.get(class_id, f"CLASS_{class_id}")

            if count == 0:
                w     = 0.0
                raw_w = 0.0
                note  = "zero-sample → masked"
            elif class_id == _NON_ENTITY_ID:
                raw_w = np.sqrt(median_count / count)
                w     = float(max(NON_ENTITY_WEIGHT_FLOOR, min(WEIGHT_MAX, raw_w)))
                note  = f"NON_ENTITY floor={NON_ENTITY_WEIGHT_FLOOR}"
            else:
                raw_w = np.sqrt(median_count / count)
                w     = float(max(WEIGHT_MIN, min(WEIGHT_MAX, raw_w)))
                if count < SPARSE_THRESHOLD and w > SPARSE_CLASS_WEIGHT_CAP:
                    note = f"sparse cap {w:.2f}→{SPARSE_CLASS_WEIGHT_CAP}"
                    w    = SPARSE_CLASS_WEIGHT_CAP
                else:
                    note = ""

            weights[class_id] = w
            pct_str = f"{freq * 100:.2f}%" if count > 0 else "0.00%"
            print(
                f"{class_id:<10} {class_name:<30} {count:<8} {pct_str:<8} "
                f"{raw_w:<8.3f} w={w:<7.4f} {note}"
            )

        active = weights[weights > 0]
        if len(active) > 0:
            max_w = active.max().item()
            min_w = active.min().item()
            ratio = max_w / max(min_w, 1e-8)
            print(f"\n📊 Weight Summary:")
            print(f"   Min weight    : {min_w:.4f}")
            print(f"   Max weight    : {max_w:.4f}")
            print(f"   Max/Min ratio : {ratio:.1f}x")
            if ratio > 50:
                print(f"   ⚠️  WARNING: Weight ratio > 50x")
            else:
                print(f"   ✅ Weight ratio healthy ({ratio:.1f}x)")

        return weights

    # =========================================================================
    # MEANINGFUL RULE SCORING & CONTEXT ENRICHMENT
    # =========================================================================
    @staticmethod
    def _score_sentence_normativity(sentence: str) -> float:
        """Return a 0.0–1.0 score for how normative/prescriptive a sentence is."""
        s = sentence.lower().strip()
        if len(s.split()) < 5:
            return 0.0

        score = 0.0

        # Strong prescriptive modals (+0.50)
        strong = {'shall', 'must', 'is required', 'are required', 'is prohibited', 'are prohibited'}
        if any(m in s for m in strong):
            score += 0.50

        # Weak prescriptive (+0.20)
        weak = {'should', 'in accordance with', 'comply with', 'conform to', 'is permitted'}
        if any(m in s for m in weak) and score < 0.40:
            score += 0.20

        # Unit‑bearing measurement (+0.15)
        if re.search(r'\d+\s*(?:mm|cm|m\b|ft|in|inch|kg|psi|°c|°f|%|hour|minute|stor)', s):
            score += 0.15

        # Compliance verbs (+0.10)
        verbs = {'install', 'provide', 'maintain', 'protect', 'equip', 'listed', 'approved'}
        if any(v in s.split() for v in verbs):
            score += 0.10

        # Penalties
        if re.search(r'\b(?:see|refer to|as shown in|figure|table|note:|definition of)\b', s):
            score -= 0.30
        if len(s.split()) < 8:
            score -= 0.20
        if s.startswith(('see', 'refer', 'note')):
            score -= 0.40

        return max(0.0, min(1.0, score))

    @staticmethod
    def _is_meaningful_rule(sentence: str) -> bool:
        """
        Preserved for any callers that still need a boolean gate.
        Now delegates to _score_sentence_normativity — a sentence is
        'meaningful' if its normativity score exceeds 0.20.
        Removes the hard filter that was discarding ~40% of candidates.
        """
        return RegulationPipeline._score_sentence_normativity(sentence) > 0.20

    def _enrich_entity_rule(self, entity_value: str, entity_type: str,
                            candidate_start: int, candidate_end: int,
                            chunk_doc, chunk_text: str, chunk_start: int = 0) -> Tuple[str, float, str]:
        """Return (best_rule, normativity, quality) using paragraph‑aware extraction."""

        # 1. Split the chunk into paragraphs (double newline)
        paragraphs = re.split(r'\n\s*\n', chunk_text)
        para_starts = []
        pos = 0
        for para in paragraphs:
            para_starts.append(pos)
            pos += len(para) + 2  # approximate, but good enough for containment

        # 2. Find the paragraph that contains the entity
        para_idx = -1
        for i, p_start in enumerate(para_starts):
            p_end = para_starts[i+1] if i+1 < len(para_starts) else len(chunk_text)
            if p_start <= candidate_start < p_end:
                para_idx = i
                break

        best_text = ""
        best_score = 0.0

        if para_idx != -1:
            para_text = paragraphs[para_idx].strip()
            para_score = self._score_sentence_normativity(para_text)
            if para_score >= 0.35:          # good enough paragraph
                best_text = para_text
                best_score = para_score
            else:
                # Paragraph is weak – fall back to sentence search inside this paragraph only
                sent_doc = self.labeler.nlp(para_text)
                sentences = list(sent_doc.sents)
                for sent in sentences:
                    sent_text = sent.text.strip()
                    if len(sent_text) < 15:
                        continue
                    if entity_value.lower() in sent_text.lower():
                        score = self._score_sentence_normativity(sent_text)
                        if score > best_score:
                            best_score = score
                            best_text = sent_text

        # 3. If still no good rule, expand to neighbouring sentences (±2) in the whole chunk
        if best_score < 0.35:
            sentences = list(chunk_doc.sents)
            containing_idx = None
            for idx, sent in enumerate(sentences):
                if sent.start_char <= candidate_start < sent.end_char:
                    containing_idx = idx
                    break
            if containing_idx is not None:
                start = max(0, containing_idx - 2)
                end = min(len(sentences), containing_idx + 3)
                for j in range(start, end):
                    sent_text = sentences[j].text.strip()
                    if len(sent_text) < 15:
                        continue
                    # Boost if the sentence shares meaningful tokens with the entity
                    score = self._score_sentence_normativity(sent_text)
                    if entity_value.lower() in sent_text.lower():
                        score += 0.15
                    if score > best_score:
                        best_score = score
                        best_text = sent_text

        # 4. Final fallback – use a generous window around the entity
        if best_score < 0.20:
            ctx_start = max(0, candidate_start - 200)
            ctx_end = min(len(chunk_text), candidate_end + 200)
            best_text = chunk_text[ctx_start:ctx_end].strip()
            best_score = self._score_sentence_normativity(best_text)

        quality = self._rule_quality_label(best_score)
        return best_text, best_score, quality

    @staticmethod
    def _rule_quality_label(score: float) -> str:
        if score >= 0.50:
            return 'strong'
        if score >= 0.30:
            return 'moderate'
        if score >= 0.10:
            return 'weak'
        return 'definition'


    def _extract_sentence_from_doc(
        self,
        doc,
        candidate_start: int,
        candidate_end:   int,
        chunk_text:      str,
    ) -> str:
        """
        PERF: extract the containing sentence from an already-built spaCy doc
        instead of calling nlp(text) fresh for every candidate.

        _extract_gat_from_chunks(); the old method is kept for callers that
        don't have a pre-built doc.
        """
        try:
            for sent in doc.sents:
                if sent.start_char <= candidate_start < sent.end_char:
                    return sent.text.strip()
        except Exception as e:
            self.logger.warning(f"Failed to extract sentence from doc: {e}")
        ctx_s = max(0, candidate_start - 100)
        ctx_e = min(len(chunk_text), candidate_end + 100)
        return chunk_text[ctx_s:ctx_e].strip()


    def _check_regex_match(self, text: str, predicted_type: str) -> bool:
        patterns = self.labeler.compiled_patterns.get(predicted_type, [])
        return any(p.search(text) for p in patterns)

    def _get_confidence_threshold(self, entity_type: str, is_novel: bool) -> float:
        if not is_novel:
            return self._ABSOLUTE_FLOOR
        floor = self._CLASS_CONFIDENCE_FLOORS.get(entity_type, 0.25)
        return max(self._ABSOLUTE_FLOOR, floor)

    def _post_process_predictions(self, predictions: List[Dict]) -> List[Dict]:
        enhanced_predictions = list(predictions)
        seen_texts = {p['entity_value'].lower().strip() for p in predictions}

        component_keywords     = ['wall', 'door', 'window', 'floor', 'ceiling',
                                  'barrier', 'partition', 'enclosure', 'shaft',
                                  'stair', 'ramp']
        material_keywords      = ['steel', 'concrete', 'wood', 'gypsum', 'masonry',
                                  'timber', 'glass', 'aluminum', 'plastic',
                                  'composite', 'brick', 'block']
        facility_keywords      = ['room', 'area', 'space', 'compartment', 'zone',
                                  'refuge', 'corridor', 'lobby', 'hall',
                                  'chamber', 'lounge']
        certification_keywords = ['listed', 'certified', 'approved', 'labeled',
                                  'fm', 'ul', 'label', 'certification', 'listing']
        building_type_keywords = ['residential', 'commercial', 'industrial',
                                  'office', 'school', 'hospital', 'warehouse',
                                  'assembly']

        _type_to_id: Dict[str, int] = REGULATION_ENTITY_TYPES
        new_candidates: List[Dict] = []

        # FIX-A: _MEASUREMENT_TYPES lives on EntityLabeler, not on this class.
        # Use the labeler's attribute; fall back to a hardcoded sentinel set so
        # post-processing never crashes even if the labeler is swapped out.
        _measurement_types: frozenset = getattr(
            self.labeler,
            '_MEASUREMENT_TYPES',
            frozenset({
                'DIMENSION', 'AREA', 'VOLUME', 'PRESSURE', 'FLOW_RATE',
                'TEMPERATURE', 'PERCENTAGE', 'FIRE_RATING', 'EGRESS_DISTANCE',
                'SEPARATION_DISTANCE', 'CLEARANCE', 'CAPACITY',
            }),
        )

        for pred in predictions:
            text = pred['entity_value'].lower()
            if len(text) < 3:
                continue

            def _add_candidate(new_type: str):
                text_lower = text.strip()
                if text_lower in seen_texts:
                    return
                new_pred               = pred.copy()
                new_pred['type']       = new_type
                new_pred['type_id']    = _type_to_id.get(new_type, pred.get('type_id'))
                new_pred['confidence'] *= 0.85
                new_pred['source']     = f'post_processed_{new_type.lower()}'
                new_candidates.append(new_pred)
                seen_texts.add(text_lower)

            if any(kw in text for kw in component_keywords):
                if pred['type'] not in ('COMPONENT', 'EQUIPMENT'):
                    _add_candidate('COMPONENT')
            if any(kw in text for kw in material_keywords):
                if pred['type'] not in ('MATERIAL', 'COMPONENT'):
                    _add_candidate('MATERIAL')
            if any(kw in text for kw in facility_keywords):
                if pred['type'] not in ('FACILITY', 'LOCATION'):
                    _add_candidate('FACILITY')
            if any(kw in text for kw in certification_keywords):
                if pred['type'] not in ('CERTIFICATION', 'STANDARD'):
                    _add_candidate('CERTIFICATION')
            if any(kw in text for kw in building_type_keywords):
                if pred['type'] not in ('BUILDING_TYPE', 'OCCUPANCY'):
                    _add_candidate('BUILDING_TYPE')

            # FIX-A: was `_MEASUREMENT_TYPES` (NameError); now `_measurement_types`
            if (any(c.isdigit() for c in text)
                    and pred['type'] not in _measurement_types):
                new_pred               = pred.copy()
                new_pred['type']       = 'DIMENSION'
                new_pred['type_id']    = _type_to_id.get('DIMENSION', pred.get('type_id'))
                new_pred['confidence'] *= 0.80
                new_pred['source']     = 'post_processed_dimension'
                text_lower             = text.strip()
                if text_lower not in seen_texts:
                    new_candidates.append(new_pred)
                    seen_texts.add(text_lower)

        enhanced_predictions.extend(new_candidates)
        if new_candidates:
            type_counts  = Counter(c['type'] for c in new_candidates)
            type_summary = ', '.join(f"{t}:{c}" for t, c in type_counts.items())
            print(f"   Post-processing added {len(new_candidates)} candidates ({type_summary})")
        return enhanced_predictions

    def _extract_regex_from_chunks(self, chunks: List[Dict]) -> List[Dict]:
        """
        Extract regex entities from all chunks.
        Rule field is the original containing sentence with newlines stripped.
        _enrich_entity_rule is still called for normativity scoring and
        rule_quality, but its best_rule output is no longer used as the Rule.
        """
        _VALIDATION_GATE: Dict[str, float] = {
            'DIMENSION':   0.10,
            'AREA':        0.10,
            'VOLUME':      0.10,
            'PRESSURE':    0.15,
            'FLOW_RATE':   0.15,
            'TEMPERATURE': 0.15,
            'STANDARD':    0.20,
            'EQUIPMENT':   0.20,
            'MATERIAL':    0.20,
        }
        _DEFAULT_GATE: float = 0.25

        all_entities:    list = []
        filtered_invalid = 0
        filtered_gate    = 0

        quality_counts: Dict[str, int] = {
            'strong': 0, 'moderate': 0, 'weak': 0, 'definition': 0
        }

        print(f"   Processing {len(chunks)} chunks with regex...")

        for chunk in tqdm(chunks, desc="Regex extraction"):
            chunk_doc   = self.labeler.nlp(chunk['text'])
            chunk_start = chunk.get('start_char', 0)

            for entity in self.labeler.extract_entities_from_chunk(
                chunk['text'], doc=chunk_doc
            ):
                if not entity.is_valid:
                    filtered_invalid += 1
                    continue

                gate = _VALIDATION_GATE.get(entity.entity_type, _DEFAULT_GATE)
                if entity.validation_score < gate:
                    filtered_gate += 1
                    continue

                # Score the original sentence for normativity/quality metadata,
                # but keep it (newlines stripped) as the Rule value.
                original_rule = entity.full_sentence
                clean_rule    = ' '.join(original_rule.split())   # collapses \n and whitespace

                _, normativity, quality = self._enrich_entity_rule(
                    entity_value    = entity.entity_value,
                    entity_type     = entity.entity_type,
                    candidate_start = entity.start,
                    candidate_end   = entity.end,
                    chunk_doc       = chunk_doc,
                    chunk_text      = chunk['text'],
                    chunk_start     = chunk_start,
                )
                quality_counts[quality] += 1

                all_entities.append({
                    'entity_value':     entity.entity_value,
                    'type':             entity.entity_type,
                    'type_id':          entity.type_id,
                    'confidence':       entity.confidence,
                    'validation_score': entity.validation_score,
                    'Rule':             clean_rule,          # original sentence, no \n
                    'rule_normativity': round(normativity, 3),
                    'rule_quality':     quality,
                    'semantic_context': entity.semantic_context,
                    'chunk_id':         chunk['chunk_id'],
                    'source':           'regex',
                    'start':            entity.start,
                    'end':              entity.end,
                    'chunk_start':      chunk_start,
                    'absolute_start':   chunk_start + entity.start,
                    'absolute_end':     chunk_start + entity.end,
                })

        total_accepted = len(all_entities)
        total_seen     = total_accepted + filtered_invalid + filtered_gate
        print(
            f"\n📊 Regex Extraction Summary:\n"
            f"   Total candidates     : {total_seen:,}\n"
            f"   Filtered (invalid)   : {filtered_invalid:,}\n"
            f"   Filtered (low score) : {filtered_gate:,}\n"
            f"   Accepted             : {total_accepted:,}\n"
            f"   Rule quality breakdown:\n"
            f"      strong     : {quality_counts['strong']:,}\n"
            f"      moderate   : {quality_counts['moderate']:,}\n"
            f"      weak       : {quality_counts['weak']:,}\n"
            f"      definition : {quality_counts['definition']:,}\n"
            f"✅ Regex extracted {total_accepted} entities"
        )
        return all_entities

    def _extract_gat_from_chunks(
        self,
        model,
        chunks:               List[Dict],
        confidence_threshold: float,
        base_threshold:       float = 0.3,
        novel_threshold:      float = 0.4,
    ) -> List[Dict]:
        """
        Run GAT inference over all chunks.

        Changes vs. original
        --------------------
        FIX-PLAUS : After the GAT predicts a class, _is_plausible() is
                    called on the (span_text, predicted_type) pair.  If the
                    prediction is implausible (e.g. GAT says PERCENTAGE but
                    the span contains no '%' or 'percent'):
                      - Priority sources  → demote to regex_fallback
                      - Novel sources     → drop entirely
                    This eliminates a whole category of confident hallucinations
                    where the GAT learns spurious correlations between span
                    position and entity class.

        All other logic (threshold gating, source classification, span
        stripping, normativity scoring, diagnostics) is unchanged.
        """
        model.eval()
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model  = model.to(device)
        all_predictions = []

        PRIORITY_SOURCES = {'training_aligned', 'measurement_regex'}

        diagnostics = {
            'total_candidates':        0,
            'total_chunks':            len(chunks),
            'non_entity_predictions':  0,
            'below_threshold':         0,
            'accepted':                0,
            'span_stripped':           0,
            'filtered_meaningless':    0,
            'implausible_demoted':     0,    # ← NEW counter
            'implausible_dropped':     0,    # ← NEW counter
            'by_source':               {},
            'by_predicted_class':      {},
            'confidence_distribution': [],
            'type_agreement':          0,
            'type_disagreement':       0,
            'regex_fallback':          0,
            'below_threshold_by_class': {},
        }

        print(f"   Processing {len(chunks)} chunks with GAT...")

        for chunk in tqdm(chunks, desc="GAT extraction"):
            candidates = self.candidate_extractor.extract_candidates(
                chunk['text'], training_mode=False
            )
            if not candidates:
                continue
            diagnostics['total_candidates'] += len(candidates)

            graph = self.graph_builder.build_graph_for_inference(chunk, candidates)
            if graph is None:
                continue

            chunk_doc          = self.labeler.nlp(chunk['text'])
            graph              = graph.to(device)
            candidate_metadata = graph.candidate_metadata

            with torch.no_grad():
                logits                    = model(graph)
                probs                     = torch.softmax(logits, dim=1)
                confidences, pred_classes = probs.max(dim=1)

            for i, candidate in enumerate(candidate_metadata):
                node_idx = i + 1
                if node_idx >= logits.size(0):
                    continue

                pred_class = pred_classes[node_idx].item()
                conf       = confidences[node_idx].item()
                diagnostics['confidence_distribution'].append(conf)

                class_name = ENTITY_ID_TO_TYPE.get(pred_class, f"CLASS_{pred_class}")
                diagnostics['by_predicted_class'][class_name] = (
                    diagnostics['by_predicted_class'].get(class_name, 0) + 1
                )

                source           = candidate.get('source', 'unknown')
                is_priority      = source in PRIORITY_SOURCES
                is_novel         = not is_priority
                expected_type_id = candidate.get('expected_type_id')
                expected_type    = candidate.get('expected_type')

                c_start     = candidate['start']
                c_end       = candidate['end']
                chunk_start = chunk.get('start_char', 0)

                full_sentence = self._extract_sentence_from_doc(
                    chunk_doc, c_start, c_end, chunk['text']
                )
                clean_rule = ' '.join(full_sentence.split())

                _, normativity, rule_quality = self._enrich_entity_rule(
                    entity_value    = candidate['text'],
                    entity_type     = ENTITY_ID_TO_TYPE.get(pred_class, 'UNKNOWN'),
                    candidate_start = c_start,
                    candidate_end   = c_end,
                    chunk_doc       = chunk_doc,
                    chunk_text      = chunk['text'],
                    chunk_start     = chunk_start,
                )

                if rule_quality == 'definition':
                    conf = max(self._ABSOLUTE_FLOOR, conf * 0.80)
                elif rule_quality == 'weak':
                    conf = max(self._ABSOLUTE_FLOOR, conf * 0.90)

                should_include  = False
                final_type      = final_type_id = None
                final_conf      = conf
                final_source    = source

                # ── Decision tree ─────────────────────────────────────────────
                if pred_class == _NON_ENTITY_ID or pred_class < 0:
                    diagnostics['non_entity_predictions'] += 1
                    if is_priority and expected_type_id is not None:
                        should_include = True
                        final_type     = expected_type
                        final_type_id  = expected_type_id
                        final_conf     = candidate.get('confidence', 0.7)
                        final_source   = 'regex_fallback'
                        diagnostics['regex_fallback'] += 1
                    else:
                        continue
                else:
                    gat_type = ENTITY_ID_TO_TYPE.get(pred_class, 'UNKNOWN')
                    if gat_type == 'UNKNOWN':
                        continue

                    # ── FIX-PLAUS: plausibility gate ─────────────────────────
                    if not _is_plausible(candidate['text'], gat_type):
                        if is_priority and expected_type_id is not None:
                            # Demote to regex label — we trust the regex more
                            # than a GAT prediction that contradicts the span text
                            should_include = True
                            final_type     = expected_type
                            final_type_id  = expected_type_id
                            final_conf     = candidate.get('confidence', 0.7)
                            final_source   = 'regex_fallback'
                            diagnostics['regex_fallback']       += 1
                            diagnostics['implausible_demoted']  += 1
                        else:
                            # Novel candidate with implausible type → drop
                            diagnostics['implausible_dropped'] += 1
                        # Skip the normal decision tree below
                        if not should_include:
                            continue
                    else:
                        # ── Normal confidence-gated acceptance ────────────────
                        class_floor = self._get_confidence_threshold(gat_type, is_novel)

                        if is_priority:
                            if expected_type_id is not None:
                                should_include = True
                                if pred_class == expected_type_id:
                                    final_type    = gat_type
                                    final_type_id = pred_class
                                    final_source  = 'gat_confirmed'
                                    final_conf    = min(1.0, conf + 0.1)
                                    diagnostics['type_agreement'] += 1
                                else:
                                    final_type    = gat_type
                                    final_type_id = pred_class
                                    final_source  = 'gat_reclassified'
                                    diagnostics['type_disagreement'] += 1
                            else:
                                if conf >= class_floor:
                                    should_include = True
                                    final_type     = gat_type
                                    final_type_id  = pred_class
                                    final_source   = 'gat_priority_aligned'
                        else:
                            effective_threshold = class_floor
                            if gat_type in {'STANDARD', 'EQUIPMENT', 'DIMENSION'}:
                                effective_threshold = max(0.15, class_floor * 0.8)
                            if conf >= effective_threshold:
                                should_include = True
                                final_type     = gat_type
                                final_type_id  = pred_class
                                final_source   = 'gat_novel'

                if not should_include:
                    diagnostics['below_threshold'] += 1
                    diagnostics['below_threshold_by_class'][class_name] = (
                        diagnostics['below_threshold_by_class'].get(class_name, 0) + 1
                    )
                    continue

                diagnostics['accepted'] += 1
                diagnostics['by_source'][final_source] = (
                    diagnostics['by_source'].get(final_source, 0) + 1
                )

                if source not in PRIORITY_SOURCES:
                    raw_value      = candidate['text']
                    abs_start      = chunk_start + c_start
                    abs_end        = chunk_start + c_end
                    stripped_value = self._strip_boundary_noise(raw_value)
                    if stripped_value and stripped_value != raw_value:
                        stripped_value, abs_start, abs_end = self._normalize_span(
                            raw_value, stripped_value, abs_start, abs_end
                        )
                        c_start = abs_start - chunk_start
                        c_end   = abs_end   - chunk_start
                        diagnostics['span_stripped'] += 1
                    else:
                        stripped_value = raw_value
                else:
                    stripped_value = candidate['text']
                    abs_start      = chunk_start + c_start
                    abs_end        = chunk_start + c_end

                semantic_context = self.labeler._extract_semantic_context(
                    candidate['text'], full_sentence, final_type, chunk_doc
                )

                all_predictions.append({
                    'entity_value':        stripped_value,
                    'type':                final_type,
                    'type_id':             final_type_id,
                    'confidence':          final_conf,
                    'start':               c_start,
                    'end':                 c_end,
                    'chunk_id':            chunk['chunk_id'],
                    'source':              final_source,
                    'chunk_start':         chunk_start,
                    'absolute_start':      abs_start,
                    'absolute_end':        abs_end,
                    'Rule':                clean_rule,
                    'rule_normativity':    round(normativity, 3),
                    'rule_quality':        rule_quality,
                    'semantic_context':    semantic_context,
                    'expected_type':       expected_type,
                    'expected_type_id':    expected_type_id,
                    'gat_predicted_class': pred_class,
                    'gat_confidence':      conf,
                })

        # ── Diagnostics printout ──────────────────────────────────────────────
        print(f"\n📊 GAT Extraction Diagnostics:")
        print(f"   Total chunks         : {diagnostics['total_chunks']}")
        print(f"   Total candidates     : {diagnostics['total_candidates']}")
        non_e   = diagnostics['non_entity_predictions']
        total_c = max(1, diagnostics['total_candidates'])
        print(f"   Predicted NON_ENTITY : {non_e} ({non_e / total_c * 100:.1f}%)")
        print(f"   Below threshold      : {diagnostics['below_threshold']}")
        print(f"   Accepted             : {diagnostics['accepted']}")
        print(f"   Spans stripped       : {diagnostics['span_stripped']}")
        print(f"\n   Plausibility filter:")
        print(f"      Implausible demoted (priority→fallback) : {diagnostics['implausible_demoted']}")
        print(f"      Implausible dropped (novel)             : {diagnostics['implausible_dropped']}")
        print(f"\n   Type Agreement Analysis:")
        print(f"      GAT confirmed regex : {diagnostics['type_agreement']}")
        print(f"      GAT disagreed       : {diagnostics['type_disagreement']}")
        print(f"      Regex fallback used : {diagnostics['regex_fallback']}")
        print(f"\n   By source:")
        for src, cnt in sorted(
            diagnostics['by_source'].items(), key=lambda x: x[1], reverse=True
        ):
            print(f"      {src}: {cnt}")
        print(f"\n   By predicted class (top 10):")
        for cls, cnt in sorted(
            diagnostics['by_predicted_class'].items(),
            key=lambda x: x[1], reverse=True,
        )[:10]:
            print(f"      {cls}: {cnt}")
        if diagnostics['below_threshold_by_class']:
            print(f"\n   Below-threshold drops by class (top 10):")
            for cls, cnt in sorted(
                diagnostics['below_threshold_by_class'].items(),
                key=lambda x: x[1], reverse=True,
            )[:10]:
                print(f"      {cls}: {cnt}")

        return all_predictions

    def extract_entities(
        self,
        text:                 str,
        method:               str   = 'both',
        model                       = None,
        confidence_threshold: float = 0.7,
        pdf_path:             str   = None,
    ) -> Dict:
        if pdf_path:
            chunks = None
            if self.text_cache_manager.has_cached_chunks(pdf_path, text):
                chunks = self.text_cache_manager.load_cached_chunks(pdf_path, text)
                if chunks:
                    print(f"\n✂️  Loaded {len(chunks)} chunks from cache")
                else:
                    chunks = None

            if chunks is None:
                chunks = self.pdf_extractor.get_chunks_with_cache(
                    pdf_path, text, self.chunker, Path(pdf_path).stem
                )
                if chunks:
                    self.text_cache_manager.save_cached_chunks(pdf_path, text, chunks)
        else:
            # FIX-D: chunk_optimal may not exist on SemanticChunker.
            # Use pdf_extractor.get_chunks_with_cache with a synthetic key
            # derived from a content hash so the cache layer still works.
            # Fall back to chunk_optimal only when it is actually present.
            if hasattr(self.chunker, 'chunk_optimal'):
                chunks = self.chunker.chunk_optimal(text, "extraction")
            else:
                # Synthesise a stable cache key from the content so repeated
                # calls to extract_entities with the same text hit the cache.
                import hashlib as _hashlib
                _content_hash = _hashlib.md5(text.encode()).hexdigest()[:12]
                _synthetic_pdf_path = f"__inline_{_content_hash}__.pdf"
                chunks = self.pdf_extractor.get_chunks_with_cache(
                    _synthetic_pdf_path, text, self.chunker, f"inline_{_content_hash}"
                )

        print(f"\n✂️  {len(chunks)} chunks for extraction")
        results = {'chunks': chunks, 'num_chunks': len(chunks)}

        if method in ['regex', 'both']:
            print("\n🔍 Extracting with regex...")
            results['regex'] = self._extract_regex_from_chunks(chunks)

        if method in ['gat', 'both']:
            if model is None:
                raise ValueError("Model is required for GAT extraction")
            print("\n🧠 Extracting with GAT...")
            results['gat'] = self._extract_gat_from_chunks(
                model, chunks, confidence_threshold
            )

        return results

    # =========================================================================
    # COMPARISON & ANALYSIS
    # =========================================================================

    def _text_normalized_overlap(
        self,
        regex_entities: List[Dict],
        gat_entities:   List[Dict],
    ) -> Dict[str, float]:
        regex_pairs     = {(e['entity_value'].lower().strip(), e['type']) for e in regex_entities}
        gat_pairs       = {(e['entity_value'].lower().strip(), e['type']) for e in gat_entities}
        gat_text_only   = {e['entity_value'].lower().strip() for e in gat_entities}
        regex_text_only = {e['entity_value'].lower().strip() for e in regex_entities}
        exact_both      = regex_pairs     & gat_pairs
        text_only_both  = regex_text_only & gat_text_only
        n_regex = max(len(regex_pairs), 1)
        return {
            'text_exact_overlap_pct':    len(exact_both)    / n_regex * 100,
            'text_agnostic_overlap_pct': len(text_only_both) / n_regex * 100,
            'exact_both':                len(exact_both),
            'text_only_both':            len(text_only_both),
            'iou_gap':                   0.0,
        }

    def _analyze_comparison(
        self,
        regex_entities: List[Dict],
        gat_entities:   List[Dict],
    ) -> Dict:
        self.logger.info("📊 Analyzing comparison with fuzzy matching...")

        strict_matches, strict_regex_only, strict_gat_only = \
            self.fuzzy_matcher.find_matches(
                regex_entities, gat_entities,
                iou_threshold=0.8, type_threshold=1.0,
            )
        relaxed_matches, relaxed_regex_only, relaxed_gat_only = \
            self.fuzzy_matcher.find_matches(
                regex_entities, gat_entities,
                iou_threshold=0.5, type_threshold=0.8,
            )

        gat_improvement = (
            len(relaxed_gat_only) / len(regex_entities) * 100
            if regex_entities else 0
        )

        regex_types = Counter([e['type'] for e in regex_entities])
        gat_types   = Counter([e['type'] for e in gat_entities])
        regex_confs = [e.get('confidence', 0) for e in regex_entities]
        gat_confs   = [e.get('confidence', 0) for e in gat_entities]

        matched_pairs = [
            {
                'regex': regex_entities[ri],
                'gat':   gat_entities[gi],
                'iou':   self.fuzzy_matcher.calculate_span_overlap(
                            regex_entities[ri], gat_entities[gi]
                         ),
            }
            for ri, gi in strict_matches
        ]

        stats = {
            'strict': {
                'total_regex':        len(regex_entities),
                'total_gat':          len(gat_entities),
                'found_by_both':      len(strict_matches),
                'regex_only':         len(strict_regex_only),
                'gat_only':           len(strict_gat_only),
                'overlap_percentage': (
                    len(strict_matches) / len(regex_entities) * 100
                    if regex_entities else 0
                ),
            },
            'relaxed': {
                'total_regex':        len(regex_entities),
                'total_gat':          len(gat_entities),
                'found_by_both':      len(relaxed_matches),
                'regex_only':         len(relaxed_regex_only),
                'gat_only':           len(relaxed_gat_only),
                'overlap_percentage': (
                    len(relaxed_matches) / len(regex_entities) * 100
                    if regex_entities else 0
                ),
            },
            'total_regex':             len(regex_entities),
            'total_gat':               len(gat_entities),
            'gat_improvement':         gat_improvement,
            'matched_pairs':           matched_pairs,
            'regex_only_entities':     [regex_entities[i] for i in strict_regex_only],
            'gat_only_entities':       [gat_entities[i]   for i in strict_gat_only],
            'regex_type_distribution': dict(regex_types),
            'gat_type_distribution':   dict(gat_types),
            'regex_avg_confidence':    float(np.mean(regex_confs)) if regex_confs else 0.0,
            'gat_avg_confidence':      float(np.mean(gat_confs))   if gat_confs   else 0.0,
            'regex_confidences':       [float(c) for c in regex_confs],
            'gat_confidences':         [float(c) for c in gat_confs],
        }

        text_stats = self._text_normalized_overlap(regex_entities, gat_entities)
        text_stats['iou_gap'] = (
            text_stats['text_exact_overlap_pct']
            - stats['relaxed']['overlap_percentage']
        )
        stats['text_normalized'] = text_stats

        per_type_miss: Dict[str, Dict] = {}
        for entity_type, regex_count in regex_types.items():
            gat_count = gat_types.get(entity_type, 0)
            missed    = sum(
                1 for i in strict_regex_only
                if regex_entities[i]['type'] == entity_type
            )
            matched   = sum(
                1 for ri, _ in strict_matches
                if regex_entities[ri]['type'] == entity_type
            )
            per_type_miss[entity_type] = {
                'regex_count': regex_count,
                'gat_count':   gat_count,
                'matched':     matched,
                'missed':      missed,
                'recall':      matched / regex_count if regex_count else 0.0,
                'gat_surplus': max(0, gat_count - regex_count),
            }
        stats['per_type_miss'] = per_type_miss

        self.logger.info(
            f"\n📈 Comparison Results (Fuzzy Matching):\n"
            f"   Strict  — both: {stats['strict']['found_by_both']:,} "
            f"({stats['strict']['overlap_percentage']:.1f}%), "
            f"regex-only: {stats['strict']['regex_only']:,}, "
            f"gat-only: {stats['strict']['gat_only']:,}\n"
            f"   Relaxed — both: {stats['relaxed']['found_by_both']:,} "
            f"({stats['relaxed']['overlap_percentage']:.1f}%), "
            f"regex-only: {stats['relaxed']['regex_only']:,}, "
            f"gat-only: {stats['relaxed']['gat_only']:,}\n"
            f"   GAT Improvement: +{stats['gat_improvement']:.1f}%"
        )

        worst_5 = sorted(per_type_miss.items(), key=lambda kv: kv[1]['recall'])[:5]
        for etype, m in worst_5:
            self.logger.info(
                f"      {etype:<28} recall={m['recall']:.2f}  "
                f"regex={m['regex_count']}  gat={m['gat_count']}  missed={m['missed']}"
            )

        return stats

    def _create_fallback_stats(
        self,
        regex_entities: List[Dict],
        gat_entities:   List[Dict],
        partial_stats:  Dict,
    ) -> Dict:
        self.logger.warning("⚠️  Using fallback comparison (exact matching)")
        regex_set   = {(e['entity_value'].lower().strip(), e['type']) for e in regex_entities}
        gat_set     = {(e['entity_value'].lower().strip(), e['type']) for e in gat_entities}
        both        = regex_set & gat_set
        regex_only  = regex_set - gat_set
        gat_only    = gat_set   - regex_set
        overlap_pct = len(both) / len(regex_set) * 100 if regex_set else 0
        improvement = len(gat_only) / len(regex_set) * 100 if regex_set else 0
        shared = {
            'total_regex':        len(regex_entities),
            'total_gat':          len(gat_entities),
            'found_by_both':      len(both),
            'regex_only':         len(regex_only),
            'gat_only':           len(gat_only),
            'overlap_percentage': overlap_pct,
        }
        fallback = {
            'strict':          shared,
            'relaxed':         shared,
            'total_regex':     len(regex_entities),
            'total_gat':       len(gat_entities),
            'gat_improvement': improvement,
            **{
                k: partial_stats.get(k, v)
                for k, v in {
                    'regex_type_distribution': {},
                    'gat_type_distribution':   {},
                    'regex_avg_confidence':    0,
                    'gat_avg_confidence':      0,
                    'regex_confidences':       [],
                    'gat_confidences':         [],
                }.items()
            },
        }
        if 'deduplication' in partial_stats:
            fallback['deduplication'] = partial_stats['deduplication']
        return fallback

    def _print_comparison_summary(self, stats: Dict):
        print("\n" + "=" * 80)
        print("📈 COMPARISON SUMMARY")
        print("=" * 80)
        strict  = stats.get('strict', {})
        relaxed = stats.get('relaxed', {})
        print(f"""
    Total Entities:
      • Regex: {stats.get('total_regex', 0):,}
      • GAT:   {stats.get('total_gat', 0):,}
      • Diff:  {stats.get('total_gat', 0) - stats.get('total_regex', 0):+,}

    Overlap Analysis (Relaxed Matching - IoU >= 0.5, Related Types):
      • Found by both: {relaxed.get('found_by_both', 0):,} ({relaxed.get('overlap_percentage', 0):.1f}%)
      • Regex only:    {relaxed.get('regex_only', 0):,}
      • GAT only:      {relaxed.get('gat_only', 0):,}

    Overlap Analysis (Strict Matching - IoU >= 0.8, Exact Type):
      • Found by both: {strict.get('found_by_both', 0):,} ({strict.get('overlap_percentage', 0):.1f}%)
      • Regex only:    {strict.get('regex_only', 0):,}
      • GAT only:      {strict.get('gat_only', 0):,}

    Performance:
      • GAT Improvement: +{stats.get('gat_improvement', 0):.1f}%

    Confidence Scores:
      • Regex avg: {stats.get('regex_avg_confidence', 0):.3f}
      • GAT avg:   {stats.get('gat_avg_confidence', 0):.3f}
        """)
        if 'deduplication' in stats:
            dedup = stats['deduplication']
            r, g  = dedup.get('regex', {}), dedup.get('gat', {})
            if r and g:
                print(f"""
    Deduplication Results:
      • Regex: {r.get('original_count',0):,} -> {r.get('final_count',0):,} (-{r.get('removal_rate',0):.1f}%)
      • GAT:   {g.get('original_count',0):,} -> {g.get('final_count',0):,} (-{g.get('removal_rate',0):.1f}%)
                """)
        print("\n📊 Top 10 Entity Types:")
        print(f"{'Type':<30} {'Regex':<10} {'GAT':<10} {'Diff':<10}")
        print("-" * 70)
        r_dist    = stats.get('regex_type_distribution', {})
        g_dist    = stats.get('gat_type_distribution', {})
        all_types = set(r_dist) | set(g_dist)
        rows      = sorted(
            [(t, r_dist.get(t, 0), g_dist.get(t, 0), r_dist.get(t, 0) + g_dist.get(t, 0))
             for t in all_types],
            key=lambda x: x[3], reverse=True,
        )
        for t, rc, gc, _ in rows[:10]:
            print(f"{t:<30} {rc:<10} {gc:<10} {gc - rc:+d}")
        print("\n" + "=" * 80)

    def compare_regex_vs_gat(
        self,
        model,
        test_pdf_path:        str   = None,
        confidence_threshold: float = 0.7,
    ) -> Dict:
        print("\n" + "=" * 80)
        print("🔬 REGEX vs GAT COMPARISON")
        print("=" * 80)

        if test_pdf_path is None:
            test_pdf_path = self.config['test_pdf_path']
        doc_id = Path(test_pdf_path).stem

        print(f"\nDocument: {Path(test_pdf_path).name}")
        print("\n📄 Extracting text from PDF...")

        # FIX: probe Drive mount before attempting extraction.
        # After a long training run the FUSE mount frequently goes stale.
        # _ensure_drive_mounted() does a fast stat probe and only calls
        # force_remount=True if the probe fails — so there is no overhead
        # when the mount is healthy.
        if not self._ensure_drive_mounted(test_pdf_path):
            raise RuntimeError(
                f"❌ Google Drive mount is unreachable and could not be "
                f"recovered.  Cannot access '{test_pdf_path}'.\n"
                f"   Manual fix: run `drive.mount('/content/drive', "
                f"force_remount=True)` in a new cell, then re-run "
                f"run_test() or compare_regex_vs_gat() directly."
            )

        # Verify the file is actually present after the mount is confirmed live
        p = Path(test_pdf_path)
        if not p.exists():
            raise FileNotFoundError(
                f"❌ PDF not found at '{test_pdf_path}' even after mount check.\n"
                f"   Verify the path in config['test_pdf_path'] is correct."
            )

        text = self._get_text_for_pdf(test_pdf_path)
        if not text or len(text.strip()) < 100:
            raise ValueError(
                f"Insufficient text extracted from '{p.name}' "
                f"({len(text) if text else 0} chars).\n"
                f"  exists={p.exists()}, size="
                f"{p.stat().st_size if p.exists() else 'N/A'} bytes, "
                f"readable={os.access(str(p), os.R_OK)}\n"
                f"  Checks: (1) FUSE/Drive mount alive, "
                f"(2) scanned PDF needs OCR, "
                f"(3) preprocessor over-cleaning."
            )
        print(f"✅ Extracted {len(text):,} characters")

        extraction_results = self.extract_entities(
            text, method='both', model=model,
            confidence_threshold=confidence_threshold,
            pdf_path=test_pdf_path,
        )
        regex_entities = extraction_results['regex']
        gat_entities   = extraction_results['gat']

        print("\n🧹 Deduplicating entities...")
        regex_clean, r_stats = self.deduplicator.deduplicate_with_stats(regex_entities)
        print(f"   Regex: {r_stats['original_count']:,} → {r_stats['final_count']:,} (-{r_stats['removal_rate']:.1f}%)")
        gat_clean, g_stats = self.deduplicator.deduplicate_with_stats(gat_entities)
        print(f"   GAT:   {g_stats['original_count']:,} → {g_stats['final_count']:,} (-{g_stats['removal_rate']:.1f}%)")

        print("\n📊 ANALYZING COMPARISON...")
        try:
            comparison_stats = self._analyze_comparison(regex_clean, gat_clean)
        except Exception as e:
            self.logger.error(f"❌ Fuzzy matching failed: {e}")
            partial = {
                'regex_type_distribution': Counter([e['type'] for e in regex_clean]),
                'gat_type_distribution':   Counter([e['type'] for e in gat_clean]),
                'regex_avg_confidence':    np.mean([e.get('confidence', 0) for e in regex_clean]) if regex_clean else 0,
                'gat_avg_confidence':      np.mean([e.get('confidence', 0) for e in gat_clean])   if gat_clean   else 0,
                'regex_confidences':       [e.get('confidence', 0) for e in regex_clean],
                'gat_confidences':         [e.get('confidence', 0) for e in gat_clean],
            }
            comparison_stats = self._create_fallback_stats(regex_clean, gat_clean, partial)

        comparison_stats['deduplication'] = {'regex': r_stats, 'gat': g_stats}
        self._print_comparison_summary(comparison_stats)
        output_paths = self._save_and_export_comparison(
            regex_clean, gat_clean, comparison_stats, doc_id
        )
        print("\n✅ Comparison complete!")
        return {
            'doc_id':         doc_id,
            'regex_entities': regex_clean,
            'gat_entities':   gat_clean,
            'stats':          comparison_stats,
            **output_paths,
        }

    def _save_and_export_comparison(
        self,
        regex_entities:   List[Dict],
        gat_entities:     List[Dict],
        comparison_stats: Dict,
        doc_id:           str,
    ) -> Dict:
        paths      = {}
        output_dir = self.comparison_dir / 'entities'
        output_dir.mkdir(exist_ok=True)

        regex_path = output_dir / f"{doc_id}_regex_entities.json"
        gat_path   = output_dir / f"{doc_id}_gat_entities.json"
        with open(regex_path, 'w') as f:
            json.dump(regex_entities, f, indent=2)
        with open(gat_path, 'w') as f:
            json.dump(gat_entities, f, indent=2)
        print(f"\n💾 Saved entity files:\n   Regex: {regex_path}\n   GAT:   {gat_path}")
        paths.update({'regex_json_path': regex_path, 'gat_json_path': gat_path})

        print("\n📊 Creating visualizations...")
        viz_dir = self.visual_dir / 'Detailed Visualizations'
        viz_dir.mkdir(exist_ok=True)
        self._create_all_visualizations(
            regex_entities, gat_entities, comparison_stats, viz_dir, doc_id
        )
        paths['visualizations_dir'] = viz_dir

        print("\n📊 Creating Excel report...")
        excel_path = self.comparison_dir / f"{doc_id}_comparison.xlsx"
        self._export_comparison_excel(regex_entities, gat_entities, comparison_stats, excel_path)
        paths['excel_path'] = excel_path
        print(f"\n   Excel: {excel_path}\n   Visualizations: {viz_dir}")
        return paths

    # =========================================================================
    # VISUALIZATION
    # =========================================================================

    def _create_all_visualizations(
        self, regex_entities, gat_entities, comparison_stats, viz_dir, doc_id,
        history=None, y_true=None, y_pred=None, label_map=None, cm_top_n=None,
    ):
        print("\n📊 Creating visualizations...")
        viz_dir = Path(viz_dir)
        viz_dir.mkdir(parents=True, exist_ok=True)

        self.visualizer.plot_comparison_dashboard(
            regex_entities, gat_entities, comparison_stats,
            filename=str(viz_dir / f"{doc_id}_dashboard.png"),
        )
        print(f"  ✅  Dashboard → {viz_dir / f'{doc_id}_dashboard.png'}")

        self.visualizer.plot_comparison_venn(
            regex_entities, gat_entities,
            relaxed_stats=comparison_stats.get('relaxed'),
            filename=str(viz_dir / f"{doc_id}_venn.png"),
        )
        print(f"  ✅  Venn diagram → {viz_dir / f'{doc_id}_venn.png'}")

        self.visualizer.plot_gat_source_breakdown(
            gat_entities, filename=str(viz_dir / f"{doc_id}_source_breakdown.png"),
        )
        print(f"  ✅  Source breakdown → {viz_dir / f'{doc_id}_source_breakdown.png'}")

        all_entities = regex_entities + gat_entities
        if all_entities:
            self.visualizer.plot_entity_distribution(
                all_entities, filename=str(viz_dir / f"{doc_id}_entity_distribution.png"),
            )
            print(f"  ✅  Entity distribution → {viz_dir / f'{doc_id}_entity_distribution.png'}")

        if history is not None:
            self.visualizer.plot_training_curves(
                history, filename=str(viz_dir / f"{doc_id}_training_curves.png")
            )
            print(f"  ✅  Training curves → {viz_dir / f'{doc_id}_training_curves.png'}")
            if history.get("phase_transitions"):
                self.visualizer.plot_curriculum_history(
                    history, filename=str(viz_dir / f"{doc_id}_curriculum_history.png")
                )
                print(f"  ✅  Curriculum history → {viz_dir / f'{doc_id}_curriculum_history.png'}")

        if y_true is not None and y_pred is not None:
            self.visualizer.plot_confusion_matrix_enhanced(
                y_true, y_pred, label_map=label_map,
                filename=str(viz_dir / f"{doc_id}_confusion_matrix.png"), top_n=cm_top_n,
            )
            print(f"  ✅  Confusion matrix → {viz_dir / f'{doc_id}_confusion_matrix.png'}")

        if y_true is not None and label_map is not None:
            counts           = Counter(y_true)
            non_entity_count = counts.pop(_NON_ENTITY_ID, 0)
            entity_count     = sum(counts.values())
            self.visualizer.plot_class_balance(
                entity_count=entity_count, non_entity_count=non_entity_count,
                class_counts=dict(counts), label_map=label_map,
                filename=str(viz_dir / f"{doc_id}_class_balance.png"),
            )
            print(f"  ✅  Class balance → {viz_dir / f'{doc_id}_class_balance.png'}")

        self._create_comparison_chart(comparison_stats, regex_entities, gat_entities, viz_dir, doc_id)
        self._create_type_comparison_table(comparison_stats, viz_dir, doc_id)
        print(f"\n✅ All visualizations saved → {viz_dir}")
        return viz_dir

    def _create_comparison_chart(self, stats, regex_entities, gat_entities, viz_dir, doc_id):
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'Regex vs GAT Comparison - {doc_id}', fontsize=16, fontweight='bold')
        relaxed = stats.get('relaxed', {})

        ax   = axes[0, 0]
        cats = ['Total\nEntities', 'Found by\nBoth', 'Regex\nOnly', 'GAT\nOnly']
        vals = [stats.get('total_regex', 0), relaxed.get('found_by_both', 0),
                relaxed.get('regex_only', 0), relaxed.get('gat_only', 0)]
        bars = ax.bar(cats, vals, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title('Entity Count Comparison', fontsize=14, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2., h, f'{int(h):,}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

        ax   = axes[0, 1]
        bars = ax.bar(['Overlap\nPercentage', 'GAT\nImprovement'],
                      [relaxed.get('overlap_percentage', 0), stats.get('gat_improvement', 0)],
                      color=['#2ecc71', '#f39c12'])
        ax.set_ylabel('Percentage (%)', fontsize=12)
        ax.set_title('Performance Metrics', fontsize=14, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2., h, f'{h:.1f}%',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

        ax      = axes[1, 0]
        r_types = stats.get('regex_type_distribution', {})
        g_types = stats.get('gat_type_distribution', {})
        all_t   = sorted({*r_types, *g_types})
        totals  = {t: r_types.get(t, 0) + g_types.get(t, 0) for t in all_t}
        top     = [t for t, _ in sorted(totals.items(), key=lambda x: x[1], reverse=True)[:10]]
        x = np.arange(len(top))
        w = 0.35
        ax.bar(x - w / 2, [r_types.get(t, 0) for t in top], w, label='Regex', color='#3498db')
        ax.bar(x + w / 2, [g_types.get(t, 0) for t in top], w, label='GAT',   color='#e74c3c')
        ax.set_title('Top 10 Entity Type Distribution', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(top, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        ax   = axes[1, 1]
        bars = ax.bar(['Regex', 'GAT'],
                      [stats.get('regex_avg_confidence', 0), stats.get('gat_avg_confidence', 0)],
                      color=['#3498db', '#e74c3c'])
        ax.set_ylabel('Average Confidence', fontsize=12)
        ax.set_title('Average Confidence Scores', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1])
        ax.grid(axis='y', alpha=0.3)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2., h, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

        plt.tight_layout()
        out = viz_dir / f"{doc_id}_comparison.png"
        plt.savefig(out, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved comparison chart: {out}")

    def _create_type_comparison_table(self, stats, viz_dir, doc_id):
        r_types = stats.get('regex_type_distribution', {})
        g_types = stats.get('gat_type_distribution', {})
        data    = sorted(
            [{'type': t, 'regex': r_types.get(t, 0), 'gat': g_types.get(t, 0),
              'diff': g_types.get(t, 0) - r_types.get(t, 0),
              'total': r_types.get(t, 0) + g_types.get(t, 0)}
             for t in sorted({*r_types, *g_types})],
            key=lambda x: x['total'], reverse=True,
        )

        fig, ax = plt.subplots(figsize=(12, max(8, len(data) * 0.4)))
        ax.axis('tight')
        ax.axis('off')
        rows = [['Entity Type', 'Regex', 'GAT', 'Difference', 'Change %']]
        for item in data[:20]:
            diff   = item['diff']
            r_base = item['regex'] or 1
            rows.append([
                item['type'], f"{item['regex']:,}", f"{item['gat']:,}",
                f"+{diff}" if diff > 0 else str(diff),
                f"+{diff/r_base*100:.1f}%" if diff > 0 else f"{diff/r_base*100:.1f}%",
            ])

        table = ax.table(cellText=rows, cellLoc='left', loc='center',
                         colWidths=[0.35, 0.15, 0.15, 0.15, 0.20])
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 2)
        for i in range(5):
            c = table[(0, i)]
            c.set_facecolor('#3498db')
            c.set_text_props(weight='bold', color='white')
        for i in range(1, len(rows)):
            diff_val = float(rows[i][3].replace('+', '').replace(',', ''))
            colour   = '#d5f4e6' if diff_val > 0 else ('#fadbd8' if diff_val < 0 else 'white')
            table[(i, 3)].set_facecolor(colour)
            table[(i, 4)].set_facecolor(colour)

        plt.title(f'Entity Type Comparison - {doc_id}', fontsize=14, fontweight='bold', pad=20)
        out = viz_dir / f"{doc_id}_type_comparison_table.png"
        plt.savefig(out, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved type comparison table: {out}")

    def _export_comparison_excel(self, regex_entities, gat_entities, stats, excel_path):
        print("\n📊 Creating Excel report...")
        relaxed = stats.get('relaxed', {})

        if not regex_entities:
            self.logger.warning("⚠️  regex_entities is empty — 'Regex Entities' sheet will be blank.")
        if not gat_entities:
            self.logger.warning("⚠️  gat_entities is empty — 'GAT Entities' sheet will be blank.")

        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            pd.DataFrame({
                'Metric': ['Total Regex', 'Total GAT', 'Found by Both (Relaxed)',
                           'Regex Only (Relaxed)', 'GAT Only (Relaxed)',
                           'Overlap % (Relaxed)', 'GAT Improvement %'],
                'Value': [
                    stats.get('total_regex', 0), stats.get('total_gat', 0),
                    relaxed.get('found_by_both', 0), relaxed.get('regex_only', 0),
                    relaxed.get('gat_only', 0),
                    f"{relaxed.get('overlap_percentage', 0):.2f}%",
                    f"{stats.get('gat_improvement', 0):.2f}%",
                ],
            }).to_excel(writer, sheet_name='Summary', index=False)

            pd.DataFrame(regex_entities or []).to_excel(writer, sheet_name='Regex Entities', index=False)
            pd.DataFrame(gat_entities   or []).to_excel(writer, sheet_name='GAT Entities',   index=False)

            novel = stats.get('gat_only_entities', [])
            if novel:
                df = pd.DataFrame(novel)
                if 'confidence' in df.columns:
                    df = df.sort_values('confidence', ascending=False)
                df.to_excel(writer, sheet_name='GAT Novel', index=False)

        print(f"✅ Excel report: {excel_path}")

    # =========================================================================
    # EVALUATION
    # =========================================================================

    def evaluate_on_training_sample(self, train_loader) -> Dict:
        print("\n" + "=" * 80)
        print("STEP 3: SANITY CHECK - EVALUATING ON TRAINING SAMPLE")
        print("=" * 80)
        sample_loader = DataLoader(train_loader.graphs[:10], batch_size=1, shuffle=False)
        metrics = self.trainer.evaluate(sample_loader)
        print(f"\n📊 Training Sample Performance:")
        print(f"   Accuracy: {metrics['accuracy']:.4f}")
        print(f"   F1 Score: {metrics['f1_score']:.4f}")
        if metrics['f1_score'] < 0.5:
            self.logger.warning("⚠️  Low performance on training sample!")
        else:
            print("✅ Model successfully learned from training data")
        return metrics

    def run_validation(self, model, confidence_threshold: float) -> Optional[Dict]:
        print("\n" + "=" * 80)
        print("STEP 4: VALIDATION - REGEX vs GAT COMPARISON")
        print("=" * 80)
        if not self.config.get('Val_pdf_path'):
            self.logger.warning("⚠️  No validation PDF specified, skipping validation")
            return None
        val_path = Path(self.config['Val_pdf_path'])
        if not val_path.exists():
            self.logger.warning(f"⚠️  Validation PDF not found: {val_path}")
            return None
        print(f"\n📄 Validation PDF: {val_path}")
        results = self.compare_regex_vs_gat(model, str(val_path), confidence_threshold)
        print(f"\n✅ Validation complete:")
        print(f"   Regex entities : {results['stats']['total_regex']}")
        print(f"   GAT entities   : {results['stats']['total_gat']}")
        print(f"   Overlap        : {results['stats']['relaxed']['overlap_percentage']:.1f}%")
        print(f"   GAT improvement: +{results['stats']['gat_improvement']:.1f}%")
        return results

    def run_test(self, model, confidence_threshold: float) -> Dict:
        print("\n" + "=" * 80)
        print("STEP 5: TEST - REGEX vs GAT COMPARISON")
        print("=" * 80)
        print(f"\n📄 Test PDF: {self.config['test_pdf_path']}")
        results = self.compare_regex_vs_gat(
            model, self.config['test_pdf_path'], confidence_threshold
        )
        print(f"\n✅ Test complete:")
        print(f"   Regex entities : {results['stats']['total_regex']}")
        print(f"   GAT entities   : {results['stats']['total_gat']}")
        print(f"   Overlap        : {results['stats']['relaxed']['overlap_percentage']:.1f}%")
        print(f"   GAT improvement: +{results['stats']['gat_improvement']:.1f}%")
        return results

    # =========================================================================
    # MAIN WORKFLOW
    # =========================================================================

    def run(self, confidence_threshold: float = 0.7) -> Dict:

        curriculum_loader, val_loader, test_loader, label_encoder = \
            self.prepare_training_data()
        self._flush_memory("prepare_training_data")

        model, history = self.train_model(curriculum_loader, val_loader, test_loader)
        self._flush_memory("train_model")

        train_metrics = self.evaluate_on_training_sample(curriculum_loader)
        final_best_f1 = self.trainer.best_f1 if self.trainer else 0.0

        # REMOVED: del val_loader, test_loader
        self._flush_memory("post-training loader cleanup")

        print("\n🔌 Checking Drive mount health before inference steps...")
        if not self._ensure_drive_mounted():
            raise RuntimeError(
                "❌ Google Drive mount is unreachable after training. "
                "Re-mount manually and call run_validation() / run_test() directly."
            )
        print("   ✅ Drive mount healthy — proceeding to inference steps.")

        val_results = self.run_validation(model, confidence_threshold)
        self._flush_memory("run_validation")

        self._ensure_drive_mounted()

        test_results = self.run_test(model, confidence_threshold)
        self._flush_memory("run_test")

        self._print_final_summary(history, train_metrics, val_results, test_results)

        return {
            'model':              model,
            'trainer':            self.trainer,
            'history':            history,
            'label_encoder':      label_encoder,
            'train_metrics':      train_metrics,
            'validation':         val_results,
            'test':               test_results,
            'best_f1':            final_best_f1,
            'curriculum_loader':  curriculum_loader,
            'val_loader':         val_loader,
            'test_loader':        test_loader,
            'train_loader':       curriculum_loader,   # alias for backward compat
        }

    def _print_final_summary(self, history, train_metrics, val_results, test_results):
        """
        FIX (#8): history['learning_rates'] can be empty if training exited
        early with no valid batches.  Accessing [-1] on an empty list raises
        IndexError.  Added guard with fallback to 0.0.
        """
        print("\n" + "=" * 80)
        print("🎉 PIPELINE COMPLETE!")
        print("=" * 80)
        print(f"\n📊 FINAL RESULTS SUMMARY:")
        print(f"\n🏋️  Training:")
        print(f"   Best val F1 Score : {self.trainer.best_f1:.4f}")
        print(f"   Total epochs      : {len(history.get('train_loss', []))}")

        # FIX (#8): guard against empty learning_rates (early exit / all-NaN epochs)
        lrs      = history.get('learning_rates', [])
        final_lr = lrs[-1] if lrs else 0.0
        print(f"   Final LR          : {final_lr:.6f}")

        if val_results:
            print(f"\n✅ Validation Performance:")
            print(f"   Total entities    : {val_results['stats']['total_gat']}")
            print(
                f"   Overlap with regex: "
                f"{val_results['stats']['relaxed']['overlap_percentage']:.1f}%"
            )
            print(f"   Novel discoveries : +{val_results['stats']['gat_improvement']:.1f}%")

        print(f"\n🧪 Test Performance:")
        print(f"   Total entities    : {test_results['stats']['total_gat']}")
        print(
            f"   Overlap with regex: "
            f"{test_results['stats']['relaxed']['overlap_percentage']:.1f}%"
        )
        print(f"   Novel discoveries : +{test_results['stats']['gat_improvement']:.1f}%")

        print(f"\n📁 Output Files:")
        print(f"   Model          : {self.model_dir}/best_model.pt")
        print(f"   Comparison     : {self.comparison_dir}")
        print(f"   Visualizations : {self.visual_dir}/Detailed Visualizations")
        print("\n" + "=" * 80)

## 4️⃣ Initialize Pipeline

In [ ]:
# Create configuration
config = create_config()

# Setup logger
logger = setup_logger(f"{config['log_dir']}/training.log")

In [ ]:
# Initialize pipeline
pipeline = RegulationPipeline(config, logger)

print("\n" + "="*80)
print("📋 CONFIGURATION")
print("="*80)
print(f"\n📂 Training folder: {config['train_pdf_dir']}")
print(f"📄 Test file: {config['test_pdf_path']}")
print(f"\n📊 Training params:")
print(f"   Epochs: {np.sum(config['epochs'])}")
print(f"   Batch size: {config['batch_size']}")
print(f"   Test split: {config['test_size']*100:.0f}%")
print(f"\n✅ Pipeline ready!")


🔤 Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO: ✅ Pipeline initialized
INFO:    GraphBuilder version : v1.1_4008c4cb
INFO:    Checkpoint dir       : /content/drive/MyDrive/RegCheckAI/Checkpoints
INFO:    Text cache dir       : /content/drive/MyDrive/RegCheckAI/Checkpoints/text_cache



📋 CONFIGURATION

📂 Training folder: /content/drive/MyDrive/RegCheckAI/PDF
📄 Test file: /content/drive/MyDrive/RegCheckAI/New Regulation/SBC-Code-801-2018-unlocked-2-001-300.pdf

📊 Training params:
   Epochs: 65
   Batch size: 32
   Test split: 20%

✅ Pipeline ready!


## 5️⃣ Run Complete Pipeline

This single cell will:
1. Extract text from all training PDFs in `/Regulations/PDF/`
2. Create chunks and label entities
3. Split into train/test (80/20)
4. Train GNN model
5. Extract entities from test file `/new_regulation/SBC-Code-801-2018.pdf`
6. Export to JSON for MCP comparison

In [ ]:
# Run the pipeline
print("\n🚀 TRAINING GAT ON REGEX LABELS...")
results = pipeline.run()

model         = results['model']
history       = results['history']
train_loader  = results['train_loader']
test_loader   = results['test_loader']
label_encoder = results['label_encoder']
train_metrics = results['train_metrics']
val_results   = results['validation']
test_results  = results['test']
trainer       = results['trainer']
best_f1       = results['best_f1']

print(f"\n✅ Training complete!")
print(f"   Best F1: {best_f1:.4f}")

print("\n🔮 USING GAT TO PREDICT BEYOND REGEX...")

print("\n✅ GAT found entities that regex missed!")


🚀 TRAINING GAT ON REGEX LABELS...

STEP 1: PREPARING CURRICULUM TRAINING DATA

💾 CHECKPOINT MANAGEMENT

📭 No checkpoint found. Starting from scratch...

📂 Training folder: /content/drive/MyDrive/RegCheckAI/PDF
   Found 291 PDF(s)

✂️  Extracting and chunking documents...


Extracting & chunking:   0%|          | 0/291 [00:00<?, ?it/s, doc=1) Proposed Connecticut State Fire]INFO: [cache hit]  1_Proposed_Connecticut_State_Fire_Prevention_Code_cleaned_6ecb4e02.txt  (126,087 chars)
INFO: [chunk hit]  1_Proposed_Connecticut_State_Fire_Prevention_Code_chunks_6ecb4e02_7d88696a.json  (51 chunks)
Extracting & chunking:   0%|          | 1/291 [00:00<04:33,  1.06it/s, doc=105152_ALLE_AHS_Fire_Life_Safety_Co]

   ✅ Chunks from cache (51)


INFO: [cache hit]  105152_ALLE_AHS_Fire_Life_Safety_Codes_022216_cleaned_57f10a89.txt  (79,664 chars)
INFO: [chunk hit]  105152_ALLE_AHS_Fire_Life_Safety_Codes_022216_chunks_57f10a89_2b7b8fa0.json  (396 chunks)
Extracting & chunking:   1%|          | 2/291 [00:01<04:09,  1.16it/s, doc=1941-National-Building-Code-of-Cana]

   ✅ Chunks from cache (396)


...

   Regex: 3,201 → 3,000 (-6.3%)


INFO: 📊 Deduplication stats:
INFO:    Exact duplicates removed:       502
INFO:    Containment duplicates removed: 3626
INFO:    Overlap duplicates removed:     75
INFO:    Cross-type merges:              296
INFO:    High-confidence preserved:      250
INFO: 📊 Analyzing comparison with fuzzy matching...


   GAT:   29,127 → 24,924 (-14.4%)

📊 ANALYZING COMPARISON...


INFO: 
📈 Comparison Results (Fuzzy Matching):
   Strict  — both: 2,988 (99.6%), regex-only: 12, gat-only: 21,936
   Relaxed — both: 2,988 (99.6%), regex-only: 12, gat-only: 21,936
   GAT Improvement: +731.2%
INFO:       HAZARD_LEVEL                 recall=0.67  regex=6  gat=362  missed=2
INFO:       FLOW_RATE                    recall=0.75  regex=4  gat=11  missed=1
INFO:       MATERIAL                     recall=0.89  regex=9  gat=1095  missed=1
INFO:       AREA                         recall=0.96  regex=131  gat=127  missed=5
INFO:       OCCUPANCY                    recall=1.00  regex=506  gat=2916  missed=2



📈 COMPARISON SUMMARY

    Total Entities:
      • Regex: 3,000
      • GAT:   24,924
      • Diff:  +21,924

    Overlap Analysis (Relaxed Matching - IoU >= 0.5, Related Types):
      • Found by both: 2,988 (99.6%)
      • Regex only:    12
      • GAT only:      21,936

    Overlap Analysis (Strict Matching - IoU >= 0.8, Exact Type):
      • Found by both: 2,988 (99.6%)
      • Regex only:    12
      • GAT only:      21,936

    Performance:
      • GAT Improvement: +731.2%

    Confidence Scores:
      • Regex avg: 0.894
      • GAT avg:   0.736
        

    Deduplication Results:
      • Regex: 3,201 -> 3,000 (-6.3%)
      • GAT:   29,127 -> 24,924 (-14.4%)
                

📊 Top 10 Entity Types:
Type                           Regex      GAT        Diff      
----------------------------------------------------------------------
EQUIPMENT                      482        4389       +3907
STANDARD                       519        4168       +3649
OCCUPANCY                      506

## 6️⃣ View Extracted Entities

In [ ]:
# 1. Check the actual validation comparison results
val_comparison = results['validation']

print("\n🔍 DETAILED VALIDATION ANALYSIS:")
print("="*80)

# Check overlap details
strict_stats = val_comparison['stats']['strict']
relaxed_stats = val_comparison['stats']['relaxed']

print(f"\nSTRICT Matching (IoU ≥ 0.8, exact type):")
print(f"  Regex entities: {strict_stats['total_regex']}")
print(f"  GAT entities: {strict_stats['total_gat']}")
print(f"  Found by both: {strict_stats['found_by_both']}")
print(f"  Regex only: {strict_stats['regex_only']}")
print(f"  GAT only: {strict_stats['gat_only']}")

print(f"\nRELAXED Matching (IoU ≥ 0.5, related types):")
print(f"  Regex entities: {relaxed_stats['total_regex']}")
print(f"  GAT entities: {relaxed_stats['total_gat']}")
print(f"  Found by both: {relaxed_stats['found_by_both']}")
print(f"  Regex only: {relaxed_stats['regex_only']}")
print(f"  GAT only: {relaxed_stats['gat_only']}")

# 2. Check novel entities - are they real or noise?
print("\n\n🆕 NOVEL GAT ENTITIES (Top 50 by confidence):")
print("="*80)

gat_only_entities = val_comparison['stats'].get('gat_only_entities', [])
gat_only_sorted = sorted(gat_only_entities, key=lambda x: x.get('confidence', 0), reverse=True)

print(f"\n{'Entity Value':<60} {'Type':<20} {'Confidence':<10}")
print("-"*90)

for entity in gat_only_sorted[:50]:
    value = entity['entity_value'][:57] + "..." if len(entity['entity_value']) > 60 else entity['entity_value']
    print(f"{value:<60} {entity['type']:<20} {entity['confidence']:.3f}")

# 3. Check what regex found that GAT missed
print("\n\n❌ REGEX ENTITIES GAT MISSED (Sample):")
print("="*80)

regex_only_entities = val_comparison['stats'].get('regex_only_entities', [])

print(f"\n{'Entity Value':<60} {'Type':<20}")
print("-"*80)

for entity in regex_only_entities[:30]:
    value = entity['entity_value'][:57] + "..." if len(entity['entity_value']) > 60 else entity['entity_value']
    print(f"{value:<60} {entity['type']:<20}")

# 4. Type distribution comparison
print("\n\n📊 TYPE DISTRIBUTION COMPARISON:")
print("="*80)

regex_types = val_comparison['stats']['regex_type_distribution']
gat_types = val_comparison['stats']['gat_type_distribution']

print(f"\n{'Type':<30} {'Regex':<10} {'GAT':<10} {'Diff':<10} {'GAT/Regex Ratio':<15}")
print("-"*80)

all_types = sorted(set(list(regex_types.keys()) + list(gat_types.keys())))

for entity_type in all_types:
    regex_count = regex_types.get(entity_type, 0)
    gat_count = gat_types.get(entity_type, 0)
    diff = gat_count - regex_count
    ratio = gat_count / regex_count if regex_count > 0 else float('inf')

    # ✅ FIXED: Correct format specifier order
    print(f"{entity_type:<30} {regex_count:<10} {gat_count:<10} {diff:+10d} {ratio:.2f}x")


🔍 DETAILED VALIDATION ANALYSIS:

STRICT Matching (IoU ≥ 0.8, exact type):
  Regex entities: 774
  GAT entities: 6236
  Found by both: 774
  Regex only: 0
  GAT only: 5462

RELAXED Matching (IoU ≥ 0.5, related types):
  Regex entities: 774
  GAT entities: 6236
  Found by both: 774
  Regex only: 0
  GAT only: 5462


🆕 NOVEL GAT ENTITIES (Top 50 by confidence):

Entity Value                                                 Type                 Confidence
------------------------------------------------------------------------------------------
Testing                                                      TESTING_REQUIREMENT  0.998
annual tests                                                 TESTING_REQUIREMENT  0.998
testing                                                      TESTING_REQUIREMENT  0.998
stations the travel distance                                 EGRESS_DISTANCE      0.998
perimeter g locations                                        LOCATION             0.998
travel distan